In [ ]:

import numpy as np
import pandas as pd
import time
from scipy.optimize import linprog

# ============================================================
# 0. SETTINGS
# ============================================================

SEED = 20261118
rng = np.random.default_rng(SEED)

N = 8                       # DMUs
M = 3                       # Inputs
S = 2                       # Outputs
K = 3                       # Events

N_CAL = 500
N_TEST = 500
TARGET_COVERAGE = 0.95

# GBU group parameter
GAMMA_LOCAL = 2.0

# EDRO calibration scales
# GE0=1.0 is deliberately selected to test an active event budget.
GE0 = 1.0
GD0 = 4.0

TOL = 1e-7

# ============================================================
# 1. NOMINAL DEA DATA
# Rows = variables; columns = DMUs
# ============================================================

X = np.array([
    [5., 8., 8., 9., 7., 9., 10., 8.],
    [8., 5., 8., 9., 9., 7., 10., 10.],
    [6., 6., 8., 9., 8., 8., 10., 9.]
])

Y = np.array([
    [10., 2., 6., 5., 5., 6., 4., 5.],
    [ 2.,10., 6., 5., 6., 5., 4., 5.]
])

assert X.shape == (M, N)
assert Y.shape == (S, N)
assert np.all(X > 0)
assert np.all(Y > 0)

# ============================================================
# 2. EVENT EXPOSURES AND VULNERABILITIES
# ============================================================

AX = np.zeros((M, K, N))

for i in range(M):
    for k in range(K):
        for j in range(N):
            AX[i, k, j] = (
                0.10 + 0.05 * ((2*i + 3*k + j) % 5)
            )

BX = 0.5 * AX

AY = np.zeros((S, K, N))
BY = np.zeros((S, K, N))

for r in range(S):
    for k in range(K):
        for j in range(N):
            AY[r, k, j] = 0.10 * (-1)**(r + j + k)
            BY[r, k, j] = 0.05 * (-1)**(r + j)

# Heterogeneous vulnerability matrix: K x N
V = np.array([
    [1.00, .80, .60, .90, .70, 1.00, .60, .80],
    [.60, 1.00, .80, .70, 1.00, .60, .90, .70],
    [.80, .60, 1.00, .80, .60, .90, .70, 1.00]
])

# ============================================================
# 3. PHYSICAL NONNEGATIVITY CHECK
#
# Conservative bound:
# baseline >= sum |a| + sum v|b|
# ============================================================

for i in range(M):
    for j in range(N):
        worst_decrease = (
            np.sum(np.abs(AX[i, :, j]))
            + np.sum(V[:, j] * np.abs(BX[i, :, j]))
        )
        assert X[i, j] > worst_decrease

for r in range(S):
    for j in range(N):
        worst_decrease = (
            np.sum(np.abs(AY[r, :, j]))
            + np.sum(V[:, j] * np.abs(BY[r, :, j]))
        )
        assert Y[r, j] > worst_decrease

print("Physical nonnegativity checks: PASSED")

# ============================================================
# 4. GENERATE COMMON CALIBRATION AND TEST SCENARIOS
# ============================================================

def generate_scenarios(number, generator):

    U = np.zeros((number, K))
    SEV = np.zeros((number, K, N))

    probabilities = np.array([0.35, 0.25, 0.30])

    for t in range(number):

        events = generator.binomial(1, probabilities)
        U[t, :] = events

        q = generator.uniform(0.20, 1.00, size=(K, N))

        SEV[t, :, :] = V * events[:, None] * q

    return U, SEV


Ucal, Scal = generate_scenarios(N_CAL, rng)
Utest, Stest = generate_scenarios(N_TEST, rng)

# ============================================================
# 5. CALIBRATION
#
# All models use the same calibration scenarios.
# Each model is calibrated to approximately the same
# in-sample scenario coverage.
# ============================================================

def empirical_quantile(values, coverage):
    return np.quantile(values, coverage, method="higher")


# EDRO joint calibration score
edro_cal_scores = np.maximum(
    Ucal.sum(axis=1) / GE0,
    Scal.sum(axis=(1, 2)) / GD0
)

Q_EDRO = empirical_quantile(
    edro_cal_scores, TARGET_COVERAGE
)

GE = GE0 * Q_EDRO
GD = GD0 * Q_EDRO

# EDRO without event budget
ablation_scores = Scal.sum(axis=(1, 2))

GD_ABLATION = empirical_quantile(
    ablation_scores, TARGET_COVERAGE
)

# BS: global budget over normalized event and severity deviations
bs_cal_scores = (
    Ucal.sum(axis=1)
    + (Scal / V[None, :, :]).sum(axis=(1, 2))
)

GAMMA_BS = empirical_quantile(
    bs_cal_scores, TARGET_COVERAGE
)

# GBU: sum of event-group norms
def gbu_scenario_score(u, sev):

    normalized = sev / V[None, :, :]
    scores = np.zeros(len(u))

    for t in range(len(u)):

        total = 0.0

        for k in range(K):

            z = np.concatenate((
                [u[t, k]],
                normalized[t, k, :]
            ))

            group_norm = max(
                np.max(z),
                np.sum(z) / GAMMA_LOCAL
            )

            total += group_norm

        scores[t] = total

    return scores


gbu_cal_scores = gbu_scenario_score(Ucal, Scal)

GAMMA_GBU = empirical_quantile(
    gbu_cal_scores, TARGET_COVERAGE
)

print("\nCALIBRATED PARAMETERS")
print("---------------------")
print(f"EDRO Gamma_E       = {GE:.6f}")
print(f"EDRO Gamma_D       = {GD:.6f}")
print(f"EDRO-ablation GD   = {GD_ABLATION:.6f}")
print(f"BS Gamma           = {GAMMA_BS:.6f}")
print(f"GBU Gamma_G        = {GAMMA_GBU:.6f}")

print("\nEvent budget active:", GE < K - TOL)

# ============================================================
# 6. BUILD THE EXACT POLYHEDRAL UNCERTAINTY SETS
#
# z = (u_1,...,u_K, s_11,...,s_KN, optional t_1,...,t_K)
#
# A z <= b, z >= 0
# ============================================================

def build_uncertainty_set(model):

    NS = K * N
    BASE_DIM = K + NS

    if model == "GBU":
        DIM = BASE_DIM + K
    else:
        DIM = BASE_DIM

    rows = []
    rhs = []

    def add(row, bound):
        rows.append(row)
        rhs.append(bound)

    # --------------------------------------------------------
    # Common individual event bounds: 0 <= u_k <= 1
    # --------------------------------------------------------

    for k in range(K):
        row = np.zeros(DIM)
        row[k] = 1.0
        add(row, 1.0)

    # --------------------------------------------------------
    # EDRO and EDRO ablation
    # --------------------------------------------------------

    if model in ("EDRO", "EDRO_ABLATION"):

        # s_kj <= v_kj u_k
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0
                row[k] = -V[k, j]

                add(row, 0.0)

        # Global event activation budget
        if model == "EDRO":

            row = np.zeros(DIM)
            row[:K] = 1.0
            add(row, GE)

        # Aggregate severity budget
        row = np.zeros(DIM)
        row[K:K+NS] = 1.0

        if model == "EDRO":
            add(row, GD)
        else:
            add(row, GD_ABLATION)

    # --------------------------------------------------------
    # BS: one global budget over normalized deviations
    # --------------------------------------------------------

    elif model == "BS":

        # s_kj <= v_kj
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0

                add(row, V[k, j])

        # Sum u_k + sum s_kj/v_kj <= Gamma_BS
        row = np.zeros(DIM)
        row[:K] = 1.0

        for k in range(K):
            for j in range(N):
                q = K + k*N + j
                row[q] = 1.0 / V[k, j]

        add(row, GAMMA_BS)

    # --------------------------------------------------------
    # GBU: one group for each event
    #
    # Group k contains:
    # u_k, s_k1/v_k1, ..., s_kN/v_kN
    #
    # N_k(z_k) = max(||z_k||_infinity,
    #                 ||z_k||_1/Gamma_local)
    #
    # Sum_k N_k(z_k) <= Gamma_GBU
    # --------------------------------------------------------

    elif model == "GBU":

        T_START = BASE_DIM

        for k in range(K):

            tk = T_START + k

            # u_k <= t_k
            row = np.zeros(DIM)
            row[k] = 1.0
            row[tk] = -1.0
            add(row, 0.0)

            # s_kj/v_kj <= t_k
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0 / V[k, j]
                row[tk] = -1.0

                add(row, 0.0)

            # (u_k + sum_j s_kj/v_kj)/Gamma_local <= t_k
            row = np.zeros(DIM)
            row[k] = 1.0 / GAMMA_LOCAL
            row[tk] = -1.0

            for j in range(N):
                q = K + k*N + j
                row[q] = 1.0 / (
                    GAMMA_LOCAL * V[k, j]
                )

            add(row, 0.0)

        # Global GBU budget
        row = np.zeros(DIM)
        row[T_START:T_START+K] = 1.0
        add(row, GAMMA_GBU)

        # Explicit local severity bounds s_kj <= v_kj
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0

                add(row, V[k, j])

    else:
        raise ValueError(f"Unknown model: {model}")

    return np.asarray(rows), np.asarray(rhs), DIM


# ============================================================
# 7. SUPPORT FUNCTION PRIMAL-DUAL VERIFICATION
# ============================================================

A_EDRO, b_EDRO, DIM_EDRO = build_uncertainty_set("EDRO")

c_test = rng.normal(size=DIM_EDRO)

# Primal:
# max c'z subject to Az <= b, z >= 0

primal = linprog(
    -c_test,
    A_ub=A_EDRO,
    b_ub=b_EDRO,
    bounds=(0, None),
    method="highs"
)

# Dual:
# min b'y subject to A'y >= c, y >= 0

dual = linprog(
    b_EDRO,
    A_ub=-A_EDRO.T,
    b_ub=-c_test,
    bounds=(0, None),
    method="highs"
)

assert primal.success, primal.message
assert dual.success, dual.message

support_primal = -primal.fun
support_dual = dual.fun

print("\nEDRO SUPPORT FUNCTION CHECK")
print("---------------------------")
print("Primal value:", support_primal)
print("Dual value:  ", support_dual)
print("Absolute gap:", abs(support_primal-support_dual))

assert abs(support_primal-support_dual) < TOL

# ============================================================
# 8. EXACT ROBUST BCC LP
#
# Input constraint:
# max_z { sum_j x_ij(z) lambda_j
#        - theta*x_io(z) } <= 0
#
# Output constraint:
# max_z { y_ro(z)
#        - sum_j y_rj(z) lambda_j } <= 0
#
# Each support function is dualized exactly.
# ============================================================

def solve_bcc(o, model):

    # ========================================================
    # NOMINAL BCC
    # ========================================================

    if model == "NOMINAL":

        NV = N + 1
        THETA = N

        objective = np.zeros(NV)
        objective[THETA] = 1.0

        AUB = []
        BUB = []

        # Input constraints
        for i in range(M):

            row = np.zeros(NV)
            row[:N] = X[i, :]
            row[THETA] = -X[i, o]

            AUB.append(row)
            BUB.append(0.0)

        # Output constraints
        for r in range(S):

            row = np.zeros(NV)
            row[:N] = -Y[r, :]

            AUB.append(row)
            BUB.append(-Y[r, o])

        AEQ = np.zeros((1, NV))
        AEQ[0, :N] = 1.0

        return linprog(
            objective,
            A_ub=np.asarray(AUB),
            b_ub=np.asarray(BUB),
            A_eq=AEQ,
            b_eq=np.array([1.0]),
            bounds=(0, None),
            method="highs"
        )

    # ========================================================
    # ROBUST BCC
    # ========================================================

    ASET, BSET, DIM = build_uncertainty_set(model)

    L = len(BSET)
    NROBUST = M + S

    THETA = N
    NV = N + 1 + NROBUST * L

    objective = np.zeros(NV)
    objective[THETA] = 1.0

    AUB = []
    BUB = []

    def dual_index(row_number, ell):
        return N + 1 + row_number * L + ell

    def add_constraint(coefficients, rhs=0.0):

        row = np.zeros(NV)

        for idx, value in coefficients.items():
            row[idx] += value

        AUB.append(row)
        BUB.append(rhs)

    # ========================================================
    # INPUT CONSTRAINTS
    #
    # Nominal residual + b'y <= 0
    # c_q(lambda,theta) - A[:,q]'y <= 0
    # ========================================================

    for i in range(M):

        row_number = i

        # Nominal input residual
        coeff = {}

        for j in range(N):
            coeff[j] = X[i, j]

        coeff[THETA] = -X[i, o]

        # Add b'y
        for ell in range(L):
            coeff[dual_index(row_number, ell)] = BSET[ell]

        add_constraint(coeff, 0.0)

        # Event-coordinate dual feasibility
        for k in range(K):

            coeff = {}

            for j in range(N):
                coeff[j] = AX[i, k, j]

            coeff[THETA] = -AX[i, k, o]

            # -A'y
            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, k]
                )

            add_constraint(coeff, 0.0)

        # Severity-coordinate dual feasibility
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                coeff = {
                    j: BX[i, k, j]
                }

                if j == o:
                    coeff[THETA] = -BX[i, k, o]

                for ell in range(L):
                    coeff[dual_index(row_number, ell)] = (
                        -ASET[ell, q]
                    )

                add_constraint(coeff, 0.0)

        # Additional GBU coordinates t_k:
        # Their uncertainty coefficient is zero.
        for q in range(K + K*N, DIM):

            coeff = {}

            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, q]
                )

            add_constraint(coeff, 0.0)

    # ========================================================
    # OUTPUT CONSTRAINTS
    #
    # y_ro - sum_j y_rj lambda_j + b'y <= 0
    #
    # c_u = a_ro - sum_j a_rj lambda_j
    # c_s = b_ro - sum_j b_rj lambda_j
    # ========================================================

    for r in range(S):

        row_number = M + r

        # Nominal output residual
        coeff = {}

        for j in range(N):
            coeff[j] = -Y[r, j]

        for ell in range(L):
            coeff[dual_index(row_number, ell)] = BSET[ell]

        add_constraint(coeff, -Y[r, o])

        # Event-coordinate dual feasibility
        for k in range(K):

            coeff = {}

            for j in range(N):
                coeff[j] = -AY[r, k, j]

            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, k]
                )

            # -sum_j a_rkj lambda_j - A'y <= -a_rko
            add_constraint(coeff, -AY[r, k, o])

        # Severity-coordinate dual feasibility
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                coeff = {
                    j: -BY[r, k, j]
                }

                for ell in range(L):
                    coeff[dual_index(row_number, ell)] = (
                        -ASET[ell, q]
                    )

                rhs = -BY[r, k, o] if j == o else 0.0

                add_constraint(coeff, rhs)

        # Additional GBU coordinates t_k
        for q in range(K + K*N, DIM):

            coeff = {}

            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, q]
                )

            add_constraint(coeff, 0.0)

    # ========================================================
    # CONVEXITY
    # ========================================================

    AEQ = np.zeros((1, NV))
    AEQ[0, :N] = 1.0

    result = linprog(
        objective,
        A_ub=np.asarray(AUB),
        b_ub=np.asarray(BUB),
        A_eq=AEQ,
        b_eq=np.array([1.0]),
        bounds=(0, None),
        method="highs"
    )

    return result


# ============================================================
# 9. SOLVE ALL MODELS FOR ALL DMUs
# ============================================================

MODELS = [
    "NOMINAL",
    "BS",
    "GBU",
    "EDRO",
    "EDRO_ABLATION"
]

records = []

total_start = time.time()

for model in MODELS:

    print(f"\nSOLVING MODEL: {model}")

    for o in range(N):

        start = time.time()

        result = solve_bcc(o, model)

        elapsed = time.time() - start

        if result.success:

            score = result.fun
            status = "optimal"

        else:

            score = np.nan
            status = result.message

        records.append({
            "model": model,
            "DMU": o + 1,
            "efficiency": score,
            "status": status,
            "runtime_seconds": elapsed
        })

        print(
            f"DMU {o+1}: "
            f"efficiency={score:.8f}"
            if result.success
            else f"DMU {o+1}: FAILED: {result.message}"
        )

results = pd.DataFrame(records)

print("\nTotal runtime:", time.time() - total_start)

# ============================================================
# 10. EFFICIENCY MATRIX AND RANKINGS
# ============================================================

pivot = results.pivot(
    index="DMU",
    columns="model",
    values="efficiency"
)

print("\nEFFICIENCY MATRIX")
print(pivot.round(6))

results["rank"] = (
    results.groupby("model")["efficiency"]
    .rank(method="min", ascending=True)
)

summary = (
    results.groupby("model")
    .agg(
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median"),
        minimum_efficiency=("efficiency", "min"),
        maximum_efficiency=("efficiency", "max"),
        mean_runtime=("runtime_seconds", "mean"),
        successful_solves=(
            "status",
            lambda x: (x == "optimal").sum()
        )
    )
)

print("\nMODEL SUMMARY")
print(summary.round(6))

# ============================================================
# 11. INCREMENTAL MODEL EFFECTS
# ============================================================

effects = pd.DataFrame(index=pivot.index)

effects["EDRO_minus_BS"] = (
    pivot["EDRO"] - pivot["BS"]
)

effects["EDRO_minus_GBU"] = (
    pivot["EDRO"] - pivot["GBU"]
)

effects["EDRO_minus_ablation"] = (
    pivot["EDRO"] - pivot["EDRO_ABLATION"]
)

effects["BS_minus_GBU"] = (
    pivot["BS"] - pivot["GBU"]
)

print("\nINCREMENTAL EFFICIENCY EFFECTS")
print(effects.round(6))

print("\nMEAN ABSOLUTE SCORE DIFFERENCES")
print(effects.abs().mean().round(6))

# ============================================================
# 12. OUT-OF-SAMPLE COVERAGE
# ============================================================

edro_test_scores = np.maximum(
    Utest.sum(axis=1) / GE0,
    Stest.sum(axis=(1, 2)) / GD0
)

edro_test_coverage = np.mean(
    edro_test_scores <= Q_EDRO
)

ablation_test_coverage = np.mean(
    Stest.sum(axis=(1, 2)) <= GD_ABLATION
)

bs_test_scores = (
    Utest.sum(axis=1)
    + (Stest / V[None, :, :]).sum(axis=(1, 2))
)

bs_test_coverage = np.mean(
    bs_test_scores <= GAMMA_BS
)

gbu_test_scores = gbu_scenario_score(Utest, Stest)

gbu_test_coverage = np.mean(
    gbu_test_scores <= GAMMA_GBU
)

coverage = pd.DataFrame({
    "model": [
        "EDRO",
        "BS",
        "GBU",
        "EDRO_ABLATION"
    ],
    "calibration_target": [
        TARGET_COVERAGE
    ] * 4,
    "test_coverage": [
        edro_test_coverage,
        bs_test_coverage,
        gbu_test_coverage,
        ablation_test_coverage
    ]
})

print("\nOUT-OF-SAMPLE COVERAGE")
print(coverage)

# ============================================================
# 13. CALIBRATION COVERAGE
# ============================================================

calibration_coverage = pd.DataFrame({
    "model": ["EDRO", "BS", "GBU", "EDRO_ABLATION"],
    "calibration_coverage": [
        np.mean(edro_cal_scores <= Q_EDRO),
        np.mean(bs_cal_scores <= GAMMA_BS),
        np.mean(gbu_cal_scores <= GAMMA_GBU),
        np.mean(ablation_scores <= GD_ABLATION)
    ]
})

print("\nIN-SAMPLE CALIBRATION COVERAGE")
print(calibration_coverage)

# ============================================================
# 14. RANKING CHANGES
# ============================================================

rank_pivot = results.pivot(
    index="DMU",
    columns="model",
    values="rank"
)

print("\nRANKINGS")
print(rank_pivot)

for model in ["BS", "GBU", "EDRO", "EDRO_ABLATION"]:

    changes = np.sum(
        rank_pivot["NOMINAL"] != rank_pivot[model]
    )

    print(
        f"Ranking changes: NOMINAL vs {model}: {changes}/{N}"
    )

# ============================================================
# 15. EXPORT ALL RESULTS
# ============================================================

results.to_csv(
    "EDRO_BCC_all_results.csv",
    index=False
)

pivot.to_csv(
    "EDRO_BCC_efficiency_matrix.csv"
)

summary.to_csv(
    "EDRO_BCC_model_summary.csv"
)

effects.to_csv(
    "EDRO_BCC_incremental_effects.csv"
)

coverage.to_csv(
    "EDRO_BCC_test_coverage.csv",
    index=False
)

calibration_coverage.to_csv(
    "EDRO_BCC_calibration_coverage.csv",
    index=False
)

rank_pivot.to_csv(
    "EDRO_BCC_rankings.csv"
)

print("\nEXPORTED FILES")
print("--------------")
print("EDRO_BCC_all_results.csv")
print("EDRO_BCC_efficiency_matrix.csv")
print("EDRO_BCC_model_summary.csv")
print("EDRO_BCC_incremental_effects.csv")
print("EDRO_BCC_test_coverage.csv")
print("EDRO_BCC_calibration_coverage.csv")
print("EDRO_BCC_rankings.csv")

print("\nEXPERIMENT COMPLETED.")


In [ ]:

# ============================================================
# EDRO: UNCERTAINTY-GEOMETRY DIAGNOSTICS
# Run AFTER the previous EDRO experiment cell.
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import linprog

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

N_DIRECTIONS = 1000
BOOTSTRAPS = 3000
SEED_DIAG = 20261119

diag_rng = np.random.default_rng(SEED_DIAG)

BASE_DIM = K + K*N

MODELS_GEOMETRY = [
    "EDRO",
    "BS",
    "GBU",
    "EDRO_ABLATION"
]

# ------------------------------------------------------------
# 2. BUILD COMMON-DIMENSION SUPPORT FUNCTION
# ------------------------------------------------------------

SETS = {}

for model in MODELS_GEOMETRY:

    A, b, dim = build_uncertainty_set(model)

    SETS[model] = (A, b, dim)

    print(
        f"{model}: inequalities={len(b)}, "
        f"dimension={dim}"
    )


def support_value(model, direction):

    A, b, dim = SETS[model]

    c = np.zeros(dim)
    c[:BASE_DIM] = direction

    result = linprog(
        -c,
        A_ub=A,
        b_ub=b,
        bounds=(0, None),
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            f"Support LP failed for {model}: {result.message}"
        )

    return -result.fun


# ------------------------------------------------------------
# 3. GENERATE RANDOM SUPPORT DIRECTIONS
#
# Both positive and negative coefficients are important.
# ------------------------------------------------------------

directions = []

for q in range(N_DIRECTIONS):

    c = diag_rng.normal(size=BASE_DIM)

    # Normalize to avoid scale effects
    c = c / np.linalg.norm(c)

    directions.append(c)

directions = np.asarray(directions)

# ------------------------------------------------------------
# 4. CALCULATE SUPPORT VALUES
# ------------------------------------------------------------

support_records = []

for q, c in enumerate(directions):

    row = {"direction": q + 1}

    for model in MODELS_GEOMETRY:
        row[model] = support_value(model, c)

    support_records.append(row)

support_df = pd.DataFrame(support_records)

print("\nFIRST TEN SUPPORT VALUES")
print(support_df.head(10).round(6))

# ------------------------------------------------------------
# 5. PAIRED SUPPORT-FUNCTION DIFFERENCES
# ------------------------------------------------------------

support_df["EDRO_minus_BS"] = (
    support_df["EDRO"] - support_df["BS"]
)

support_df["EDRO_minus_GBU"] = (
    support_df["EDRO"] - support_df["GBU"]
)

support_df["EDRO_minus_ablation"] = (
    support_df["EDRO"] - support_df["EDRO_ABLATION"]
)

support_df["BS_minus_GBU"] = (
    support_df["BS"] - support_df["GBU"]
)

# ------------------------------------------------------------
# 6. BOOTSTRAP CONFIDENCE INTERVALS
# ------------------------------------------------------------

def bootstrap_mean_ci(values, B=3000, seed=123):

    values = np.asarray(values)
    n = len(values)

    generator = np.random.default_rng(seed)

    boot_means = np.empty(B)

    for b in range(B):

        sample = generator.choice(
            values,
            size=n,
            replace=True
        )

        boot_means[b] = np.mean(sample)

    return (
        np.mean(values),
        np.quantile(boot_means, 0.025),
        np.quantile(boot_means, 0.975)
    )


comparison_columns = [
    "EDRO_minus_BS",
    "EDRO_minus_GBU",
    "EDRO_minus_ablation",
    "BS_minus_GBU"
]

comparison_results = []

for col in comparison_columns:

    mean, lower, upper = bootstrap_mean_ci(
        support_df[col].values,
        B=BOOTSTRAPS,
        seed=SEED_DIAG
    )

    comparison_results.append({
        "comparison": col,
        "mean_difference": mean,
        "CI_lower": lower,
        "CI_upper": upper,
        "positive_fraction": np.mean(
            support_df[col] > 1e-9
        ),
        "negative_fraction": np.mean(
            support_df[col] < -1e-9
        ),
        "equal_fraction": np.mean(
            np.abs(support_df[col]) <= 1e-9
        )
    })

comparison_results = pd.DataFrame(comparison_results)

print("\nSUPPORT-FUNCTION COMPARISONS")
print(comparison_results.round(6))

# ------------------------------------------------------------
# 7. APPROXIMATE SET-INCLUSION DIAGNOSTICS
#
# If h_A(c) <= h_B(c) for every direction c,
# then A is contained in B.
#
# Random directions provide evidence, not a proof.
# ------------------------------------------------------------

inclusion_results = []

for A_model, B_model in [
    ("EDRO", "BS"),
    ("EDRO", "GBU"),
    ("BS", "EDRO"),
    ("GBU", "EDRO"),
    ("EDRO", "EDRO_ABLATION"),
    ("EDRO_ABLATION", "EDRO")
]:

    difference = (
        support_df[A_model].values
        - support_df[B_model].values
    )

    inclusion_results.append({
        "candidate_subset": A_model,
        "candidate_superset": B_model,
        "max_support_difference": np.max(difference),
        "mean_support_difference": np.mean(difference),
        "directions_with_violation": np.sum(
            difference > 1e-8
        ),
        "violation_fraction": np.mean(
            difference > 1e-8
        )
    })

inclusion_df = pd.DataFrame(inclusion_results)

print("\nAPPROXIMATE SET-INCLUSION DIAGNOSTICS")
print(inclusion_df.round(6))

# ------------------------------------------------------------
# 8. SUPPORT FUNCTION DIFFERENCE DISTRIBUTIONS
# ------------------------------------------------------------

distribution_summary = support_df[
    comparison_columns
].agg([
    "mean",
    "std",
    "min",
    "median",
    "max"
]).T

print("\nSUPPORT DIFFERENCE DISTRIBUTIONS")
print(distribution_summary.round(6))

# ------------------------------------------------------------
# 9. WILSON CONFIDENCE INTERVAL FOR COVERAGE
# ------------------------------------------------------------

def wilson_interval(successes, total, z=1.96):

    p = successes / total

    denominator = 1 + z**2 / total

    center = (
        p + z**2 / (2*total)
    ) / denominator

    half_width = (
        z * np.sqrt(
            p*(1-p)/total
            + z**2/(4*total**2)
        ) / denominator
    )

    return center-half_width, center+half_width


coverage_data = {
    "EDRO": np.mean(edro_test_scores <= Q_EDRO),
    "BS": np.mean(bs_test_scores <= GAMMA_BS),
    "GBU": np.mean(gbu_test_scores <= GAMMA_GBU),
    "EDRO_ABLATION": np.mean(
        Stest.sum(axis=(1, 2)) <= GD_ABLATION
    )
}

coverage_ci_records = []

for model, coverage_value in coverage_data.items():

    if model == "EDRO":
        indicator = edro_test_scores <= Q_EDRO

    elif model == "BS":
        indicator = bs_test_scores <= GAMMA_BS

    elif model == "GBU":
        indicator = gbu_test_scores <= GAMMA_GBU

    else:
        indicator = (
            Stest.sum(axis=(1, 2)) <= GD_ABLATION
        )

    successes = int(np.sum(indicator))
    lower, upper = wilson_interval(
        successes,
        len(indicator)
    )

    coverage_ci_records.append({
        "model": model,
        "coverage": coverage_value,
        "CI_lower": lower,
        "CI_upper": upper,
        "covered_scenarios": successes,
        "total_scenarios": len(indicator)
    })

coverage_ci = pd.DataFrame(coverage_ci_records)

print("\nOUT-OF-SAMPLE COVERAGE WITH 95% WILSON INTERVALS")
print(coverage_ci.round(6))

# ------------------------------------------------------------
# 10. EXPORT
# ------------------------------------------------------------

support_df.to_csv(
    "EDRO_support_direction_results.csv",
    index=False
)

comparison_results.to_csv(
    "EDRO_support_comparison.csv",
    index=False
)

inclusion_df.to_csv(
    "EDRO_approximate_set_inclusion.csv",
    index=False
)

distribution_summary.to_csv(
    "EDRO_support_difference_distribution.csv"
)

coverage_ci.to_csv(
    "EDRO_coverage_confidence_intervals.csv",
    index=False
)

print("\nGEOMETRY DIAGNOSTICS COMPLETED.")


In [ ]:

# ============================================================
# CORRECTED EXACT POLYHEDRAL CONTAINMENT TESTS
# EDRO versus adapted GBU
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from itertools import product

TOL_CONTAIN = 1e-8

# ------------------------------------------------------------
# 1. BUILD SETS
# ------------------------------------------------------------

A_E, b_E, dim_E = build_uncertainty_set("EDRO")
A_G, b_G, dim_G = build_uncertainty_set("GBU")

BASE_DIM = K + K*N

# ------------------------------------------------------------
# 2. EDRO SUBSET GBU: EXACT TEST
#
# GBU group norm:
#
# max{u_k, max_j(s_kj/v_kj),
#     (u_k + sum_j s_kj/v_kj)/Gamma_local}
#
# Maximize the sum of these norms over EDRO.
#
# The sum of maxima equals the maximum over all
# combinations of active expressions.
# There are (N+2)^K combinations.
# ------------------------------------------------------------

def edro_group_candidates(k):

    candidates = []

    # Candidate 1: u_k
    c = np.zeros(dim_E)
    c[k] = 1.0
    candidates.append(c)

    # Candidates 2,...,N+1: s_kj/v_kj
    for j in range(N):

        c = np.zeros(dim_E)
        q = K + k*N + j
        c[q] = 1.0 / V[k, j]

        candidates.append(c)

    # Last candidate:
    # (u_k + sum_j s_kj/v_kj)/Gamma_local
    c = np.zeros(dim_E)
    c[k] = 1.0 / GAMMA_LOCAL

    for j in range(N):
        q = K + k*N + j
        c[q] = 1.0 / (
            GAMMA_LOCAL * V[k, j]
        )

    candidates.append(c)

    return candidates


candidate_lists = [
    edro_group_candidates(k)
    for k in range(K)
]

max_GBU_norm_over_EDRO = -np.inf
best_active_combination = None

for active_indices in product(
    *[range(len(x)) for x in candidate_lists]
):

    c_total = np.zeros(dim_E)

    for k, active_index in enumerate(active_indices):
        c_total += candidate_lists[k][active_index]

    result = linprog(
        -c_total,
        A_ub=A_E,
        b_ub=b_E,
        bounds=(0, None),
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            "EDRO support LP failed: " + result.message
        )

    value = -result.fun

    if value > max_GBU_norm_over_EDRO:
        max_GBU_norm_over_EDRO = value
        best_active_combination = active_indices

EDRO_subset_GBU = (
    max_GBU_norm_over_EDRO <= GAMMA_GBU + TOL_CONTAIN
)

print("\nEDRO SUBSET GBU")
print("----------------")
print("Maximum GBU norm over EDRO:",
      max_GBU_norm_over_EDRO)
print("GBU budget:", GAMMA_GBU)
print("Containment:", EDRO_subset_GBU)
print("Maximizing active expressions:",
      best_active_combination)

# ------------------------------------------------------------
# 3. GBU SUBSET EDRO: EXACT TEST
#
# Every EDRO defining inequality must hold for
# every point in GBU.
#
# For each inequality a'z <= b, solve:
# max a'z over GBU.
# ------------------------------------------------------------

def maximize_over_GBU(c):

    result = linprog(
        -c,
        A_ub=A_G,
        b_ub=b_G,
        bounds=(0, None),
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            "GBU support LP failed: " + result.message
        )

    return -result.fun


violations = []

# Event budget
c = np.zeros(dim_G)
c[:K] = 1.0

value = maximize_over_GBU(c)

violations.append({
    "constraint": "sum(u) <= Gamma_E",
    "maximum_lhs": value,
    "limit": GE,
    "maximum_violation": value - GE
})

# Aggregate severity budget
c = np.zeros(dim_G)
c[K:K+K*N] = 1.0

value = maximize_over_GBU(c)

violations.append({
    "constraint": "sum(s) <= Gamma_D",
    "maximum_lhs": value,
    "limit": GD,
    "maximum_violation": value - GD
})

# Event-severity linking constraints
for k in range(K):
    for j in range(N):

        c = np.zeros(dim_G)
        q = K + k*N + j

        c[q] = 1.0
        c[k] = -V[k, j]

        value = maximize_over_GBU(c)

        violations.append({
            "constraint": f"s[{k},{j}] <= v[{k},{j}]*u[{k}]",
            "maximum_lhs": value,
            "limit": 0.0,
            "maximum_violation": value
        })

violation_df = pd.DataFrame(violations)

GBU_subset_EDRO = (
    violation_df["maximum_violation"].max()
    <= TOL_CONTAIN
)

print("\nGBU SUBSET EDRO")
print("----------------")
print(violation_df.round(10).to_string(index=False))
print("\nMaximum violation:",
      violation_df["maximum_violation"].max())
print("Containment:", GBU_subset_EDRO)

# ------------------------------------------------------------
# 4. ANALYTICAL EDRO SUBSET BS CHECK
#
# For every EDRO point:
# sum(u) + sum(s/v) <= 2 sum(u) <= 2 Gamma_E.
# ------------------------------------------------------------

edro_implied_bs_budget = 2.0 * GE

EDRO_subset_BS = (
    edro_implied_bs_budget <= GAMMA_BS + TOL_CONTAIN
)

print("\nEDRO SUBSET BS")
print("---------------")
print("Upper bound on BS budget expression:",
      edro_implied_bs_budget)
print("BS budget:", GAMMA_BS)
print("Containment:", EDRO_subset_BS)

# ------------------------------------------------------------
# 5. EDRO SUBSET ABLATION
#
# The ablation removes the event budget but retains
# the EDRO linking and severity constraints.
# ------------------------------------------------------------

print("\nEDRO SUBSET EDRO ABLATION: TRUE BY CONSTRUCTION")

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

containment_summary = pd.DataFrame([
    {
        "candidate_subset": "EDRO",
        "candidate_superset": "GBU",
        "containment": EDRO_subset_GBU
    },
    {
        "candidate_subset": "GBU",
        "candidate_superset": "EDRO",
        "containment": GBU_subset_EDRO
    },
    {
        "candidate_subset": "EDRO",
        "candidate_superset": "BS",
        "containment": EDRO_subset_BS
    },
    {
        "candidate_subset": "EDRO",
        "candidate_superset": "EDRO_ABLATION",
        "containment": True
    }
])

print("\nFINAL CONTAINMENT SUMMARY")
print(containment_summary.to_string(index=False))

containment_summary.to_csv(
    "EDRO_exact_containment_summary.csv",
    index=False
)

violation_df.to_csv(
    "EDRO_GBU_containment_violations.csv",
    index=False
)

print("\nCORRECTED CONTAINMENT TESTS COMPLETED.")


In [ ]:

# ============================================================
# COMPLETE CONTAINMENT TESTS
# EDRO vs GBU vs LINKED GBU vs BS
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from itertools import product

TOL_CONTAIN = 1e-8

# ============================================================
# 1. BUILD ALL UNCERTAINTY SETS
# ============================================================

A_E, b_E, dim_E = build_uncertainty_set("EDRO")
A_G, b_G, dim_G = build_uncertainty_set("GBU")

# ============================================================
# 2. BUILD LINKED GBU
#
# Original GBU constraints plus:
# s[k,j] <= V[k,j] * u[k]
# ============================================================

def build_linked_GBU_set():

    A, b, dim = build_uncertainty_set("GBU")

    A_link = []

    for k in range(K):
        for j in range(N):

            row = np.zeros(dim)

            row[K + k*N + j] = 1.0
            row[k] = -V[k, j]

            A_link.append(row)

    A_link = np.asarray(A_link)

    A_new = np.vstack([A, A_link])
    b_new = np.concatenate([b, np.zeros(K*N)])

    return A_new, b_new, dim


A_GL, b_GL, dim_GL = build_linked_GBU_set()

print("All uncertainty sets constructed.")
print("EDRO dimension:", dim_E)
print("GBU dimension:", dim_G)
print("Linked GBU dimension:", dim_GL)

# ============================================================
# 3. EDRO SUBSET GBU AND LINKED GBU
#
# For each event:
#
# N_k(u,s) = max{
#   u_k,
#   max_j(s_kj/V_kj),
#   (u_k + sum_j(s_kj/V_kj))/GAMMA_LOCAL
# }
#
# Maximize sum_k N_k over EDRO.
#
# Exact enumeration of active expressions.
# ============================================================

def edro_group_candidates(k):

    candidates = []

    # u_k
    c = np.zeros(dim_E)
    c[k] = 1.0
    candidates.append(c)

    # s_kj / V_kj
    for j in range(N):

        if V[k, j] <= 0:
            raise ValueError("All V[k,j] must be positive.")

        c = np.zeros(dim_E)
        q = K + k*N + j
        c[q] = 1.0 / V[k, j]

        candidates.append(c)

    # (u_k + sum_j s_kj/V_kj) / GAMMA_LOCAL
    c = np.zeros(dim_E)
    c[k] = 1.0 / GAMMA_LOCAL

    for j in range(N):
        q = K + k*N + j
        c[q] = 1.0 / (GAMMA_LOCAL * V[k, j])

    candidates.append(c)

    return candidates


candidate_lists = [
    edro_group_candidates(k)
    for k in range(K)
]

max_GBU_norm_over_EDRO = -np.inf
best_active_combination = None
best_EDRO_point = None

for active_indices in product(
    *[range(len(x)) for x in candidate_lists]
):

    c_total = np.zeros(dim_E)

    for k, idx in enumerate(active_indices):
        c_total += candidate_lists[k][idx]

    res = linprog(
        -c_total,
        A_ub=A_E,
        b_ub=b_E,
        bounds=(0, None),
        method="highs"
    )

    if not res.success:
        raise RuntimeError(
            "EDRO support LP failed: " + res.message
        )

    value = -res.fun

    if value > max_GBU_norm_over_EDRO:
        max_GBU_norm_over_EDRO = value
        best_active_combination = active_indices
        best_EDRO_point = res.x.copy()


EDRO_subset_GBU = (
    max_GBU_norm_over_EDRO
    <= GAMMA_GBU + TOL_CONTAIN
)

# EDRO already satisfies the linking constraints.
# Therefore the same group-norm maximum tests
# EDRO subset LINKED GBU.

EDRO_subset_linked_GBU = EDRO_subset_GBU

# ============================================================
# 4. GENERAL SUPPORT FUNCTION
# ============================================================

def maximize_over_set(c, A, b, dim, label):

    res = linprog(
        -c,
        A_ub=A,
        b_ub=b,
        bounds=(0, None),
        method="highs"
    )

    if not res.success:
        raise RuntimeError(
            label + " support LP failed: " + res.message
        )

    return -res.fun, res.x


# ============================================================
# 5. TEST GBU SUBSET EDRO
# ============================================================

def test_subset_EDRO(A, b, dim, label):

    violations = []

    # Event budget: sum(u) <= GE
    c = np.zeros(dim)
    c[:K] = 1.0

    value, _ = maximize_over_set(
        c, A, b, dim, label
    )

    violations.append({
        "constraint": "sum(u) <= GE",
        "maximum_lhs": value,
        "limit": GE,
        "maximum_violation": value - GE
    })

    # Aggregate severity budget: sum(s) <= GD
    c = np.zeros(dim)
    c[K:K+K*N] = 1.0

    value, _ = maximize_over_set(
        c, A, b, dim, label
    )

    violations.append({
        "constraint": "sum(s) <= GD",
        "maximum_lhs": value,
        "limit": GD,
        "maximum_violation": value - GD
    })

    # Event-severity links:
    # s[k,j] - V[k,j]*u[k] <= 0
    for k in range(K):
        for j in range(N):

            c = np.zeros(dim)

            c[K + k*N + j] = 1.0
            c[k] = -V[k, j]

            value, _ = maximize_over_set(
                c, A, b, dim, label
            )

            violations.append({
                "constraint": f"link[{k},{j}]",
                "maximum_lhs": value,
                "limit": 0.0,
                "maximum_violation": value
            })

    df = pd.DataFrame(violations)

    contained = (
        df["maximum_violation"].max()
        <= TOL_CONTAIN
    )

    return contained, df


GBU_subset_EDRO, GBU_violations = test_subset_EDRO(
    A_G, b_G, dim_G, "GBU"
)

Linked_GBU_subset_EDRO, Linked_GBU_violations = (
    test_subset_EDRO(A_GL, b_GL, dim_GL, "Linked GBU")
)

# ============================================================
# 6. TEST EDRO SUBSET BS
#
# EDRO implies:
# sum(u) + sum(s/V)
# <= 2 sum(u)
# <= 2 GE.
# ============================================================

EDRO_BS_upper_bound = 2.0 * GE

EDRO_subset_BS = (
    EDRO_BS_upper_bound
    <= GAMMA_BS + TOL_CONTAIN
)

# ============================================================
# 7. TEST EDRO SUBSET EDRO ABLATION
#
# True by construction: the ablation removes the
# event budget but retains all other EDRO constraints.
# ============================================================

EDRO_subset_ablation = True

# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

print("\n" + "="*65)
print("EDRO SUBSET GBU")
print("="*65)

print("Maximum GBU group norm over EDRO:",
      max_GBU_norm_over_EDRO)

print("GBU budget:", GAMMA_GBU)
print("Containment:", EDRO_subset_GBU)
print("Maximizing active expressions:",
      best_active_combination)

print("\n" + "="*65)
print("GBU SUBSET EDRO")
print("="*65)

print(GBU_violations.round(10).to_string(index=False))

print("Maximum violation:",
      GBU_violations["maximum_violation"].max())

print("Containment:", GBU_subset_EDRO)

print("\n" + "="*65)
print("EDRO SUBSET LINKED GBU")
print("="*65)

print("Maximum group norm over EDRO:",
      max_GBU_norm_over_EDRO)

print("Linked GBU budget:", GAMMA_GBU)
print("Containment:", EDRO_subset_linked_GBU)

print("\n" + "="*65)
print("LINKED GBU SUBSET EDRO")
print("="*65)

print(Linked_GBU_violations.round(10).to_string(index=False))

print("Maximum violation:",
      Linked_GBU_violations["maximum_violation"].max())

print("Containment:", Linked_GBU_subset_EDRO)

print("\n" + "="*65)
print("EDRO SUBSET BS")
print("="*65)

print("EDRO-implied BS budget upper bound:",
      EDRO_BS_upper_bound)

print("BS budget:", GAMMA_BS)
print("Containment:", EDRO_subset_BS)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

summary = pd.DataFrame([
    ["EDRO", "GBU", EDRO_subset_GBU],
    ["GBU", "EDRO", GBU_subset_EDRO],
    ["EDRO", "LINKED_GBU", EDRO_subset_linked_GBU],
    ["LINKED_GBU", "EDRO", Linked_GBU_subset_EDRO],
    ["EDRO", "BS", EDRO_subset_BS],
    ["EDRO", "EDRO_ABLATION", EDRO_subset_ablation]
], columns=[
    "candidate_subset",
    "candidate_superset",
    "containment"
])

print("\n" + "="*65)
print("FINAL EXACT CONTAINMENT SUMMARY")
print("="*65)

print(summary.to_string(index=False))

# ============================================================
# 10. EXPORT
# ============================================================

summary.to_csv(
    "EDRO_linked_GBU_containment_summary.csv",
    index=False
)

GBU_violations.to_csv(
    "EDRO_GBU_containment_violations.csv",
    index=False
)

Linked_GBU_violations.to_csv(
    "EDRO_linked_GBU_containment_violations.csv",
    index=False
)

print("\nAll containment tests completed.")
print("Three CSV files exported.")


In [ ]:

import numpy as np
import pandas as pd
import time
from scipy.optimize import linprog

# ============================================================
# 0. SETTINGS
# ============================================================

SEED = 20261118
rng = np.random.default_rng(SEED)

N = 8                       # DMUs
M = 3                       # Inputs
S = 2                       # Outputs
K = 3                       # Events

N_CAL = 500
N_TEST = 500
TARGET_COVERAGE = 0.95

# GBU group parameter
GAMMA_LOCAL = 2.0

# EDRO calibration scales
# GE0=1.0 is deliberately selected to test an active event budget.
GE0 = 1.0
GD0 = 4.0

TOL = 1e-7

# ============================================================
# 1. NOMINAL DEA DATA
# Rows = variables; columns = DMUs
# ============================================================

X = np.array([
    [5., 8., 8., 9., 7., 9., 10., 8.],
    [8., 5., 8., 9., 9., 7., 10., 10.],
    [6., 6., 8., 9., 8., 8., 10., 9.]
])

Y = np.array([
    [10., 2., 6., 5., 5., 6., 4., 5.],
    [ 2.,10., 6., 5., 6., 5., 4., 5.]
])

assert X.shape == (M, N)
assert Y.shape == (S, N)
assert np.all(X > 0)
assert np.all(Y > 0)

# ============================================================
# 2. EVENT EXPOSURES AND VULNERABILITIES
# ============================================================

AX = np.zeros((M, K, N))

for i in range(M):
    for k in range(K):
        for j in range(N):
            AX[i, k, j] = (
                0.10 + 0.05 * ((2*i + 3*k + j) % 5)
            )

BX = 0.5 * AX

AY = np.zeros((S, K, N))
BY = np.zeros((S, K, N))

for r in range(S):
    for k in range(K):
        for j in range(N):
            AY[r, k, j] = 0.10 * (-1)**(r + j + k)
            BY[r, k, j] = 0.05 * (-1)**(r + j)

# Heterogeneous vulnerability matrix: K x N
V = np.array([
    [1.00, .80, .60, .90, .70, 1.00, .60, .80],
    [.60, 1.00, .80, .70, 1.00, .60, .90, .70],
    [.80, .60, 1.00, .80, .60, .90, .70, 1.00]
])

# ============================================================
# 3. PHYSICAL NONNEGATIVITY CHECK
#
# Conservative bound:
# baseline >= sum |a| + sum v|b|
# ============================================================

for i in range(M):
    for j in range(N):
        worst_decrease = (
            np.sum(np.abs(AX[i, :, j]))
            + np.sum(V[:, j] * np.abs(BX[i, :, j]))
        )
        assert X[i, j] > worst_decrease

for r in range(S):
    for j in range(N):
        worst_decrease = (
            np.sum(np.abs(AY[r, :, j]))
            + np.sum(V[:, j] * np.abs(BY[r, :, j]))
        )
        assert Y[r, j] > worst_decrease

print("Physical nonnegativity checks: PASSED")

# ============================================================
# 4. GENERATE COMMON CALIBRATION AND TEST SCENARIOS
# ============================================================

def generate_scenarios(number, generator):

    U = np.zeros((number, K))
    SEV = np.zeros((number, K, N))

    probabilities = np.array([0.35, 0.25, 0.30])

    for t in range(number):

        events = generator.binomial(1, probabilities)
        U[t, :] = events

        q = generator.uniform(0.20, 1.00, size=(K, N))

        SEV[t, :, :] = V * events[:, None] * q

    return U, SEV


Ucal, Scal = generate_scenarios(N_CAL, rng)
Utest, Stest = generate_scenarios(N_TEST, rng)

# ============================================================
# 5. CALIBRATION
#
# All models use the same calibration scenarios.
# Each model is calibrated to approximately the same
# in-sample scenario coverage.
# ============================================================

def empirical_quantile(values, coverage):
    return np.quantile(values, coverage, method="higher")


# EDRO joint calibration score
edro_cal_scores = np.maximum(
    Ucal.sum(axis=1) / GE0,
    Scal.sum(axis=(1, 2)) / GD0
)

Q_EDRO = empirical_quantile(
    edro_cal_scores, TARGET_COVERAGE
)

GE = GE0 * Q_EDRO
GD = GD0 * Q_EDRO

# EDRO without event budget
ablation_scores = Scal.sum(axis=(1, 2))

GD_ABLATION = empirical_quantile(
    ablation_scores, TARGET_COVERAGE
)

# BS: global budget over normalized event and severity deviations
bs_cal_scores = (
    Ucal.sum(axis=1)
    + (Scal / V[None, :, :]).sum(axis=(1, 2))
)

GAMMA_BS = empirical_quantile(
    bs_cal_scores, TARGET_COVERAGE
)

# GBU: sum of event-group norms
def gbu_scenario_score(u, sev):

    normalized = sev / V[None, :, :]
    scores = np.zeros(len(u))

    for t in range(len(u)):

        total = 0.0

        for k in range(K):

            z = np.concatenate((
                [u[t, k]],
                normalized[t, k, :]
            ))

            group_norm = max(
                np.max(z),
                np.sum(z) / GAMMA_LOCAL
            )

            total += group_norm

        scores[t] = total

    return scores


gbu_cal_scores = gbu_scenario_score(Ucal, Scal)

GAMMA_GBU = empirical_quantile(
    gbu_cal_scores, TARGET_COVERAGE
)

print("\nCALIBRATED PARAMETERS")
print("---------------------")
print(f"EDRO Gamma_E       = {GE:.6f}")
print(f"EDRO Gamma_D       = {GD:.6f}")
print(f"EDRO-ablation GD   = {GD_ABLATION:.6f}")
print(f"BS Gamma           = {GAMMA_BS:.6f}")
print(f"GBU Gamma_G        = {GAMMA_GBU:.6f}")

print("\nEvent budget active:", GE < K - TOL)

# ============================================================
# 6. BUILD THE EXACT POLYHEDRAL UNCERTAINTY SETS
#
# z = (u_1,...,u_K, s_11,...,s_KN, optional t_1,...,t_K)
#
# A z <= b, z >= 0
# ============================================================

def build_uncertainty_set(model):

    NS = K * N
    BASE_DIM = K + NS

    if model in ("GBU", "GBU_LINKED"):
        DIM = BASE_DIM + K
    else:
        DIM = BASE_DIM

    rows = []
    rhs = []

    def add(row, bound):
        rows.append(row)
        rhs.append(bound)

    # --------------------------------------------------------
    # Common individual event bounds: 0 <= u_k <= 1
    # --------------------------------------------------------

    for k in range(K):
        row = np.zeros(DIM)
        row[k] = 1.0
        add(row, 1.0)

    # --------------------------------------------------------
    # EDRO and EDRO ablation
    # --------------------------------------------------------

    if model in ("EDRO", "EDRO_ABLATION"):

        # s_kj <= v_kj u_k
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0
                row[k] = -V[k, j]

                add(row, 0.0)

        # Global event activation budget
        if model == "EDRO":

            row = np.zeros(DIM)
            row[:K] = 1.0
            add(row, GE)

        # Aggregate severity budget
        row = np.zeros(DIM)
        row[K:K+NS] = 1.0

        if model == "EDRO":
            add(row, GD)
        else:
            add(row, GD_ABLATION)

    # --------------------------------------------------------
    # BS: one global budget over normalized deviations
    # --------------------------------------------------------

    elif model == "BS":

        # s_kj <= v_kj
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0

                add(row, V[k, j])

        # Sum u_k + sum s_kj/v_kj <= Gamma_BS
        row = np.zeros(DIM)
        row[:K] = 1.0

        for k in range(K):
            for j in range(N):
                q = K + k*N + j
                row[q] = 1.0 / V[k, j]

        add(row, GAMMA_BS)

    # --------------------------------------------------------
    # GBU: one group for each event
    #
    # Group k contains:
    # u_k, s_k1/v_k1, ..., s_kN/v_kN
    #
    # N_k(z_k) = max(||z_k||_infinity,
    #                 ||z_k||_1/Gamma_local)
    #
    # Sum_k N_k(z_k) <= Gamma_GBU
    # --------------------------------------------------------

    elif model in ("GBU", "GBU_LINKED"):

        T_START = BASE_DIM

        for k in range(K):

            tk = T_START + k

            # u_k <= t_k
            row = np.zeros(DIM)
            row[k] = 1.0
            row[tk] = -1.0
            add(row, 0.0)

            # s_kj/v_kj <= t_k
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0 / V[k, j]
                row[tk] = -1.0

                add(row, 0.0)

            # (u_k + sum_j s_kj/v_kj)/Gamma_local <= t_k
            row = np.zeros(DIM)
            row[k] = 1.0 / GAMMA_LOCAL
            row[tk] = -1.0

            for j in range(N):
                q = K + k*N + j
                row[q] = 1.0 / (
                    GAMMA_LOCAL * V[k, j]
                )

            add(row, 0.0)

        # Global GBU budget
        row = np.zeros(DIM)
        row[T_START:T_START+K] = 1.0
        add(row, GAMMA_GBU)

        # Explicit local severity bounds s_kj <= v_kj
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                row = np.zeros(DIM)
                row[q] = 1.0

                add(row, V[k, j])

        # Linked GBU additionally requires:
        # s_kj <= v_kj * u_k.
        # This is the only geometric difference from GBU.
        if model == "GBU_LINKED":
            for k in range(K):
                for j in range(N):

                    q = K + k*N + j

                    row = np.zeros(DIM)
                    row[q] = 1.0
                    row[k] = -V[k, j]

                    add(row, 0.0)

    else:
        raise ValueError(f"Unknown model: {model}")

    return np.asarray(rows), np.asarray(rhs), DIM


# ============================================================
# 7. SUPPORT FUNCTION PRIMAL-DUAL VERIFICATION
# ============================================================

A_EDRO, b_EDRO, DIM_EDRO = build_uncertainty_set("EDRO")

c_test = rng.normal(size=DIM_EDRO)

# Primal:
# max c'z subject to Az <= b, z >= 0

primal = linprog(
    -c_test,
    A_ub=A_EDRO,
    b_ub=b_EDRO,
    bounds=(0, None),
    method="highs"
)

# Dual:
# min b'y subject to A'y >= c, y >= 0

dual = linprog(
    b_EDRO,
    A_ub=-A_EDRO.T,
    b_ub=-c_test,
    bounds=(0, None),
    method="highs"
)

assert primal.success, primal.message
assert dual.success, dual.message

support_primal = -primal.fun
support_dual = dual.fun

print("\nEDRO SUPPORT FUNCTION CHECK")
print("---------------------------")
print("Primal value:", support_primal)
print("Dual value:  ", support_dual)
print("Absolute gap:", abs(support_primal-support_dual))

assert abs(support_primal-support_dual) < TOL

# ============================================================
# 8. EXACT ROBUST BCC LP
#
# Input constraint:
# max_z { sum_j x_ij(z) lambda_j
#        - theta*x_io(z) } <= 0
#
# Output constraint:
# max_z { y_ro(z)
#        - sum_j y_rj(z) lambda_j } <= 0
#
# Each support function is dualized exactly.
# ============================================================

def solve_bcc(o, model):

    # ========================================================
    # NOMINAL BCC
    # ========================================================

    if model == "NOMINAL":

        NV = N + 1
        THETA = N

        objective = np.zeros(NV)
        objective[THETA] = 1.0

        AUB = []
        BUB = []

        # Input constraints
        for i in range(M):

            row = np.zeros(NV)
            row[:N] = X[i, :]
            row[THETA] = -X[i, o]

            AUB.append(row)
            BUB.append(0.0)

        # Output constraints
        for r in range(S):

            row = np.zeros(NV)
            row[:N] = -Y[r, :]

            AUB.append(row)
            BUB.append(-Y[r, o])

        AEQ = np.zeros((1, NV))
        AEQ[0, :N] = 1.0

        return linprog(
            objective,
            A_ub=np.asarray(AUB),
            b_ub=np.asarray(BUB),
            A_eq=AEQ,
            b_eq=np.array([1.0]),
            bounds=(0, None),
            method="highs"
        )

    # ========================================================
    # ROBUST BCC
    # ========================================================

    ASET, BSET, DIM = build_uncertainty_set(model)

    L = len(BSET)
    NROBUST = M + S

    THETA = N
    NV = N + 1 + NROBUST * L

    objective = np.zeros(NV)
    objective[THETA] = 1.0

    AUB = []
    BUB = []

    def dual_index(row_number, ell):
        return N + 1 + row_number * L + ell

    def add_constraint(coefficients, rhs=0.0):

        row = np.zeros(NV)

        for idx, value in coefficients.items():
            row[idx] += value

        AUB.append(row)
        BUB.append(rhs)

    # ========================================================
    # INPUT CONSTRAINTS
    #
    # Nominal residual + b'y <= 0
    # c_q(lambda,theta) - A[:,q]'y <= 0
    # ========================================================

    for i in range(M):

        row_number = i

        # Nominal input residual
        coeff = {}

        for j in range(N):
            coeff[j] = X[i, j]

        coeff[THETA] = -X[i, o]

        # Add b'y
        for ell in range(L):
            coeff[dual_index(row_number, ell)] = BSET[ell]

        add_constraint(coeff, 0.0)

        # Event-coordinate dual feasibility
        for k in range(K):

            coeff = {}

            for j in range(N):
                coeff[j] = AX[i, k, j]

            coeff[THETA] = -AX[i, k, o]

            # -A'y
            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, k]
                )

            add_constraint(coeff, 0.0)

        # Severity-coordinate dual feasibility
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                coeff = {
                    j: BX[i, k, j]
                }

                if j == o:
                    coeff[THETA] = -BX[i, k, o]

                for ell in range(L):
                    coeff[dual_index(row_number, ell)] = (
                        -ASET[ell, q]
                    )

                add_constraint(coeff, 0.0)

        # Additional GBU coordinates t_k:
        # Their uncertainty coefficient is zero.
        for q in range(K + K*N, DIM):

            coeff = {}

            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, q]
                )

            add_constraint(coeff, 0.0)

    # ========================================================
    # OUTPUT CONSTRAINTS
    #
    # y_ro - sum_j y_rj lambda_j + b'y <= 0
    #
    # c_u = a_ro - sum_j a_rj lambda_j
    # c_s = b_ro - sum_j b_rj lambda_j
    # ========================================================

    for r in range(S):

        row_number = M + r

        # Nominal output residual
        coeff = {}

        for j in range(N):
            coeff[j] = -Y[r, j]

        for ell in range(L):
            coeff[dual_index(row_number, ell)] = BSET[ell]

        add_constraint(coeff, -Y[r, o])

        # Event-coordinate dual feasibility
        for k in range(K):

            coeff = {}

            for j in range(N):
                coeff[j] = -AY[r, k, j]

            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, k]
                )

            # -sum_j a_rkj lambda_j - A'y <= -a_rko
            add_constraint(coeff, -AY[r, k, o])

        # Severity-coordinate dual feasibility
        for k in range(K):
            for j in range(N):

                q = K + k*N + j

                coeff = {
                    j: -BY[r, k, j]
                }

                for ell in range(L):
                    coeff[dual_index(row_number, ell)] = (
                        -ASET[ell, q]
                    )

                rhs = -BY[r, k, o] if j == o else 0.0

                add_constraint(coeff, rhs)

        # Additional GBU coordinates t_k
        for q in range(K + K*N, DIM):

            coeff = {}

            for ell in range(L):
                coeff[dual_index(row_number, ell)] = (
                    -ASET[ell, q]
                )

            add_constraint(coeff, 0.0)

    # ========================================================
    # CONVEXITY
    # ========================================================

    AEQ = np.zeros((1, NV))
    AEQ[0, :N] = 1.0

    result = linprog(
        objective,
        A_ub=np.asarray(AUB),
        b_ub=np.asarray(BUB),
        A_eq=AEQ,
        b_eq=np.array([1.0]),
        bounds=(0, None),
        method="highs"
    )

    return result


# ============================================================
# 9. SOLVE ALL MODELS FOR ALL DMUs
# ============================================================

MODELS = [
    "NOMINAL",
    "BS",
    "GBU",
    "GBU_LINKED",
    "EDRO",
    "EDRO_ABLATION"
]

records = []

total_start = time.time()

for model in MODELS:

    print(f"\nSOLVING MODEL: {model}")

    for o in range(N):

        start = time.time()

        result = solve_bcc(o, model)

        elapsed = time.time() - start

        if result.success:

            score = result.fun
            status = "optimal"

        else:

            score = np.nan
            status = result.message

        records.append({
            "model": model,
            "DMU": o + 1,
            "efficiency": score,
            "status": status,
            "runtime_seconds": elapsed
        })

        print(
            f"DMU {o+1}: "
            f"efficiency={score:.8f}"
            if result.success
            else f"DMU {o+1}: FAILED: {result.message}"
        )

results = pd.DataFrame(records)

print("\nTotal runtime:", time.time() - total_start)

# ============================================================
# 10. EFFICIENCY MATRIX AND RANKINGS
# ============================================================

pivot = results.pivot(
    index="DMU",
    columns="model",
    values="efficiency"
)

print("\nEFFICIENCY MATRIX")
print(pivot.round(6))

results["rank"] = (
    results.groupby("model")["efficiency"]
    .rank(method="min", ascending=True)
)

summary = (
    results.groupby("model")
    .agg(
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median"),
        minimum_efficiency=("efficiency", "min"),
        maximum_efficiency=("efficiency", "max"),
        mean_runtime=("runtime_seconds", "mean"),
        successful_solves=(
            "status",
            lambda x: (x == "optimal").sum()
        )
    )
)

print("\nMODEL SUMMARY")
print(summary.round(6))

# ============================================================
# 11. INCREMENTAL MODEL EFFECTS
# ============================================================

effects = pd.DataFrame(index=pivot.index)

effects["EDRO_minus_BS"] = (
    pivot["EDRO"] - pivot["BS"]
)

effects["EDRO_minus_GBU"] = (
    pivot["EDRO"] - pivot["GBU"]
)

effects["EDRO_minus_GBU_LINKED"] = (
    pivot["EDRO"] - pivot["GBU_LINKED"]
)

effects["GBU_LINKED_minus_GBU"] = (
    pivot["GBU_LINKED"] - pivot["GBU"]
)

effects["EDRO_minus_ablation"] = (
    pivot["EDRO"] - pivot["EDRO_ABLATION"]
)

effects["BS_minus_GBU"] = (
    pivot["BS"] - pivot["GBU"]
)

print("\nINCREMENTAL EFFICIENCY EFFECTS")
print(effects.round(6))

print("\nMEAN ABSOLUTE SCORE DIFFERENCES")
print(effects.abs().mean().round(6))

# ============================================================
# 12. OUT-OF-SAMPLE COVERAGE
# ============================================================

edro_test_scores = np.maximum(
    Utest.sum(axis=1) / GE0,
    Stest.sum(axis=(1, 2)) / GD0
)

edro_test_coverage = np.mean(
    edro_test_scores <= Q_EDRO
)

ablation_test_coverage = np.mean(
    Stest.sum(axis=(1, 2)) <= GD_ABLATION
)

bs_test_scores = (
    Utest.sum(axis=1)
    + (Stest / V[None, :, :]).sum(axis=(1, 2))
)

bs_test_coverage = np.mean(
    bs_test_scores <= GAMMA_BS
)

gbu_test_scores = gbu_scenario_score(Utest, Stest)

gbu_test_coverage = np.mean(
    gbu_test_scores <= GAMMA_GBU
)

# The generated calibration/test scenarios satisfy s_kj <= v_kj*u_k
# by construction. Therefore linked GBU has exactly the same
# scenario score, calibration budget, and scenario coverage as GBU.
linked_gbu_test_coverage = np.mean(
    gbu_test_scores <= GAMMA_GBU
)

coverage = pd.DataFrame({
    "model": [
        "EDRO",
        "BS",
        "GBU",
        "GBU_LINKED",
        "EDRO_ABLATION"
    ],
    "calibration_target": [
        TARGET_COVERAGE
    ] * 5,
    "test_coverage": [
        edro_test_coverage,
        bs_test_coverage,
        gbu_test_coverage,
        linked_gbu_test_coverage,
        ablation_test_coverage
    ]
})

print("\nOUT-OF-SAMPLE COVERAGE")
print(coverage)

# ============================================================
# 13. CALIBRATION COVERAGE
# ============================================================

calibration_coverage = pd.DataFrame({
    "model": [
        "EDRO", "BS", "GBU", "GBU_LINKED", "EDRO_ABLATION"
    ],
    "calibration_coverage": [
        np.mean(edro_cal_scores <= Q_EDRO),
        np.mean(bs_cal_scores <= GAMMA_BS),
        np.mean(gbu_cal_scores <= GAMMA_GBU),
        np.mean(gbu_cal_scores <= GAMMA_GBU),
        np.mean(ablation_scores <= GD_ABLATION)
    ]
})

print("\nIN-SAMPLE CALIBRATION COVERAGE")
print(calibration_coverage)

# ============================================================
# 14. RANKING CHANGES
# ============================================================

rank_pivot = results.pivot(
    index="DMU",
    columns="model",
    values="rank"
)

print("\nRANKINGS")
print(rank_pivot)

for model in ["BS", "GBU", "GBU_LINKED", "EDRO", "EDRO_ABLATION"]:

    changes = np.sum(
        rank_pivot["NOMINAL"] != rank_pivot[model]
    )

    print(
        f"Ranking changes: NOMINAL vs {model}: {changes}/{N}"
    )

# ============================================================
# 15. EXPORT ALL RESULTS
# ============================================================

results.to_csv(
    "EDRO_BCC_all_results.csv",
    index=False
)

pivot.to_csv(
    "EDRO_BCC_efficiency_matrix.csv"
)

summary.to_csv(
    "EDRO_BCC_model_summary.csv"
)

effects.to_csv(
    "EDRO_BCC_incremental_effects.csv"
)

coverage.to_csv(
    "EDRO_BCC_test_coverage.csv",
    index=False
)

calibration_coverage.to_csv(
    "EDRO_BCC_calibration_coverage.csv",
    index=False
)

rank_pivot.to_csv(
    "EDRO_BCC_rankings.csv"
)

print("\nEXPORTED FILES")
print("--------------")
print("EDRO_BCC_all_results.csv")
print("EDRO_BCC_efficiency_matrix.csv")
print("EDRO_BCC_model_summary.csv")
print("EDRO_BCC_incremental_effects.csv")
print("EDRO_BCC_test_coverage.csv")
print("EDRO_BCC_calibration_coverage.csv")
print("EDRO_BCC_rankings.csv")

print("\nNOTE ON LINKED GBU CALIBRATION")
print("-------------------------------")
print("Calibration and test scenarios are generated with")
print("SEV = V * event * q, so all scenarios satisfy the")
print("linking constraints. Thus GBU and GBU_LINKED have")
print("the same calibrated Gamma_GBU and the same scenario coverage.")
print("Their robust DEA scores can nevertheless differ because")
print("their uncertainty sets differ outside the sampled scenarios.")

print("\nEXPERIMENT COMPLETED.")


In [ ]:
"""Repeated-seed evaluation of EDRO versus BS/GBU comparators.

Self-contained: the single-seed EDRO experiment is embedded below.
Run this file in Deepnote or ordinary Python. It performs 100 independent
calibration/test replications, each with 500 calibration and 500 test cases.
"""
import io
import contextlib
import time
import numpy as np
import pandas as pd

N_SEEDS = 100
FIRST_SEED = 20261118
BOOTSTRAP_REPS = 10000

BASE_SCRIPT = '\nimport numpy as np\nimport pandas as pd\nimport time\nfrom scipy.optimize import linprog\n\n# ============================================================\n# 0. SETTINGS\n# ============================================================\n\nSEED = 20261118\nrng = np.random.default_rng(SEED)\n\nN = 8                       # DMUs\nM = 3                       # Inputs\nS = 2                       # Outputs\nK = 3                       # Events\n\nN_CAL = 500\nN_TEST = 500\nTARGET_COVERAGE = 0.95\n\n# GBU group parameter\nGAMMA_LOCAL = 2.0\n\n# EDRO calibration scales\n# GE0=1.0 is deliberately selected to test an active event budget.\nGE0 = 1.0\nGD0 = 4.0\n\nTOL = 1e-7\n\n# ============================================================\n# 1. NOMINAL DEA DATA\n# Rows = variables; columns = DMUs\n# ============================================================\n\nX = np.array([\n    [5., 8., 8., 9., 7., 9., 10., 8.],\n    [8., 5., 8., 9., 9., 7., 10., 10.],\n    [6., 6., 8., 9., 8., 8., 10., 9.]\n])\n\nY = np.array([\n    [10., 2., 6., 5., 5., 6., 4., 5.],\n    [ 2.,10., 6., 5., 6., 5., 4., 5.]\n])\n\nassert X.shape == (M, N)\nassert Y.shape == (S, N)\nassert np.all(X > 0)\nassert np.all(Y > 0)\n\n# ============================================================\n# 2. EVENT EXPOSURES AND VULNERABILITIES\n# ============================================================\n\nAX = np.zeros((M, K, N))\n\nfor i in range(M):\n    for k in range(K):\n        for j in range(N):\n            AX[i, k, j] = (\n                0.10 + 0.05 * ((2*i + 3*k + j) % 5)\n            )\n\nBX = 0.5 * AX\n\nAY = np.zeros((S, K, N))\nBY = np.zeros((S, K, N))\n\nfor r in range(S):\n    for k in range(K):\n        for j in range(N):\n            AY[r, k, j] = 0.10 * (-1)**(r + j + k)\n            BY[r, k, j] = 0.05 * (-1)**(r + j)\n\n# Heterogeneous vulnerability matrix: K x N\nV = np.array([\n    [1.00, .80, .60, .90, .70, 1.00, .60, .80],\n    [.60, 1.00, .80, .70, 1.00, .60, .90, .70],\n    [.80, .60, 1.00, .80, .60, .90, .70, 1.00]\n])\n\n# ============================================================\n# 3. PHYSICAL NONNEGATIVITY CHECK\n#\n# Conservative bound:\n# baseline >= sum |a| + sum v|b|\n# ============================================================\n\nfor i in range(M):\n    for j in range(N):\n        worst_decrease = (\n            np.sum(np.abs(AX[i, :, j]))\n            + np.sum(V[:, j] * np.abs(BX[i, :, j]))\n        )\n        assert X[i, j] > worst_decrease\n\nfor r in range(S):\n    for j in range(N):\n        worst_decrease = (\n            np.sum(np.abs(AY[r, :, j]))\n            + np.sum(V[:, j] * np.abs(BY[r, :, j]))\n        )\n        assert Y[r, j] > worst_decrease\n\nprint("Physical nonnegativity checks: PASSED")\n\n# ============================================================\n# 4. GENERATE COMMON CALIBRATION AND TEST SCENARIOS\n# ============================================================\n\ndef generate_scenarios(number, generator):\n\n    U = np.zeros((number, K))\n    SEV = np.zeros((number, K, N))\n\n    probabilities = np.array([0.35, 0.25, 0.30])\n\n    for t in range(number):\n\n        events = generator.binomial(1, probabilities)\n        U[t, :] = events\n\n        q = generator.uniform(0.20, 1.00, size=(K, N))\n\n        SEV[t, :, :] = V * events[:, None] * q\n\n    return U, SEV\n\n\nUcal, Scal = generate_scenarios(N_CAL, rng)\nUtest, Stest = generate_scenarios(N_TEST, rng)\n\n# ============================================================\n# 5. CALIBRATION\n#\n# All models use the same calibration scenarios.\n# Each model is calibrated to approximately the same\n# in-sample scenario coverage.\n# ============================================================\n\ndef empirical_quantile(values, coverage):\n    return np.quantile(values, coverage, method="higher")\n\n\n# EDRO joint calibration score\nedro_cal_scores = np.maximum(\n    Ucal.sum(axis=1) / GE0,\n    Scal.sum(axis=(1, 2)) / GD0\n)\n\nQ_EDRO = empirical_quantile(\n    edro_cal_scores, TARGET_COVERAGE\n)\n\nGE = GE0 * Q_EDRO\nGD = GD0 * Q_EDRO\n\n# EDRO without event budget\nablation_scores = Scal.sum(axis=(1, 2))\n\nGD_ABLATION = empirical_quantile(\n    ablation_scores, TARGET_COVERAGE\n)\n\n# BS: global budget over normalized event and severity deviations\nbs_cal_scores = (\n    Ucal.sum(axis=1)\n    + (Scal / V[None, :, :]).sum(axis=(1, 2))\n)\n\nGAMMA_BS = empirical_quantile(\n    bs_cal_scores, TARGET_COVERAGE\n)\n\n# GBU: sum of event-group norms\ndef gbu_scenario_score(u, sev):\n\n    normalized = sev / V[None, :, :]\n    scores = np.zeros(len(u))\n\n    for t in range(len(u)):\n\n        total = 0.0\n\n        for k in range(K):\n\n            z = np.concatenate((\n                [u[t, k]],\n                normalized[t, k, :]\n            ))\n\n            group_norm = max(\n                np.max(z),\n                np.sum(z) / GAMMA_LOCAL\n            )\n\n            total += group_norm\n\n        scores[t] = total\n\n    return scores\n\n\ngbu_cal_scores = gbu_scenario_score(Ucal, Scal)\n\nGAMMA_GBU = empirical_quantile(\n    gbu_cal_scores, TARGET_COVERAGE\n)\n\nprint("\\nCALIBRATED PARAMETERS")\nprint("---------------------")\nprint(f"EDRO Gamma_E       = {GE:.6f}")\nprint(f"EDRO Gamma_D       = {GD:.6f}")\nprint(f"EDRO-ablation GD   = {GD_ABLATION:.6f}")\nprint(f"BS Gamma           = {GAMMA_BS:.6f}")\nprint(f"GBU Gamma_G        = {GAMMA_GBU:.6f}")\n\nprint("\\nEvent budget active:", GE < K - TOL)\n\n# ============================================================\n# 6. BUILD THE EXACT POLYHEDRAL UNCERTAINTY SETS\n#\n# z = (u_1,...,u_K, s_11,...,s_KN, optional t_1,...,t_K)\n#\n# A z <= b, z >= 0\n# ============================================================\n\ndef build_uncertainty_set(model):\n\n    NS = K * N\n    BASE_DIM = K + NS\n\n    if model in ("GBU", "GBU_LINKED"):\n        DIM = BASE_DIM + K\n    else:\n        DIM = BASE_DIM\n\n    rows = []\n    rhs = []\n\n    def add(row, bound):\n        rows.append(row)\n        rhs.append(bound)\n\n    # --------------------------------------------------------\n    # Common individual event bounds: 0 <= u_k <= 1\n    # --------------------------------------------------------\n\n    for k in range(K):\n        row = np.zeros(DIM)\n        row[k] = 1.0\n        add(row, 1.0)\n\n    # --------------------------------------------------------\n    # EDRO and EDRO ablation\n    # --------------------------------------------------------\n\n    if model in ("EDRO", "EDRO_ABLATION"):\n\n        # s_kj <= v_kj u_k\n        for k in range(K):\n            for j in range(N):\n\n                q = K + k*N + j\n\n                row = np.zeros(DIM)\n                row[q] = 1.0\n                row[k] = -V[k, j]\n\n                add(row, 0.0)\n\n        # Global event activation budget\n        if model == "EDRO":\n\n            row = np.zeros(DIM)\n            row[:K] = 1.0\n            add(row, GE)\n\n        # Aggregate severity budget\n        row = np.zeros(DIM)\n        row[K:K+NS] = 1.0\n\n        if model == "EDRO":\n            add(row, GD)\n        else:\n            add(row, GD_ABLATION)\n\n    # --------------------------------------------------------\n    # BS: one global budget over normalized deviations\n    # --------------------------------------------------------\n\n    elif model == "BS":\n\n        # s_kj <= v_kj\n        for k in range(K):\n            for j in range(N):\n\n                q = K + k*N + j\n\n                row = np.zeros(DIM)\n                row[q] = 1.0\n\n                add(row, V[k, j])\n\n        # Sum u_k + sum s_kj/v_kj <= Gamma_BS\n        row = np.zeros(DIM)\n        row[:K] = 1.0\n\n        for k in range(K):\n            for j in range(N):\n                q = K + k*N + j\n                row[q] = 1.0 / V[k, j]\n\n        add(row, GAMMA_BS)\n\n    # --------------------------------------------------------\n    # GBU: one group for each event\n    #\n    # Group k contains:\n    # u_k, s_k1/v_k1, ..., s_kN/v_kN\n    #\n    # N_k(z_k) = max(||z_k||_infinity,\n    #                 ||z_k||_1/Gamma_local)\n    #\n    # Sum_k N_k(z_k) <= Gamma_GBU\n    # --------------------------------------------------------\n\n    elif model in ("GBU", "GBU_LINKED"):\n\n        T_START = BASE_DIM\n\n        for k in range(K):\n\n            tk = T_START + k\n\n            # u_k <= t_k\n            row = np.zeros(DIM)\n            row[k] = 1.0\n            row[tk] = -1.0\n            add(row, 0.0)\n\n            # s_kj/v_kj <= t_k\n            for j in range(N):\n\n                q = K + k*N + j\n\n                row = np.zeros(DIM)\n                row[q] = 1.0 / V[k, j]\n                row[tk] = -1.0\n\n                add(row, 0.0)\n\n            # (u_k + sum_j s_kj/v_kj)/Gamma_local <= t_k\n            row = np.zeros(DIM)\n            row[k] = 1.0 / GAMMA_LOCAL\n            row[tk] = -1.0\n\n            for j in range(N):\n                q = K + k*N + j\n                row[q] = 1.0 / (\n                    GAMMA_LOCAL * V[k, j]\n                )\n\n            add(row, 0.0)\n\n        # Global GBU budget\n        row = np.zeros(DIM)\n        row[T_START:T_START+K] = 1.0\n        add(row, GAMMA_GBU)\n\n        # Explicit local severity bounds s_kj <= v_kj\n        for k in range(K):\n            for j in range(N):\n\n                q = K + k*N + j\n\n                row = np.zeros(DIM)\n                row[q] = 1.0\n\n                add(row, V[k, j])\n\n        # Linked GBU additionally requires:\n        # s_kj <= v_kj * u_k.\n        # This is the only geometric difference from GBU.\n        if model == "GBU_LINKED":\n            for k in range(K):\n                for j in range(N):\n\n                    q = K + k*N + j\n\n                    row = np.zeros(DIM)\n                    row[q] = 1.0\n                    row[k] = -V[k, j]\n\n                    add(row, 0.0)\n\n    else:\n        raise ValueError(f"Unknown model: {model}")\n\n    return np.asarray(rows), np.asarray(rhs), DIM\n\n\n# ============================================================\n# 7. SUPPORT FUNCTION PRIMAL-DUAL VERIFICATION\n# ============================================================\n\nA_EDRO, b_EDRO, DIM_EDRO = build_uncertainty_set("EDRO")\n\nc_test = rng.normal(size=DIM_EDRO)\n\n# Primal:\n# max c\'z subject to Az <= b, z >= 0\n\nprimal = linprog(\n    -c_test,\n    A_ub=A_EDRO,\n    b_ub=b_EDRO,\n    bounds=(0, None),\n    method="highs"\n)\n\n# Dual:\n# min b\'y subject to A\'y >= c, y >= 0\n\ndual = linprog(\n    b_EDRO,\n    A_ub=-A_EDRO.T,\n    b_ub=-c_test,\n    bounds=(0, None),\n    method="highs"\n)\n\nassert primal.success, primal.message\nassert dual.success, dual.message\n\nsupport_primal = -primal.fun\nsupport_dual = dual.fun\n\nprint("\\nEDRO SUPPORT FUNCTION CHECK")\nprint("---------------------------")\nprint("Primal value:", support_primal)\nprint("Dual value:  ", support_dual)\nprint("Absolute gap:", abs(support_primal-support_dual))\n\nassert abs(support_primal-support_dual) < TOL\n\n# ============================================================\n# 8. EXACT ROBUST BCC LP\n#\n# Input constraint:\n# max_z { sum_j x_ij(z) lambda_j\n#        - theta*x_io(z) } <= 0\n#\n# Output constraint:\n# max_z { y_ro(z)\n#        - sum_j y_rj(z) lambda_j } <= 0\n#\n# Each support function is dualized exactly.\n# ============================================================\n\ndef solve_bcc(o, model):\n\n    # ========================================================\n    # NOMINAL BCC\n    # ========================================================\n\n    if model == "NOMINAL":\n\n        NV = N + 1\n        THETA = N\n\n        objective = np.zeros(NV)\n        objective[THETA] = 1.0\n\n        AUB = []\n        BUB = []\n\n        # Input constraints\n        for i in range(M):\n\n            row = np.zeros(NV)\n            row[:N] = X[i, :]\n            row[THETA] = -X[i, o]\n\n            AUB.append(row)\n            BUB.append(0.0)\n\n        # Output constraints\n        for r in range(S):\n\n            row = np.zeros(NV)\n            row[:N] = -Y[r, :]\n\n            AUB.append(row)\n            BUB.append(-Y[r, o])\n\n        AEQ = np.zeros((1, NV))\n        AEQ[0, :N] = 1.0\n\n        return linprog(\n            objective,\n            A_ub=np.asarray(AUB),\n            b_ub=np.asarray(BUB),\n            A_eq=AEQ,\n            b_eq=np.array([1.0]),\n            bounds=(0, None),\n            method="highs"\n        )\n\n    # ========================================================\n    # ROBUST BCC\n    # ========================================================\n\n    ASET, BSET, DIM = build_uncertainty_set(model)\n\n    L = len(BSET)\n    NROBUST = M + S\n\n    THETA = N\n    NV = N + 1 + NROBUST * L\n\n    objective = np.zeros(NV)\n    objective[THETA] = 1.0\n\n    AUB = []\n    BUB = []\n\n    def dual_index(row_number, ell):\n        return N + 1 + row_number * L + ell\n\n    def add_constraint(coefficients, rhs=0.0):\n\n        row = np.zeros(NV)\n\n        for idx, value in coefficients.items():\n            row[idx] += value\n\n        AUB.append(row)\n        BUB.append(rhs)\n\n    # ========================================================\n    # INPUT CONSTRAINTS\n    #\n    # Nominal residual + b\'y <= 0\n    # c_q(lambda,theta) - A[:,q]\'y <= 0\n    # ========================================================\n\n    for i in range(M):\n\n        row_number = i\n\n        # Nominal input residual\n        coeff = {}\n\n        for j in range(N):\n            coeff[j] = X[i, j]\n\n        coeff[THETA] = -X[i, o]\n\n        # Add b\'y\n        for ell in range(L):\n            coeff[dual_index(row_number, ell)] = BSET[ell]\n\n        add_constraint(coeff, 0.0)\n\n        # Event-coordinate dual feasibility\n        for k in range(K):\n\n            coeff = {}\n\n            for j in range(N):\n                coeff[j] = AX[i, k, j]\n\n            coeff[THETA] = -AX[i, k, o]\n\n            # -A\'y\n            for ell in range(L):\n                coeff[dual_index(row_number, ell)] = (\n                    -ASET[ell, k]\n                )\n\n            add_constraint(coeff, 0.0)\n\n        # Severity-coordinate dual feasibility\n        for k in range(K):\n            for j in range(N):\n\n                q = K + k*N + j\n\n                coeff = {\n                    j: BX[i, k, j]\n                }\n\n                if j == o:\n                    coeff[THETA] = -BX[i, k, o]\n\n                for ell in range(L):\n                    coeff[dual_index(row_number, ell)] = (\n                        -ASET[ell, q]\n                    )\n\n                add_constraint(coeff, 0.0)\n\n        # Additional GBU coordinates t_k:\n        # Their uncertainty coefficient is zero.\n        for q in range(K + K*N, DIM):\n\n            coeff = {}\n\n            for ell in range(L):\n                coeff[dual_index(row_number, ell)] = (\n                    -ASET[ell, q]\n                )\n\n            add_constraint(coeff, 0.0)\n\n    # ========================================================\n    # OUTPUT CONSTRAINTS\n    #\n    # y_ro - sum_j y_rj lambda_j + b\'y <= 0\n    #\n    # c_u = a_ro - sum_j a_rj lambda_j\n    # c_s = b_ro - sum_j b_rj lambda_j\n    # ========================================================\n\n    for r in range(S):\n\n        row_number = M + r\n\n        # Nominal output residual\n        coeff = {}\n\n        for j in range(N):\n            coeff[j] = -Y[r, j]\n\n        for ell in range(L):\n            coeff[dual_index(row_number, ell)] = BSET[ell]\n\n        add_constraint(coeff, -Y[r, o])\n\n        # Event-coordinate dual feasibility\n        for k in range(K):\n\n            coeff = {}\n\n            for j in range(N):\n                coeff[j] = -AY[r, k, j]\n\n            for ell in range(L):\n                coeff[dual_index(row_number, ell)] = (\n                    -ASET[ell, k]\n                )\n\n            # -sum_j a_rkj lambda_j - A\'y <= -a_rko\n            add_constraint(coeff, -AY[r, k, o])\n\n        # Severity-coordinate dual feasibility\n        for k in range(K):\n            for j in range(N):\n\n                q = K + k*N + j\n\n                coeff = {\n                    j: -BY[r, k, j]\n                }\n\n                for ell in range(L):\n                    coeff[dual_index(row_number, ell)] = (\n                        -ASET[ell, q]\n                    )\n\n                rhs = -BY[r, k, o] if j == o else 0.0\n\n                add_constraint(coeff, rhs)\n\n        # Additional GBU coordinates t_k\n        for q in range(K + K*N, DIM):\n\n            coeff = {}\n\n            for ell in range(L):\n                coeff[dual_index(row_number, ell)] = (\n                    -ASET[ell, q]\n                )\n\n            add_constraint(coeff, 0.0)\n\n    # ========================================================\n    # CONVEXITY\n    # ========================================================\n\n    AEQ = np.zeros((1, NV))\n    AEQ[0, :N] = 1.0\n\n    result = linprog(\n        objective,\n        A_ub=np.asarray(AUB),\n        b_ub=np.asarray(BUB),\n        A_eq=AEQ,\n        b_eq=np.array([1.0]),\n        bounds=(0, None),\n        method="highs"\n    )\n\n    return result\n\n\n# ============================================================\n# 9. SOLVE ALL MODELS FOR ALL DMUs\n# ============================================================\n\nMODELS = [\n    "NOMINAL",\n    "BS",\n    "GBU",\n    "GBU_LINKED",\n    "EDRO",\n    "EDRO_ABLATION"\n]\n\nrecords = []\n\ntotal_start = time.time()\n\nfor model in MODELS:\n\n    print(f"\\nSOLVING MODEL: {model}")\n\n    for o in range(N):\n\n        start = time.time()\n\n        result = solve_bcc(o, model)\n\n        elapsed = time.time() - start\n\n        if result.success:\n\n            score = result.fun\n            status = "optimal"\n\n        else:\n\n            score = np.nan\n            status = result.message\n\n        records.append({\n            "model": model,\n            "DMU": o + 1,\n            "efficiency": score,\n            "status": status,\n            "runtime_seconds": elapsed\n        })\n\n        print(\n            f"DMU {o+1}: "\n            f"efficiency={score:.8f}"\n            if result.success\n            else f"DMU {o+1}: FAILED: {result.message}"\n        )\n\nresults = pd.DataFrame(records)\n\nprint("\\nTotal runtime:", time.time() - total_start)\n\n# ============================================================\n# 10. EFFICIENCY MATRIX AND RANKINGS\n# ============================================================\n\npivot = results.pivot(\n    index="DMU",\n    columns="model",\n    values="efficiency"\n)\n\nprint("\\nEFFICIENCY MATRIX")\nprint(pivot.round(6))\n\nresults["rank"] = (\n    results.groupby("model")["efficiency"]\n    .rank(method="min", ascending=True)\n)\n\nsummary = (\n    results.groupby("model")\n    .agg(\n        mean_efficiency=("efficiency", "mean"),\n        median_efficiency=("efficiency", "median"),\n        minimum_efficiency=("efficiency", "min"),\n        maximum_efficiency=("efficiency", "max"),\n        mean_runtime=("runtime_seconds", "mean"),\n        successful_solves=(\n            "status",\n            lambda x: (x == "optimal").sum()\n        )\n    )\n)\n\nprint("\\nMODEL SUMMARY")\nprint(summary.round(6))\n\n# ============================================================\n# 11. INCREMENTAL MODEL EFFECTS\n# ============================================================\n\neffects = pd.DataFrame(index=pivot.index)\n\neffects["EDRO_minus_BS"] = (\n    pivot["EDRO"] - pivot["BS"]\n)\n\neffects["EDRO_minus_GBU"] = (\n    pivot["EDRO"] - pivot["GBU"]\n)\n\neffects["EDRO_minus_GBU_LINKED"] = (\n    pivot["EDRO"] - pivot["GBU_LINKED"]\n)\n\neffects["GBU_LINKED_minus_GBU"] = (\n    pivot["GBU_LINKED"] - pivot["GBU"]\n)\n\neffects["EDRO_minus_ablation"] = (\n    pivot["EDRO"] - pivot["EDRO_ABLATION"]\n)\n\neffects["BS_minus_GBU"] = (\n    pivot["BS"] - pivot["GBU"]\n)\n\nprint("\\nINCREMENTAL EFFICIENCY EFFECTS")\nprint(effects.round(6))\n\nprint("\\nMEAN ABSOLUTE SCORE DIFFERENCES")\nprint(effects.abs().mean().round(6))\n\n# ============================================================\n# 12. OUT-OF-SAMPLE COVERAGE\n# ============================================================\n\nedro_test_scores = np.maximum(\n    Utest.sum(axis=1) / GE0,\n    Stest.sum(axis=(1, 2)) / GD0\n)\n\nedro_test_coverage = np.mean(\n    edro_test_scores <= Q_EDRO\n)\n\nablation_test_coverage = np.mean(\n    Stest.sum(axis=(1, 2)) <= GD_ABLATION\n)\n\nbs_test_scores = (\n    Utest.sum(axis=1)\n    + (Stest / V[None, :, :]).sum(axis=(1, 2))\n)\n\nbs_test_coverage = np.mean(\n    bs_test_scores <= GAMMA_BS\n)\n\ngbu_test_scores = gbu_scenario_score(Utest, Stest)\n\ngbu_test_coverage = np.mean(\n    gbu_test_scores <= GAMMA_GBU\n)\n\n# The generated calibration/test scenarios satisfy s_kj <= v_kj*u_k\n# by construction. Therefore linked GBU has exactly the same\n# scenario score, calibration budget, and scenario coverage as GBU.\nlinked_gbu_test_coverage = np.mean(\n    gbu_test_scores <= GAMMA_GBU\n)\n\ncoverage = pd.DataFrame({\n    "model": [\n        "EDRO",\n        "BS",\n        "GBU",\n        "GBU_LINKED",\n        "EDRO_ABLATION"\n    ],\n    "calibration_target": [\n        TARGET_COVERAGE\n    ] * 5,\n    "test_coverage": [\n        edro_test_coverage,\n        bs_test_coverage,\n        gbu_test_coverage,\n        linked_gbu_test_coverage,\n        ablation_test_coverage\n    ]\n})\n\nprint("\\nOUT-OF-SAMPLE COVERAGE")\nprint(coverage)\n\n# ============================================================\n# 13. CALIBRATION COVERAGE\n# ============================================================\n\ncalibration_coverage = pd.DataFrame({\n    "model": [\n        "EDRO", "BS", "GBU", "GBU_LINKED", "EDRO_ABLATION"\n    ],\n    "calibration_coverage": [\n        np.mean(edro_cal_scores <= Q_EDRO),\n        np.mean(bs_cal_scores <= GAMMA_BS),\n        np.mean(gbu_cal_scores <= GAMMA_GBU),\n        np.mean(gbu_cal_scores <= GAMMA_GBU),\n        np.mean(ablation_scores <= GD_ABLATION)\n    ]\n})\n\nprint("\\nIN-SAMPLE CALIBRATION COVERAGE")\nprint(calibration_coverage)\n\n# ============================================================\n# 14. RANKING CHANGES\n# ============================================================\n\nrank_pivot = results.pivot(\n    index="DMU",\n    columns="model",\n    values="rank"\n)\n\nprint("\\nRANKINGS")\nprint(rank_pivot)\n\nfor model in ["BS", "GBU", "GBU_LINKED", "EDRO", "EDRO_ABLATION"]:\n\n    changes = np.sum(\n        rank_pivot["NOMINAL"] != rank_pivot[model]\n    )\n\n    print(\n        f"Ranking changes: NOMINAL vs {model}: {changes}/{N}"\n    )\n\n# ============================================================\n# 15. EXPORT ALL RESULTS\n# ============================================================\n\nresults.to_csv(\n    "EDRO_BCC_all_results.csv",\n    index=False\n)\n\npivot.to_csv(\n    "EDRO_BCC_efficiency_matrix.csv"\n)\n\nsummary.to_csv(\n    "EDRO_BCC_model_summary.csv"\n)\n\neffects.to_csv(\n    "EDRO_BCC_incremental_effects.csv"\n)\n\ncoverage.to_csv(\n    "EDRO_BCC_test_coverage.csv",\n    index=False\n)\n\ncalibration_coverage.to_csv(\n    "EDRO_BCC_calibration_coverage.csv",\n    index=False\n)\n\nrank_pivot.to_csv(\n    "EDRO_BCC_rankings.csv"\n)\n\nprint("\\nEXPORTED FILES")\nprint("--------------")\nprint("EDRO_BCC_all_results.csv")\nprint("EDRO_BCC_efficiency_matrix.csv")\nprint("EDRO_BCC_model_summary.csv")\nprint("EDRO_BCC_incremental_effects.csv")\nprint("EDRO_BCC_test_coverage.csv")\nprint("EDRO_BCC_calibration_coverage.csv")\nprint("EDRO_BCC_rankings.csv")\n\nprint("\\nNOTE ON LINKED GBU CALIBRATION")\nprint("-------------------------------")\nprint("Calibration and test scenarios are generated with")\nprint("SEV = V * event * q, so all scenarios satisfy the")\nprint("linking constraints. Thus GBU and GBU_LINKED have")\nprint("the same calibrated Gamma_GBU and the same scenario coverage.")\nprint("Their robust DEA scores can nevertheless differ because")\nprint("their uncertainty sets differ outside the sampled scenarios.")\n\nprint("\\nEXPERIMENT COMPLETED.")\n'


def bootstrap_mean_ci(values, rng, reps=BOOTSTRAP_REPS):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (np.nan, np.nan, np.nan)
    draws = rng.choice(values, size=(reps, len(values)), replace=True).mean(axis=1)
    return (float(values.mean()), float(np.quantile(draws, 0.025)),
            float(np.quantile(draws, 0.975)))


def main():
    seed_records = []
    dmu_records = []
    start_all = time.time()

    for rep in range(N_SEEDS):
        seed = FIRST_SEED + rep
        source = BASE_SCRIPT.replace('SEED = 20261118', f'SEED = {seed}', 1)
        if 'SEED = 20261118' not in BASE_SCRIPT:
            raise RuntimeError('Seed declaration not found in embedded experiment.')

        # The support primal-dual check has already been verified separately;
        # omit its two extra LPs in every replication to reduce runtime.
        check_start = source.index('# 7. SUPPORT FUNCTION PRIMAL-DUAL VERIFICATION')
        check_end = source.index('# 8. EXACT ROBUST BCC LP')
        source = source[:check_start] + '# Repeated-run support check omitted.\n\n' + source[check_end:]

        # Avoid rewriting the single-run CSV files on every replication.
        export_start = source.index('# 15. EXPORT ALL RESULTS')
        source = source[:export_start]

        namespace = {'__name__': f'edro_replication_{rep}'}
        captured = io.StringIO()
        with contextlib.redirect_stdout(captured):
            exec(compile(source, f'<EDRO seed {seed}>', 'exec'), namespace)

        results = namespace['results']
        pivot = namespace['pivot']
        coverage = namespace['coverage'].set_index('model')['test_coverage']
        rank_pivot = namespace['rank_pivot']
        effects = namespace['effects']

        if not (results['status'] == 'optimal').all():
            raise RuntimeError(f'Non-optimal solve at seed {seed}.')

        row = {
            'seed': seed,
            'EDRO_mean_efficiency': float(pivot['EDRO'].mean()),
            'GBU_LINKED_mean_efficiency': float(pivot['GBU_LINKED'].mean()),
            'GBU_mean_efficiency': float(pivot['GBU'].mean()),
            'BS_mean_efficiency': float(pivot['BS'].mean()),
            'EDRO_ABLATION_mean_efficiency': float(pivot['EDRO_ABLATION'].mean()),
            'EDRO_test_coverage': float(coverage['EDRO']),
            'GBU_LINKED_test_coverage': float(coverage['GBU_LINKED']),
            'GBU_test_coverage': float(coverage['GBU']),
            'BS_test_coverage': float(coverage['BS']),
            'EDRO_ABLATION_test_coverage': float(coverage['EDRO_ABLATION']),
            'EDRO_minus_linked_GBU_mean_eff': float((pivot['EDRO'] - pivot['GBU_LINKED']).mean()),
            'EDRO_minus_linked_GBU_MAE': float((pivot['EDRO'] - pivot['GBU_LINKED']).abs().mean()),
            'EDRO_minus_ablation_MAE': float((pivot['EDRO'] - pivot['EDRO_ABLATION']).abs().mean()),
            'EDRO_minus_BS_MAE': float((pivot['EDRO'] - pivot['BS']).abs().mean()),
            'EDRO_minus_linked_GBU_coverage': float(coverage['EDRO'] - coverage['GBU_LINKED']),
            'EDRO_minus_ablation_coverage': float(coverage['EDRO'] - coverage['EDRO_ABLATION']),
            'EDRO_rank_changes_vs_nominal': int((rank_pivot['EDRO'] != rank_pivot['NOMINAL']).sum()),
            'linked_GBU_rank_changes_vs_nominal': int((rank_pivot['GBU_LINKED'] != rank_pivot['NOMINAL']).sum()),
            'EDRO_minus_ablation_rank_change': int((rank_pivot['EDRO'] != rank_pivot['NOMINAL']).sum()) - int((rank_pivot['EDRO_ABLATION'] != rank_pivot['NOMINAL']).sum()),
            'event_budget_active': bool(namespace['GE'] < namespace['K'] - namespace['TOL']),
            'Gamma_E': float(namespace['GE']),
            'Gamma_D': float(namespace['GD']),
            'Gamma_GBU': float(namespace['GAMMA_GBU']),
            'successful_solves': int((results['status'] == 'optimal').sum()),
            'total_solves': int(len(results)),
        }
        seed_records.append(row)

        for dmu in pivot.index:
            dmu_records.append({
                'seed': seed,
                'DMU': int(dmu),
                'EDRO': float(pivot.loc[dmu, 'EDRO']),
                'GBU_LINKED': float(pivot.loc[dmu, 'GBU_LINKED']),
                'EDRO_ABLATION': float(pivot.loc[dmu, 'EDRO_ABLATION']),
                'BS': float(pivot.loc[dmu, 'BS']),
                'EDRO_minus_linked_GBU': float(pivot.loc[dmu, 'EDRO'] - pivot.loc[dmu, 'GBU_LINKED']),
                'EDRO_minus_ablation': float(pivot.loc[dmu, 'EDRO'] - pivot.loc[dmu, 'EDRO_ABLATION']),
            })

        if (rep + 1) % 10 == 0 or rep == 0:
            print(f'Completed {rep + 1}/{N_SEEDS} replications (seed {seed}).')
            # Checkpoint raw results so a long run still leaves usable output.
            pd.DataFrame(seed_records).to_csv('EDRO_100seed_per_seed_checkpoint.csv', index=False)
            pd.DataFrame(dmu_records).to_csv('EDRO_100seed_per_DMU_checkpoint.csv', index=False)

    per_seed = pd.DataFrame(seed_records)
    per_dmu = pd.DataFrame(dmu_records)

    # Paired differences are computed within each seed before averaging.
    metrics = [
        'EDRO_minus_linked_GBU_mean_eff',
        'EDRO_minus_linked_GBU_MAE',
        'EDRO_minus_ablation_MAE',
        'EDRO_minus_BS_MAE',
        'EDRO_minus_linked_GBU_coverage',
        'EDRO_minus_ablation_coverage',
        'EDRO_mean_efficiency',
        'GBU_LINKED_mean_efficiency',
        'EDRO_test_coverage',
        'GBU_LINKED_test_coverage',
        'EDRO_rank_changes_vs_nominal',
        'linked_GBU_rank_changes_vs_nominal',
        'Gamma_E', 'Gamma_D', 'Gamma_GBU'
    ]

    ci_rng = np.random.default_rng(FIRST_SEED + 987654)
    summary_rows = []
    for metric in metrics:
        mean, lo, hi = bootstrap_mean_ci(per_seed[metric].to_numpy(), ci_rng)
        summary_rows.append({
            'metric': metric,
            'mean_across_seeds': mean,
            'bootstrap_95CI_lower': lo,
            'bootstrap_95CI_upper': hi,
            'seed_sd': float(per_seed[metric].std(ddof=1)),
            'n_seeds': int(per_seed[metric].notna().sum())
        })
    summary = pd.DataFrame(summary_rows)

    # DMU-level paired averages across seeds, useful for detecting whether
    # the effect is concentrated in a small number of DMUs.
    dmu_summary = per_dmu.groupby('DMU').agg(
        EDRO_mean=('EDRO', 'mean'),
        linked_GBU_mean=('GBU_LINKED', 'mean'),
        EDRO_ablation_mean=('EDRO_ABLATION', 'mean'),
        EDRO_minus_linked_GBU_mean=('EDRO_minus_linked_GBU', 'mean'),
        EDRO_minus_linked_GBU_MAE=('EDRO_minus_linked_GBU', lambda x: x.abs().mean()),
        EDRO_minus_ablation_mean=('EDRO_minus_ablation', 'mean'),
        EDRO_minus_ablation_MAE=('EDRO_minus_ablation', lambda x: x.abs().mean()),
    ).reset_index()

    per_seed.to_csv('EDRO_100seed_per_seed.csv', index=False)
    summary.to_csv('EDRO_100seed_summary_CI.csv', index=False)
    per_dmu.to_csv('EDRO_100seed_per_DMU.csv', index=False)
    dmu_summary.to_csv('EDRO_100seed_DMU_summary.csv', index=False)

    print('\nREPEATED-SEED SUMMARY (means and paired bootstrap 95% CIs)')
    print(summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print('\nDMU-LEVEL SUMMARY')
    print(dmu_summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print('\nDIAGNOSTICS')
    print('Replications:', len(per_seed))
    print('All 48 solves optimal in every seed:', bool((per_seed['successful_solves'] == per_seed['total_solves']).all()))
    print('Event budget active in all seeds:', bool(per_seed['event_budget_active'].all()))
    print('Total elapsed seconds:', round(time.time() - start_all, 2))
    print('\nEXPORTED FILES')
    print('EDRO_100seed_per_seed.csv')
    print('EDRO_100seed_summary_CI.csv')
    print('EDRO_100seed_per_DMU.csv')
    print('EDRO_100seed_DMU_summary.csv')


if __name__ == '__main__':
    main()


In [ ]:

# ============================================================
# EVENT-DRIVEN ROBUST DEA (EDRO)
# Complete reproducible experimental implementation
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.sparse import lil_matrix, csr_matrix
from pathlib import Path

# ------------------------------------------------------------
# 1. EXPERIMENT SETTINGS
# ------------------------------------------------------------

SEED = 20261118
REPS = 10                 # Increase to 100 after checking runtime
N = 40                    # Number of DMUs
T = 120                   # Number of periods
K = 4                     # Number of event groups

N_EVAL_TIMES = 12
N_EVAL_DMUS = 20

GAMMA_BS = 2.0
GAMMA_GBU = 1.5
GAMMA_EDRO = 1.5

NOISE = 0.12

OUTPUT_DIR = Path("EDRO_results")
OUTPUT_DIR.mkdir(exist_ok=True)

METHODS = ["Nominal", "BS", "GBU", "EDRO"]


# ------------------------------------------------------------
# 2. DATA GENERATION
# ------------------------------------------------------------

def generate_data(seed):

    rng = np.random.default_rng(seed)

    events = rng.normal(size=(T, K))

    base_x = rng.uniform(3.0, 6.0, N)
    base_y = rng.uniform(3.0, 7.0, N)

    ax = rng.uniform(0.05, 0.22, size=(K, N))
    ay = rng.uniform(0.05, 0.22, size=(K, N))

    X = np.zeros((T, N))
    Y = np.zeros((T, N))

    for t in range(T):

        X[t] = (
            base_x
            + events[t] @ ax
            + rng.normal(0, NOISE, N)
        )

        Y[t] = (
            base_y
            + events[t] @ ay
            + rng.normal(0, NOISE, N)
        )

    # Ensure strictly positive data
    X = np.maximum(X, 0.1)
    Y = np.maximum(Y, 0.1)

    return X, Y, events, ax, ay


# ------------------------------------------------------------
# 3. ESTIMATE EVENT EXPOSURES
# ------------------------------------------------------------

def estimate_exposures(X, events, train_end):

    E = np.column_stack([
        np.ones(train_end),
        events[:train_end]
    ])

    coefficients = np.linalg.lstsq(
        E, X[:train_end], rcond=None
    )[0]

    # Absolute event sensitivities
    D = np.abs(coefficients[1:, :])

    return D


# ------------------------------------------------------------
# 4. CALIBRATE EVENT INTENSITIES
# ------------------------------------------------------------

def calibrate_events(events, calibration_start, calibration_end):

    calibration = events[
        calibration_start:calibration_end
    ]

    intensity = np.mean(np.abs(calibration), axis=0)

    intensity = intensity / (
        np.mean(intensity) + 1e-10
    )

    return np.clip(intensity, 0.5, 2.0)


def event_severity(event, calibration_scale):

    severity = (
        0.5
        + np.abs(event) /
        (2.0 * calibration_scale + 1e-10)
    )

    return np.clip(severity, 0.5, 2.0)


# ------------------------------------------------------------
# 5. CONSTRUCT UNCERTAINTY GROUPS
# ------------------------------------------------------------

def construct_uncertainty(method, D, event, calibration_scale):

    """
    Returns:
        groups: list of (deviation_vector, budget)

    Each deviation vector has one coefficient per reference DMU.
    """

    if method == "Nominal":
        return []

    if method == "BS":

        # One pooled budget across all event exposures
        pooled = np.sum(D, axis=0) / np.sqrt(D.shape[0])

        return [(pooled, GAMMA_BS)]

    if method == "GBU":

        # Separate local budgets for each event group
        return [
            (D[k].copy(), GAMMA_GBU)
            for k in range(K)
        ]

    if method == "EDRO":

        severity = event_severity(
            event, calibration_scale
        )

        groups = []

        for k in range(K):

            # Event-dependent uncertainty magnitude
            deviation = severity[k] * D[k]

            # Event-dependent uncertainty budget
            budget = min(
                N,
                GAMMA_EDRO * severity[k]
            )

            groups.append((deviation, budget))

        return groups

    raise ValueError("Unknown method: " + method)


# ------------------------------------------------------------
# 6. ROBUST INPUT-ORIENTED BCC DEA
# ------------------------------------------------------------

def solve_robust_bcc(x0, y0, Xref, Yref, groups):

    """
    Exact LP dualization of independent budgeted
    input-coefficient uncertainty groups.

    Minimize theta

    Subject to:

        sum_j Xref[j]*lambda[j]
        + sum_h [Gamma_h*pi_h + sum_j w[h,j]]
        <= theta*x0

        sum_j Yref[j]*lambda[j] >= y0

        sum_j lambda[j] = 1

        pi[h] + w[h,j] >= D[h,j]*lambda[j]

        lambda, pi, w, theta >= 0
    """

    n = len(Xref)
    H = len(groups)

    # Variable positions:
    # theta: 0
    # lambda: 1,...,n
    # pi_h and w_hj follow

    theta_idx = 0
    lambda_start = 1

    next_idx = 1 + n

    pi_idx = []
    w_idx = []

    for h in range(H):

        pi_idx.append(next_idx)
        next_idx += 1

        w_idx.append(np.arange(next_idx, next_idx + n))
        next_idx += n

    nv = next_idx

    c = np.zeros(nv)
    c[theta_idx] = 1.0

    # Inequality constraints:
    # 1 output constraint
    # 1 robust input constraint
    # H*n dual constraints

    n_ub = 2 + H * n

    A_ub = lil_matrix((n_ub, nv))
    b_ub = np.zeros(n_ub)

    # Output constraint:
    # -sum(lambda_j*Yref_j) <= -y0

    A_ub[0, lambda_start:lambda_start+n] = -Yref
    b_ub[0] = -y0

    # Robust input constraint

    A_ub[1, theta_idx] = -x0

    A_ub[
        1, lambda_start:lambda_start+n
    ] = Xref

    for h, (deviation, budget) in enumerate(groups):

        A_ub[1, pi_idx[h]] = budget

        A_ub[
            1, w_idx[h]
        ] = np.ones(n)

    # Dual constraints:
    # deviation_j * lambda_j - pi_h - w_hj <= 0

    row = 2

    for h, (deviation, budget) in enumerate(groups):

        for j in range(n):

            A_ub[row, lambda_start+j] = deviation[j]
            A_ub[row, pi_idx[h]] = -1.0
            A_ub[row, w_idx[h][j]] = -1.0

            row += 1

    # Convexity constraint:
    # sum(lambda_j) = 1

    A_eq = lil_matrix((1, nv))
    A_eq[0, lambda_start:lambda_start+n] = 1.0

    b_eq = np.array([1.0])

    bounds = [(0, None)] * nv

    result = linprog(
        c,
        A_ub=csr_matrix(A_ub),
        b_ub=b_ub,
        A_eq=csr_matrix(A_eq),
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:

        return {
            "score": np.nan,
            "success": False,
            "status": result.message
        }

    return {
        "score": float(result.x[theta_idx]),
        "success": True,
        "status": "optimal"
    }


# ------------------------------------------------------------
# 7. ONE REPLICATION
# ------------------------------------------------------------

def run_replication(seed):

    X, Y, events, true_ax, true_ay = generate_data(seed)

    train_end = 72
    calibration_end = 96

    # Chronological separation
    Xtrain = X[:train_end]
    Ytrain = Y[:train_end]

    # Reference technology: training-period average
    Xref = np.mean(Xtrain, axis=0)
    Yref = np.mean(Ytrain, axis=0)

    # Estimate uncertainty using training data only
    D = estimate_exposures(
        X, events, train_end
    )

    # Calibration data
    calibration_scale = np.quantile(
        np.abs(events[train_end:calibration_end]),
        0.90,
        axis=0
    )

    calibration_scale = np.maximum(
        calibration_scale, 0.1
    )

    # Test period
    test_indices = np.linspace(
        calibration_end,
        T-1,
        N_EVAL_TIMES,
        dtype=int
    )

    # Fixed sample of DMUs
    rng = np.random.default_rng(seed + 1000)

    eval_dmus = rng.choice(
        N,
        size=min(N_EVAL_DMUS, N),
        replace=False
    )

    records = []

    failures = []

    for tt in test_indices:

        for j in eval_dmus:

            x0 = X[tt, j]
            y0 = Y[tt, j]

            for method in METHODS:

                groups = construct_uncertainty(
                    method,
                    D,
                    events[tt],
                    calibration_scale
                )

                result = solve_robust_bcc(
                    x0,
                    y0,
                    Xref,
                    Yref,
                    groups
                )

                if result["success"]:

                    records.append({
                        "seed": seed,
                        "time": tt,
                        "DMU": j,
                        "method": method,
                        "score": result["score"],
                        "success": 1
                    })

                else:

                    failures.append({
                        "seed": seed,
                        "time": tt,
                        "DMU": j,
                        "method": method,
                        "status": result["status"]
                    })

                    records.append({
                        "seed": seed,
                        "time": tt,
                        "DMU": j,
                        "method": method,
                        "score": np.nan,
                        "success": 0
                    })

    return pd.DataFrame(records), pd.DataFrame(failures)


# ------------------------------------------------------------
# 8. RUN MONTE CARLO EXPERIMENT
# ------------------------------------------------------------

def run_monte_carlo():

    all_results = []
    all_failures = []

    for r in range(REPS):

        seed = SEED + r

        result, failures = run_replication(seed)

        all_results.append(result)
        all_failures.append(failures)

        if (r + 1) % 1 == 0:

            valid = result["score"].notna().mean()

            print(
                f"Completed {r+1}/{REPS} replications "
                f"(seed {seed}), "
                f"valid LPs={valid:.3f}"
            )

    results = pd.concat(
        all_results,
        ignore_index=True
    )

    failures = pd.concat(
        all_failures,
        ignore_index=True
    )

    return results, failures


# ------------------------------------------------------------
# 9. PERFORMANCE SUMMARY
# ------------------------------------------------------------

def summarize(results):

    summary = (
        results.groupby("method")
        .agg(
            mean_score=("score", "mean"),
            std_score=("score", "std"),
            median_score=("score", "median"),
            min_score=("score", "min"),
            max_score=("score", "max"),
            valid_rate=("success", "mean"),
            observations=("score", "count")
        )
        .reset_index()
    )

    return summary


# ------------------------------------------------------------
# 10. PAIRED METHOD COMPARISONS
# ------------------------------------------------------------

def paired_comparisons(results):

    pivot = results.pivot_table(
        index=["seed", "time", "DMU"],
        columns="method",
        values="score"
    )

    comparisons = []

    for method in ["BS", "GBU", "EDRO"]:

        paired = pivot[["Nominal", method]].dropna()

        difference = (
            paired[method] - paired["Nominal"]
        )

        comparisons.append({
            "comparison": method + " - Nominal",
            "n": len(difference),
            "mean_difference": difference.mean(),
            "std_difference": difference.std(),
            "median_difference": difference.median(),
            "positive_fraction": (
                difference > 0
            ).mean()
        })

    return pd.DataFrame(comparisons)


# ------------------------------------------------------------
# 11. MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    print("=" * 65)
    print("EVENT-DRIVEN ROBUST DEA EXPERIMENT")
    print("=" * 65)

    results, failures = run_monte_carlo()

    summary = summarize(results)

    comparisons = paired_comparisons(results)

    print("\n1. AGGREGATE RESULTS")
    print(summary.to_string(index=False))

    print("\n2. PAIRED COMPARISONS")
    print(comparisons.to_string(index=False))

    print("\n3. SOLVER FAILURES")

    if len(failures):

        print(
            failures.groupby("method")
            .size()
            .to_string()
        )

    else:

        print("No solver failures.")

    # Save outputs
    results.to_csv(
        OUTPUT_DIR / "individual_results.csv",
        index=False
    )

    summary.to_csv(
        OUTPUT_DIR / "aggregate_results.csv",
        index=False
    )

    comparisons.to_csv(
        OUTPUT_DIR / "paired_comparisons.csv",
        index=False
    )

    failures.to_csv(
        OUTPUT_DIR / "solver_failures.csv",
        index=False
    )

    print("\nFiles saved in:", OUTPUT_DIR.resolve())

    print("\nEXPERIMENT COMPLETED")


In [ ]:

# ============================================================
# EDRO vs BS vs GBU
# FAIR-BUDGET ROBUST DEA EXPERIMENT
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import linprog
from scipy.sparse import lil_matrix, csr_matrix

from pathlib import Path

# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 20261118
REPS = 10

N = 40
T = 120
K = 4

TRAIN_END = 72
CAL_END = 96

N_EVAL_TIMES = 12
N_EVAL_DMUS = 20

TOTAL_GAMMA = 2.0

NOISE = 0.12
EPS = 1e-9

METHODS = ["Nominal", "BS", "GBU", "EDRO"]

OUT = Path("EDRO_fair_results")
OUT.mkdir(exist_ok=True)

# ============================================================
# 2. DATA GENERATION
# ============================================================

def generate_data(seed):

    rng = np.random.default_rng(seed)

    events = rng.normal(size=(T, K))

    base_x = rng.uniform(3.0, 6.0, N)
    base_y = rng.uniform(3.0, 7.0, N)

    true_ax = rng.uniform(0.05, 0.22, (K, N))
    true_ay = rng.uniform(0.05, 0.22, (K, N))

    X = np.zeros((T, N))
    Y = np.zeros((T, N))

    for t in range(T):

        X[t] = (
            base_x
            + events[t] @ true_ax
            + rng.normal(0, NOISE, N)
        )

        Y[t] = (
            base_y
            + events[t] @ true_ay
            + rng.normal(0, NOISE, N)
        )

    X = np.maximum(X, 0.1)
    Y = np.maximum(Y, 0.1)

    return X, Y, events, true_ax, true_ay


# ============================================================
# 3. ESTIMATE EVENT EXPOSURES
# ============================================================

def estimate_exposures(X, events):

    E = np.column_stack([
        np.ones(TRAIN_END),
        events[:TRAIN_END]
    ])

    coefficients = np.linalg.lstsq(
        E,
        X[:TRAIN_END],
        rcond=None
    )[0]

    # Estimated absolute event sensitivities
    D = np.abs(coefficients[1:, :])

    return D


# ============================================================
# 4. EVENT CALIBRATION
# ============================================================

def calibrate_event_intensity(events):

    calibration = events[TRAIN_END:CAL_END]

    scale = np.quantile(
        np.abs(calibration),
        0.90,
        axis=0
    )

    return np.maximum(scale, 0.1)


def get_event_severity(event, scale):

    severity = np.abs(event) / scale

    severity = np.maximum(severity, 0.05)

    return severity


# ============================================================
# 5. EQUAL-BUDGET UNCERTAINTY SETS
# ============================================================

def construct_blocks(method, D, event, scale):

    """
    Each block is a matrix A of uncertain coefficients.

    For a block A and budget Gamma:

       max sum_{r,j} A[r,j]*lambda[j]*q[r,j]

       subject to:
           0 <= q[r,j] <= 1
           sum(q[r,j]) <= Gamma

    Total uncertainty budget is equal across methods.
    """

    if method == "Nominal":
        return []

    if method == "BS":

        # One pooled budget across all event-DMU pairs
        return [(D.copy(), TOTAL_GAMMA)]

    if method == "GBU":

        # Equal local budget for each event group
        local_gamma = TOTAL_GAMMA / K

        return [
            (D[k:k+1, :].copy(), local_gamma)
            for k in range(K)
        ]

    if method == "EDRO":

        # Event-dependent budget allocation
        severity = get_event_severity(event, scale)

        gamma = (
            TOTAL_GAMMA
            * severity / np.sum(severity)
        )

        return [
            (D[k:k+1, :].copy(), gamma[k])
            for k in range(K)
        ]

    raise ValueError("Unknown method")


# ============================================================
# 6. EXACT ROBUST BCC LP
# ============================================================

def solve_robust_bcc(x0, y0, Xref, Yref, blocks):

    """
    Input-oriented BCC robust DEA.

    Minimize theta.

    Constraints:

       sum_j Xref[j]*lambda[j]
       + sum_b [Gamma_b*pi_b + sum_{r,j} w_b[r,j]]
       <= theta*x0

       sum_j Yref[j]*lambda[j] >= y0

       sum_j lambda[j] = 1

       pi_b + w_b[r,j] >= A_b[r,j]*lambda[j]

       theta, lambda, pi, w >= 0.

    Returns theta and reference intensities lambda.
    """

    n = len(Xref)

    # Variable indices
    theta_idx = 0
    lam_start = 1

    next_idx = 1 + n

    block_indices = []

    for A, gamma in blocks:

        m = A.shape[0]

        pi_idx = next_idx
        next_idx += 1

        w_idx = np.arange(
            next_idx,
            next_idx + m*n
        ).reshape(m, n)

        next_idx += m*n

        block_indices.append(
            (A, gamma, pi_idx, w_idx)
        )

    nv = next_idx

    # Objective
    c = np.zeros(nv)
    c[theta_idx] = 1.0

    n_dual = sum(
        A.shape[0] * n
        for A, gamma in blocks
    )

    n_ub = 2 + n_dual

    A_ub = lil_matrix((n_ub, nv))
    b_ub = np.zeros(n_ub)

    # Output constraint
    A_ub[0, lam_start:lam_start+n] = -Yref
    b_ub[0] = -y0

    # Robust input constraint
    A_ub[1, theta_idx] = -x0

    A_ub[
        1, lam_start:lam_start+n
    ] = Xref

    for A, gamma, pi_idx, w_idx in block_indices:

        A_ub[1, pi_idx] = gamma

        A_ub[
            1, w_idx.ravel()
        ] = np.ones(w_idx.size)

    # Uncertainty dual constraints
    row = 2

    for A, gamma, pi_idx, w_idx in block_indices:

        for r in range(A.shape[0]):

            for j in range(n):

                A_ub[row, lam_start+j] = A[r, j]

                A_ub[row, pi_idx] = -1.0

                A_ub[row, w_idx[r, j]] = -1.0

                row += 1

    # BCC convexity
    A_eq = lil_matrix((1, nv))

    A_eq[
        0, lam_start:lam_start+n
    ] = np.ones(n)

    b_eq = np.array([1.0])

    bounds = [(0, None)] * nv

    result = linprog(
        c,
        A_ub=csr_matrix(A_ub),
        b_ub=b_ub,
        A_eq=csr_matrix(A_eq),
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:

        return {
            "success": False,
            "score": np.nan,
            "lambda": None,
            "message": result.message
        }

    lam = result.x[
        lam_start:lam_start+n
    ]

    return {
        "success": True,
        "score": float(result.fun),
        "lambda": lam,
        "message": "optimal"
    }


# ============================================================
# 7. REALIZED EVENT SHOCK AND PROTECTION
# ============================================================

def calculate_protection(
    method,
    lam,
    D,
    event,
    scale,
    true_ax
):

    """
    Compare robust uncertainty allowance with
    the realized adverse event shock.

    Uses the same reference intensity lambda
    returned by the robust LP.
    """

    blocks = construct_blocks(
        method, D, event, scale
    )

    # Realized positive input shock
    realized_by_event = np.maximum(
        event[:, None] * true_ax,
        0.0
    )

    realized = np.sum(
        realized_by_event @ lam
    )

    # Robust support allowance
    allowance = 0.0

    for A, gamma in blocks:

        values = A * lam[None, :]

        flat = np.sort(
            values.ravel()
        )[::-1]

        budget = min(float(gamma), len(flat))

        whole = int(np.floor(budget))
        fraction = budget - whole

        support = np.sum(flat[:whole])

        if whole < len(flat):
            support += fraction * flat[whole]

        allowance += support

    covered = (
        allowance + EPS >= realized
    )

    return {
        "realized_shock": realized,
        "allowance": allowance,
        "covered": int(covered),
        "protection_ratio": (
            allowance / max(realized, EPS)
        )
    }


# ============================================================
# 8. ONE MONTE CARLO REPLICATION
# ============================================================

def run_replication(seed):

    X, Y, events, true_ax, true_ay = generate_data(seed)

    D = estimate_exposures(X, events)

    scale = calibrate_event_intensity(events)

    # Evaluation dates
    test_times = np.linspace(
        CAL_END,
        T-1,
        N_EVAL_TIMES,
        dtype=int
    )

    rng = np.random.default_rng(seed + 10000)

    eval_dmus = rng.choice(
        N,
        size=min(N_EVAL_DMUS, N),
        replace=False
    )

    records = []
    failures = []

    for tt in test_times:

        # Contemporaneous reference technology.
        # Every evaluated DMU belongs to its reference set.
        Xref = X[tt].copy()
        Yref = Y[tt].copy()

        for j in eval_dmus:

            x0 = X[tt, j]
            y0 = Y[tt, j]

            for method in METHODS:

                blocks = construct_blocks(
                    method,
                    D,
                    events[tt],
                    scale
                )

                result = solve_robust_bcc(
                    x0,
                    y0,
                    Xref,
                    Yref,
                    blocks
                )

                if not result["success"]:

                    failures.append({
                        "seed": seed,
                        "time": tt,
                        "DMU": j,
                        "method": method,
                        "message": result["message"]
                    })

                    records.append({
                        "seed": seed,
                        "time": tt,
                        "DMU": j,
                        "method": method,
                        "score": np.nan,
                        "realized_shock": np.nan,
                        "allowance": np.nan,
                        "covered": np.nan,
                        "protection_ratio": np.nan
                    })

                    continue

                protection = calculate_protection(
                    method,
                    result["lambda"],
                    D,
                    events[tt],
                    scale,
                    true_ax
                )

                records.append({
                    "seed": seed,
                    "time": tt,
                    "DMU": j,
                    "method": method,
                    "score": result["score"],
                    **protection
                })

    return (
        pd.DataFrame(records),
        pd.DataFrame(failures)
    )


# ============================================================
# 9. MONTE CARLO
# ============================================================

def run_experiment():

    all_results = []
    all_failures = []

    for r in range(REPS):

        seed = SEED + r

        results, failures = run_replication(seed)

        all_results.append(results)
        all_failures.append(failures)

        valid = results["score"].notna().mean()

        print(
            f"Completed {r+1}/{REPS} "
            f"(seed {seed}), valid={valid:.3f}"
        )

    results = pd.concat(
        all_results,
        ignore_index=True
    )

    failures = pd.concat(
        all_failures,
        ignore_index=True
    )

    return results, failures


# ============================================================
# 10. SUMMARY
# ============================================================

def aggregate_results(results):

    summary = (
        results.groupby("method")
        .agg(
            mean_score=("score", "mean"),
            std_score=("score", "std"),
            median_score=("score", "median"),
            valid_rate=("score", lambda x: x.notna().mean()),
            mean_allowance=("allowance", "mean"),
            mean_realized_shock=("realized_shock", "mean"),
            coverage=("covered", "mean"),
            mean_protection_ratio=("protection_ratio", "mean"),
            observations=("score", "count")
        )
        .reset_index()
    )

    return summary


# ============================================================
# 11. PAIRED COMPARISONS
# ============================================================

def paired_comparisons(results):

    comparisons = []

    metrics = [
        "score",
        "allowance",
        "protection_ratio",
        "covered"
    ]

    for metric in metrics:

        pivot = results.pivot_table(
            index=["seed", "time", "DMU"],
            columns="method",
            values=metric
        )

        for method in ["BS", "GBU", "EDRO"]:

            pair = pivot[
                ["GBU", method]
            ].dropna()

            if len(pair) == 0:
                continue

            diff = pair[method] - pair["GBU"]

            comparisons.append({
                "metric": metric,
                "comparison": method + " minus GBU",
                "n": len(diff),
                "mean_difference": diff.mean(),
                "median_difference": diff.median(),
                "positive_fraction": (diff > 0).mean()
            })

    return pd.DataFrame(comparisons)


# ============================================================
# 12. EVENT-INTENSITY ANALYSIS
# ============================================================

def event_analysis(results, events):

    rows = []

    for seed in results["seed"].unique():

        X, Y, E, ax, ay = generate_data(seed)

        scale = calibrate_event_intensity(E)

        for tt in results.loc[
            results["seed"] == seed, "time"
        ].unique():

            severity = np.mean(
                get_event_severity(E[tt], scale)
            )

            rows.append({
                "seed": seed,
                "time": tt,
                "event_intensity": severity
            })

    intensity = pd.DataFrame(rows)

    merged = results.merge(
        intensity,
        on=["seed", "time"],
        how="left"
    )

    merged["intensity_group"] = pd.qcut(
        merged["event_intensity"],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop"
    )

    return (
        merged.groupby(
            ["method", "intensity_group"],
            observed=True
        )
        .agg(
            mean_score=("score", "mean"),
            mean_allowance=("allowance", "mean"),
            coverage=("covered", "mean"),
            mean_ratio=("protection_ratio", "mean")
        )
        .reset_index()
    )


# ============================================================
# 13. MAIN
# ============================================================

if __name__ == "__main__":

    print("=" * 70)
    print("FAIR-BUDGET EDRO / GBU / BS ROBUST DEA")
    print("=" * 70)

    results, failures = run_experiment()

    summary = aggregate_results(results)

    comparisons = paired_comparisons(results)

    intensity = event_analysis(results, None)

    print("\n1. AGGREGATE RESULTS")
    print(summary.to_string(index=False))

    print("\n2. PAIRED COMPARISONS")
    print(comparisons.to_string(index=False))

    print("\n3. EVENT INTENSITY ANALYSIS")
    print(intensity.to_string(index=False))

    print("\n4. SOLVER FAILURES")

    if len(failures):
        print(
            failures.groupby("method")
            .size()
            .to_string()
        )
    else:
        print("No solver failures.")

    results.to_csv(
        OUT / "individual_results.csv",
        index=False
    )

    summary.to_csv(
        OUT / "aggregate_results.csv",
        index=False
    )

    comparisons.to_csv(
        OUT / "paired_comparisons.csv",
        index=False
    )

    intensity.to_csv(
        OUT / "event_intensity.csv",
        index=False
    )

    failures.to_csv(
        OUT / "solver_failures.csv",
        index=False
    )

    print("\nSaved all files in:", OUT.resolve())
    print("\nEXPERIMENT FINISHED")


In [ ]:
# ============================================================
# FINAL EDRO vs GBU vs BS ROBUST DEA EXPERIMENT
# Common event-level uncertainty model; 100 Monte Carlo runs
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linprog
from scipy.sparse import lil_matrix, csr_matrix

# ---------------------------- SETTINGS ----------------------------

SEED = 20261118
REPS = 100                 # use 3-5 first only if you want a quick smoke test
N, T, K = 40, 120, 4
TRAIN_END = 72
CAL_END = 96
N_EVAL_TIMES = 6
N_EVAL_DMUS = 5

TOTAL_GAMMA = 2.0
NOISE = 0.12
LOOKBACK = 12
BOOTSTRAP_B = 5000
EPS = 1e-10

# Two fixed event groups, used identically by GBU and EDRO.
EVENT_GROUPS = [np.array([0, 1]), np.array([2, 3])]
METHODS = ["Nominal", "BS", "GBU", "EDRO"]

OUT = Path("EDRO_final_results")
OUT.mkdir(exist_ok=True)


# ------------------------- DATA GENERATION -------------------------

def generate_data(seed):
    rng = np.random.default_rng(seed)
    events = rng.normal(size=(T, K))

    base_x = rng.uniform(3.0, 6.0, N)
    base_y = rng.uniform(3.0, 7.0, N)

    # Positive input sensitivities: positive event realizations are adverse.
    true_ax = rng.uniform(0.05, 0.22, size=(K, N))
    true_ay = rng.uniform(0.02, 0.10, size=(K, N))

    X = np.empty((T, N))
    Y = np.empty((T, N))
    for t in range(T):
        X[t] = base_x + events[t] @ true_ax + rng.normal(0, NOISE, N)
        Y[t] = base_y + events[t] @ true_ay + rng.normal(0, NOISE, N)

    return (np.maximum(X, 0.1), np.maximum(Y, 0.1),
            events, true_ax, true_ay)


# ---------------------- TRAIN / CALIBRATE --------------------------

def fit_event_models(X, Y, events):
    """Estimate event loadings on training data; remove fitted event effects."""
    E = np.column_stack([np.ones(TRAIN_END), events[:TRAIN_END]])

    bx = np.linalg.lstsq(E, X[:TRAIN_END], rcond=None)[0][1:, :]
    by = np.linalg.lstsq(E, Y[:TRAIN_END], rcond=None)[0][1:, :]

    # Event-adjusted reference technology; the test event is not left in
    # the baseline technology and then counted a second time as a shock.
    X_adj = X - events @ bx
    Y_adj = Y - events @ by

    # Calibration scale is estimated only on the calibration window.
    scale = np.quantile(
        np.abs(events[TRAIN_END:CAL_END]), 0.90, axis=0
    )
    scale = np.maximum(scale, 0.10)

    # C[k,j] is the estimated input exposure to a unit standardized event.
    C = np.abs(bx) * scale[:, None]
    return bx, by, X_adj, Y_adj, scale, C


# ---------------------- BUDGET ALLOCATION --------------------------

def edro_group_budgets(events, scale, t):
    """
    Allocate the fixed total budget using only lagged event information.
    No current or future test event is used to choose EDRO's budgets.
    """
    start = max(TRAIN_END, t - LOOKBACK)
    history = events[start:t]

    if len(history) == 0:
        event_severity = np.ones(K)
    else:
        event_severity = np.mean(np.abs(history) / scale[None, :], axis=0)

    group_severity = np.array([
        max(float(np.mean(event_severity[g])), 0.05)
        for g in EVENT_GROUPS
    ])
    gamma = TOTAL_GAMMA * group_severity / group_severity.sum()

    # Each group has two event coordinates, so its budget cannot exceed 2.
    # With strictly positive severities and total budget 2, this is satisfied.
    return gamma


def make_blocks(method, C, events, scale, t):
    """
    A block is (event_indices, Gamma_block).
    The uncertainty variables are event-level, common across all DMUs:
        0 <= q_k <= 1, sum_{k in block} q_k <= Gamma_block.
    """
    if method == "Nominal":
        return []

    if method == "BS":
        return [(np.arange(K), TOTAL_GAMMA)]

    if method == "GBU":
        local_gamma = TOTAL_GAMMA / len(EVENT_GROUPS)
        return [(g.copy(), local_gamma) for g in EVENT_GROUPS]

    if method == "EDRO":
        gamma = edro_group_budgets(events, scale, t)
        return [(EVENT_GROUPS[h].copy(), float(gamma[h]))
                for h in range(len(EVENT_GROUPS))]

    raise ValueError(f"Unknown method: {method}")


# ----------------------- SUPPORT FUNCTION --------------------------

def budget_support(event_coefficients, gamma):
    """Exact BS support for nonnegative event coefficients."""
    c = np.maximum(np.asarray(event_coefficients, dtype=float), 0.0)
    if len(c) == 0 or gamma <= 0:
        return 0.0

    gamma = min(float(gamma), float(len(c)))
    ordered = np.sort(c)[::-1]
    whole = int(np.floor(gamma))
    frac = gamma - whole

    value = float(ordered[:whole].sum())
    if whole < len(ordered):
        value += frac * float(ordered[whole])
    return value


def allowance_for_lambda(C, lam, blocks):
    """Evaluate exactly the same support function used in the robust LP."""
    event_coeff = C @ lam
    return sum(
        budget_support(event_coeff[idx], gamma)
        for idx, gamma in blocks
    )


# ----------------------- ROBUST BCC LP -----------------------------

def solve_robust_bcc(x0, y0, Xref, Yref, C, blocks):
    """
    Input-oriented BCC DEA:
        min theta
        Yref' lambda >= y0
        Xref' lambda + support_U(C lambda) <= theta*x0
        1'lambda = 1; theta,lambda >= 0.

    For each event block, the support-function LP dual has one pi and
    one w_k per event coordinate:
        pi + w_k >= sum_j C[k,j] lambda_j.
    """
    n = len(Xref)

    theta_idx = 0
    lam_start = 1
    next_idx = 1 + n
    block_vars = []

    for idx, gamma in blocks:
        pi_idx = next_idx
        next_idx += 1
        w_idx = np.arange(next_idx, next_idx + len(idx))
        next_idx += len(idx)
        block_vars.append((idx, gamma, pi_idx, w_idx))

    nvar = next_idx
    c_obj = np.zeros(nvar)
    c_obj[theta_idx] = 1.0

    n_dual_rows = sum(len(idx) for idx, _ in blocks)
    A_ub = lil_matrix((2 + n_dual_rows, nvar))
    b_ub = np.zeros(2 + n_dual_rows)

    # Output constraint.
    A_ub[0, lam_start:lam_start+n] = -Yref
    b_ub[0] = -y0

    # Robust input constraint.
    A_ub[1, theta_idx] = -x0
    A_ub[1, lam_start:lam_start+n] = Xref

    for idx, gamma, pi_idx, w_idx in block_vars:
        A_ub[1, pi_idx] = gamma
        A_ub[1, w_idx] = 1.0

    # Dual constraints for each event coordinate.
    row = 2
    for idx, gamma, pi_idx, w_idx in block_vars:
        for r, k in enumerate(idx):
            A_ub[row, lam_start:lam_start+n] = C[k, :]
            A_ub[row, pi_idx] = -1.0
            A_ub[row, w_idx[r]] = -1.0
            row += 1

    A_eq = lil_matrix((1, nvar))
    A_eq[0, lam_start:lam_start+n] = 1.0
    b_eq = np.array([1.0])

    result = linprog(
        c_obj,
        A_ub=csr_matrix(A_ub),
        b_ub=b_ub,
        A_eq=csr_matrix(A_eq),
        b_eq=b_eq,
        bounds=[(0, None)] * nvar,
        method="highs",
    )

    if not result.success:
        return {"success": False, "score": np.nan, "lambda": None,
                "message": result.message}

    lam = result.x[lam_start:lam_start+n]
    return {"success": True, "score": float(result.fun),
            "lambda": lam, "message": "optimal"}


# --------------------- HELD-OUT STRESS TEST ------------------------

def realized_adverse_shock(event, true_ax, lam):
    """
    Same event-level shock geometry as the uncertainty model.
    Negative event realizations do not increase input requirements.
    """
    adverse_event = np.maximum(event, 0.0)
    return float(np.sum(adverse_event * (true_ax @ lam)))


def run_replication(seed):
    X, Y, events, true_ax, true_ay = generate_data(seed)
    bx, by, X_adj, Y_adj, scale, C = fit_event_models(X, Y, events)

    test_times = np.linspace(
        CAL_END, T - 1, N_EVAL_TIMES, dtype=int
    )
    rng = np.random.default_rng(seed + 10000)
    eval_dmus = rng.choice(N, size=min(N_EVAL_DMUS, N), replace=False)

    records, failures = [], []

    for t in test_times:
        # Event-adjusted contemporaneous technology. The evaluated DMU
        # remains in the reference set, so the BCC LP is feasible.
        Xref = np.maximum(X_adj[t].copy(), 0.1)
        Yref = np.maximum(Y_adj[t].copy(), 0.1)

        # Current shock is used only for out-of-sample stress evaluation.
        current_intensity = float(np.mean(np.abs(events[t]) / scale))

        for j in eval_dmus:
            x0, y0 = Xref[j], Yref[j]

            for method in METHODS:
                blocks = make_blocks(method, C, events, scale, t)
                sol = solve_robust_bcc(
                    x0, y0, Xref, Yref, C, blocks
                )

                if not sol["success"]:
                    failures.append({
                        "seed": seed, "time": t, "DMU": int(j),
                        "method": method, "message": sol["message"]
                    })
                    records.append({
                        "seed": seed, "time": t, "DMU": int(j),
                        "method": method, "score": np.nan,
                        "allowance": np.nan, "realized_shock": np.nan,
                        "covered": np.nan, "event_intensity": current_intensity
                    })
                    continue

                lam = sol["lambda"]
                allowance = allowance_for_lambda(C, lam, blocks)
                realized = realized_adverse_shock(events[t], true_ax, lam)

                records.append({
                    "seed": seed, "time": t, "DMU": int(j),
                    "method": method,
                    "score": sol["score"],
                    "allowance": allowance,
                    "realized_shock": realized,
                    "covered": int(allowance + EPS >= realized),
                    "event_intensity": current_intensity,
                })

    return pd.DataFrame(records), pd.DataFrame(failures)


# -------------------------- EXPERIMENT -----------------------------

def run_experiment():
    result_frames, failure_frames = [], []

    for r in range(REPS):
        seed = SEED + r
        res, fail = run_replication(seed)
        result_frames.append(res)
        failure_frames.append(fail)

        print(
            f"Completed {r+1}/{REPS} (seed {seed}), "
            f"valid={res['score'].notna().mean():.3f}"
        )

    results = pd.concat(result_frames, ignore_index=True)
    failures = pd.concat(failure_frames, ignore_index=True)
    return results, failures


# --------------------------- SUMMARIES -----------------------------

def aggregate_results(results):
    return (
        results.groupby("method")
        .agg(
            mean_score=("score", "mean"),
            sd_score=("score", "std"),
            mean_allowance=("allowance", "mean"),
            median_allowance=("allowance", "median"),
            mean_realized_shock=("realized_shock", "mean"),
            coverage=("covered", "mean"),
            valid_rate=("score", lambda s: s.notna().mean()),
            observations=("score", "count"),
        )
        .reindex(METHODS)
        .reset_index()
    )


def paired_comparisons(results):
    """
    Cluster bootstrap by Monte Carlo replication (seed), not by individual
    DMU-time rows. This respects within-replication dependence.
    """
    metrics = ["score", "allowance", "covered"]
    rows = []
    rng = np.random.default_rng(SEED + 999)

    for metric in metrics:
        seed_means = (
            results.groupby(["seed", "method"])[metric]
            .mean().unstack("method")
        )

        for method in ["BS", "EDRO"]:
            pair = seed_means[[method, "GBU"]].dropna()
            diff = (pair[method] - pair["GBU"]).to_numpy()
            if len(diff) == 0:
                continue

            boot = np.empty(BOOTSTRAP_B)
            for b in range(BOOTSTRAP_B):
                sample = rng.choice(diff, size=len(diff), replace=True)
                boot[b] = np.mean(sample)

            rows.append({
                "metric": metric,
                "comparison": f"{method} minus GBU",
                "replications": len(diff),
                "mean_difference": float(np.mean(diff)),
                "median_difference": float(np.median(diff)),
                "ci95_low": float(np.quantile(boot, 0.025)),
                "ci95_high": float(np.quantile(boot, 0.975)),
                "positive_fraction": float(np.mean(diff > 0)),
            })

    return pd.DataFrame(rows)


def event_intensity_analysis(results):
    # Assign terciles at the seed-date level, then merge back to DMU rows.
    date_level = (
        results[["seed", "time", "event_intensity"]]
        .drop_duplicates(["seed", "time"])
        .copy()
    )

    ranks = date_level["event_intensity"].rank(method="first")
    date_level["intensity_group"] = pd.qcut(
        ranks, q=3, labels=["Low", "Medium", "High"]
    )

    merged = results.merge(
        date_level[["seed", "time", "intensity_group"]],
        on=["seed", "time"], how="left"
    )

    return (
        merged.groupby(["method", "intensity_group"], observed=True)
        .agg(
            mean_score=("score", "mean"),
            mean_allowance=("allowance", "mean"),
            coverage=("covered", "mean"),
            observations=("score", "count"),
        )
        .reset_index()
    )


# ------------------------------- MAIN ------------------------------

if __name__ == "__main__":
    print("=" * 72)
    print("FINAL EDRO / GBU / BS ROBUST DEA EXPERIMENT")
    print(f"Replications={REPS}; total budget={TOTAL_GAMMA}; "
          f"test dates={N_EVAL_TIMES}; evaluated DMUs={N_EVAL_DMUS}")
    print("=" * 72)

    results, failures = run_experiment()
    summary = aggregate_results(results)
    comparisons = paired_comparisons(results)
    intensity = event_intensity_analysis(results)

    print("\n1. AGGREGATE RESULTS")
    print(summary.to_string(index=False))

    print("\n2. PAIRED CLUSTER-BOOTSTRAP COMPARISONS")
    print(comparisons.to_string(index=False))

    print("\n3. EVENT-INTENSITY ANALYSIS")
    print(intensity.to_string(index=False))

    print("\n4. SOLVER FAILURES")
    if len(failures):
        print(failures.groupby(["method", "message"]).size().to_string())
    else:
        print("No solver failures.")

    results.to_csv(OUT / "individual_results.csv", index=False)
    summary.to_csv(OUT / "aggregate_results.csv", index=False)
    comparisons.to_csv(OUT / "paired_comparisons.csv", index=False)
    intensity.to_csv(OUT / "event_intensity.csv", index=False)
    failures.to_csv(OUT / "solver_failures.csv", index=False)

    print(f"\nSaved all outputs in: {OUT.resolve()}")
    print("EXPERIMENT FINISHED")


In [ ]:
# ================================================================
# REAL-DATA EDRO APPLICATION: U.S. ELECTRIC POWER PLANTS
# Official sources: EIA-923 (monthly operations) + EIA-860 (generators)
#
# Run in Deepnote/Jupyter:
#   %run download_and_build_eia_edro_data.py
#
# Outputs:
#   eia_edro_project/
#       raw/eia923/                 annual official ZIP archives
#       raw/eia860/
#       extracted/eia923/
#       extracted/eia860/
#       processed/eia923_monthly_plant.csv
#       processed/eia860_plant_capacity.csv
#       processed/plant_month_panel.csv
#       processed/data_quality_report.txt
#       processed/download_manifest.csv
#
# Main DEA variables:
#   Inputs:  fuel consumption (physical units, retained by fuel type),
#            installed nameplate capacity (MW)
#   Output:  net electricity generation (MWh)
#
# IMPORTANT:
# - This script does not fabricate missing values or convert missing data to 0.
# - EIA reporting formats vary across years. It catalogs workbook sheets and
#   uses column-name matching; review the quality report before estimating DEA.
# - Plant-level fuel consumption and generation may be fuel-specific. The
#   panel preserves fuel category to avoid mixing unlike technologies/fuels.
# ================================================================

from pathlib import Path
from zipfile import ZipFile, BadZipFile
from urllib.parse import urljoin
import csv
import re
import sys
import time
import subprocess
import warnings

# Install lightweight dependencies if missing (Deepnote usually has these).
def ensure_package(import_name, package_name=None):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            package_name or import_name, "-q"
        ])

ensure_package("requests")
ensure_package("bs4", "beautifulsoup4")
ensure_package("pandas")
ensure_package("openpyxl")
ensure_package("xlrd")

import requests
import pandas as pd
from bs4 import BeautifulSoup

YEARS = list(range(2020, 2026))
BASE = Path("eia_edro_project")
RAW = BASE / "raw"
EXTRACTED = BASE / "extracted"
PROCESSED = BASE / "processed"
for p in [RAW, EXTRACTED, PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

INDEX = {
    "eia923": "https://www.eia.gov/electricity/data/eia923/",
    "eia860": "https://www.eia.gov/electricity/data/eia860/",
}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; academic-research-data-download/1.0)"
}
TIMEOUT = 180
SESSION = requests.Session()
SESSION.headers.update(HEADERS)


def say(msg):
    print(msg, flush=True)


def get_year_from_text(text):
    """Find a four-digit annual vintage in link text or URL."""
    hits = re.findall(r"(?<!\d)(20(?:1\d|2[0-6]))(?!\d)", str(text))
    if not hits:
        return None
    # Prefer the latest year if a link label contains a range.
    return max(map(int, hits))


def discover_archives(kind, index_url):
    say(f"\n[{kind.upper()}] Reading official index: {index_url}")
    r = SESSION.get(index_url, timeout=TIMEOUT)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    found = {}

    # Find ZIP links and infer their year from URL + visible text.
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full = urljoin(index_url, href)
        if ".zip" not in full.lower():
            continue
        label = " ".join([
            a.get_text(" ", strip=True),
            href,
            a.parent.get_text(" ", strip=True) if a.parent else ""
        ])
        year = get_year_from_text(label)
        if year in YEARS:
            # Keep the most likely annual archive; reject ancillary files.
            low = full.lower()
            if kind == "eia923":
                likely = ("923" in low or "f923" in low or "eia923" in low)
            else:
                likely = ("860" in low or "eia860" in low)
            if likely:
                found[year] = full

    # Some EIA index pages link to a subpage containing annual ZIPs.
    if len(found) < 3:
        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            if not href.lower().endswith((".php", ".html", ".htm")):
                continue
            label = (a.get_text(" ", strip=True) + " " + href).lower()
            if any(str(y) in label for y in YEARS):
                sub_url = urljoin(index_url, href)
                try:
                    sr = SESSION.get(sub_url, timeout=TIMEOUT)
                    sr.raise_for_status()
                    sub = BeautifulSoup(sr.text, "html.parser")
                    for link in sub.find_all("a", href=True):
                        zurl = urljoin(sub_url, link["href"].strip())
                        if ".zip" not in zurl.lower():
                            continue
                        zlabel = link.get_text(" ", strip=True) + " " + zurl
                        yr = get_year_from_text(zlabel)
                        if yr in YEARS:
                            found[yr] = zurl
                except Exception as exc:
                    say(f"  Sub-index skipped ({sub_url}): {exc}")

    say(f"  Discovered annual archives: {len(found)}")
    for y in sorted(found):
        say(f"    {y}: {found[y]}")
    return found


def download_archive(url, dest):
    if dest.exists() and dest.stat().st_size > 100_000:
        # Validate existing archive before reusing it.
        try:
            with ZipFile(dest) as z:
                z.namelist()
            say(f"  Already downloaded: {dest.name}")
            return "existing", dest.stat().st_size
        except BadZipFile:
            dest.unlink()

    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, 4):
        try:
            say(f"  Downloading {dest.name}, attempt {attempt}")
            with SESSION.get(url, stream=True, timeout=TIMEOUT) as r:
                r.raise_for_status()
                total = 0
                with open(part, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            total += len(chunk)
            if total < 50_000:
                raise RuntimeError(
                    f"Download too small ({total:,} bytes); likely not a ZIP archive."
                )
            with ZipFile(part) as z:
                bad = z.testzip()
                if bad:
                    raise BadZipFile(f"Corrupt member: {bad}")
            part.replace(dest)
            return "downloaded", total
        except Exception:
            if part.exists():
                part.unlink()
            if attempt == 3:
                raise
            time.sleep(attempt * 3)


def extract_archive(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / "_EXTRACTED_OK.txt"
    if marker.exists():
        return "existing"
    with ZipFile(archive) as z:
        z.extractall(destination)
    marker.write_text("Extraction completed.\n", encoding="utf-8")
    return "extracted"


def download_all():
    manifest = []
    archive_map = {}
    for kind, index_url in INDEX.items():
        links = discover_archives(kind, index_url)
        archive_map[kind] = {}
        for year in YEARS:
            url = links.get(year)
            if not url:
                manifest.append([kind, year, "", "", "not_found_on_index"])
                continue
            dest = RAW / kind / f"{kind}_{year}.zip"
            try:
                status, size = download_archive(url, dest)
                extract_status = extract_archive(
                    dest, EXTRACTED / kind / str(year)
                )
                archive_map[kind][year] = EXTRACTED / kind / str(year)
                manifest.append([
                    kind, year, url, size, f"{status};{extract_status}"
                ])
            except Exception as exc:
                say(f"  ERROR {kind} {year}: {exc}")
                manifest.append([kind, year, url, "", f"error:{exc}"])

    pd.DataFrame(
        manifest,
        columns=["dataset", "year", "source_url", "bytes", "status"]
    ).to_csv(PROCESSED / "download_manifest.csv", index=False)
    return archive_map


def clean_col(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")


def _row_looks_like_header(values, kind):
    vals = [clean_col(v) for v in values if pd.notna(v)]
    joined = " ".join(vals)
    has_plant = any(x in joined for x in [
        "plant_id", "plant_code", "plant_number", "plant_num", "plant_identifier"
    ])
    if kind == "eia923":
        has_measure = any(x in joined for x in [
            "net_generation", "netgen", "elec_fuel_consumption",
            "fuel_consumption", "fuel_consumed"
        ])
        return has_plant and has_measure
    if kind == "eia860":
        has_measure = any(x in joined for x in [
            "nameplate_capacity", "summer_capacity",
            "winter_capacity", "capacity_mw"
        ])
        return has_plant and has_measure
    return False


def _read_excel_sheet_with_header(path, sheet, kind, book=None):
    raw = (book.parse(sheet_name=sheet, header=None, dtype=object)
           if book is not None else
           pd.read_excel(path, sheet_name=sheet, header=None, dtype=object))
    if raw.empty:
        return None
    header_idx = None
    for i in range(min(50, len(raw))):
        if _row_looks_like_header(raw.iloc[i].tolist(), kind):
            header_idx = i
            break
    if header_idx is None:
        return None
    headers = []
    for j, value in enumerate(raw.iloc[header_idx].tolist()):
        headers.append(
            str(value).strip()
            if pd.notna(value) and str(value).strip()
            else f"unnamed_{j}"
        )
    df = raw.iloc[header_idx + 1:].copy()
    df.columns = headers
    df = df.dropna(how="all")
    return df if not df.empty else None


def read_tabular_file(path, kind):
    """Read EIA workbooks/CSVs with automatic detection of embedded headers."""
    out = []
    suffix = path.suffix.lower()
    try:
        if suffix in [".xlsx", ".xlsm", ".xls"]:
            with pd.ExcelFile(path) as book:
                for sheet in book.sheet_names:
                    try:
                        df = _read_excel_sheet_with_header(path, sheet, kind, book=book)
                        if df is not None and not df.empty:
                            out.append((sheet, df))
                        del df
                    except Exception as exc:
                        say(f"    Could not read {path.name}/{sheet}: {exc}")
        elif suffix == ".csv":
            for enc in ["utf-8-sig", "cp1252", "latin1"]:
                try:
                    raw = pd.read_csv(
                        path, encoding=enc, header=None,
                        low_memory=False, dtype=object
                    )
                    header_idx = None
                    for i in range(min(50, len(raw))):
                        if _row_looks_like_header(raw.iloc[i].tolist(), kind):
                            header_idx = i
                            break
                    if header_idx is not None:
                        headers = [
                            str(v).strip()
                            if pd.notna(v) and str(v).strip()
                            else f"unnamed_{j}"
                            for j, v in enumerate(raw.iloc[header_idx].tolist())
                        ]
                        df = raw.iloc[header_idx + 1:].copy()
                        df.columns = headers
                        df = df.dropna(how="all")
                        if not df.empty:
                            out.append((path.stem, df))
                        break
                except Exception:
                    continue
    except Exception as exc:
        say(f"    Could not inspect {path}: {exc}")
    return out


def walk_data_files(root):
    allowed = {".xlsx", ".xls", ".xlsm", ".csv"}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in allowed:
            if not p.name.startswith("~$"):
                yield p


def find_column(columns, patterns):
    """Return first original column matching one of the regex patterns."""
    pairs = [(c, clean_col(c)) for c in columns]
    for pat in patterns:
        rx = re.compile(pat)
        for original, normalized in pairs:
            if rx.fullmatch(normalized) or rx.search(normalized):
                return original
    return None


def numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False)
              .str.replace(" ", "", regex=False),
        errors="coerce"
    )



def safe_nullable_integer(series):
    """Convert only whole-number values; fractional/malformed values become NA."""
    values = pd.to_numeric(series, errors="coerce")
    rounded = values.round()
    valid = values.notna() & ((values - rounded).abs() < 1e-8)
    result = pd.Series(pd.NA, index=values.index, dtype="Int64")
    if valid.any():
        result.loc[valid] = rounded.loc[valid].astype("int64")
    return result

def standardize_eia923_sheet(df, source_file, sheet, year_hint):
    cols = list(df.columns)
    c_year = find_column(cols, [r"year", r"report_year"])
    c_month = find_column(cols, [r"month", r"report_month"])
    c_plant = find_column(cols, [
        r"plant_id", r"plant_code", r"plant_number", r"plant_num"
    ])
    c_gen = find_column(cols, [
        r"netgen.*", r"net_generation.*", r"net_generation_mwh.*"
    ])
    c_fuel = find_column(cols, [
        r"elec_fuel_cons.*", r"fuel_consumption.*",
        r"fuel_consumed.*", r"fuel_consumption_quantity.*"
    ])
    c_fuel_type = find_column(cols, [
        r"aer_fuel_type.*", r"energy_source.*", r"fuel_type.*",
        r"reported_fuel_type.*"
    ])
    c_plant_name = find_column(cols, [r"plant_name"])
    c_state = find_column(cols, [r"plant_state", r"state"])
    c_prime = find_column(cols, [r"prime_mover.*"])

    if c_plant is None or (c_gen is None and c_fuel is None):
        return None

    d = pd.DataFrame()
    d["plant_id"] = safe_nullable_integer(numeric(df[c_plant]))
    d["year"] = numeric(df[c_year]) if c_year else year_hint
    d["month"] = numeric(df[c_month]) if c_month else pd.NA
    d["net_generation_mwh"] = numeric(df[c_gen]) if c_gen else pd.NA
    d["fuel_consumption"] = numeric(df[c_fuel]) if c_fuel else pd.NA
    d["fuel_type"] = df[c_fuel_type].astype(str) if c_fuel_type else "Unknown"
    d["plant_name"] = df[c_plant_name].astype(str) if c_plant_name else ""
    d["state"] = df[c_state].astype(str) if c_state else ""
    d["prime_mover"] = df[c_prime].astype(str) if c_prime else ""
    d["source_file"] = str(source_file)
    d["source_sheet"] = str(sheet)
    d = d.dropna(subset=["plant_id", "year"])
    d["year"] = safe_nullable_integer(numeric(d["year"]))
    d["month"] = safe_nullable_integer(numeric(d["month"]))
    d = d[d["year"].between(min(YEARS), max(YEARS))]
    d = d[d["month"].between(1, 12) | d["month"].isna()]
    return d if not d.empty else None


def standardize_eia860_sheet(df, source_file, sheet, year_hint):
    cols = list(df.columns)
    c_plant = find_column(cols, [
        r"plant_code", r"plant_id", r"plant_number", r"plant_num"
    ])
    c_capacity = find_column(cols, [
        r"nameplate_capacity.*", r"summer_capacity.*",
        r"winter_capacity.*", r"capacity_mw"
    ])
    if c_plant is None or c_capacity is None:
        return None

    c_year = find_column(cols, [r"report_year", r"year"])
    c_name = find_column(cols, [r"plant_name"])
    c_state = find_column(cols, [r"plant_state", r"state"])
    c_fuel = find_column(cols, [
        r"energy_source_1", r"energy_source", r"fuel_type"
    ])
    c_tech = find_column(cols, [
        r"technology", r"technology_description"
    ])
    c_status = find_column(cols, [r"status", r"operational_status"])

    d = pd.DataFrame()
    d["plant_id"] = safe_nullable_integer(numeric(df[c_plant]))
    d["year"] = numeric(df[c_year]) if c_year else year_hint
    d["capacity_mw"] = numeric(df[c_capacity])
    d["plant_name"] = df[c_name].astype(str) if c_name else ""
    d["state"] = df[c_state].astype(str) if c_state else ""
    d["primary_fuel"] = df[c_fuel].astype(str) if c_fuel else "Unknown"
    d["technology"] = df[c_tech].astype(str) if c_tech else "Unknown"
    d["status"] = df[c_status].astype(str) if c_status else ""
    d["source_file"] = str(source_file)
    d["source_sheet"] = str(sheet)
    d = d.dropna(subset=["plant_id", "capacity_mw"])
    d["year"] = safe_nullable_integer(numeric(d["year"]))
    d = d[d["year"].between(min(YEARS), max(YEARS))]
    return d if not d.empty else None


def extract_year_hint(path):
    y = get_year_from_text(path.name)
    if y:
        return y
    for part in path.parts:
        if part.isdigit() and int(part) in YEARS:
            return int(part)
    return None


def process_workbooks():
    """Laptop-friendly: process one calendar year at a time and write checkpoints."""
    say("\n[PROCESSING] Laptop mode: one year at a time (2020-2025).")
    all_ops_paths = []
    all_cap_paths = []

    for year in YEARS:
        say(f"\n========== PROCESSING YEAR {year} ==========")
        year_ops = []
        year_cap = []

        for kind in ["eia923", "eia860"]:
            root = EXTRACTED / kind / str(year)
            if not root.exists():
                say(f"  {kind}: no extracted folder for {year}; skipping")
                continue

            files = list(walk_data_files(root))
            say(f"  {kind}: {len(files)} files for {year}")
            for i, path in enumerate(files, 1):
                say(f"    [{kind} {year}] file {i}/{len(files)}: {path.name}")
                try:
                    for sheet, df in read_tabular_file(path, kind):
                        if kind == "eia923":
                            z = standardize_eia923_sheet(df, path, sheet, year)
                            if z is not None:
                                year_ops.append(z)
                        else:
                            z = standardize_eia860_sheet(df, path, sheet, year)
                            if z is not None:
                                year_cap.append(z)
                        del df
                except Exception as exc:
                    say(f"      File skipped: {exc}")

        # Save this year's standardized rows immediately, so a later crash
        # does not lose completed years.
        if year_ops:
            ops_y = pd.concat(year_ops, ignore_index=True).drop_duplicates()
            ops_path = PROCESSED / f"eia923_standardized_{year}.csv"
            ops_y.to_csv(ops_path, index=False)
            all_ops_paths.append(ops_path)
            del ops_y, year_ops
            say(f"  Saved EIA-923 checkpoint for {year}: {ops_path.name}")
        else:
            say(f"  WARNING: no standardized EIA-923 rows recognized for {year}")

        if year_cap:
            cap_y = pd.concat(year_cap, ignore_index=True).drop_duplicates()
            cap_path = PROCESSED / f"eia860_standardized_{year}.csv"
            cap_y.to_csv(cap_path, index=False)
            all_cap_paths.append(cap_path)
            del cap_y, year_cap
            say(f"  Saved EIA-860 checkpoint for {year}: {cap_path.name}")
        else:
            say(f"  WARNING: no standardized EIA-860 rows recognized for {year}")

        import gc
        gc.collect()

    # Combine the small yearly checkpoint files only after all years finish.
    ops_frames = [pd.read_csv(f, low_memory=False) for f in all_ops_paths]
    cap_frames = [pd.read_csv(f, low_memory=False) for f in all_cap_paths]
    if not ops_frames:
        raise RuntimeError(
            "No EIA-923 tables were recognized in 2020-2025. "
            "Please upload workbook_sheet_catalog.csv and download_manifest.csv."
        )
    if not cap_frames:
        raise RuntimeError(
            "No EIA-860 capacity tables were recognized in 2020-2025. "
            "Please upload the downloaded workbook catalog and manifest."
        )

    ops = pd.concat(ops_frames, ignore_index=True).drop_duplicates()
    cap = pd.concat(cap_frames, ignore_index=True).drop_duplicates()
    del ops_frames, cap_frames

    group_cols = ["plant_id", "year", "month", "fuel_type"]
    monthly = (
        ops.groupby(group_cols, dropna=False, as_index=False)
           .agg(
               net_generation_mwh=("net_generation_mwh", "sum"),
               fuel_consumption=("fuel_consumption", "sum"),
               plant_name=("plant_name", "first"),
               state=("state", "first"),
               prime_mover=("prime_mover", "first"),
               source_file=("source_file", "first")
           )
    )

    cap["capacity_mw"] = numeric(cap["capacity_mw"])
    cap_plant = (
        cap.groupby(["plant_id", "year"], as_index=False)
           .agg(
               capacity_mw=("capacity_mw", "sum"),
               plant_name=("plant_name", "first"),
               state=("state", "first"),
               primary_fuel=("primary_fuel", "first"),
               technology=("technology", "first"),
               status=("status", "first")
           )
    )

    monthly.to_csv(PROCESSED / "eia923_monthly_plant.csv", index=False)
    cap_plant.to_csv(PROCESSED / "eia860_plant_capacity.csv", index=False)

    panel = monthly.merge(
        cap_plant, on=["plant_id", "year"], how="left",
        suffixes=("", "_eia860"), validate="many_to_one"
    )
    panel["year"] = safe_nullable_integer(panel["year"])
    panel["month"] = safe_nullable_integer(panel["month"])
    panel = panel.dropna(subset=["year"])
    panel["month_date"] = pd.to_datetime(
        dict(
            year=panel["year"].astype("int64"),
            month=panel["month"].fillna(1).astype("int64"),
            day=1
        ), errors="coerce"
    )
    panel = panel.sort_values(["plant_id", "year", "month", "fuel_type"])
    panel.to_csv(PROCESSED / "plant_month_panel.csv", index=False)

    report = [
        "EIA POWER-PLANT DATA QUALITY REPORT",
        "=" * 60,
        "Period: 2020-2025",
        f"EIA-923 standardized rows: {len(ops):,}",
        f"EIA-923 plant-month-fuel rows: {len(monthly):,}",
        f"EIA-860 standardized generator rows: {len(cap):,}",
        f"EIA-860 plant-year rows: {len(cap_plant):,}",
        f"Merged panel rows: {len(panel):,}",
        f"Unique plants in panel: {panel.plant_id.nunique():,}",
        f"Date range: {panel.month_date.min()} to {panel.month_date.max()}",
        f"Missing net generation: {panel.net_generation_mwh.isna().mean():.1%}",
        f"Missing fuel consumption: {panel.fuel_consumption.isna().mean():.1%}",
        f"Missing capacity after merge: {panel.capacity_mw.isna().mean():.1%}",
        "\nRows by year:",
        panel.groupby("year").size().to_string(),
        "\nTop fuel categories:",
        panel["fuel_type"].value_counts().head(20).to_string(),
        "\nTop states:",
        panel["state"].value_counts().head(20).to_string(),
        "\nCautions:",
        "- This is a preliminary source-data panel, not a DEA-ready sample.",
        "- Verify fuel consumption units before pooling fuel types.",
        "- Restrict DEA comparisons to reasonably homogeneous technologies.",
        "- Do not impute missing values as zero without a documented rule."
    ]
    (PROCESSED / "data_quality_report.txt").write_text(
        "\n".join(report), encoding="utf-8"
    )
    say("\n".join(report))
    return panel


def main():
    say("EIA REAL-DATA DOWNLOAD AND PANEL BUILD")
    say("Period requested: 2020-2025 (laptop mode)")
    say("Official EIA source pages:")
    say("  EIA-923: " + INDEX["eia923"])
    say("  EIA-860: " + INDEX["eia860"])

    download_all()
    panel = process_workbooks()

    say("\nSUCCESS: source data downloaded and preliminary panel created.")
    say(f"Project folder: {BASE.resolve()}")
    say(f"Panel file: {(PROCESSED / 'plant_month_panel.csv').resolve()}")
    say(f"Panel dimensions: {panel.shape[0]:,} rows x {panel.shape[1]} columns")
    say("\nNext, inspect data_quality_report.txt and workbook_sheet_catalog.csv.")
    say("Zip the eia_edro_project folder and upload it for the DEA/EDRO stage.")


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path

for p in Path(".").rglob("download_manifest.csv"):
    print("FOUND:", p.resolve())

for p in Path(".").rglob("*catalog*.csv"):
    print("CATALOG:", p.resolve())

In [ ]:
from IPython.display import display, FileLink

display(FileLink(
    "/datasets/_deepnote_work/eia_edro_project/processed/download_manifest.csv"
))

display(FileLink(
    "/datasets/_deepnote_work/eia_edro_project/processed/workbook_sheet_catalog.csv"
))

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("/datasets/_deepnote_work/eia_edro_project/processed")

for name in ["download_manifest.csv", "workbook_sheet_catalog.csv"]:
    p = folder / name
    print("\n" + "=" * 60)
    print(name, "| Exists:", p.exists())

    if p.exists():
        df = pd.read_csv(p)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print(df.head(15).to_string(index=False))

In [ ]:
from pathlib import Path
import requests
import zipfile
import pandas as pd
import numpy as np
import re
import io
import gc
import time
import warnings

warnings.filterwarnings("ignore")

BASE = Path("/datasets/_deepnote_work/eia_edro_project")
RAW = BASE / "raw" / "eia860"
EXTRACTED = BASE / "extracted" / "eia860"
PROCESSED = BASE / "processed"

for folder in [RAW, EXTRACTED, PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

YEARS = range(2020, 2026)

def say(message):
    print(time.strftime("%H:%M:%S"), message)

def clean_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    return x.strip("_")

say("EIA-860 recovery setup completed.")
print("Project:", BASE)
def eia860_url(year):
    if year == 2025:
        return (
            f"https://www.eia.gov/electricity/data/"
            f"eia860/xls/eia860{year}.zip"
        )
    return (
        f"https://www.eia.gov/electricity/data/"
        f"eia860/archive/xls/eia860{year}.zip"
    )

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

download_log = []

for year in YEARS:
    target = RAW / f"eia860{year}.zip"

    if target.exists() and zipfile.is_zipfile(target):
        say(f"{year}: existing ZIP reused")
        download_log.append({
            "year": year,
            "status": "reused",
            "file": str(target)
        })
        continue

    url = eia860_url(year)
    say(f"{year}: downloading EIA-860")

    try:
        response = session.get(url, timeout=120)
        response.raise_for_status()

        if not response.content.startswith(b"PK"):
            raise ValueError("Downloaded response is not a ZIP archive")

        target.write_bytes(response.content)

        if not zipfile.is_zipfile(target):
            raise ValueError("Invalid ZIP archive")

        say(f"{year}: download successful")

        download_log.append({
            "year": year,
            "status": "downloaded",
            "file": str(target)
        })

    except Exception as exc:
        say(f"{year}: DOWNLOAD ERROR: {exc}")
        download_log.append({
            "year": year,
            "status": "failed",
            "file": str(target),
            "error": str(exc)
        })

pd.DataFrame(download_log).to_csv(
    PROCESSED / "eia860_download_log.csv",
    index=False
)

print(pd.DataFrame(download_log).to_string(index=False))
extracted_files = []

for year in YEARS:
    archive = RAW / f"eia860{year}.zip"
    destination = EXTRACTED / str(year)
    destination.mkdir(parents=True, exist_ok=True)

    if not archive.exists() or not zipfile.is_zipfile(archive):
        say(f"{year}: ZIP unavailable; skipped")
        continue

    say(f"{year}: extracting archive")

    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if member.is_dir():
                continue

            suffix = Path(member.filename).suffix.lower()

            if suffix not in [".xlsx", ".xls", ".xlsm", ".csv"]:
                continue

            # Avoid extracting unrelated documentation files.
            if not any(k in member.filename.lower()
                       for k in ["generator", "plant", "eia860"]):
                continue

            z.extract(member, destination)

    for f in destination.rglob("*"):
        if f.is_file() and f.suffix.lower() in [
            ".xlsx", ".xls", ".xlsm", ".csv"
        ]:
            extracted_files.append({
                "year": year,
                "path": str(f),
                "filename": f.name
            })

    say(f"{year}: extraction complete")

file_table = pd.DataFrame(extracted_files)

file_table.to_csv(
    PROCESSED / "eia860_extracted_files.csv",
    index=False
)

print("Extracted tabular files:", len(file_table))
print(file_table.head(20).to_string(index=False))
def find_header_row(preview):
    for i in range(len(preview)):
        row = [clean_name(x) for x in preview.iloc[i].tolist()]
        joined = " ".join(row)

        has_plant = (
            "plant_code" in joined
            or "plant_id" in joined
            or "plant_number" in joined
        )

        has_capacity = (
            "nameplate_capacity" in joined
            or "summer_capacity" in joined
            or "winter_capacity" in joined
            or "capacity_mw" in joined
        )

        if has_plant and has_capacity:
            return i

    return None


def standardize_capacity(df, year, source):
    df.columns = [clean_name(c) for c in df.columns]

    plant_candidates = [
        c for c in df.columns
        if re.search(r"plant_(code|id|number)", c)
    ]

    capacity_candidates = [
        c for c in df.columns
        if re.search(
            r"nameplate.*capacity|capacity.*nameplate|"
            r"summer.*capacity|winter.*capacity",
            c
        )
    ]

    if not plant_candidates or not capacity_candidates:
        return None

    plant_col = plant_candidates[0]

    # Prefer nameplate capacity where available.
    nameplate = [
        c for c in capacity_candidates
        if "nameplate" in c
    ]

    capacity_col = (
        nameplate[0] if nameplate
        else capacity_candidates[0]
    )

    out = pd.DataFrame({
        "year": year,
        "plant_id": pd.to_numeric(
            df[plant_col], errors="coerce"
        ),
        "capacity_mw": pd.to_numeric(
            df[capacity_col]
            .astype(str)
            .str.replace(",", "", regex=False),
            errors="coerce"
        ),
        "source_file": Path(source).name
    })

    out = out.dropna(subset=["plant_id", "capacity_mw"])
    out = out[out["capacity_mw"] >= 0]

    return out


capacity_frames = []
catalog = []

for row in file_table.itertuples(index=False):
    path = Path(row.path)
    year = int(row.year)

    say(f"Inspecting {year}: {path.name}")

    try:
        if path.suffix.lower() == ".csv":
            preview = pd.read_csv(
                path, header=None, nrows=60, dtype=str
            )
            header_row = find_header_row(preview)

            if header_row is None:
                continue

            df = pd.read_csv(path, header=header_row)

            result = standardize_capacity(df, year, path)

            if result is not None and len(result):
                capacity_frames.append(result)

        else:
            with pd.ExcelFile(path) as book:
                for sheet in book.sheet_names:
                    preview = book.parse(
                        sheet_name=sheet,
                        header=None,
                        nrows=60,
                        dtype=str
                    )

                    header_row = find_header_row(preview)

                    if header_row is None:
                        continue

                    df = book.parse(
                        sheet_name=sheet,
                        header=header_row
                    )

                    result = standardize_capacity(
                        df, year, path
                    )

                    catalog.append({
                        "year": year,
                        "file": path.name,
                        "sheet": sheet,
                        "header_row": header_row,
                        "recognized": result is not None,
                        "rows": 0 if result is None else len(result)
                    })

                    if result is not None and len(result):
                        capacity_frames.append(result)

                    del df, preview
                    gc.collect()

    except Exception as exc:
        say(f"READ ERROR: {path.name}: {exc}")

    gc.collect()

pd.DataFrame(catalog).to_csv(
    PROCESSED / "eia860_capacity_sheet_catalog.csv",
    index=False
)

print("Recognized capacity tables:", len(capacity_frames))
if not capacity_frames:
    raise RuntimeError(
        "No EIA-860 capacity tables recognized. "
        "Please paste the contents of "
        "eia860_capacity_sheet_catalog.csv and "
        "eia860_extracted_files.csv."
    )

capacity_raw = pd.concat(
    capacity_frames,
    ignore_index=True
)

# Generator-level capacity is aggregated to plant level.
capacity_panel = (
    capacity_raw
    .groupby(["year", "plant_id"], as_index=False)
    .agg(
        capacity_mw=("capacity_mw", "sum"),
        generator_records=("capacity_mw", "size")
    )
)

capacity_panel = capacity_panel.sort_values(
    ["year", "plant_id"]
)

output = PROCESSED / "eia860_plant_capacity_2020_2025.csv"

capacity_panel.to_csv(output, index=False)

print("\nEIA-860 CAPACITY PANEL")
print("Rows:", len(capacity_panel))
print("Plants:", capacity_panel["plant_id"].nunique())
print("Years:", sorted(capacity_panel["year"].unique()))
print("\nRows per year:")
print(capacity_panel.groupby("year").size())

print("\nSummary:")
print(capacity_panel["capacity_mw"].describe())

print("\nSaved:", output)

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = BASE / "processed"

print("CSV FILES IN PROCESSED FOLDER\n")

for p in sorted(PROCESSED.glob("*.csv")):
    print(f"\nFILE: {p.name}")
    print(f"SIZE: {p.stat().st_size / 1e6:.2f} MB")

    try:
        df = pd.read_csv(p, nrows=3)
        print("COLUMNS:")
        print(list(df.columns))
        print("SAMPLE:")
        print(df.head(2).to_string(index=False))
    except Exception as e:
        print("Could not inspect:", e)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = BASE / "processed"

# --------------------------------------------------
# 1. Load EIA-923 annual files
# --------------------------------------------------
files = sorted(PROCESSED.glob("eia923_standardized_*.csv"))

if not files:
    raise FileNotFoundError("No eia923_standardized_YEAR.csv files found.")

frames = []
for f in files:
    d = pd.read_csv(f)
    d["plant_id"] = pd.to_numeric(d["plant_id"], errors="coerce")
    d["year"] = pd.to_numeric(d["year"], errors="coerce")
    d["net_generation_mwh"] = pd.to_numeric(
        d["net_generation_mwh"], errors="coerce"
    )
    d["fuel_consumption"] = pd.to_numeric(
        d["fuel_consumption"], errors="coerce"
    )
    frames.append(d)

eia = pd.concat(frames, ignore_index=True)

print("EIA-923 OVERVIEW")
print("Rows:", len(eia))
print("Years:", sorted(eia["year"].dropna().unique()))
print("Missing month (%):", round(100 * eia["month"].isna().mean(), 2))
print("Missing generation (%):",
      round(100 * eia["net_generation_mwh"].isna().mean(), 2))
print("Missing fuel consumption (%):",
      round(100 * eia["fuel_consumption"].isna().mean(), 2))

# --------------------------------------------------
# 2. Check whether records are annual or monthly
# --------------------------------------------------
print("\nRECORDS PER PLANT-YEAR")
print(
    eia.groupby(["year", "plant_id"]).size()
       .describe()
       .round(2)
)

print("\nMONTH VALUES")
print(eia["month"].value_counts(dropna=False).head(20))

# --------------------------------------------------
# 3. Fuel types and prime movers
# --------------------------------------------------
print("\nFUEL TYPES")
print(eia["fuel_type"].value_counts(dropna=False).head(30))

print("\nPRIME MOVERS")
print(eia["prime_mover"].value_counts(dropna=False).head(30))

print("\nFUEL TYPE x PRIME MOVER")
print(pd.crosstab(
    eia["fuel_type"],
    eia["prime_mover"],
    dropna=False
).to_string())

# --------------------------------------------------
# 4. Inspect NG combined-cycle observations
# --------------------------------------------------
ngcc = eia[
    eia["fuel_type"].astype(str).str.strip().str.upper().eq("NG")
    & eia["prime_mover"].astype(str).str.strip().str.upper().eq("CC")
].copy()

print("\nNGCC RECORDS")
print("Rows:", len(ngcc))
print("Plants:", ngcc["plant_id"].nunique())
print("Years:", sorted(ngcc["year"].dropna().unique()))

print("\nNGCC SAMPLE")
print(ngcc[
    ["plant_id", "year", "net_generation_mwh",
     "fuel_consumption", "fuel_type", "prime_mover",
     "plant_name", "state"]
].head(15).to_string(index=False))

# --------------------------------------------------
# 5. Check duplicate plant-year-fuel-technology records
# --------------------------------------------------
keys = ["plant_id", "year", "fuel_type", "prime_mover"]

print("\nDUPLICATE KEY COUNTS")
print("All EIA-923:", eia.duplicated(keys).sum())
print("NGCC:", ngcc.duplicated(keys).sum())

# --------------------------------------------------
# 6. Inspect capacity data
# --------------------------------------------------
cap_file = PROCESSED / "eia860_plant_capacity_2020_2025.csv"

if not cap_file.exists():
    raise FileNotFoundError(cap_file)

cap = pd.read_csv(cap_file)
cap["plant_id"] = pd.to_numeric(cap["plant_id"], errors="coerce")
cap["year"] = pd.to_numeric(cap["year"], errors="coerce")
cap["capacity_mw"] = pd.to_numeric(cap["capacity_mw"], errors="coerce")

print("\nCAPACITY OVERVIEW")
print("Rows:", len(cap))
print("Years:", sorted(cap["year"].dropna().unique()))
print("Plants:", cap["plant_id"].nunique())
print(cap["capacity_mw"].describe().round(2))

# --------------------------------------------------
# 7. Check EIA-923 / EIA-860 plant-year matching
# --------------------------------------------------
eia_keys = eia[["plant_id", "year"]].dropna().drop_duplicates()
cap_keys = cap[["plant_id", "year"]].dropna().drop_duplicates()

matched = eia_keys.merge(
    cap_keys,
    on=["plant_id", "year"],
    how="inner"
)

print("\nPLANT-YEAR MATCHING")
print("EIA-923 plant-years:", len(eia_keys))
print("Matched plant-years:", len(matched))
print("Match rate (%):",
      round(100 * len(matched) / max(len(eia_keys), 1), 2))

# --------------------------------------------------
# 8. Save the NGCC subset for inspection
# --------------------------------------------------
ngcc.to_csv(PROCESSED / "ngcc_eia923_inspection.csv", index=False)

print("\nSaved:")
print(PROCESSED / "ngcc_eia923_inspection.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = BASE / "processed"

# Load all standardized EIA-923 files
eia = pd.concat(
    [pd.read_csv(p) for p in sorted(
        PROCESSED.glob("eia923_standardized_*.csv")
    )],
    ignore_index=True
)

for c in ["plant_id", "year", "net_generation_mwh", "fuel_consumption"]:
    eia[c] = pd.to_numeric(eia[c], errors="coerce")

eia["fuel_type"] = eia["fuel_type"].astype(str).str.strip().str.upper()
eia["prime_mover"] = eia["prime_mover"].astype(str).str.strip().str.upper()

# --------------------------------------------------
# 1. Correct NGCC cohort: NG fuel + CA prime mover
# --------------------------------------------------
ngcc = eia[
    (eia["fuel_type"] == "NG") &
    (eia["prime_mover"] == "CA")
].copy()

print("NGCC (NG + CA)")
print("Rows:", len(ngcc))
print("Plants:", ngcc["plant_id"].nunique())
print("Plant-years:", ngcc[["plant_id", "year"]].drop_duplicates().shape[0])
print("\nRows by year:")
print(ngcc.groupby("year").size())

print("\nGeneration and fuel consumption:")
print(ngcc[["net_generation_mwh", "fuel_consumption"]].describe())

# --------------------------------------------------
# 2. Inspect duplicate keys
# --------------------------------------------------
keys = ["plant_id", "year", "fuel_type", "prime_mover"]

dup = eia[eia.duplicated(keys, keep=False)].copy()

print("\nDUPLICATE RECORD DIAGNOSTICS")
print("Rows belonging to duplicated keys:", len(dup))
print("Number of duplicated keys:", dup.groupby(keys).ngroups)

print("\nFirst duplicated records:")
cols = keys + [
    "net_generation_mwh", "fuel_consumption",
    "plant_name", "state", "source_file", "source_sheet"
]
print(dup[cols].head(20).to_string(index=False))

# Are the rows exact duplicates across the available data?
print("\nExact duplicate rows across all columns:", eia.duplicated().sum())

# Compare duplicate-key totals and within-key variation
dup_summary = (
    dup.groupby(keys, dropna=False)
       .agg(
           records=("plant_id", "size"),
           generation_sum=("net_generation_mwh", "sum"),
           generation_min=("net_generation_mwh", "min"),
           generation_max=("net_generation_mwh", "max"),
           fuel_sum=("fuel_consumption", "sum"),
           fuel_min=("fuel_consumption", "min"),
           fuel_max=("fuel_consumption", "max")
       )
       .reset_index()
)

print("\nDUPLICATE-KEY SUMMARY")
print(dup_summary.head(15).to_string(index=False))

# --------------------------------------------------
# 3. Check whether NGCC plant-years have multiple rows
# --------------------------------------------------
ngcc_summary = (
    ngcc.groupby(["plant_id", "year"])
        .agg(
            records=("plant_id", "size"),
            generation=("net_generation_mwh", "sum"),
            fuel=("fuel_consumption", "sum"),
            generation_missing=("net_generation_mwh",
                                lambda x: x.isna().sum())
        )
        .reset_index()
)

print("\nNGCC PLANT-YEAR SUMMARY")
print(ngcc_summary.describe().round(2).to_string())

print("\nNGCC plant-years with multiple records:")
print(
    ngcc_summary[ngcc_summary["records"] > 1]
    .sort_values("records", ascending=False)
    .head(15)
    .to_string(index=False)
)

# --------------------------------------------------
# 4. Inspect EIA-860 generator workbook files
# --------------------------------------------------
extracted = BASE / "extracted" / "eia860"

print("\nEIA-860 GENERATOR WORKBOOKS")
for p in sorted(extracted.rglob("*Generator*.xlsx"))[:20]:
    print(p)

print("\nEIA-860 capacity catalog, generator sheets:")
catalog_file = PROCESSED / "eia860_capacity_sheet_catalog.csv"
catalog = pd.read_csv(catalog_file)

print(
    catalog[
        catalog["file"].astype(str).str.contains("Generator", case=False)
    ].head(20).to_string(index=False)
)

# --------------------------------------------------
# 5. Save diagnostics
# --------------------------------------------------
ngcc.to_csv(PROCESSED / "ngcc_raw_records.csv", index=False)
ngcc_summary.to_csv(PROCESSED / "ngcc_plant_year_diagnostic.csv", index=False)
dup_summary.to_csv(PROCESSED / "eia923_duplicate_key_diagnostic.csv", index=False)

print("\nDiagnostic files saved in:", PROCESSED)

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = BASE / "processed"

# --------------------------------------------------
# A. Inspect EIA-923 standardized records
# --------------------------------------------------
eia = pd.read_csv(PROCESSED / "eia923_standardized_2025.csv")

plant_id = 10745
year = 2025

print("\n1. STANDARDIZED EIA-923 RECORDS")
print(
    eia[pd.to_numeric(eia["plant_id"], errors="coerce") == plant_id]
    .to_string(index=False)
)

# Find the original workbook
raw_dir = BASE / "extracted" / "eia923" / str(year)
workbooks = list(raw_dir.glob("*.xlsx"))

print("\n2. ORIGINAL EIA-923 WORKBOOKS")
for p in workbooks:
    print(p.name)

# Inspect the original generation/fuel sheet
for p in workbooks:
    try:
        xl = pd.ExcelFile(p)
        matching_sheets = [
            s for s in xl.sheet_names
            if "Generation and Fuel Data" in s
        ]

        for sheet in matching_sheets:
            raw = pd.read_excel(p, sheet_name=sheet)

            # Locate plant ID column
            plant_col = next(
                (c for c in raw.columns
                 if str(c).strip().lower() in
                 ["plant id", "plant_id", "plant code"]),
                None
            )

            if plant_col is None:
                continue

            selected = raw[
                pd.to_numeric(raw[plant_col], errors="coerce")
                == plant_id
            ]

            if len(selected):
                print("\n3. ORIGINAL EIA-923 ROWS")
                print("Workbook:", p.name)
                print("Sheet:", sheet)
                print("Columns:", list(raw.columns))
                print(selected.to_string(index=False))

    except Exception as ex:
        print("Could not inspect", p.name, ex)

# --------------------------------------------------
# B. Inspect EIA-860 generator-level records
# --------------------------------------------------
print("\n4. EIA-860 GENERATOR RECORDS")

gen_file = (
    BASE / "extracted" / "eia860" /
    str(year) / f"3_1_Generator_Y{year}.xlsx"
)

xl = pd.ExcelFile(gen_file)
print("Sheets:", xl.sheet_names)

for sheet in xl.sheet_names:
    try:
        g = pd.read_excel(gen_file, sheet_name=sheet, header=1)

        plant_col = next(
            (c for c in g.columns
             if "plant" in str(c).lower()
             and ("code" in str(c).lower() or "id" in str(c).lower())),
            None
        )

        if plant_col is None:
            continue

        selected = g[
            pd.to_numeric(g[plant_col], errors="coerce")
            == plant_id
        ]

        if len(selected):
            print("\nSheet:", sheet)
            print("Columns:", list(g.columns))
            print(selected.to_string(index=False))

    except Exception as ex:
        print("Could not inspect sheet", sheet, ex)

# --------------------------------------------------
# C. Show prime mover and fuel combinations
# --------------------------------------------------
print("\n5. PRIME MOVER COUNTS FOR NATURAL GAS")

all_eia = pd.concat(
    [pd.read_csv(p) for p in sorted(
        PROCESSED.glob("eia923_standardized_*.csv")
    )],
    ignore_index=True
)

ng = all_eia[
    all_eia["fuel_type"].astype(str).str.strip().str.upper() == "NG"
]

print(ng["prime_mover"].value_counts(dropna=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = BASE / "processed"
EIA860 = BASE / "extracted" / "eia860"

YEARS = range(2020, 2026)

# ---------------------------------------------------------
# 1. Build NGCC capacity from EIA-860 generator workbooks
# ---------------------------------------------------------
capacity_frames = []

for year in YEARS:

    path = EIA860 / str(year) / f"3_1_Generator_Y{year}.xlsx"

    if not path.exists():
        print("Missing:", path)
        continue

    gen = pd.read_excel(path, sheet_name="Operable", header=1)

    # Normalize relevant fields
    gen["Plant Code"] = pd.to_numeric(
        gen["Plant Code"], errors="coerce"
    )

    gen["Summer Capacity (MW)"] = pd.to_numeric(
        gen["Summer Capacity (MW)"], errors="coerce"
    )

    gen["Technology"] = gen["Technology"].astype(str).str.strip()
    gen["Energy Source 1"] = (
        gen["Energy Source 1"].astype(str).str.strip().str.upper()
    )
    gen["Status"] = gen["Status"].astype(str).str.strip().str.upper()

    # Strict NGCC technology definition
    ngcc_gen = gen[
        gen["Technology"].str.contains(
            "Natural Gas Fired Combined Cycle",
            case=False,
            na=False
        )
        & (gen["Energy Source 1"] == "NG")
        & (gen["Status"] == "OP")
    ].copy()

    ngcc_gen = ngcc_gen.dropna(
        subset=["Plant Code", "Summer Capacity (MW)"]
    )

    # Plant-level capacity
    cap_year = (
        ngcc_gen.groupby("Plant Code")
        .agg(
            capacity_mw=("Summer Capacity (MW)", "sum"),
            generator_count=("Plant Code", "size"),
            plant_name=("Plant Name", "first"),
            state=("State", "first")
        )
        .reset_index()
        .rename(columns={"Plant Code": "plant_id"})
    )

    cap_year["year"] = year
    capacity_frames.append(cap_year)

    print(
        year,
        "| NGCC generators:", len(ngcc_gen),
        "| NGCC plants:", len(cap_year)
    )

if not capacity_frames:
    raise RuntimeError("No NGCC generator capacity data were found.")

capacity = pd.concat(capacity_frames, ignore_index=True)

# ---------------------------------------------------------
# 2. Load EIA-923 and retain Page 1 only
# ---------------------------------------------------------
eia_frames = []

for year in YEARS:

    path = PROCESSED / f"eia923_standardized_{year}.csv"

    if not path.exists():
        print("Missing:", path)
        continue

    d = pd.read_csv(path)

    # Keep the generation-and-fuel schedule only
    d["source_sheet"] = d["source_sheet"].astype(str)

    d = d[
        d["source_sheet"].str.contains(
            "Page 1 Generation and Fuel Data",
            case=False,
            na=False
        )
    ].copy()

    d["plant_id"] = pd.to_numeric(d["plant_id"], errors="coerce")
    d["year"] = pd.to_numeric(d["year"], errors="coerce")
    d["net_generation_mwh"] = pd.to_numeric(
        d["net_generation_mwh"], errors="coerce"
    )
    d["fuel_consumption"] = pd.to_numeric(
        d["fuel_consumption"], errors="coerce"
    )

    d["fuel_type"] = d["fuel_type"].astype(str).str.strip().str.upper()
    d["prime_mover"] = (
        d["prime_mover"].astype(str).str.strip().str.upper()
    )

    # Natural gas records for the two NGCC components
    d = d[
        (d["fuel_type"] == "NG")
        & (d["prime_mover"].isin(["CA", "CT"]))
    ].copy()

    eia_frames.append(d)

eia_ngcc = pd.concat(eia_frames, ignore_index=True)

print("\nEIA-923 Page 1 NG / CA-CT records:", len(eia_ngcc))

# ---------------------------------------------------------
# 3. Keep only plants identified as NGCC in EIA-860
# ---------------------------------------------------------
valid_keys = capacity[["plant_id", "year"]].drop_duplicates()

eia_ngcc = eia_ngcc.merge(
    valid_keys,
    on=["plant_id", "year"],
    how="inner"
)

# Do not sum Page 3 boiler records.
# Aggregate Page 1 plant/prime-mover records only.
panel = (
    eia_ngcc.groupby(["plant_id", "year"], as_index=False)
    .agg(
        generation_mwh=("net_generation_mwh", "sum"),
        fuel_consumption=("fuel_consumption", "sum"),
        generation_missing=("net_generation_mwh",
                            lambda x: x.isna().sum()),
        source_records=("plant_id", "size"),
        plant_name=("plant_name", "first"),
        state=("state", "first")
    )
)

# ---------------------------------------------------------
# 4. Merge capacity
# ---------------------------------------------------------
panel = panel.merge(
    capacity,
    on=["plant_id", "year"],
    how="inner",
    suffixes=("", "_capacity")
)

# ---------------------------------------------------------
# 5. Clean observations
# ---------------------------------------------------------
print("\nBefore cleaning:", len(panel))

# Remove missing or nonpositive inputs and outputs.
# Negative generation is not economically meaningful for this
# production frontier and is excluded rather than set to zero.
panel = panel[
    panel["generation_missing"].eq(0)
    & panel["generation_mwh"].notna()
    & panel["fuel_consumption"].notna()
    & panel["capacity_mw"].notna()
    & (panel["generation_mwh"] > 0)
    & (panel["fuel_consumption"] > 0)
    & (panel["capacity_mw"] > 0)
].copy()

# Remove non-finite values
numeric_cols = [
    "generation_mwh",
    "fuel_consumption",
    "capacity_mw"
]

panel = panel[
    np.isfinite(panel[numeric_cols]).all(axis=1)
].copy()

panel = panel.sort_values(
    ["year", "plant_id"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 6. Derived productivity indicator
# ---------------------------------------------------------
panel["heat_rate_mmbtu_per_mwh"] = (
    panel["fuel_consumption"] / panel["generation_mwh"]
)

# ---------------------------------------------------------
# 7. Save
# ---------------------------------------------------------
output = PROCESSED / "ngcc_plant_year_panel.csv"
panel.to_csv(output, index=False)

print("\nCLEAN NGCC PANEL")
print("Observations:", len(panel))
print("Plants:", panel["plant_id"].nunique())
print("Years:", sorted(panel["year"].unique()))

print("\nObservations by year:")
print(panel.groupby("year").size())

print("\nDescriptive statistics:")
print(
    panel[
        ["capacity_mw", "fuel_consumption",
         "generation_mwh", "heat_rate_mmbtu_per_mwh"]
    ].describe().round(3)
)

print("\nSaved:", output)

In [ ]:
import pandas as pd
import os
import glob

BASE = "/datasets/_deepnote_work/eia_edro_project"

# Load panel
panel_path = os.path.join(BASE, "processed", "ngcc_plant_year_panel.csv")
df = pd.read_csv(panel_path)

df["heat_rate"] = df["fuel_consumption"] / df["generation_mwh"]

print("AVAILABLE COLUMNS:")
print(df.columns.tolist())

# Quantiles
print("\nFUEL AND GENERATION QUANTILES:")
print(
    df[["fuel_consumption", "generation_mwh", "heat_rate"]]
    .quantile([0, .01, .05, .25, .50, .75, .95, .99, 1])
)

# Largest heat rates
cols = [
    c for c in [
        "plant_id", "year", "plant_name", "prime_mover",
        "fuel_consumption", "generation_mwh", "heat_rate"
    ] if c in df.columns
]

print("\nTEN LARGEST HEAT-RATE OBSERVATIONS:")
print(df.nlargest(10, "heat_rate")[cols].to_string(index=False))

# Inspect original EIA-923 workbook
folder = os.path.join(BASE, "extracted", "eia923", "2025")
files = glob.glob(os.path.join(folder, "*.xlsx"))

print("\nORIGINAL WORKBOOKS:")
for f in files:
    print(os.path.basename(f))

if not files:
    raise FileNotFoundError("No original 2025 EIA-923 workbook found.")

workbook = files[0]
xls = pd.ExcelFile(workbook)

print("\nWORKBOOK SHEETS:")
for s in xls.sheet_names:
    print(s)

for sheet in xls.sheet_names:
    if "Page 1 Generation and Fuel Data" in sheet:
        raw = pd.read_excel(
            workbook, sheet_name=sheet,
            header=None, nrows=8
        )
        print("\nPAGE 1 SHEET:", sheet)
        print(raw.to_string(index=False, header=False))

In [ ]:
import pandas as pd
import os
import glob

BASE = "/datasets/_deepnote_work/eia_edro_project"
folder = os.path.join(BASE, "extracted", "eia923", "2024")

files = glob.glob(os.path.join(folder, "*Schedules_2_3_4_5*.xlsx"))
if not files:
    files = glob.glob(os.path.join(folder, "*.xlsx"))

if not files:
    raise FileNotFoundError("No original EIA-923 workbook found.")

workbook = files[0]
sheet = "Page 1 Generation and Fuel Data"

# Read the first 30 rows without assuming a header
raw = pd.read_excel(
    workbook, sheet_name=sheet, header=None, nrows=30
)

print("Rows containing likely column names:\n")

for i in range(len(raw)):
    vals = [
        str(v).strip()
        for v in raw.iloc[i].tolist()
        if pd.notna(v)
    ]
    joined = " | ".join(vals)

    if any(k in joined.lower() for k in
           ["plant id", "netgen", "elec_mmbtu", "tot_mmbtu"]):
        print(f"ROW {i}: {joined[:1500]}")

print("\nFirst 30 rows inspected.")

In [ ]:
import pandas as pd
import os
import glob

BASE = "/datasets/_deepnote_work/eia_edro_project"

# Original 2024 EIA-923 workbook
folder = os.path.join(BASE, "extracted", "eia923", "2024")
files = glob.glob(os.path.join(folder, "*Schedules_2_3_4_5*.xlsx"))

if not files:
    raise FileNotFoundError("Original EIA-923 workbook not found.")

workbook = files[0]

original = pd.read_excel(
    workbook,
    sheet_name="Page 1 Generation and Fuel Data",
    header=5
)

print("ORIGINAL COLUMN NAMES:")
for i, c in enumerate(original.columns):
    print(i, repr(c))

# Identify plant 55405
original["Plant Id"] = pd.to_numeric(
    original["Plant Id"], errors="coerce"
)

plant = original[original["Plant Id"] == 55405]

print("\nORIGINAL RECORDS FOR PLANT 55405:")
print("Number of records:", len(plant))

# Display relevant fields, including annual totals
keywords = [
    "Plant Id", "Plant Name", "Reported Prime Mover",
    "Reported Fuel Type Code", "Physical Unit Label",
    "Total Fuel Consumption", "Electric Fuel Consumption",
    "Net Generation", "Tot_MMBtu", "Elec_MMBtu", "Netgen"
]

cols = [
    c for c in original.columns
    if any(k.lower() in str(c).lower() for k in keywords)
]

print(plant[cols].to_string(index=False))

# Compare standardized data
standardized = pd.read_csv(
    os.path.join(BASE, "processed", "eia923_standardized_2024.csv")
)

print("\nSTANDARDIZED RECORDS FOR PLANT 55405:")
print(
    standardized[
        pd.to_numeric(standardized["plant_id"], errors="coerce") == 55405
    ].to_string(index=False)
)

In [ ]:

import pandas as pd
import numpy as np
import os
import glob

BASE = "/datasets/_deepnote_work/eia_edro_project"
YEARS = range(2020, 2026)

all_years = []

for year in YEARS:

    # --------------------------------------------------
    # 1. Identify operating NGCC generators in EIA-860
    # --------------------------------------------------
    eia860_folder = os.path.join(
        BASE, "extracted", "eia860", str(year)
    )

    generator_files = glob.glob(
        os.path.join(eia860_folder, "*3_1_Generator*.xlsx")
    )

    if not generator_files:
        raise FileNotFoundError(
            f"EIA-860 generator workbook not found for {year}"
        )

    gen = pd.read_excel(
        generator_files[0],
        sheet_name="Operable",
        header=1
    )

    gen.columns = gen.columns.astype(str).str.strip()

    gen["Technology"] = gen["Technology"].astype(str)
    gen["Energy Source 1"] = (
        gen["Energy Source 1"].astype(str).str.strip()
    )
    gen["Status"] = gen["Status"].astype(str).str.strip()

    ngcc = gen[
        gen["Technology"].str.contains(
            "Natural Gas Fired Combined Cycle",
            case=False, na=False
        )
        & gen["Energy Source 1"].eq("NG")
        & gen["Status"].eq("OP")
    ].copy()

    ngcc["Plant Code"] = pd.to_numeric(
        ngcc["Plant Code"], errors="coerce"
    )

    ngcc["Summer Capacity (MW)"] = pd.to_numeric(
        ngcc["Summer Capacity (MW)"], errors="coerce"
    )

    capacity = (
        ngcc.dropna(subset=["Plant Code"])
        .groupby("Plant Code")
        .agg(
            capacity_mw=("Summer Capacity (MW)", "sum"),
            generator_count=("Plant Code", "size"),
            plant_name=("Plant Name", "first"),
            state=("Plant State", "first")
        )
        .reset_index()
        .rename(columns={"Plant Code": "plant_id"})
    )

    capacity["year"] = year

    # --------------------------------------------------
    # 2. Read original annual EIA-923 Page 1 data
    # --------------------------------------------------
    eia923_folder = os.path.join(
        BASE, "extracted", "eia923", str(year)
    )

    eia923_files = glob.glob(
        os.path.join(
            eia923_folder,
            "*Schedules_2_3_4_5*.xlsx"
        )
    )

    if not eia923_files:
        raise FileNotFoundError(
            f"EIA-923 workbook not found for {year}"
        )

    raw = pd.read_excel(
        eia923_files[0],
        sheet_name="Page 1 Generation and Fuel Data",
        header=5
    )

    # Normalize line breaks in column names
    raw.columns = (
        raw.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

    required = [
        "Plant Id",
        "Reported Prime Mover",
        "Reported Fuel Type Code",
        "Elec Fuel Consumption MMBtu",
        "Net Generation (Megawatthours)"
    ]

    missing = [c for c in required if c not in raw.columns]

    if missing:
        raise ValueError(
            f"Missing EIA-923 columns in {year}: {missing}"
        )

    raw["Plant Id"] = pd.to_numeric(
        raw["Plant Id"], errors="coerce"
    )

    raw["Reported Fuel Type Code"] = (
        raw["Reported Fuel Type Code"]
        .astype(str).str.strip()
    )

    raw["Reported Prime Mover"] = (
        raw["Reported Prime Mover"]
        .astype(str).str.strip()
    )

    raw["Elec Fuel Consumption MMBtu"] = pd.to_numeric(
        raw["Elec Fuel Consumption MMBtu"],
        errors="coerce"
    )

    raw["Net Generation (Megawatthours)"] = pd.to_numeric(
        raw["Net Generation (Megawatthours)"],
        errors="coerce"
    )

    # Keep natural gas records reported under CA or CT.
    # These are the relevant NGCC prime-mover components.
    fuel = raw[
        raw["Reported Fuel Type Code"].eq("NG")
        & raw["Reported Prime Mover"].isin(["CA", "CT"])
        & raw["Plant Id"].isin(capacity["plant_id"])
    ].copy()

    # --------------------------------------------------
    # 3. Aggregate annual electricity fuel and generation
    # --------------------------------------------------
    # These are annual summary columns, NOT monthly columns.
    annual = (
        fuel.groupby("Plant Id", as_index=False)
        .agg(
            fuel_consumption_mmbtu=(
                "Elec Fuel Consumption MMBtu", "sum"
            ),
            generation_mwh=(
                "Net Generation (Megawatthours)", "sum"
            ),
            source_records=("Plant Id", "size")
        )
        .rename(columns={"Plant Id": "plant_id"})
    )

    # --------------------------------------------------
    # 4. Merge with NGCC capacity
    # --------------------------------------------------
    panel_year = capacity.merge(
        annual, on="plant_id", how="inner"
    )

    panel_year["year"] = year

    panel_year = panel_year[
        (panel_year["capacity_mw"] > 0)
        & (panel_year["fuel_consumption_mmbtu"] > 0)
        & (panel_year["generation_mwh"] > 0)
    ].copy()

    panel_year["heat_rate_mmbtu_per_mwh"] = (
        panel_year["fuel_consumption_mmbtu"]
        / panel_year["generation_mwh"]
    )

    all_years.append(panel_year)

    print(
        year,
        "| NGCC plants:", len(capacity),
        "| matched:", len(panel_year)
    )

# ------------------------------------------------------
# 5. Combine years and save corrected panel
# ------------------------------------------------------
corrected = pd.concat(all_years, ignore_index=True)

corrected = corrected.replace(
    [np.inf, -np.inf], np.nan
)

corrected = corrected.dropna(
    subset=[
        "capacity_mw",
        "fuel_consumption_mmbtu",
        "generation_mwh",
        "heat_rate_mmbtu_per_mwh"
    ]
)

output = os.path.join(
    BASE, "processed",
    "ngcc_plant_year_panel_corrected.csv"
)

corrected.to_csv(output, index=False)

print("\nCORRECTED NGCC PANEL")
print("Observations:", len(corrected))
print("Unique plants:", corrected["plant_id"].nunique())
print("Years:", sorted(corrected["year"].unique()))

print("\nObservations by year:")
print(corrected.groupby("year").size())

print("\nCorrected heat-rate distribution:")
print(
    corrected["heat_rate_mmbtu_per_mwh"]
    .describe(percentiles=[.01, .05, .25, .50, .75, .95, .99])
)

print("\nTen highest heat rates:")
print(
    corrected.nlargest(10, "heat_rate_mmbtu_per_mwh")[
        [
            "plant_id", "year", "plant_name",
            "fuel_consumption_mmbtu",
            "generation_mwh",
            "heat_rate_mmbtu_per_mwh"
        ]
    ].to_string(index=False)
)

print("\nSaved:", output)


In [ ]:

import pandas as pd
import numpy as np
import os
import glob

BASE = "/datasets/_deepnote_work/eia_edro_project"
YEARS = range(2020, 2026)

all_years = []

for year in YEARS:

    print(f"\nProcessing {year}...")

    # ==================================================
    # 1. EIA-860: Identify operating NGCC generators
    # ==================================================
    eia860_folder = os.path.join(
        BASE, "extracted", "eia860", str(year)
    )

    generator_files = glob.glob(
        os.path.join(eia860_folder, "*3_1_Generator*.xlsx")
    )

    if not generator_files:
        raise FileNotFoundError(
            f"EIA-860 generator workbook not found for {year}"
        )

    gen = pd.read_excel(
        generator_files[0],
        sheet_name="Operable",
        header=1
    )

    gen.columns = (
        gen.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

    # Normalize possible column-name variations
    state_col = next(
        (c for c in ["Plant State", "State"]
         if c in gen.columns),
        None
    )

    required_gen = [
        "Plant Code",
        "Technology",
        "Energy Source 1",
        "Status",
        "Summer Capacity (MW)",
        "Plant Name"
    ]

    missing = [c for c in required_gen if c not in gen.columns]

    if missing:
        raise ValueError(
            f"EIA-860 {year}: missing columns {missing}. "
            f"Available: {gen.columns.tolist()}"
        )

    if state_col is None:
        print("Warning: no state column found.")
        gen["_state"] = np.nan
        state_col = "_state"

    gen["Technology"] = gen["Technology"].astype(str)
    gen["Energy Source 1"] = (
        gen["Energy Source 1"].astype(str).str.strip()
    )
    gen["Status"] = gen["Status"].astype(str).str.strip()

    ngcc = gen[
        gen["Technology"].str.contains(
            "Natural Gas Fired Combined Cycle",
            case=False,
            na=False
        )
        & gen["Energy Source 1"].eq("NG")
        & gen["Status"].eq("OP")
    ].copy()

    ngcc["Plant Code"] = pd.to_numeric(
        ngcc["Plant Code"], errors="coerce"
    )

    ngcc["Summer Capacity (MW)"] = pd.to_numeric(
        ngcc["Summer Capacity (MW)"], errors="coerce"
    )

    capacity = (
        ngcc.dropna(subset=["Plant Code"])
        .groupby("Plant Code")
        .agg(
            capacity_mw=("Summer Capacity (MW)", "sum"),
            generator_count=("Plant Code", "size"),
            plant_name=("Plant Name", "first"),
            state=(state_col, "first")
        )
        .reset_index()
        .rename(columns={"Plant Code": "plant_id"})
    )

    capacity["year"] = year

    # ==================================================
    # 2. EIA-923: Read original annual Page 1 data
    # ==================================================
    eia923_folder = os.path.join(
        BASE, "extracted", "eia923", str(year)
    )

    eia923_files = glob.glob(
        os.path.join(
            eia923_folder,
            "*Schedules_2_3_4_5*.xlsx"
        )
    )

    if not eia923_files:
        raise FileNotFoundError(
            f"EIA-923 workbook not found for {year}"
        )

    raw = pd.read_excel(
        eia923_files[0],
        sheet_name="Page 1 Generation and Fuel Data",
        header=5
    )

    raw.columns = (
        raw.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

    required_eia923 = [
        "Plant Id",
        "Reported Prime Mover",
        "Reported Fuel Type Code",
        "Elec Fuel Consumption MMBtu",
        "Net Generation (Megawatthours)"
    ]

    missing = [
        c for c in required_eia923
        if c not in raw.columns
    ]

    if missing:
        raise ValueError(
            f"EIA-923 {year}: missing columns {missing}. "
            f"Available: {raw.columns.tolist()}"
        )

    raw["Plant Id"] = pd.to_numeric(
        raw["Plant Id"], errors="coerce"
    )

    raw["Reported Fuel Type Code"] = (
        raw["Reported Fuel Type Code"]
        .astype(str).str.strip()
    )

    raw["Reported Prime Mover"] = (
        raw["Reported Prime Mover"]
        .astype(str).str.strip()
    )

    raw["Elec Fuel Consumption MMBtu"] = pd.to_numeric(
        raw["Elec Fuel Consumption MMBtu"],
        errors="coerce"
    )

    raw["Net Generation (Megawatthours)"] = pd.to_numeric(
        raw["Net Generation (Megawatthours)"],
        errors="coerce"
    )

    # Keep NG fuel records for CA and CT prime movers
    fuel = raw[
        raw["Reported Fuel Type Code"].eq("NG")
        & raw["Reported Prime Mover"].isin(["CA", "CT"])
        & raw["Plant Id"].isin(capacity["plant_id"])
    ].copy()

    # ==================================================
    # 3. Aggregate annual electricity fuel and generation
    # ==================================================
    annual = (
        fuel.groupby("Plant Id", as_index=False)
        .agg(
            fuel_consumption_mmbtu=(
                "Elec Fuel Consumption MMBtu", "sum"
            ),
            generation_mwh=(
                "Net Generation (Megawatthours)", "sum"
            ),
            source_records=("Plant Id", "size")
        )
        .rename(columns={"Plant Id": "plant_id"})
    )

    # ==================================================
    # 4. Merge with capacity
    # ==================================================
    panel_year = capacity.merge(
        annual,
        on="plant_id",
        how="inner"
    )

    panel_year["year"] = year

    # Remove invalid or zero observations
    panel_year = panel_year[
        (panel_year["capacity_mw"] > 0)
        & (panel_year["fuel_consumption_mmbtu"] > 0)
        & (panel_year["generation_mwh"] > 0)
    ].copy()

    panel_year["heat_rate_mmbtu_per_mwh"] = (
        panel_year["fuel_consumption_mmbtu"]
        / panel_year["generation_mwh"]
    )

    all_years.append(panel_year)

    print(
        f"{year}: NGCC generators={len(ngcc)}, "
        f"NGCC plants={len(capacity)}, "
        f"valid plant-year observations={len(panel_year)}"
    )

# ======================================================
# 5. Combine all six years
# ======================================================
corrected = pd.concat(
    all_years,
    ignore_index=True
)

corrected = corrected.replace(
    [np.inf, -np.inf], np.nan
)

corrected = corrected.dropna(
    subset=[
        "capacity_mw",
        "fuel_consumption_mmbtu",
        "generation_mwh",
        "heat_rate_mmbtu_per_mwh"
    ]
)

# ======================================================
# 6. Save corrected panel
# ======================================================
output = os.path.join(
    BASE,
    "processed",
    "ngcc_plant_year_panel_corrected.csv"
)

corrected.to_csv(output, index=False)

# ======================================================
# 7. Report results
# ======================================================
print("\n" + "=" * 60)
print("CORRECTED NGCC PANEL")
print("=" * 60)

print("Observations:", len(corrected))
print("Unique plants:", corrected["plant_id"].nunique())
print("Years:", sorted(corrected["year"].unique()))

print("\nObservations by year:")
print(corrected.groupby("year").size())

print("\nDescriptive statistics:")
print(
    corrected[
        [
            "capacity_mw",
            "fuel_consumption_mmbtu",
            "generation_mwh",
            "heat_rate_mmbtu_per_mwh"
        ]
    ].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    ).round(3)
)

print("\nTen highest heat rates:")
print(
    corrected.nlargest(
        10, "heat_rate_mmbtu_per_mwh"
    )[
        [
            "plant_id",
            "year",
            "plant_name",
            "fuel_consumption_mmbtu",
            "generation_mwh",
            "heat_rate_mmbtu_per_mwh"
        ]
    ].to_string(index=False)
)

print("\nSaved file:")
print(output)


In [ ]:

import pandas as pd
import numpy as np
import os
import glob
import calendar

BASE = "/datasets/_deepnote_work/eia_edro_project"

path = os.path.join(
    BASE, "processed",
    "ngcc_plant_year_panel_corrected.csv"
)

df = pd.read_csv(path)

# --------------------------------------------------
# 1. Calculate annual capacity factors
# --------------------------------------------------
df["hours_year"] = df["year"].apply(
    lambda y: 8784 if calendar.isleap(int(y)) else 8760
)

df["capacity_factor"] = (
    df["generation_mwh"]
    / (df["capacity_mw"] * df["hours_year"])
)

print("CAPACITY FACTOR DISTRIBUTION")
print(
    df["capacity_factor"]
    .describe(percentiles=[.01, .05, .25, .50, .75, .95, .99])
    .round(4)
)

print("\nHEAT RATE BY CAPACITY-FACTOR GROUP")
df["cf_group"] = pd.cut(
    df["capacity_factor"],
    bins=[0, .05, .10, .20, .40, .60, .80, 1.01],
    labels=[
        "0-5%", "5-10%", "10-20%", "20-40%",
        "40-60%", "60-80%", "80-100%"
    ]
)

print(
    df.groupby("cf_group", observed=True)
    ["heat_rate_mmbtu_per_mwh"]
    .agg(["count", "median", "mean", "max"])
    .round(3)
)

# Inspect the highest heat-rate observations
print("\nHIGHEST HEAT-RATE OBSERVATIONS")
print(
    df.nlargest(15, "heat_rate_mmbtu_per_mwh")[
        [
            "plant_id", "year", "plant_name",
            "capacity_mw", "generation_mwh",
            "capacity_factor",
            "heat_rate_mmbtu_per_mwh"
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# 2. Check mixed-technology plants in EIA-860
# --------------------------------------------------
mix_results = []

for year in sorted(df["year"].unique()):

    folder = os.path.join(
        BASE, "extracted", "eia860", str(int(year))
    )

    files = glob.glob(
        os.path.join(folder, "*3_1_Generator*.xlsx")
    )

    if not files:
        print("Missing EIA-860 file:", year)
        continue

    gen = pd.read_excel(
        files[0], sheet_name="Operable", header=1
    )

    gen.columns = (
        gen.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

    gen["Plant Code"] = pd.to_numeric(
        gen["Plant Code"], errors="coerce"
    )

    gen["Summer Capacity (MW)"] = pd.to_numeric(
        gen["Summer Capacity (MW)"], errors="coerce"
    )

    gen["Status"] = gen["Status"].astype(str).str.strip()
    gen["Technology"] = gen["Technology"].astype(str)

    operating = gen[
        gen["Status"].eq("OP")
        & gen["Plant Code"].notna()
    ].copy()

    # Total operating capacity of each plant
    total = (
        operating.groupby("Plant Code")[
            "Summer Capacity (MW)"
        ].sum()
        .rename("total_operating_capacity_mw")
    )

    # NGCC operating capacity
    ngcc = operating[
        operating["Technology"].str.contains(
            "Natural Gas Fired Combined Cycle",
            case=False, na=False
        )
        & operating["Energy Source 1"].astype(str).str.strip().eq("NG")
    ]

    ngcc_capacity = (
        ngcc.groupby("Plant Code")[
            "Summer Capacity (MW)"
        ].sum()
        .rename("ngcc_capacity_mw")
    )

    shares = pd.concat(
        [total, ngcc_capacity], axis=1
    ).reset_index()

    shares = shares.rename(
        columns={"Plant Code": "plant_id"}
    )

    shares["year"] = int(year)

    shares["ngcc_capacity_share"] = (
        shares["ngcc_capacity_mw"]
        / shares["total_operating_capacity_mw"]
    )

    mix_results.append(shares)

mix = pd.concat(mix_results, ignore_index=True)

# Merge technology shares with the DEA panel
df = df.merge(
    mix[
        [
            "plant_id", "year",
            "total_operating_capacity_mw",
            "ngcc_capacity_share"
        ]
    ],
    on=["plant_id", "year"],
    how="left"
)

print("\nNGCC CAPACITY SHARE DISTRIBUTION")
print(
    df["ngcc_capacity_share"]
    .describe(percentiles=[.01, .05, .25, .50, .75, .95, .99])
    .round(4)
)

print("\nPLANT-YEAR COUNTS BY NGCC CAPACITY SHARE")
df["technology_group"] = pd.cut(
    df["ngcc_capacity_share"],
    bins=[0, .50, .80, .90, .95, 1.001],
    labels=[
        "Below 50%",
        "50-80%",
        "80-90%",
        "90-95%",
        "95-100%"
    ],
    include_lowest=True
)

print(
    df.groupby("technology_group", observed=False)
    .agg(
        observations=("plant_id", "size"),
        plants=("plant_id", "nunique"),
        median_heat_rate=("heat_rate_mmbtu_per_mwh", "median")
    )
    .round(3)
)

print("\nLOWEST NGCC-SHARE OBSERVATIONS")
print(
    df.nsmallest(15, "ngcc_capacity_share")[
        [
            "plant_id", "year", "plant_name",
            "capacity_mw",
            "total_operating_capacity_mw",
            "ngcc_capacity_share"
        ]
    ].to_string(index=False)
)

# Save validation dataset
validation_path = os.path.join(
    BASE, "processed",
    "ngcc_plant_year_validation.csv"
)

df.to_csv(validation_path, index=False)

print("\nValidation file saved:", validation_path)


In [ ]:
import pandas as pd
import numpy as np
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

panel = pd.read_csv(
    os.path.join(PROC, "ngcc_plant_year_panel_corrected.csv")
)

validation = pd.read_csv(
    os.path.join(PROC, "ngcc_plant_year_validation.csv")
)

df = panel.merge(
    validation[
        ["plant_id", "year", "capacity_factor", "ngcc_capacity_share",
         "total_operating_capacity_mw", "ngcc_capacity_mw"]
    ],
    on=["plant_id", "year"],
    how="left"
)

# 1. Capacity-factor anomalies
print("\nCAPACITY FACTOR ABOVE 1")
print("Count:", (df["capacity_factor"] > 1).sum())

print("\nCAPACITY FACTOR ABOVE 1.05")
print("Count:", (df["capacity_factor"] > 1.05).sum())

print("\nCAPACITY FACTOR ABOVE 1.20")
print("Count:", (df["capacity_factor"] > 1.20).sum())

print("\nALL OBSERVATIONS ABOVE 1")
print(
    df.loc[df["capacity_factor"] > 1,
           ["plant_id", "year", "plant_name", "capacity_mw",
            "generation_mwh", "capacity_factor",
            "ngcc_capacity_share"]]
    .sort_values("capacity_factor", ascending=False)
    .to_string(index=False)
)

# 2. Compare capacity-factor distributions by technology purity
df["purity_group"] = np.select(
    [
        df["ngcc_capacity_share"] >= 0.95,
        df["ngcc_capacity_share"] >= 0.80,
        df["ngcc_capacity_share"] >= 0.50
    ],
    ["At least 95%", "80-95%", "50-80%"],
    default="Below 50%"
)

print("\nCAPACITY FACTOR BY NGCC PURITY")
print(
    df.groupby("purity_group")["capacity_factor"]
      .agg(["count", "mean", "median", "max"])
      .round(4)
)

# 3. Check whether high capacity factors are concentrated
# among plants with very low NGCC capacity shares
print("\nOBSERVATIONS WITH CAPACITY FACTOR ABOVE 1")
print(
    df.loc[df["capacity_factor"] > 1]
      .groupby("purity_group")
      .size()
)

# 4. Candidate homogeneous sample
pure = df[df["ngcc_capacity_share"] >= 0.95].copy()

print("\nCANDIDATE PURE-NGCC SAMPLE")
print("Observations:", len(pure))
print("Plants:", pure["plant_id"].nunique())
print("Years:", sorted(pure["year"].unique()))
print("Capacity factor above 1:", (pure["capacity_factor"] > 1).sum())

print("\nPURE-NGCC CAPACITY FACTOR DISTRIBUTION")
print(pure["capacity_factor"].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
).round(4))

# 5. Save diagnostic sample, without deleting any observations
pure.to_csv(
    os.path.join(PROC, "ngcc_pure_candidate_panel.csv"),
    index=False
)

print("\nSaved candidate sample: ngcc_pure_candidate_panel.csv")

In [ ]:

import pandas as pd
import numpy as np
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# Load the corrected panel and validation results
panel = pd.read_csv(
    os.path.join(PROC, "ngcc_plant_year_panel_corrected.csv")
)

validation = pd.read_csv(
    os.path.join(PROC, "ngcc_plant_year_validation.csv")
)

# Merge only columns that actually exist in the validation file
merge_cols = [
    "plant_id",
    "year",
    "capacity_factor",
    "ngcc_capacity_share",
    "total_operating_capacity_mw"
]

df = panel.merge(
    validation[merge_cols],
    on=["plant_id", "year"],
    how="left"
)

# --------------------------------------------------
# 1. Capacity-factor anomalies
# --------------------------------------------------

print("\nCAPACITY FACTOR ABOVE 1")
print("Count:", (df["capacity_factor"] > 1).sum())

print("\nCAPACITY FACTOR ABOVE 1.05")
print("Count:", (df["capacity_factor"] > 1.05).sum())

print("\nCAPACITY FACTOR ABOVE 1.20")
print("Count:", (df["capacity_factor"] > 1.20).sum())

print("\nALL OBSERVATIONS ABOVE 1")

print(
    df.loc[
        df["capacity_factor"] > 1,
        [
            "plant_id", "year", "plant_name",
            "capacity_mw", "generation_mwh",
            "capacity_factor", "ngcc_capacity_share"
        ]
    ]
    .sort_values("capacity_factor", ascending=False)
    .to_string(index=False)
)

# --------------------------------------------------
# 2. Capacity factor by NGCC technology purity
# --------------------------------------------------

df["purity_group"] = np.select(
    [
        df["ngcc_capacity_share"] >= 0.95,
        df["ngcc_capacity_share"] >= 0.80,
        df["ngcc_capacity_share"] >= 0.50
    ],
    [
        "At least 95%",
        "80-95%",
        "50-80%"
    ],
    default="Below 50%"
)

print("\nCAPACITY FACTOR BY NGCC PURITY")

print(
    df.groupby("purity_group")["capacity_factor"]
      .agg(["count", "mean", "median", "max"])
      .round(4)
)

# --------------------------------------------------
# 3. Distribution of anomalous observations
# --------------------------------------------------

print("\nOBSERVATIONS WITH CAPACITY FACTOR ABOVE 1")

print(
    df.loc[df["capacity_factor"] > 1]
      .groupby("purity_group")
      .size()
)

# --------------------------------------------------
# 4. Candidate homogeneous NGCC sample
# --------------------------------------------------

pure = df[
    df["ngcc_capacity_share"] >= 0.95
].copy()

print("\nCANDIDATE PURE-NGCC SAMPLE")

print("Observations:", len(pure))
print("Plants:", pure["plant_id"].nunique())
print("Years:", sorted(pure["year"].unique()))

print(
    "Capacity factor above 1:",
    (pure["capacity_factor"] > 1).sum()
)

print("\nPURE-NGCC CAPACITY FACTOR DISTRIBUTION")

print(
    pure["capacity_factor"].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    ).round(4)
)

# --------------------------------------------------
# 5. Save candidate sample
# --------------------------------------------------

output_path = os.path.join(
    PROC,
    "ngcc_pure_candidate_panel.csv"
)

pure.to_csv(output_path, index=False)

print("\nSaved candidate sample:")
print(output_path)


In [ ]:

import pandas as pd
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_pure_candidate_panel.csv")
)

# Identify observations with capacity factors above 1
anomalies = df[df["capacity_factor"] > 1].copy()

cols = [
    "plant_id",
    "year",
    "plant_name",
    "capacity_mw",
    "generation_mwh",
    "fuel_consumption_mmbtu",
    "capacity_factor",
    "heat_rate_mmbtu_per_mwh",
    "ngcc_capacity_share",
    "total_operating_capacity_mw"
]

print("NUMBER OF ANOMALOUS OBSERVATIONS:", len(anomalies))

print("\nDETAILS")
print(
    anomalies[cols]
    .sort_values("capacity_factor", ascending=False)
    .to_string(index=False)
)

# Check whether anomalies persist across multiple years
print("\nANOMALIES BY PLANT")

print(
    anomalies.groupby(["plant_id", "plant_name"])
    .agg(
        years=("year", lambda x: sorted(x.tolist())),
        observations=("year", "count"),
        maximum_capacity_factor=("capacity_factor", "max"),
        median_heat_rate=("heat_rate_mmbtu_per_mwh", "median")
    )
    .to_string()
)

# Save the diagnostic
anomalies[cols].to_csv(
    os.path.join(PROC, "ngcc_capacity_factor_anomalies.csv"),
    index=False
)

print("\nSaved ngcc_capacity_factor_anomalies.csv")


In [ ]:

import pandas as pd
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_pure_candidate_panel.csv")
)

before = len(df)

# Exclude Dearborn across all years
df = df[df["plant_id"] != 62289].copy()

# Exclude CPV Fairview in 2025 only
df = df[
    ~((df["plant_id"] == 60589) & (df["year"] == 2025))
].copy()

# Retain Oregon State University and other observations
df = df.sort_values(["plant_id", "year"]).reset_index(drop=True)

print("PROVISIONAL PRIMARY SAMPLE")
print("Original observations:", before)
print("Final observations:", len(df))
print("Excluded observations:", before - len(df))
print("Unique plants:", df["plant_id"].nunique())
print("Years:", sorted(df["year"].unique()))

print("\nOBSERVATIONS PER YEAR")
print(df.groupby("year").size())

print("\nCAPACITY FACTOR DISTRIBUTION")
print(
    df["capacity_factor"].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    ).round(4)
)

print("\nOBSERVATIONS WITH CAPACITY FACTOR ABOVE 1.01")
print(
    df.loc[
        df["capacity_factor"] > 1.01,
        ["plant_id", "year", "plant_name", "capacity_factor"]
    ].to_string(index=False)
)

print("\nHEAT RATE DISTRIBUTION")
print(
    df["heat_rate_mmbtu_per_mwh"].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    ).round(4)
)

output = os.path.join(PROC, "ngcc_primary_panel.csv")
df.to_csv(output, index=False)

print("\nSaved:", output)


In [ ]:

import pandas as pd
import numpy as np
import os
from scipy.optimize import linprog

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# Load frozen provisional sample
df = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
)

# DEA variables
input_cols = [
    "capacity_mw",
    "fuel_consumption_mmbtu"
]
output_col = "generation_mwh"

# Check data
required = input_cols + [output_col, "plant_id", "year"]

assert all(c in df.columns for c in required), \
    "A required DEA variable is missing."

assert (df[input_cols + [output_col]] > 0).all().all(), \
    "DEA inputs and output must be positive."

# Store efficiency results
df["nominal_vrs_efficiency"] = np.nan
df["dea_status"] = ""

# --------------------------------------------------
# Annual input-oriented VRS DEA
# --------------------------------------------------

for year in sorted(df["year"].unique()):

    idx = df.index[df["year"] == year]
    data = df.loc[idx].copy()

    X = data[input_cols].to_numpy(dtype=float)
    Y = data[[output_col]].to_numpy(dtype=float)

    n, m = X.shape
    s = Y.shape[1]

    # Scale variables by annual medians.
    # This does not change DEA efficiency scores.
    X = X / np.median(X, axis=0)
    Y = Y / np.median(Y, axis=0)

    # Input constraints:
    # X_peer' lambda - theta*x_o <= 0
    # Output constraints:
    # -Y_peer' lambda <= -y_o
    A_input = np.hstack([
        X.T,
        -X.T * 0  # placeholder replaced below
    ])

    # Build constraints separately for clarity
    A1 = np.hstack([
        X.T,
        np.zeros((m, 1))
    ])

    A2 = np.hstack([
        -Y.T,
        np.zeros((s, 1))
    ])

    A_eq = np.zeros((1, n + 1))
    A_eq[0, :n] = 1.0

    b_eq = np.array([1.0])

    bounds = [(0, None)] * n + [(0, None)]

    scores = []

    for i in range(n):

        # Input constraints depend on the evaluated DMU
        A1_i = A1.copy()
        A1_i[:, -1] = -X[i, :]

        A_ub = np.vstack([A1_i, A2])

        b_ub = np.concatenate([
            np.zeros(m),
            -Y[i, :]
        ])

        # Minimize theta
        c = np.zeros(n + 1)
        c[-1] = 1.0

        result = linprog(
            c,
            A_ub=A_ub,
            b_ub=b_ub,
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=bounds,
            method="highs"
        )

        if result.success:
            scores.append(result.x[-1])
        else:
            scores.append(np.nan)

    df.loc[idx, "nominal_vrs_efficiency"] = scores

    print(
        f"Year {year}: "
        f"{n} DMUs, "
        f"{np.isfinite(scores).sum()} successful solutions"
    )

# --------------------------------------------------
# Results
# --------------------------------------------------

print("\nNOMINAL VRS-DEA SUMMARY")

print(
    df["nominal_vrs_efficiency"].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    ).round(4)
)

print("\nANNUAL EFFICIENCY SUMMARY")

print(
    df.groupby("year")["nominal_vrs_efficiency"]
      .agg(["count", "mean", "std", "median", "min", "max"])
      .round(4)
)

print("\nEFFICIENT DMUS BY YEAR")

print(
    df.assign(
        efficient=df["nominal_vrs_efficiency"] >= 0.99999
    )
    .groupby("year")["efficient"]
    .agg(["sum", "count", "mean"])
    .round(4)
)

# Save results
output = os.path.join(
    PROC,
    "ngcc_nominal_vrs_dea.csv"
)

df.to_csv(output, index=False)

print("\nSaved:", output)


In [ ]:

import pandas as pd
import numpy as np
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# --------------------------------------------------
# 1. Download monthly Henry Hub natural gas prices
# --------------------------------------------------

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MHHNGSP"

gas = pd.read_csv(url)

gas.columns = ["date", "gas_price"]

gas["date"] = pd.to_datetime(gas["date"], errors="coerce")
gas["gas_price"] = pd.to_numeric(
    gas["gas_price"], errors="coerce"
)

gas = gas.dropna()
gas = gas[gas["gas_price"] > 0].copy()

gas["year"] = gas["date"].dt.year

gas.to_csv(
    os.path.join(PROC, "henry_hub_monthly_prices.csv"),
    index=False
)

print("MONTHLY GAS PRICE DATA")
print("Observations:", len(gas))
print("Date range:", gas["date"].min(), "to", gas["date"].max())

# --------------------------------------------------
# 2. Construct annual gas-price indicators
# --------------------------------------------------

annual = (
    gas.groupby("year")
       .agg(
           annual_gas_price=("gas_price", "mean"),
           gas_price_sd=("gas_price", "std"),
           gas_price_max=("gas_price", "max"),
           months_available=("gas_price", "count")
       )
       .reset_index()
       .sort_values("year")
)

annual["gas_price_cv"] = (
    annual["gas_price_sd"] / annual["annual_gas_price"]
)

# Three-year historical reference, excluding event year
annual["historical_price_baseline"] = (
    annual["annual_gas_price"]
    .shift(1)
    .rolling(window=3, min_periods=3)
    .mean()
)

# Positive price stress only
annual["gas_event_severity"] = np.maximum(
    0,
    np.log(
        annual["annual_gas_price"] /
        annual["historical_price_baseline"]
    )
)

print("\nANNUAL GAS PRICE INDICATORS")
print(
    annual[
        [
            "year",
            "annual_gas_price",
            "historical_price_baseline",
            "gas_event_severity",
            "gas_price_cv",
            "months_available"
        ]
    ].round(4).to_string(index=False)
)

annual.to_csv(
    os.path.join(PROC, "annual_gas_price_events.csv"),
    index=False
)

# --------------------------------------------------
# 3. Shift event information to the following year
# --------------------------------------------------

# Event severity in year t-1 is available for decisions in year t.
gas_lag = annual[
    ["year", "annual_gas_price", "gas_event_severity", "gas_price_cv"]
].copy()

gas_lag["year"] = gas_lag["year"] + 1

gas_lag = gas_lag.rename(
    columns={
        "annual_gas_price": "previous_year_gas_price",
        "gas_event_severity": "lagged_gas_event_severity",
        "gas_price_cv": "lagged_gas_price_cv"
    }
)

# --------------------------------------------------
# 4. Merge with the primary plant panel
# --------------------------------------------------

panel = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
)

panel["plant_id"] = panel["plant_id"].astype(int)
panel["year"] = panel["year"].astype(int)

panel = panel.sort_values(
    ["plant_id", "year"]
).copy()

panel = panel.merge(
    gas_lag,
    on="year",
    how="left"
)

# --------------------------------------------------
# 5. Construct strictly lagged plant exposure
# --------------------------------------------------

panel["lagged_heat_rate"] = (
    panel.groupby("plant_id")["heat_rate_mmbtu_per_mwh"]
         .shift(1)
)

# Normalize exposure within each target year
year_median_exposure = (
    panel.groupby("year")["lagged_heat_rate"]
         .transform("median")
)

panel["gas_exposure"] = (
    panel["lagged_heat_rate"] / year_median_exposure
)

# Preliminary gas-related EDRO signal
panel["gas_event_exposure"] = (
    panel["lagged_gas_event_severity"] *
    panel["gas_exposure"]
)

# --------------------------------------------------
# 6. Diagnostics
# --------------------------------------------------

print("\nMERGED EVENT-EXPOSURE PANEL")

print(
    panel[
        [
            "year",
            "lagged_gas_event_severity",
            "gas_exposure",
            "gas_event_exposure"
        ]
    ].describe().round(4)
)

print("\nANNUAL EVENT AND EXPOSURE SUMMARY")

print(
    panel.groupby("year").agg(
        observations=("plant_id", "size"),
        gas_stress=("lagged_gas_event_severity", "first"),
        available_exposures=("gas_exposure", "count"),
        median_exposure=("gas_exposure", "median"),
        mean_event_exposure=("gas_event_exposure", "mean")
    ).round(4)
)

print("\nMISSING LAGGED EXPOSURES BY YEAR")

print(
    panel.groupby("year")["gas_exposure"]
         .apply(lambda x: x.isna().sum())
)

# Save
output = os.path.join(
    PROC,
    "ngcc_gas_event_exposure_panel.csv"
)

panel.to_csv(output, index=False)

print("\nSaved:", output)


In [ ]:
import pandas as pd

base = "https://www.ncei.noaa.gov/data/nclimgrid-daily/access/averages/2025/"

for variable in ["tmax", "tmin"]:
    filename = f"{variable}-202501-ste-scaled.csv"
    url = base + filename

    print("\n", filename)
    df = pd.read_csv(url)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(8).to_string(index=False))

In [ ]:
import requests
import pandas as pd
from io import StringIO

BASE_URLS = [
    "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/",
    "https://noaa-nclimgrid-daily-pds.s3.us-east-1.amazonaws.com/",
    "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/EpiNOAA/csv/"
]

filename = "max-202501-ste-scaled.csv"

for base in BASE_URLS:
    url = base + filename
    try:
        r = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        print("\nURL:", url)
        print("HTTP:", r.status_code)
        print("Content type:", r.headers.get("Content-Type"))
        print("Bytes:", len(r.content))

        if r.status_code == 200 and len(r.content) > 100:
            df = pd.read_csv(StringIO(r.text), header=None)
            print("SUCCESS")
            print("Shape:", df.shape)
            print(df.head(3).to_string(index=False, header=False))
            break

    except Exception as e:
        print("ERROR:", str(e))
else:
    print("\nNone of these URLs worked.")

In [ ]:
import requests
import xml.etree.ElementTree as ET

BUCKET = "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/"

def list_s3(prefix="", token=None):
    params = {
        "list-type": "2",
        "prefix": prefix,
        "max-keys": "1000"
    }
    if token:
        params["continuation-token"] = token

    r = requests.get(BUCKET, params=params, timeout=60)
    print("HTTP:", r.status_code, "| Prefix:", prefix)
    r.raise_for_status()

    root = ET.fromstring(r.content)
    ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}

    keys = [
        x.findtext("s:Key", namespaces=ns)
        for x in root.findall(".//s:Contents", ns)
    ]
    prefixes = [
        x.findtext("s:Prefix", namespaces=ns)
        for x in root.findall(".//s:CommonPrefixes", ns)
    ]

    return root, keys, prefixes

# Inspect the bucket's top-level folders
root, keys, prefixes = list_s3()

print("\nTOP-LEVEL FOLDERS:")
for p in prefixes:
    print(p)

print("\nTOP-LEVEL FILES:")
for k in keys[:20]:
    print(k)

# Inspect the documented EpiNOAA CSV folder
root, keys, prefixes = list_s3("EpiNOAA/csv/")

print("\nEpiNOAA CSV FILES:", len(keys))
for k in keys[:30]:
    print(k)

print("\nFOLDERS:")
for p in prefixes:
    print(p)

# Search for the exact 2025 file
matches = [k for k in keys if "202501" in k and "ste" in k.lower()]
print("\nMATCHING JANUARY 2025 STATE FILES:")
for k in matches:
    print(BUCKET + k)

In [ ]:
import requests
import xml.etree.ElementTree as ET

url = "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/"

params = {
    "list-type": "2",
    "prefix": "EpiNOAA/v1-0-0/csv/",
    "delimiter": "/",
    "max-keys": "1000"
}

r = requests.get(url, params=params, timeout=60)
print("HTTP:", r.status_code)
r.raise_for_status()

root = ET.fromstring(r.content)
ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}

folders = [
    x.findtext("s:Prefix", namespaces=ns)
    for x in root.findall(".//s:CommonPrefixes", ns)
]

print("Available NOAA data folders:")
for folder in folders:
    print(folder)

print("\nFiles directly in this directory:")
for item in root.findall(".//s:Contents", ns):
    print(item.findtext("s:Key", namespaces=ns))

In [ ]:
import requests
import xml.etree.ElementTree as ET

BUCKET = "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/"

params = {
    "list-type": "2",
    "prefix": "EpiNOAA/v1-0-0/csv/ste/",
    "max-keys": "1000"
}

r = requests.get(BUCKET, params=params, timeout=60)
print("HTTP:", r.status_code)
r.raise_for_status()

root = ET.fromstring(r.content)
ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}

keys = [
    x.findtext("s:Key", namespaces=ns)
    for x in root.findall(".//s:Contents", ns)
]

print("Number of files:", len(keys))
print("\nFiles containing 202501:")
for k in keys:
    if "202501" in k:
        print(k)

print("\nFirst 25 actual filenames:")
for k in keys[:25]:
    print(k)

print("\nLast 25 actual filenames:")
for k in keys[-25:]:
    print(k)

print("\nMore files available:", 
      root.findtext("s:IsTruncated", namespaces=ns))

In [ ]:
import requests
import pandas as pd
from io import BytesIO

url = (
    "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/"
    "EpiNOAA/v1-0-0/csv/ste/202501-scaled.csv"
)

r = requests.get(url, timeout=60)
print("HTTP status:", r.status_code)
r.raise_for_status()

# NOAA files may be headerless; inspect both interpretations.
df = pd.read_csv(BytesIO(r.content), header=None)

print("\nShape:", df.shape)
print("\nFirst 12 rows:")
display(df.head(12))

print("\nLast 5 rows:")
display(df.tail(5))

print("\nFirst 2500 characters of raw file:")
print(r.text[:2500])

In [ ]:
import os
import requests
import pandas as pd
from io import BytesIO

# ------------------------------------------------------------
# 1. Paths and download settings
# ------------------------------------------------------------
BASE = "/datasets/_deepnote_work/eia_edro_project"
OUT = os.path.join(BASE, "processed")
os.makedirs(OUT, exist_ok=True)

BUCKET = (
    "https://noaa-nclimgrid-daily-pds.s3.amazonaws.com/"
    "EpiNOAA/v1-0-0/csv/ste/"
)

YEARS = range(2019, 2026)

session = requests.Session()
frames = []
failed = []

# ------------------------------------------------------------
# 2. Download monthly state-level daily temperature files
# ------------------------------------------------------------
for year in YEARS:
    for month in range(1, 13):
        filename = f"{year}{month:02d}-scaled.csv"
        url = BUCKET + filename

        try:
            response = session.get(url, timeout=90)
            response.raise_for_status()

            df = pd.read_csv(BytesIO(response.content))

            required = {
                "region_type", "fips", "state_name",
                "postal_code", "date", "tmax", "tmin"
            }

            missing = required.difference(df.columns)
            if missing:
                raise ValueError(
                    f"Missing columns {sorted(missing)}; "
                    f"found {list(df.columns)}"
                )

            df = df[
                [
                    "region_type", "fips", "state_name",
                    "postal_code", "date", "tmax", "tmin"
                ]
            ].copy()

            df["date"] = pd.to_datetime(df["date"])
            df["year"] = df["date"].dt.year
            df["month"] = df["date"].dt.month

            df["tmax"] = pd.to_numeric(df["tmax"], errors="coerce")
            df["tmin"] = pd.to_numeric(df["tmin"], errors="coerce")

            frames.append(df)

        except Exception as e:
            failed.append((filename, str(e)))

    print(f"Completed {year}")

# ------------------------------------------------------------
# 3. Combine and validate
# ------------------------------------------------------------
if failed:
    print("\nFailed downloads:")
    for item in failed:
        print(item)

if not frames:
    raise RuntimeError("No NOAA temperature files were downloaded.")

temperature = pd.concat(frames, ignore_index=True)

temperature = temperature.drop_duplicates(
    subset=["postal_code", "date"]
)

temperature = temperature.sort_values(
    ["postal_code", "date"]
).reset_index(drop=True)

# Basic validity checks
temperature = temperature[
    temperature["date"].between("2019-01-01", "2025-12-31")
].copy()

temperature["temperature_range_check"] = (
    temperature["tmax"] >= temperature["tmin"]
)

print("\nDATA VALIDATION")
print("----------------------------")
print("Rows:", len(temperature))
print("States:", temperature["postal_code"].nunique())
print("Date range:", temperature["date"].min(), "to",
      temperature["date"].max())
print("Missing Tmax:", temperature["tmax"].isna().sum())
print("Missing Tmin:", temperature["tmin"].isna().sum())
print("Tmax below Tmin:",
      (~temperature["temperature_range_check"]).sum())
print("Duplicate state-date pairs:",
      temperature.duplicated(["postal_code", "date"]).sum())

print("\nObservations by year:")
print(temperature.groupby("year").size())

print("\nObservations by state:")
print(
    temperature.groupby("postal_code")["date"]
    .nunique()
    .sort_values()
)

# ------------------------------------------------------------
# 4. Save daily dataset
# ------------------------------------------------------------
daily_file = os.path.join(
    OUT, "state_daily_temperature_2019_2025.csv"
)

temperature.drop(
    columns=["temperature_range_check"]
).to_csv(daily_file, index=False)

print("\nSaved:", daily_file)

# ------------------------------------------------------------
# 5. Create annual state-level summary
# ------------------------------------------------------------
annual_temperature = (
    temperature.groupby(["postal_code", "state_name", "year"])
    .agg(
        mean_tmax=("tmax", "mean"),
        mean_tmin=("tmin", "mean"),
        p95_tmax=("tmax", lambda x: x.quantile(0.95)),
        max_tmax=("tmax", "max"),
        observed_days=("date", "nunique")
    )
    .reset_index()
)

annual_file = os.path.join(
    OUT, "state_annual_temperature_summary_2019_2025.csv"
)

annual_temperature.to_csv(annual_file, index=False)

print("\nAnnual summary saved:", annual_file)
print("\nAnnual summary preview:")
display(annual_temperature.head(15))

In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# ------------------------------------------------------------
# 1. Load daily temperature data
# ------------------------------------------------------------
temp_file = os.path.join(
    PROC, "state_daily_temperature_2019_2025.csv"
)
temp = pd.read_csv(temp_file, parse_dates=["date"])

temp["year"] = temp["date"].dt.year
temp["tmax"] = pd.to_numeric(temp["tmax"], errors="coerce")

# ------------------------------------------------------------
# 2. Calculate state-specific, strictly historical thresholds
#    For event year t, use only observations from t-1, t-2, t-3.
# ------------------------------------------------------------
records = []

for state, sdf in temp.groupby("postal_code"):
    sdf = sdf.sort_values("date").copy()

    for year in range(2020, 2026):
        history = sdf[
            (sdf["year"] >= max(2019, year - 3)) &
            (sdf["year"] < year)
        ]["tmax"].dropna()

        current = sdf[sdf["year"] == year]["tmax"].dropna()

        if len(history) < 300 or len(current) == 0:
            continue

        threshold = history.quantile(0.90)
        exceedance = (current - threshold).clip(lower=0)

        records.append({
            "postal_code": state,
            "state_name": sdf["state_name"].iloc[0],
            "year": year,
            "historical_tmax_p90": threshold,
            "heat_event_days": int((current > threshold).sum()),
            "heat_degree_days": float(exceedance.sum()),
            "heat_event_severity": float(exceedance.sum() / len(current)),
            "observed_days": int(len(current))
        })

heat_events = pd.DataFrame(records)

# ------------------------------------------------------------
# 3. Add one-year lag
#    Plant-year t receives the event severity from year t-1.
# ------------------------------------------------------------
heat_events = heat_events.sort_values(
    ["postal_code", "year"]
)

heat_events["lagged_heat_event_severity"] = (
    heat_events.groupby("postal_code")["heat_event_severity"].shift(1)
)

heat_events["lagged_heat_event_days"] = (
    heat_events.groupby("postal_code")["heat_event_days"].shift(1)
)

heat_events["lagged_heat_degree_days"] = (
    heat_events.groupby("postal_code")["heat_degree_days"].shift(1)
)

# Save annual event data
event_file = os.path.join(
    PROC, "state_annual_temperature_events.csv"
)
heat_events.to_csv(event_file, index=False)

# ------------------------------------------------------------
# 4. Merge lagged event severity into plant-year panel
# ------------------------------------------------------------
panel_file = os.path.join(PROC, "ngcc_primary_panel.csv")
panel = pd.read_csv(panel_file)

# Identify the plant-state column
state_candidates = [
    "postal_code", "state_abbr", "state_code",
    "state", "state_name"
]

state_col = next(
    (c for c in state_candidates if c in panel.columns),
    None
)

if state_col is None:
    raise ValueError(
        "Could not identify the state column. "
        f"Panel columns are: {list(panel.columns)}"
    )

if "year" not in panel.columns:
    raise ValueError("The plant panel has no 'year' column.")

# Convert state names to postal abbreviations where necessary
name_to_code = (
    temp[["state_name", "postal_code"]]
    .drop_duplicates()
    .set_index("state_name")["postal_code"]
    .to_dict()
)

panel["postal_code"] = panel[state_col].astype(str).str.strip().str.upper()

panel["postal_code"] = panel["postal_code"].map(
    lambda x: name_to_code.get(x.title(), x)
)

panel["year"] = pd.to_numeric(
    panel["year"], errors="coerce"
).astype("Int64")

# Merge lagged state event information
merge_cols = [
    "postal_code", "year",
    "lagged_heat_event_severity",
    "lagged_heat_event_days",
    "lagged_heat_degree_days"
]

panel = panel.merge(
    heat_events[merge_cols],
    on=["postal_code", "year"],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 5. Validation and save
# ------------------------------------------------------------
print("ANNUAL STATE HEAT EVENTS")
print("------------------------")
print("Rows:", len(heat_events))
print("States:", heat_events["postal_code"].nunique())
print("Years:", sorted(heat_events["year"].unique()))
display(
    heat_events.groupby("year").agg(
        states=("postal_code", "nunique"),
        mean_severity=("heat_event_severity", "mean"),
        median_severity=("heat_event_severity", "median"),
        mean_hot_days=("heat_event_days", "mean"),
        max_severity=("heat_event_severity", "max")
    ).round(4)
)

print("\nPLANT-YEAR MERGE")
print("----------------")
print("Plant-year observations:", len(panel))
print("Matched lagged heat events:",
      panel["lagged_heat_event_severity"].notna().sum())
print("Missing lagged heat events:",
      panel["lagged_heat_event_severity"].isna().sum())

print("\nMatched observations by year:")
print(
    panel.groupby("year")["lagged_heat_event_severity"]
    .agg(["count", "mean", "median"])
)

merged_file = os.path.join(
    PROC, "ngcc_temperature_event_exposure_panel.csv"
)
panel.to_csv(merged_file, index=False)

print("\nSaved annual events:", event_file)
print("Saved plant panel:", merged_file)


In [ ]:
import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

temp_panel = pd.read_csv(
    os.path.join(PROC, "ngcc_temperature_event_exposure_panel.csv")
)

gas_panel = pd.read_csv(
    os.path.join(PROC, "ngcc_gas_event_exposure_panel.csv")
)

# ------------------------------------------------------------
# 1. Identify temperature-unmatched plants
# ------------------------------------------------------------
missing_temp = temp_panel[
    temp_panel["lagged_heat_event_severity"].isna()
].copy()

print("TEMPERATURE-UNMATCHED OBSERVATIONS")
print("----------------------------------")
print("Rows:", len(missing_temp))
print("By year:")
print(missing_temp.groupby("year").size())

state_col = "postal_code"
if state_col in missing_temp.columns:
    print("\nUnmatched states:")
    print(missing_temp[state_col].value_counts(dropna=False))

id_col = "plant_id" if "plant_id" in missing_temp.columns else None
if id_col:
    print("\nUnmatched plants:")
    cols = [id_col, "year"]
    for c in ["plant_name", state_col]:
        if c in missing_temp.columns:
            cols.append(c)
    display(
        missing_temp[cols]
        .drop_duplicates()
        .sort_values(["year", id_col])
    )

# ------------------------------------------------------------
# 2. Inspect gas panel structure
# ------------------------------------------------------------
print("\nGAS PANEL")
print("---------")
print("Rows:", len(gas_panel))
print("Columns:", list(gas_panel.columns))

print("\nPotential event/exposure columns:")
print([
    c for c in gas_panel.columns
    if any(s in c.lower() for s in
           ["gas", "event", "exposure", "severity", "heat"])
])

# ------------------------------------------------------------
# 3. Check plant-year key compatibility
# ------------------------------------------------------------
keys = ["plant_id", "year"]

if all(c in temp_panel.columns and c in gas_panel.columns for c in keys):
    temp_keys = temp_panel[keys].drop_duplicates()
    gas_keys = gas_panel[keys].drop_duplicates()

    print("\nPLANT-YEAR KEY CHECK")
    print("-------------------")
    print("Temperature unique keys:", len(temp_keys))
    print("Gas unique keys:", len(gas_keys))
    print("Temperature keys absent from gas:",
          len(temp_keys.merge(gas_keys, on=keys, how="left",
                              indicator=True).query("_merge == 'left_only'")))
    print("Gas keys absent from temperature:",
          len(gas_keys.merge(temp_keys, on=keys, how="left",
                             indicator=True).query("_merge == 'left_only'")))
else:
    print("\nCould not check keys. Verify plant identifier columns.")

In [ ]:
import pandas as pd
import os

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

df = pd.read_csv(
    os.path.join(PROC, "ngcc_temperature_event_exposure_panel.csv")
)

print("Alaska observations:")
display(
    df.loc[
        (df["postal_code"] == "AK") &
        (df["year"] >= 2021),
        ["plant_id", "plant_name", "postal_code", "year",
         "lagged_heat_event_severity"]
    ].sort_values(["plant_id", "year"])
)

print("\nAll observations with missing lagged heat severity after 2020:")
display(
    df.loc[
        (df["year"] >= 2021) &
        df["lagged_heat_event_severity"].isna(),
        ["plant_id", "plant_name", "postal_code", "year"]
    ].sort_values(["postal_code", "plant_id", "year"])
)

In [ ]:

import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

# Load the two panels
temp = pd.read_csv(
    os.path.join(PROC, "ngcc_temperature_event_exposure_panel.csv")
)

gas = pd.read_csv(
    os.path.join(PROC, "ngcc_gas_event_exposure_panel.csv")
)

# Retain only the gas variables needed for the combined panel
gas_cols = [
    "plant_id", "year",
    "lagged_gas_event_severity",
    "lagged_heat_rate",
    "gas_exposure"
]

missing_cols = [c for c in gas_cols if c not in gas.columns]
if missing_cols:
    raise ValueError(f"Missing gas-panel columns: {missing_cols}")

gas_small = gas[gas_cols].copy()

# Merge on plant-year
panel = temp.merge(
    gas_small,
    on=["plant_id", "year"],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 1. Define a common plant exposure proxy
# ------------------------------------------------------------
# Use the existing gas exposure calculation:
# lagged plant heat rate / annual median lagged plant heat rate.
#
# For plants without a lagged heat rate, use neutral exposure = 1.
# This avoids treating unavailable plant history as zero exposure.

panel["plant_exposure"] = (
    pd.to_numeric(panel["gas_exposure"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
)

# ------------------------------------------------------------
# 2. Construct comparable event-exposure scores
# ------------------------------------------------------------
panel["gas_event_exposure_final"] = (
    pd.to_numeric(
        panel["lagged_gas_event_severity"], errors="coerce"
    ) * panel["plant_exposure"]
)

panel["heat_event_exposure_final"] = (
    pd.to_numeric(
        panel["lagged_heat_event_severity"], errors="coerce"
    ) * panel["plant_exposure"]
)

# ------------------------------------------------------------
# 3. Define primary evaluation sample
# ------------------------------------------------------------
# Exclude 2020 because lagged temperature events are unavailable.
# Exclude Alaska because this NOAA dataset covers the contiguous US.

panel["year"] = pd.to_numeric(panel["year"], errors="coerce")

evaluation = panel[
    (panel["year"] >= 2021) &
    (panel["postal_code"] != "AK")
].copy()

# Require both event signals to be available
evaluation = evaluation.dropna(
    subset=[
        "gas_event_exposure_final",
        "heat_event_exposure_final"
    ]
)

# ------------------------------------------------------------
# 4. Validation
# ------------------------------------------------------------
print("COMBINED EVENT-EXPOSURE PANEL")
print("--------------------------------")
print("Full plant-year panel:", len(panel))
print("Evaluation observations:", len(evaluation))
print("Unique plants:", evaluation["plant_id"].nunique())
print("Years:", sorted(evaluation["year"].unique()))
print("States:", evaluation["postal_code"].nunique())

print("\nObservations by year:")
print(evaluation.groupby("year").size())

print("\nEvent exposure summary:")
display(
    evaluation[
        [
            "gas_event_exposure_final",
            "heat_event_exposure_final",
            "plant_exposure"
        ]
    ].describe().round(4)
)

print("\nAnnual event exposure:")
display(
    evaluation.groupby("year").agg(
        observations=("plant_id", "size"),
        mean_gas_exposure=("gas_event_exposure_final", "mean"),
        mean_heat_exposure=("heat_event_exposure_final", "mean"),
        median_gas_exposure=("gas_event_exposure_final", "median"),
        median_heat_exposure=("heat_event_exposure_final", "median")
    ).round(4)
)

# ------------------------------------------------------------
# 5. Save both full and evaluation panels
# ------------------------------------------------------------
full_out = os.path.join(
    PROC, "ngcc_gas_temperature_event_panel.csv"
)

eval_out = os.path.join(
    PROC, "ngcc_gas_temperature_event_evaluation.csv"
)

panel.to_csv(full_out, index=False)
evaluation.to_csv(eval_out, index=False)

print("\nSaved full panel:", full_out)
print("Saved evaluation panel:", eval_out)


In [ ]:

import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

INPUT = os.path.join(
    PROC, "ngcc_gas_temperature_event_evaluation.csv"
)

df = pd.read_csv(INPUT)

GAMMA = 3.0

groups = {
    "gas": "gas_event_exposure_final",
    "heat": "heat_event_exposure_final"
}

# ------------------------------------------------------------
# 1. Normalize each event group by its annual cross-sectional
#    95th percentile, with an upper cap of one.
# ------------------------------------------------------------
for group, col in groups.items():
    df[col] = pd.to_numeric(df[col], errors="coerce").clip(lower=0)

    q95 = df.groupby("year")[col].transform(
        lambda x: x.quantile(0.95)
    )

    df[f"{group}_normalized"] = np.where(
        q95 > 0,
        (df[col] / q95).clip(upper=1),
        0.0
    )

# ------------------------------------------------------------
# 2. Calculate annual group intensities and budget shares
# ------------------------------------------------------------
annual = (
    df.groupby("year")
    .agg(
        observations=("plant_id", "size"),
        gas_intensity=("gas_normalized", "mean"),
        heat_intensity=("heat_normalized", "mean"),
        gas_raw_mean=("gas_event_exposure_final", "mean"),
        heat_raw_mean=("heat_event_exposure_final", "mean")
    )
    .reset_index()
)

total_intensity = (
    annual["gas_intensity"] + annual["heat_intensity"]
)

annual["gas_budget_share"] = np.where(
    total_intensity > 0,
    annual["gas_intensity"] / total_intensity,
    0.5
)

annual["heat_budget_share"] = np.where(
    total_intensity > 0,
    annual["heat_intensity"] / total_intensity,
    0.5
)

annual["Gamma_gas"] = GAMMA * annual["gas_budget_share"]
annual["Gamma_heat"] = GAMMA * annual["heat_budget_share"]

# ------------------------------------------------------------
# 3. Validate the allocation
# ------------------------------------------------------------
annual["Gamma_total"] = (
    annual["Gamma_gas"] + annual["Gamma_heat"]
)

print("EDRO ANNUAL BUDGET ALLOCATION")
print("--------------------------------")
print(annual.round(4).to_string(index=False))

print("\nValidation:")
print("Total budget:", GAMMA)
print("Maximum budget deviation:",
      (annual["Gamma_total"] - GAMMA).abs().max())

print("\nGas budget range:",
      annual["Gamma_gas"].min(),
      annual["Gamma_gas"].max())

print("Heat budget range:",
      annual["Gamma_heat"].min(),
      annual["Gamma_heat"].max())

# ------------------------------------------------------------
# 4. Save annual allocation and plant-level normalized scores
# ------------------------------------------------------------
annual.to_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv"),
    index=False
)

df.to_csv(
    os.path.join(PROC, "ngcc_edro_normalized_exposure_panel.csv"),
    index=False
)

print("\nSaved:")
print(os.path.join(PROC, "edro_annual_budget_allocation.csv"))
print(os.path.join(PROC, "ngcc_edro_normalized_exposure_panel.csv"))


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# ------------------------------------------------------------
# 1. Load the plant-year panel and annual gas prices
# ------------------------------------------------------------
plants = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
)

prices = pd.read_csv(
    os.path.join(PROC, "annual_gas_price_events.csv")
)

print("Plant panel columns:")
print(plants.columns.tolist())

print("\nAnnual price columns:")
print(prices.columns.tolist())

# Identify annual price column
price_candidates = [
    "annual_gas_price",
    "annual_mean_price",
    "mean_gas_price",
    "gas_price",
    "price"
]

price_col = next(
    (c for c in price_candidates if c in prices.columns),
    None
)

if price_col is None:
    raise ValueError(
        "Annual gas price column not identified. "
        f"Available columns: {prices.columns.tolist()}"
    )

required_plant_cols = [
    "plant_id", "year", "capacity_mw",
    "fuel_consumption_mmbtu", "generation_mwh"
]

missing = [
    c for c in required_plant_cols
    if c not in plants.columns
]

if missing:
    raise ValueError(f"Missing plant columns: {missing}")

if "year" not in prices.columns:
    raise ValueError("Annual price file has no year column.")

# ------------------------------------------------------------
# 2. Prepare and validate annual prices
# ------------------------------------------------------------
prices["year"] = pd.to_numeric(
    prices["year"], errors="coerce"
)

prices[price_col] = pd.to_numeric(
    prices[price_col], errors="coerce"
)

prices = prices[
    ["year", price_col]
].dropna()

prices = prices.rename(
    columns={price_col: "henry_hub_price_usd_per_mmbtu"}
)

prices = prices.drop_duplicates("year")

if (prices["henry_hub_price_usd_per_mmbtu"] <= 0).any():
    raise ValueError("Gas prices must be strictly positive.")

# ------------------------------------------------------------
# 3. Merge prices into plant-year panel
# ------------------------------------------------------------
plants["year"] = pd.to_numeric(
    plants["year"], errors="coerce"
)

df = plants.merge(
    prices,
    on="year",
    how="left",
    validate="many_to_one",
    indicator=True
)

unmatched = df[df["_merge"] != "both"]

if len(unmatched):
    print("\nUnmatched plant-year observations:")
    print(unmatched[["plant_id", "year"]].to_string(index=False))
    raise ValueError("Some plant-year observations lack annual gas prices.")

df = df.drop(columns="_merge")

# ------------------------------------------------------------
# 4. Construct the fuel-cost input
# ------------------------------------------------------------
df["fuel_cost_usd_proxy"] = (
    df["fuel_consumption_mmbtu"]
    * df["henry_hub_price_usd_per_mmbtu"]
)

# Retain original physical inputs and output
df["capacity_input_mw"] = df["capacity_mw"]
df["fuel_input_mmbtu"] = df["fuel_consumption_mmbtu"]
df["generation_output_mwh"] = df["generation_mwh"]

# ------------------------------------------------------------
# 5. Validate values
# ------------------------------------------------------------
check_cols = [
    "capacity_input_mw",
    "fuel_input_mmbtu",
    "henry_hub_price_usd_per_mmbtu",
    "fuel_cost_usd_proxy",
    "generation_output_mwh"
]

for col in check_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nFUEL-COST INPUT VALIDATION")
print("--------------------------------")
print("Observations:", len(df))
print("Plants:", df["plant_id"].nunique())
print("Years:", sorted(df["year"].dropna().unique()))

print("\nMissing values:")
print(df[check_cols].isna().sum())

print("\nNon-positive values:")
print((df[check_cols] <= 0).sum())

print("\nDescriptive statistics:")
print(df[check_cols].describe().round(3).to_string())

print("\nAnnual Henry Hub prices:")
print(
    prices.sort_values("year").to_string(index=False)
)

print("\nAnnual mean fuel-cost proxy:")
print(
    df.groupby("year")["fuel_cost_usd_proxy"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
    .to_string()
)

# ------------------------------------------------------------
# 6. Create evaluation panel (2021-2025, contiguous states)
# ------------------------------------------------------------
evaluation = df[
    (df["year"] >= 2021) &
    (df["state"] != "AK")
].copy()

if len(evaluation) != 2051:
    raise ValueError(
        f"Expected 2051 evaluation observations, got {len(evaluation)}"
    )

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------
full_path = os.path.join(
    PROC, "ngcc_fuel_cost_dea_panel.csv"
)

eval_path = os.path.join(
    PROC, "ngcc_fuel_cost_dea_evaluation.csv"
)

df.to_csv(full_path, index=False)
evaluation.to_csv(eval_path, index=False)

print("\nSaved full panel:", full_path)
print("Saved evaluation panel:", eval_path)
print("\nEvaluation observations:", len(evaluation))
print("Evaluation plants:", evaluation["plant_id"].nunique())


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# Load the frozen primary panel
df = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
)

df = df.sort_values(["plant_id", "year"]).copy()

# Check required variables
required = [
    "plant_id", "year", "heat_rate_mmbtu_per_mwh",
    "capacity_factor"
]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Keep valid positive observations
for col in ["heat_rate_mmbtu_per_mwh", "capacity_factor"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    (df["heat_rate_mmbtu_per_mwh"] > 0) &
    (df["capacity_factor"] > 0)
].copy()

# Log changes are relative changes and are less scale-dependent
df["log_heat_rate"] = np.log(df["heat_rate_mmbtu_per_mwh"])
df["log_capacity_factor"] = np.log(df["capacity_factor"])

df["gas_variability"] = (
    df.groupby("plant_id")["log_heat_rate"].diff().abs()
)

df["heat_variability"] = (
    df.groupby("plant_id")["log_capacity_factor"].diff().abs()
)

# Keep only genuine consecutive-year changes
previous_year = df.groupby("plant_id")["year"].shift(1)
df.loc[df["year"] - previous_year != 1,
       ["gas_variability", "heat_variability"]] = np.nan

# ------------------------------------------------------------
# 1. Overall calibration diagnostics
# ------------------------------------------------------------
def summarize(series):
    x = series.dropna()
    return pd.Series({
        "transitions": len(x),
        "mean": x.mean(),
        "median": x.median(),
        "q75": x.quantile(0.75),
        "q90": x.quantile(0.90),
        "q95": x.quantile(0.95),
        "maximum": x.max()
    })

overall = pd.DataFrame({
    "gas_heat_rate_change": summarize(df["gas_variability"]),
    "heat_capacity_factor_change": summarize(df["heat_variability"])
}).T

print("OVERALL OPERATING-VARIABILITY DIAGNOSTICS")
print(overall.round(4).to_string())

# ------------------------------------------------------------
# 2. Calibration diagnostics by transition year
# ------------------------------------------------------------
annual_rows = []

for year, group in df.groupby("year"):
    for variable, label in [
        ("gas_variability", "gas_heat_rate_change"),
        ("heat_variability", "heat_capacity_factor_change")
    ]:
        s = group[variable].dropna()

        annual_rows.append({
            "year": year,
            "measure": label,
            "transitions": len(s),
            "median": s.median() if len(s) else np.nan,
            "q90": s.quantile(0.90) if len(s) else np.nan,
            "q95": s.quantile(0.95) if len(s) else np.nan
        })

annual = pd.DataFrame(annual_rows)

print("\nVARIABILITY BY TRANSITION YEAR")
print(
    annual.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 3. Candidate full-sample diagnostic scales
# ------------------------------------------------------------
delta_gas = df["gas_variability"].quantile(0.90)
delta_heat = df["heat_variability"].quantile(0.90)

print("\nCANDIDATE FULL-SAMPLE 90TH-PERCENTILE SCALES")
print(f"Gas-related scale:  {delta_gas:.4f}")
print(f"Heat-related scale: {delta_heat:.4f}")

print("\nEquivalent approximate percentage changes:")
print(f"Gas-related:  {(np.exp(delta_gas)-1)*100:.2f}%")
print(f"Heat-related: {(np.exp(delta_heat)-1)*100:.2f}%")

# ------------------------------------------------------------
# 4. Save diagnostics
# ------------------------------------------------------------
overall.to_csv(
    os.path.join(PROC, "uncertainty_calibration_overall.csv")
)

annual.to_csv(
    os.path.join(PROC, "uncertainty_calibration_by_year.csv"),
    index=False
)

df.to_csv(
    os.path.join(PROC, "uncertainty_calibration_panel.csv"),
    index=False
)

print("\nSaved:")
print(" - uncertainty_calibration_overall.csv")
print(" - uncertainty_calibration_by_year.csv")
print(" - uncertainty_calibration_panel.csv")


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# Load primary plant panel
df = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
).sort_values(["plant_id", "year"])

# Load event data
gas = pd.read_csv(
    os.path.join(PROC, "annual_gas_price_events.csv")
)

temp = pd.read_csv(
    os.path.join(PROC, "state_annual_temperature_events.csv")
)

print("Temperature columns:", temp.columns.tolist())
print("Gas columns:", gas.columns.tolist())

# Identify event columns
gas_col = "gas_event_severity"
temp_candidates = [
    "heat_event_severity",
    "heat_degree_days",
    "heat_event_days"
]

temp_col = next(
    (c for c in temp_candidates if c in temp.columns),
    None
)

if gas_col not in gas.columns:
    raise ValueError("gas_event_severity not found.")

if temp_col is None:
    raise ValueError("No recognized heat event variable found.")

# Prepare event data
gas_small = gas[["year", gas_col]].copy()
gas_small["year"] = pd.to_numeric(gas_small["year"])
gas_small[gas_col] = pd.to_numeric(
    gas_small[gas_col], errors="coerce"
)

# Find state identifier in temperature data
state_candidates = ["state", "postal_code", "state_abbr"]
state_col = next(
    (c for c in state_candidates if c in temp.columns),
    None
)

if state_col is None:
    raise ValueError("Temperature state identifier not found.")

temp_small = temp[
    [state_col, "year", temp_col]
].copy()

temp_small = temp_small.rename(
    columns={state_col: "state"}
)

temp_small["year"] = pd.to_numeric(
    temp_small["year"], errors="coerce"
)

temp_small[temp_col] = pd.to_numeric(
    temp_small[temp_col], errors="coerce"
)

# Merge current-year event indicators
df = df.merge(
    gas_small,
    on="year",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    temp_small,
    on=["state", "year"],
    how="left",
    validate="many_to_one"
)

# Compute signed annual plant-level changes
df["log_heat_rate"] = np.log(
    df["heat_rate_mmbtu_per_mwh"]
)

df["log_capacity_factor"] = np.log(
    df["capacity_factor"]
)

for col in ["log_heat_rate", "log_capacity_factor"]:
    df["change_" + col] = (
        df.groupby("plant_id")[col].diff()
    )

previous_year = df.groupby("plant_id")["year"].shift(1)

for col in ["change_log_heat_rate", "change_log_capacity_factor"]:
    df.loc[
        df["year"] - previous_year != 1, col
    ] = np.nan

# Evaluation sample: consecutive-year changes
d = df[df["year"].between(2021, 2025)].copy()

# Remove missing event observations
d = d.dropna(subset=[
    "change_log_heat_rate",
    "change_log_capacity_factor",
    gas_col,
    temp_col
])

# Correlation diagnostics
pairs = [
    ("change_log_heat_rate", gas_col,
     "Heat-rate change vs gas-price event"),
    ("change_log_capacity_factor", temp_col,
     "Capacity-factor change vs heat event"),
    ("change_log_capacity_factor", gas_col,
     "Capacity-factor change vs gas-price event"),
    ("change_log_heat_rate", temp_col,
     "Heat-rate change vs heat event")
]

print("\nEVENT-OPERATING CHANGE CORRELATIONS")
print("-----------------------------------")

for x, y, label in pairs:
    z = d[[x, y]].dropna()

    if len(z) < 10 or z[x].nunique() < 2 or z[y].nunique() < 2:
        print(label, ": insufficient variation")
        continue

    pr = pearsonr(z[x], z[y])
    sr = spearmanr(z[x], z[y])

    print(f"\n{label}")
    print("Observations:", len(z))
    print("Pearson correlation:", round(pr.statistic, 4))
    print("Pearson p-value:", round(pr.pvalue, 4))
    print("Spearman correlation:", round(sr.statistic, 4))
    print("Spearman p-value:", round(sr.pvalue, 4))

# Annual means by event severity
print("\nANNUAL OPERATING CHANGES AND EVENT SEVERITY")

annual = d.groupby("year").agg(
    observations=("plant_id", "size"),
    mean_gas_event=(gas_col, "mean"),
    mean_heat_event=(temp_col, "mean"),
    mean_heat_rate_change=("change_log_heat_rate", "mean"),
    median_heat_rate_change=("change_log_heat_rate", "median"),
    mean_capacity_factor_change=(
        "change_log_capacity_factor", "mean"
    ),
    median_capacity_factor_change=(
        "change_log_capacity_factor", "median"
    )
)

print(annual.round(4).to_string())

# Save
out = os.path.join(
    PROC, "event_operating_change_diagnostics.csv"
)

annual.to_csv(out)

print("\nSaved:", out)


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
df = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
).sort_values(["plant_id", "year"])

gas = pd.read_csv(
    os.path.join(PROC, "annual_gas_price_events.csv")
)

temp = pd.read_csv(
    os.path.join(PROC, "state_annual_temperature_events.csv")
)

# ------------------------------------------------------------
# 2. Construct correctly lagged gas event signal
#    Event in year t-1 is assigned to target year t
# ------------------------------------------------------------
gas_lag = gas[["year", "gas_event_severity"]].copy()

gas_lag["year"] = pd.to_numeric(gas_lag["year"])
gas_lag["gas_event_severity"] = pd.to_numeric(
    gas_lag["gas_event_severity"], errors="coerce"
)

gas_lag["year"] = gas_lag["year"] + 1

gas_lag = gas_lag.rename(
    columns={
        "gas_event_severity": "lagged_gas_event_severity"
    }
)

# ------------------------------------------------------------
# 3. Use the already lagged temperature event signal
# ------------------------------------------------------------
temp_lag = temp[
    ["postal_code", "year", "lagged_heat_event_severity"]
].copy()

temp_lag = temp_lag.rename(
    columns={"postal_code": "state"}
)

# ------------------------------------------------------------
# 4. Merge lagged event signals
# ------------------------------------------------------------
df["year"] = pd.to_numeric(df["year"])

df = df.merge(
    gas_lag,
    on="year",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    temp_lag,
    on=["state", "year"],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 5. Calculate consecutive-year operating changes
# ------------------------------------------------------------
df["log_heat_rate"] = np.log(
    df["heat_rate_mmbtu_per_mwh"]
)

df["log_capacity_factor"] = np.log(
    df["capacity_factor"]
)

df["change_log_heat_rate"] = (
    df.groupby("plant_id")["log_heat_rate"].diff()
)

df["change_log_capacity_factor"] = (
    df.groupby("plant_id")["log_capacity_factor"].diff()
)

previous_year = df.groupby("plant_id")["year"].shift(1)

consecutive = (df["year"] - previous_year) == 1

df.loc[
    ~consecutive,
    ["change_log_heat_rate", "change_log_capacity_factor"]
] = np.nan

# Evaluation period
d = df[df["year"].between(2021, 2025)].copy()

print("Lagged event availability:")
print(
    d[[
        "lagged_gas_event_severity",
        "lagged_heat_event_severity"
    ]].notna().sum()
)

d = d.dropna(subset=[
    "change_log_heat_rate",
    "change_log_capacity_factor",
    "lagged_gas_event_severity",
    "lagged_heat_event_severity"
]).copy()

print("\nComplete observations:", len(d))

# ------------------------------------------------------------
# 6. Pooled and within-year correlations
# ------------------------------------------------------------
pairs = [
    (
        "change_log_heat_rate",
        "lagged_gas_event_severity",
        "Heat-rate change vs lagged gas event"
    ),
    (
        "change_log_capacity_factor",
        "lagged_heat_event_severity",
        "Capacity-factor change vs lagged heat event"
    ),
    (
        "change_log_capacity_factor",
        "lagged_gas_event_severity",
        "Capacity-factor change vs lagged gas event"
    ),
    (
        "change_log_heat_rate",
        "lagged_heat_event_severity",
        "Heat-rate change vs lagged heat event"
    )
]

def report_corr(data, x, y, label):
    z = data[[x, y]].dropna()

    print(f"\n{label}")
    print("N:", len(z))

    if len(z) < 10 or z[x].nunique() < 2 or z[y].nunique() < 2:
        print("Insufficient variation.")
        return

    p = pearsonr(z[x], z[y])
    s = spearmanr(z[x], z[y])

    print("Pearson r:", round(p.statistic, 4))
    print("Pearson p:", round(p.pvalue, 5))
    print("Spearman rho:", round(s.statistic, 4))
    print("Spearman p:", round(s.pvalue, 5))

print("\nPOOLED CORRELATIONS")
for x, y, label in pairs:
    report_corr(d, x, y, label)

# Within-year demeaned correlations remove common annual means
within = d.copy()

variables = [
    "change_log_heat_rate",
    "change_log_capacity_factor",
    "lagged_gas_event_severity",
    "lagged_heat_event_severity"
]

for col in variables:
    within[col + "_within"] = (
        within[col] -
        within.groupby("year")[col].transform("mean")
    )

print("\nWITHIN-YEAR CORRELATIONS")
for x, y, label in pairs:
    report_corr(
        within,
        x + "_within",
        y + "_within",
        label + " (year-demeaned)"
    )

# ------------------------------------------------------------
# 7. Annual descriptive statistics
# ------------------------------------------------------------
annual = d.groupby("year").agg(
    observations=("plant_id", "size"),
    lagged_gas_event=("lagged_gas_event_severity", "mean"),
    lagged_heat_event=("lagged_heat_event_severity", "mean"),
    mean_heat_rate_change=("change_log_heat_rate", "mean"),
    median_heat_rate_change=("change_log_heat_rate", "median"),
    mean_capacity_factor_change=(
        "change_log_capacity_factor", "mean"
    ),
    median_capacity_factor_change=(
        "change_log_capacity_factor", "median"
    )
)

print("\nANNUAL LAGGED EVENT DIAGNOSTICS")
print(annual.round(4).to_string())

# Save
annual.to_csv(
    os.path.join(PROC, "lagged_event_operating_diagnostics.csv")
)

d.to_csv(
    os.path.join(PROC, "lagged_event_operating_panel.csv"),
    index=False
)

print("\nSaved lagged diagnostic files.")


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "uncertainty_calibration_panel.csv")
)

# Retain consecutive-year observations
df = df.sort_values(["plant_id", "year"])

previous_year = df.groupby("plant_id")["year"].shift(1)
consecutive = (df["year"] - previous_year) == 1

df.loc[
    ~consecutive,
    ["gas_variability", "heat_variability"]
] = np.nan

# Plant-level historical variability
plant_stats = (
    df.groupby("plant_id")
    .agg(
        transitions=("gas_variability", "count"),
        gas_scale=("gas_variability", "median"),
        heat_scale=("heat_variability", "median"),
        gas_q90=("gas_variability", lambda x: x.quantile(0.90)),
        heat_q90=("heat_variability", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

plant_stats = plant_stats[
    (plant_stats["transitions"] >= 2)
].copy()

print("PLANT-LEVEL VARIABILITY")
print("-----------------------")
print("Plants:", len(plant_stats))

for col in ["gas_scale", "heat_scale", "gas_q90", "heat_q90"]:
    print(f"\n{col}")
    print(
        plant_stats[col].describe(
            percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
        ).round(4)
    )

# Cross-sectional heterogeneity
print("\nCROSS-PLANT HETEROGENEITY")
for col in ["gas_scale", "heat_scale"]:
    x = plant_stats[col].dropna()

    print(
        col,
        "| coefficient of variation:",
        round(x.std() / x.mean(), 4) if x.mean() != 0 else np.nan,
        "| distinct values:",
        x.nunique()
    )

# Save
out = os.path.join(
    PROC, "plant_uncertainty_heterogeneity.csv"
)

plant_stats.to_csv(out, index=False)

print("\nSaved:", out)


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
plants = pd.read_csv(
    os.path.join(PROC, "ngcc_primary_panel.csv")
)

history = pd.read_csv(
    os.path.join(PROC, "uncertainty_calibration_panel.csv")
)

plants = plants.sort_values(["plant_id", "year"]).copy()
history = history.sort_values(["plant_id", "year"]).copy()

# Keep contiguous US for consistency with temperature data
plants = plants[plants["state"] != "AK"].copy()

# ------------------------------------------------------------
# 2. Build strictly historical plant-specific coefficients
# ------------------------------------------------------------
records = []

for year in range(2022, 2026):

    # Only transitions completed before the target year
    past = history[
        (history["year"] < year) &
        history["gas_variability"].notna() &
        history["heat_variability"].notna()
    ].copy()

    if len(past) == 0:
        continue

    # Pooled historical fallback and upper caps
    pooled = {
        "gas": past["gas_variability"].median(),
        "heat": past["heat_variability"].median()
    }

    caps = {
        "gas": past["gas_variability"].quantile(0.95),
        "heat": past["heat_variability"].quantile(0.95)
    }

    # Plant-specific historical statistics
    plant_hist = (
        past.groupby("plant_id")
        .agg(
            n_gas=("gas_variability", "count"),
            gas_median=("gas_variability", "median"),
            n_heat=("heat_variability", "count"),
            heat_median=("heat_variability", "median")
        )
        .reset_index()
    )

    target = plants[plants["year"] == year].copy()

    target = target.merge(
        plant_hist,
        on="plant_id",
        how="left"
    )

    # Shrink noisy plant-specific estimates toward historical
    # pooled medians. Three pseudo-observations provide the
    # shrinkage strength.
    for group in ["gas", "heat"]:

        n = target[f"n_{group}"].fillna(0)

        observed = target[f"{group}_median"]
        fallback = pooled[group]

        weight = n / (n + 3.0)

        estimate = (
            weight * observed.fillna(fallback)
            + (1 - weight) * fallback
        )

        target[f"delta_{group}"] = np.minimum(
            estimate,
            caps[group]
        )

    # --------------------------------------------------------
    # 3. Convert log variability into relative uncertainty
    # --------------------------------------------------------

    # Fuel-input increase corresponding to heat-rate variability
    target["gas_relative_radius"] = (
        np.exp(target["delta_gas"]) - 1
    )

    # Output reduction corresponding to capacity-factor variability
    target["heat_relative_radius"] = (
        1 - np.exp(-target["delta_heat"])
    )

    # Absolute uncertainty magnitudes
    target["fuel_uncertainty_radius"] = (
        target["fuel_consumption_mmbtu"]
        * target["gas_relative_radius"]
    )

    target["output_uncertainty_radius"] = (
        target["generation_mwh"]
        * target["heat_relative_radius"]
    )

    records.append(target)

# ------------------------------------------------------------
# 4. Combine annual calibration panels
# ------------------------------------------------------------
if not records:
    raise ValueError("No valid historical calibration records.")

cal = pd.concat(records, ignore_index=True)

# ------------------------------------------------------------
# 5. Validate
# ------------------------------------------------------------
print("LEAKAGE-SAFE CALIBRATION")
print("------------------------")
print("Observations:", len(cal))
print("Plants:", cal["plant_id"].nunique())
print("Years:", sorted(cal["year"].unique()))

print("\nMissing calibration values:")
print(
    cal[[
        "delta_gas",
        "delta_heat",
        "gas_relative_radius",
        "heat_relative_radius",
        "fuel_uncertainty_radius",
        "output_uncertainty_radius"
    ]].isna().sum()
)

print("\nRelative uncertainty statistics:")
print(
    cal[[
        "gas_relative_radius",
        "heat_relative_radius"
    ]].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    ).round(4).to_string()
)

print("\nAnnual calibration summaries:")
print(
    cal.groupby("year").agg(
        observations=("plant_id", "size"),
        gas_radius_median=("gas_relative_radius", "median"),
        gas_radius_q90=("gas_relative_radius",
                        lambda x: x.quantile(0.90)),
        heat_radius_median=("heat_relative_radius", "median"),
        heat_radius_q90=("heat_relative_radius",
                         lambda x: x.quantile(0.90))
    ).round(4).to_string()
)

# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------
output = os.path.join(
    PROC, "ngcc_lagged_uncertainty_coefficients.csv"
)

cal.to_csv(output, index=False)

print("\nSaved:", output)


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.optimize import linprog

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# ------------------------------------------------------------
# 1. Load calibrated panel and event-responsive budgets
# ------------------------------------------------------------
panel_path = os.path.join(
    PROC, "ngcc_lagged_uncertainty_coefficients.csv"
)

budget_path = os.path.join(
    PROC, "edro_annual_budget_allocation.csv"
)

df = pd.read_csv(panel_path)
budgets = pd.read_csv(budget_path)

required = [
    "plant_id", "year", "capacity_mw",
    "fuel_consumption_mmbtu", "generation_mwh",
    "fuel_uncertainty_radius", "output_uncertainty_radius"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing calibration columns: {missing}")

# Identify budget columns robustly
def find_column(data, candidates, description):
    normalized = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for candidate in candidates:
        key = candidate.lower().replace("_", "").replace(" ", "")
        if key in normalized:
            return normalized[key]
    raise ValueError(
        f"Could not identify {description}. "
        f"Available columns: {data.columns.tolist()}"
    )

year_col = find_column(budgets, ["year"], "budget year")
gas_budget_col = find_column(
    budgets,
    ["gamma_gas", "Gamma gas", "gas_gamma", "gamma_g"],
    "gas budget"
)
heat_budget_col = find_column(
    budgets,
    ["gamma_heat", "Gamma heat", "heat_gamma", "gamma_h"],
    "heat budget"
)

budgets = budgets.rename(columns={
    year_col: "year",
    gas_budget_col: "gamma_gas",
    heat_budget_col: "gamma_heat"
})

df["year"] = pd.to_numeric(df["year"]).astype(int)
budgets["year"] = pd.to_numeric(budgets["year"]).astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

if df[["gamma_gas", "gamma_heat"]].isna().any().any():
    raise ValueError("Missing annual EDRO budget allocations.")

# Keep 2022-2025, contiguous US
df = df[
    df["year"].between(2022, 2025) &
    (df["state"] != "AK")
].copy()

# Validate uncertainty coefficients
numeric_cols = [
    "capacity_mw", "fuel_consumption_mmbtu",
    "generation_mwh", "fuel_uncertainty_radius",
    "output_uncertainty_radius", "gamma_gas", "gamma_heat"
]

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if df[numeric_cols].isna().any().any():
    raise ValueError("Missing or invalid numerical model inputs.")

if (df[["capacity_mw", "fuel_consumption_mmbtu",
        "generation_mwh"]] <= 0).any().any():
    raise ValueError("DEA inputs and output must be positive.")

if (df[["fuel_uncertainty_radius",
        "output_uncertainty_radius"]] < 0).any().any():
    raise ValueError("Uncertainty radii cannot be negative.")

# Check EDRO budget total
annual_budgets = (
    df.groupby("year")[["gamma_gas", "gamma_heat"]]
    .first()
)

if not np.allclose(
    annual_budgets.sum(axis=1).values, 3.0, atol=1e-5
):
    raise ValueError(
        "EDRO gas and heat budgets must sum to 3 in each year."
    )

print("INPUT VALIDATION")
print("----------------")
print("Observations:", len(df))
print("Plants:", df["plant_id"].nunique())
print("Years:", sorted(df["year"].unique()))
print("\nAnnual budgets:")
print(annual_budgets.round(4).to_string())

# ------------------------------------------------------------
# 2. Robust input-oriented VRS DEA solver
# ------------------------------------------------------------
def solve_robust_dea(
    K, F, Y, radius_F, radius_Y,
    gamma_gas, gamma_heat
):
    """
    Robust input-oriented VRS DEA.

    Fuel constraint:
        sum(F_j lambda_j) + BS_support_gas <= theta * F_o

    Output constraint:
        sum(Y_j lambda_j) - BS_support_heat >= Y_o

    Capacity remains deterministic.
    """

    n = len(K)

    # Variables:
    # theta
    # lambda[0:n]
    # pi_gas
    # q_gas[0:n]
    # pi_heat
    # q_heat[0:n]

    i_theta = 0
    i_lam = 1
    i_pi_g = i_lam + n
    i_q_g = i_pi_g + 1
    i_pi_h = i_q_g + n
    i_q_h = i_pi_h + 1
    nvars = i_q_h + n

    c = np.zeros(nvars)
    c[i_theta] = 1.0

    A_ub = []
    b_ub = []

    # Constraints are scaled to improve numerical stability.
    Kscale = np.median(K)
    Fscale = np.median(F)
    Yscale = np.median(Y)

    Ks = K / Kscale
    Fs = F / Fscale
    Ys = Y / Yscale
    rFs = radius_F / Fscale
    rYs = radius_Y / Yscale

    # Capacity constraint:
    # sum(K_j lambda_j) <= theta * K_o
    for o in range(n):
        row = np.zeros(nvars)
        row[i_theta] = -Ks[o]
        row[i_lam:i_lam+n] = Ks
        A_ub.append(row)
        b_ub.append(0.0)

    # Robust fuel-input constraint:
    # sum(F_j lambda_j) + Gamma_g*pi_g + sum(q_g)
    # <= theta * F_o
    for o in range(n):
        row = np.zeros(nvars)
        row[i_theta] = -Fs[o]
        row[i_lam:i_lam+n] = Fs
        row[i_pi_g] = gamma_gas
        row[i_q_g:i_q_g+n] = 1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # q_gj >= radius_F_j * lambda_j - pi_g
    for j in range(n):
        row = np.zeros(nvars)
        row[i_lam+j] = rFs[j]
        row[i_pi_g] = -1.0
        row[i_q_g+j] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # Robust output constraint:
    # sum(Y_j lambda_j) - Gamma_h*pi_h - sum(q_h)
    # >= Y_o
    for o in range(n):
        row = np.zeros(nvars)
        row[i_lam:i_lam+n] = -Ys
        row[i_pi_h] = gamma_heat
        row[i_q_h:i_q_h+n] = 1.0
        A_ub.append(row)
        b_ub.append(-Ys[o])

    # q_hj >= radius_Y_j * lambda_j - pi_h
    for j in range(n):
        row = np.zeros(nvars)
        row[i_lam+j] = rYs[j]
        row[i_pi_h] = -1.0
        row[i_q_h+j] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # VRS convexity: sum(lambda) = 1
    A_eq = np.zeros((1, nvars))
    A_eq[0, i_lam:i_lam+n] = 1.0
    b_eq = np.array([1.0])

    bounds = [(0, None)] * nvars

    result = linprog(
        c,
        A_ub=np.asarray(A_ub),
        b_ub=np.asarray(b_ub),
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:
        return np.nan, result.message

    return result.x[i_theta], "optimal"


# ------------------------------------------------------------
# 3. Solve four models on each annual reference set
# ------------------------------------------------------------
results = []

methods = [
    ("Nominal", 0.0, 0.0),
    ("BS_separate", 3.0, 3.0),
    ("GBU_equal", 1.5, 1.5)
]

for year, group in df.groupby("year"):

    group = group.sort_values("plant_id").reset_index(drop=True)

    K = group["capacity_mw"].to_numpy(dtype=float)
    F = group["fuel_consumption_mmbtu"].to_numpy(dtype=float)
    Y = group["generation_mwh"].to_numpy(dtype=float)

    rF = group["fuel_uncertainty_radius"].to_numpy(dtype=float)
    rY = group["output_uncertainty_radius"].to_numpy(dtype=float)

    gamma_g = float(group["gamma_gas"].iloc[0])
    gamma_h = float(group["gamma_heat"].iloc[0])

    print(f"\nSolving year {year}: {len(group)} DMUs")

    for method, fixed_g, fixed_h in methods:

        # Nominal uses zero uncertainty radii
        if method == "Nominal":
            method_rF = np.zeros_like(rF)
            method_rY = np.zeros_like(rY)
            g, h = 0.0, 0.0
        else:
            method_rF = rF
            method_rY = rY
            g, h = fixed_g, fixed_h

        for o in range(len(group)):
            score, status = solve_robust_dea(
                K, F, Y,
                method_rF, method_rY,
                g, h
            )

            results.append({
                "plant_id": group.loc[o, "plant_id"],
                "year": year,
                "method": method,
                "efficiency": score,
                "status": status,
                "gamma_gas": g,
                "gamma_heat": h
            })

    # EDRO
    for o in range(len(group)):
        score, status = solve_robust_dea(
            K, F, Y, rF, rY, gamma_g, gamma_h
        )

        results.append({
            "plant_id": group.loc[o, "plant_id"],
            "year": year,
            "method": "EDRO",
            "efficiency": score,
            "status": status,
            "gamma_gas": gamma_g,
            "gamma_heat": gamma_h
        })

    print("Finished year", year)

# ------------------------------------------------------------
# 4. Save detailed results
# ------------------------------------------------------------
out = pd.DataFrame(results)

out_path = os.path.join(
    PROC, "robust_dea_all_results.csv"
)

out.to_csv(out_path, index=False)

# ------------------------------------------------------------
# 5. Summary table
# ------------------------------------------------------------
summary = (
    out.groupby(["year", "method"])
    .agg(
        observations=("efficiency", "count"),
        successful=("status", lambda x: (x == "optimal").sum()),
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median"),
        std_efficiency=("efficiency", "std"),
        efficient_dmus=("efficiency", lambda x: (x <= 1.000001).sum())
    )
    .reset_index()
)

summary_path = os.path.join(
    PROC, "robust_dea_summary.csv"
)

summary.to_csv(summary_path, index=False)

print("\nROBUST DEA SUMMARY")
print("------------------")
print(summary.round(4).to_string(index=False))

print("\nSaved detailed results:", out_path)
print("Saved summary:", summary_path)


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

res = pd.read_csv(
    os.path.join(PROC, "robust_dea_all_results.csv")
)

print("1. RAW DATA")
print(df[[
    "year", "plant_id", "capacity_mw",
    "fuel_consumption_mmbtu", "generation_mwh"
]].head(8).to_string(index=False))

print("\n2. INPUT DISTRIBUTIONS")
print(df[[
    "capacity_mw", "fuel_consumption_mmbtu",
    "generation_mwh"
]].describe().to_string())

print("\n3. NOMINAL RESULTS")
nom = res[res.method == "Nominal"]
print(nom.groupby("year").efficiency.describe().to_string())

print("\n4. ROBUST FAILURE MESSAGES")
rob = res[res.method != "Nominal"]
print(rob.status.value_counts(dropna=False).to_string())
print("\nExamples:")
print(rob[["year", "method", "status"]].head(12).to_string(index=False))

print("\n5. UNCERTAINTY COLUMN NAMES")
print([
    c for c in df.columns
    if "radius" in c.lower() or "uncertainty" in c.lower()
])

print("\n6. SAVED NOMINAL BENCHMARK")
benchmark = pd.read_csv(
    os.path.join(PROC, "ngcc_nominal_vrs_dea.csv")
)
print(benchmark.columns.tolist())
print(benchmark.head(3).to_string(index=False))


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.optimize import linprog

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)
benchmark = pd.read_csv(
    os.path.join(PROC, "ngcc_nominal_vrs_dea.csv")
)

# Use the exact calibration sample for 2022
g = df[(df.year == 2022) & (df.state != "AK")].copy()
g = g.sort_values("plant_id").reset_index(drop=True)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)

n = len(g)

# Absolute uncertainty radii
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)


def solve_one_dea(K, F, Y, rF, rY, target,
                  gamma_gas=0.0, gamma_heat=0.0):

    n = len(K)

    # Variable indices
    it = 0
    il = 1
    ipg = il + n
    iqg = ipg + 1
    iph = iqg + n
    iqh = iph + 1
    nv = iqh + n

    # Scale each physical dimension separately
    ks = np.median(K)
    fs = np.median(F)
    ys = np.median(Y)

    k = K / ks
    f = F / fs
    y = Y / ys
    rf = rF / fs
    ry = rY / ys

    c = np.zeros(nv)
    c[it] = 1.0

    rows = []
    rhs = []

    # Capacity constraint for THIS target only
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Robust fuel constraint for THIS target only
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f
    row[ipg] = gamma_gas
    row[iqg:iqg+n] = 1.0
    rows.append(row)
    rhs.append(0.0)

    # Gas uncertainty excess variables
    for j in range(n):
        row = np.zeros(nv)
        row[il+j] = rf[j]
        row[ipg] = -1.0
        row[iqg+j] = -1.0
        rows.append(row)
        rhs.append(0.0)

    # Robust output constraint for THIS target only
    row = np.zeros(nv)
    row[il:il+n] = -y
    row[iph] = gamma_heat
    row[iqh:iqh+n] = 1.0
    rows.append(row)
    rhs.append(-y[target])

    # Heat uncertainty excess variables
    for j in range(n):
        row = np.zeros(nv)
        row[il+j] = ry[j]
        row[iph] = -1.0
        row[iqh+j] = -1.0
        rows.append(row)
        rhs.append(0.0)

    # VRS convexity
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    bounds = [(0, None)] * nv

    # If a budget is zero, fix its auxiliary variables at zero
    if gamma_gas == 0:
        bounds[ipg] = (0, 0)
        for j in range(n):
            bounds[iqg+j] = (0, 0)

    if gamma_heat == 0:
        bounds[iph] = (0, 0)
        for j in range(n):
            bounds[iqh+j] = (0, 0)

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=bounds,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message


# --------------------------------------------------
# Validate nominal DEA for all 2022 target plants
# --------------------------------------------------
scores = []

for o in range(n):
    score, status = solve_one_dea(
        K, F, Y,
        np.zeros(n), np.zeros(n),
        target=o
    )
    scores.append({
        "plant_id": g.loc[o, "plant_id"],
        "year": 2022,
        "new_efficiency": score,
        "status": status
    })

    if (o + 1) % 50 == 0:
        print(f"Validated {o+1}/{n}")

new = pd.DataFrame(scores)

# Compare against frozen nominal benchmark
old = benchmark[benchmark.year == 2022][
    ["plant_id", "nominal_vrs_efficiency"]
].copy()

check = new.merge(old, on="plant_id", how="inner")

check["absolute_difference"] = (
    check.new_efficiency - check.nominal_vrs_efficiency
).abs()

print("\nVALIDATION RESULTS")
print("------------------")
print("New successful:", (new.status == "optimal").sum())
print("Matched benchmark observations:", len(check))
print("Mean new efficiency:", check.new_efficiency.mean())
print("Mean original efficiency:",
      check.nominal_vrs_efficiency.mean())
print("Maximum absolute difference:",
      check.absolute_difference.max())
print("Correlation:",
      check[["new_efficiency", "nominal_vrs_efficiency"]]
      .corr().iloc[0, 1])

print("\nLargest differences:")
print(
    check.nlargest(10, "absolute_difference")[
        ["plant_id", "new_efficiency",
         "nominal_vrs_efficiency", "absolute_difference"]
    ].to_string(index=False)
)

check.to_csv(
    os.path.join(PROC, "dea_nominal_validation_2022.csv"),
    index=False
)


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

# Identify annual budget columns
def col_match(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for name in candidates:
        key = name.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(
        f"Budget column not found. Available: {data.columns.tolist()}"
    )

budgets = budgets.rename(columns={
    col_match(budgets, ["year"]): "year",
    col_match(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    col_match(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

df = df[
    df.year.between(2022, 2025) &
    (df.state != "AK")
].copy()

df = df.sort_values(["year", "plant_id"]).reset_index(drop=True)

# Load nominal benchmark: no need to solve it again
nominal = pd.read_csv(
    os.path.join(PROC, "ngcc_nominal_vrs_dea.csv")
)

all_results = []

# Keep any earlier successfully saved years if this cell is rerun
output_file = os.path.join(PROC, "robust_dea_corrected_results.csv")

for year, group in df.groupby("year"):

    group = group.sort_values("plant_id").reset_index(drop=True)

    K = group.capacity_mw.to_numpy(float)
    F = group.fuel_consumption_mmbtu.to_numpy(float)
    Y = group.generation_mwh.to_numpy(float)

    rF = group.fuel_uncertainty_radius.to_numpy(float)
    rY = group.output_uncertainty_radius.to_numpy(float)

    gamma_edro_g = float(group.gamma_gas.iloc[0])
    gamma_edro_h = float(group.gamma_heat.iloc[0])

    # Validate annual allocation
    assert np.isclose(gamma_edro_g + gamma_edro_h, 3.0)

    print(f"\nYEAR {year}: {len(group)} DMUs", flush=True)

    year_results = []

    models = [
        ("GBU_equal", 1.5, 1.5),
        ("BS_separate", 3.0, 3.0),
        ("EDRO", gamma_edro_g, gamma_edro_h)
    ]

    for method, gamma_g, gamma_h in models:

        print(
            f"  Starting {method}: "
            f"Gamma gas={gamma_g:.4f}, heat={gamma_h:.4f}",
            flush=True
        )

        success = 0
        failures = 0

        for o in range(len(group)):

            score, status = solve_one_dea(
                K, F, Y, rF, rY,
                target=o,
                gamma_gas=gamma_g,
                gamma_heat=gamma_h
            )

            if status == "optimal":
                success += 1
            else:
                failures += 1

            year_results.append({
                "plant_id": group.loc[o, "plant_id"],
                "year": year,
                "method": method,
                "efficiency": score,
                "status": status,
                "gamma_gas": gamma_g,
                "gamma_heat": gamma_h
            })

            if (o + 1) % 50 == 0:
                print(
                    f"    {method}: {o+1}/{len(group)} "
                    f"(successful={success}, failed={failures})",
                    flush=True
                )

        print(
            f"  Completed {method}: "
            f"{success} successful, {failures} failed",
            flush=True
        )

    # Save after each year
    all_results.extend(year_results)

    partial = pd.DataFrame(all_results)
    partial.to_csv(output_file, index=False)

    print(f"Saved through {year}: {output_file}", flush=True)


# --------------------------------------------------
# Summary
# --------------------------------------------------
results = pd.DataFrame(all_results)

summary = (
    results.groupby(["year", "method"])
    .agg(
        total=("plant_id", "size"),
        successful=("status", lambda x: (x == "optimal").sum()),
        failed=("status", lambda x: (x != "optimal").sum()),
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median"),
        std_efficiency=("efficiency", "std"),
        share_above_one=("efficiency", lambda x: (x > 1.000001).mean())
    )
    .reset_index()
)

summary.to_csv(
    os.path.join(PROC, "robust_dea_corrected_summary.csv"),
    index=False
)

print("\nFINAL SUMMARY")
print(summary.round(4).to_string(index=False))
print("\nSaved:", output_file)


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)
budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def getcol(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for name in candidates:
        key = name.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    getcol(budgets, ["year"]): "year",
    getcol(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    getcol(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df.year.astype(int)
budgets["year"] = budgets.year.astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year", how="left", validate="many_to_one"
)

old = pd.read_csv(
    os.path.join(PROC, "robust_dea_corrected_results.csv")
)

# Retain all previously successful results except EDRO 2024-25
keep = old[
    ~(
        (old.method == "EDRO") &
        (old.year.isin([2024, 2025]))
    )
].copy()

new_edro = []

for year in [2024, 2025]:

    group = df[
        (df.year == year) & (df.state != "AK")
    ].sort_values("plant_id").reset_index(drop=True)

    K = group.capacity_mw.to_numpy(float)
    F = group.fuel_consumption_mmbtu.to_numpy(float)
    Y = group.generation_mwh.to_numpy(float)
    rF = group.fuel_uncertainty_radius.to_numpy(float)
    rY = group.output_uncertainty_radius.to_numpy(float)

    gg = float(group.gamma_gas.iloc[0])
    gh = float(group.gamma_heat.iloc[0])

    print(f"\nEDRO {year}: gas={gg}, heat={gh}", flush=True)

    for o in range(len(group)):

        score, status = solve_one_dea(
            K, F, Y, rF, rY,
            target=o,
            gamma_gas=gg,
            gamma_heat=gh
        )

        new_edro.append({
            "plant_id": group.loc[o, "plant_id"],
            "year": year,
            "method": "EDRO",
            "efficiency": score,
            "status": status,
            "gamma_gas": gg,
            "gamma_heat": gh
        })

        if (o + 1) % 50 == 0:
            print(
                f"  {o+1}/{len(group)} completed",
                flush=True
            )

    yr = pd.DataFrame(new_edro)
    print(
        f"Year {year}: "
        f"{(yr[yr.year == year].status == 'optimal').sum()} "
        f"successful, "
        f"{(yr[yr.year == year].status != 'optimal').sum()} failed",
        flush=True
    )

combined = pd.concat(
    [keep, pd.DataFrame(new_edro)],
    ignore_index=True
)

combined.to_csv(
    os.path.join(PROC, "robust_dea_corrected_results.csv"),
    index=False
)

summary = (
    combined.groupby(["year", "method"])
    .agg(
        total=("plant_id", "size"),
        successful=("status", lambda x: (x == "optimal").sum()),
        failed=("status", lambda x: (x != "optimal").sum()),
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median")
    )
    .reset_index()
)

summary.to_csv(
    os.path.join(PROC, "robust_dea_corrected_summary.csv"),
    index=False
)

print("\nUPDATED SUMMARY")
print(summary.round(4).to_string(index=False))


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.optimize import linprog

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

def solve_one_dea_corrected(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    # Scale physical quantities
    ks, fs, ys = np.median(K), np.median(F), np.median(Y)

    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    # Variables: theta, lambda, then only active uncertainty blocks
    it = 0
    il = 1
    next_idx = il + n

    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg = next_idx
        iqg = next_idx + 1
        next_idx += n + 1

    if gamma_heat > 0:
        iph = next_idx
        iqh = next_idx + 1
        next_idx += n + 1

    nv = next_idx

    c = np.zeros(nv)
    c[it] = 1.0

    rows, rhs = [], []

    # Capacity constraint
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Fuel constraint
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    # Gas support-function constraints only if budget > 0
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Output constraint
    row = np.zeros(nv)
    row[il:il+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Heat support-function constraints only if budget > 0
    if gamma_heat > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = ry[j]
            row[iph] = -1.0
            row[iqh+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # VRS constraint
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=[(0, None)] * nv,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message


# --------------------------------------------------
# Test one 2024 DMU
# --------------------------------------------------
df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def find_col(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for x in candidates:
        key = x.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    find_col(budgets, ["year"]): "year",
    find_col(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    find_col(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df.year.astype(int)
budgets["year"] = budgets.year.astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

g = df[
    (df.year == 2024) & (df.state != "AK")
].sort_values("plant_id").reset_index(drop=True)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)

gamma_g = float(g.gamma_gas.iloc[0])
gamma_h = float(g.gamma_heat.iloc[0])

print("2024 budgets:", gamma_g, gamma_h)
print("Number of DMUs:", len(g))

# Test first DMU
score, status = solve_one_dea_corrected(
    K, F, Y, rF, rY,
    target=0,
    gamma_gas=gamma_g,
    gamma_heat=gamma_h
)

print("\nTEST RESULT")
print("Plant:", g.loc[0, "plant_id"])
print("Score:", score)
print("Status:", status)

# Also test nominal self-consistency
nominal_score, nominal_status = solve_one_dea_corrected(
    K, F, Y,
    np.zeros_like(rF), np.zeros_like(rY),
    target=0,
    gamma_gas=0,
    gamma_heat=0
)

print("\nNOMINAL TEST")
print("Score:", nominal_score)
print("Status:", nominal_status)


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Reuse the corrected solver from the previous cell.
# It must be named solve_one_dea_corrected.

years_to_run = [2024, 2025]
all_results = []

for year in years_to_run:

    g = df[
        (df.year == year) & (df.state != "AK")
    ].copy().reset_index(drop=True)

    if len(g) == 0:
        print(f"No observations for {year}")
        continue

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    gamma_g = float(g.gamma_gas.iloc[0])
    gamma_h = float(g.gamma_heat.iloc[0])

    print(f"\nEDRO {year}: gas={gamma_g}, heat={gamma_h}")
    successful = 0

    for i in range(len(g)):
        score, status = solve_one_dea_corrected(
            K, F, Y, rF, rY,
            target=i,
            gamma_gas=gamma_g,
            gamma_heat=gamma_h
        )

        if status == "optimal":
            successful += 1

        all_results.append({
            "year": year,
            "method": "EDRO",
            "plant_id": g.loc[i, "plant_id"],
            "efficiency": score,
            "status": status,
            "gamma_gas": gamma_g,
            "gamma_heat": gamma_h
        })

        if (i + 1) % 50 == 0:
            print(f"{i+1}/{len(g)} completed")

    print(
        f"Year {year}: {successful} successful, "
        f"{len(g)-successful} failed"
    )

# Save separately
out = pd.DataFrame(all_results)

out_path = os.path.join(
    PROC, "robust_dea_edro_zero_budget_repaired.csv"
)
out.to_csv(out_path, index=False)

print("\nSAVED:", out_path)

# Summary
summary = (
    out.groupby(["year", "method"])
       .agg(
           observations=("plant_id", "size"),
           successful=("status", lambda x: (x == "optimal").sum()),
           mean_efficiency=("efficiency", "mean"),
           median_efficiency=("efficiency", "median"),
           sd_efficiency=("efficiency", "std"),
           minimum_efficiency=("efficiency", "min"),
           maximum_efficiency=("efficiency", "max")
       )
       .reset_index()
)

display(summary)


In [ ]:

import os
import pandas as pd
import numpy as np

old_path = os.path.join(PROC, "robust_dea_corrected_results.csv")
new_path = os.path.join(PROC, "robust_dea_edro_zero_budget_repaired.csv")

old = pd.read_csv(old_path)
new = pd.read_csv(new_path)

# Show the failed DMUs
failed = new[new["status"] != "optimal"].copy()

print("FAILED EDRO DMUs")
display(failed[["year", "plant_id", "status"]])

# Replace only EDRO 2024-2025 in the old results
old["year"] = old["year"].astype(int)
old = old[
    ~(
        old["method"].astype(str).str.upper().eq("EDRO")
        & old["year"].isin([2024, 2025])
    )
].copy()

combined = pd.concat([old, new], ignore_index=True)

combined_path = os.path.join(
    PROC, "robust_dea_corrected_results_final.csv"
)
combined.to_csv(combined_path, index=False)

# Compare all methods in 2022-2025
comparison = (
    combined[combined["year"].isin([2022, 2023, 2024, 2025])]
    .groupby(["year", "method"])
    .agg(
        observations=("plant_id", "size"),
        successful=("efficiency", "count"),
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median"),
        sd_efficiency=("efficiency", "std"),
        minimum=("efficiency", "min"),
        maximum=("efficiency", "max")
    )
    .reset_index()
)

print("\nMETHOD COMPARISON")
display(comparison.round(4))

print("\nFINAL RESULTS SAVED:")
print(combined_path)


In [ ]:

# Inspect the failed plants in the original evaluation data
failed_ids = [(2024, 56407), (2025, 710), (2025, 56407)]

diagnostics = []

for year, plant_id in failed_ids:
    r = df[
        (df.year == year) &
        (df.plant_id == plant_id)
    ]

    if len(r) == 0:
        print("Not found:", year, plant_id)
        continue

    r = r.iloc[0]

    diagnostics.append({
        "year": year,
        "plant_id": plant_id,
        "state": r.get("state", None),
        "capacity_mw": r.get("capacity_mw", np.nan),
        "fuel_mmbtu": r.get("fuel_consumption_mmbtu", np.nan),
        "generation_mwh": r.get("generation_mwh", np.nan),
        "capacity_factor": r.get("capacity_factor", np.nan),
        "heat_rate": r.get("heat_rate", np.nan),
        "fuel_uncertainty_radius": r.get("fuel_uncertainty_radius", np.nan),
        "output_uncertainty_radius": r.get("output_uncertainty_radius", np.nan),
        "gamma_gas": r.get("gamma_gas", np.nan),
        "gamma_heat": r.get("gamma_heat", np.nan)
    })

display(pd.DataFrame(diagnostics).round(5))


In [ ]:

# Detailed diagnostic for the three infeasible observations

for year, plant_id in [(2024, 56407), (2025, 710), (2025, 56407)]:
    r = df[
        (df.year == year) &
        (df.plant_id == plant_id)
    ]

    print(f"\nYEAR={year}, PLANT={plant_id}")

    if r.empty:
        print("Not found")
        continue

    cols = [
        c for c in [
            "year", "plant_id", "state",
            "capacity_mw", "fuel_consumption_mmbtu",
            "generation_mwh", "capacity_factor", "heat_rate",
            "fuel_uncertainty_radius", "output_uncertainty_radius",
            "gas_relative_radius", "heat_relative_radius",
            "gamma_gas", "gamma_heat"
        ] if c in r.columns
    ]

    display(r[cols].T)

    # Compare with nominal score if available
    nominal_path = os.path.join(PROC, "ngcc_nominal_vrs_dea.csv")
    nominal = pd.read_csv(nominal_path)

    nom = nominal[
        (nominal.year == year) &
        (nominal.plant_id == plant_id)
    ]

    if not nom.empty:
        print("Nominal DEA record:")
        display(nom.T)


In [ ]:

# Detailed diagnostic for the three infeasible observations

for year, plant_id in [(2024, 56407), (2025, 710), (2025, 56407)]:
    r = df[
        (df.year == year) &
        (df.plant_id == plant_id)
    ]

    print(f"\nYEAR={year}, PLANT={plant_id}")

    if r.empty:
        print("Not found")
        continue

    cols = [
        c for c in [
            "year", "plant_id", "state",
            "capacity_mw", "fuel_consumption_mmbtu",
            "generation_mwh", "capacity_factor", "heat_rate",
            "fuel_uncertainty_radius", "output_uncertainty_radius",
            "gas_relative_radius", "heat_relative_radius",
            "gamma_gas", "gamma_heat"
        ] if c in r.columns
    ]

    display(r[cols].T)

    # Compare with nominal score if available
    nominal_path = os.path.join(PROC, "ngcc_nominal_vrs_dea.csv")
    nominal = pd.read_csv(nominal_path)

    nom = nominal[
        (nominal.year == year) &
        (nominal.plant_id == plant_id)
    ]

    if not nom.empty:
        print("Nominal DEA record:")
        display(nom.T)


In [ ]:

nominal_path = os.path.join(PROC, "ngcc_nominal_vrs_dea.csv")
nominal = pd.read_csv(nominal_path)

for year, plant_id in [(2024, 56407), (2025, 710), (2025, 56407)]:
    print(f"\n{'='*45}\nYEAR {year} | PLANT {plant_id}")

    r = df[(df.year == year) & (df.plant_id == plant_id)]
    if r.empty:
        print("Not found in calibration data")
        continue

    r = r.iloc[0]

    for c in df.columns:
        if any(s in c.lower() for s in [
            "radius", "delta", "uncertainty", "capacity_factor",
            "heat_rate", "generation", "fuel_consumption"
        ]):
            print(f"{c}: {r[c]}")

    n = nominal[
        (nominal.year == year) &
        (nominal.plant_id == plant_id)
    ]

    if not n.empty:
        print("\nNOMINAL DEA:")
        print(n.to_string(index=False))
    else:
        print("\nNo nominal DEA record found")


In [ ]:

from scipy.optimize import linprog
import numpy as np
import pandas as pd

failed_ids = [(2024, 56407), (2025, 710), (2025, 56407)]
diagnostics = []

for year, plant_id in failed_ids:

    g = df[
        (df.year == year) & (df.state != "AK")
    ].reset_index(drop=True)

    target_rows = g.index[g.plant_id == plant_id].tolist()
    if not target_rows:
        continue

    target = target_rows[0]

    Y = g.generation_mwh.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    ys = np.median(Y)
    y = Y / ys
    ry = rY / ys

    gamma = float(g.gamma_heat.iloc[0])
    n = len(g)

    # Maximize robust attainable output:
    # max sum(y_j lambda_j) - gamma*pi - sum(q_j)
    # subject to q_j >= ry_j*lambda_j - pi
    c = np.r_[-y, gamma, np.ones(n)]

    A_ub = []
    b_ub = []

    for j in range(n):
        row = np.zeros(2*n + 1)
        row[j] = ry[j]
        row[n] = -1
        row[n+1+j] = -1
        A_ub.append(row)
        b_ub.append(0)

    A_eq = np.zeros((1, 2*n + 1))
    A_eq[0, :n] = 1

    result = linprog(
        c,
        A_ub=np.asarray(A_ub),
        b_ub=np.asarray(b_ub),
        A_eq=A_eq,
        b_eq=[1.0],
        bounds=[(0, None)] * (2*n + 1),
        method="highs"
    )

    if result.success:
        max_robust_output = -result.fun * ys
        target_output = Y[target]
        shortfall = target_output - max_robust_output

        diagnostics.append({
            "year": year,
            "plant_id": plant_id,
            "target_output": target_output,
            "max_robust_output": max_robust_output,
            "shortfall_mwh": shortfall,
            "shortfall_percent": 100 * shortfall / target_output,
            "output_feasible": max_robust_output >= target_output - 1e-6
        })

display(pd.DataFrame(diagnostics).round(3))


In [ ]:

import os
import pandas as pd
import numpy as np

path = os.path.join(PROC, "robust_dea_corrected_results_final.csv")
res = pd.read_csv(path)

res["success"] = (
    res["status"].astype(str).str.lower().isin(["optimal", "0"])
    & res["efficiency"].notna()
)

summary = (
    res.groupby(["year", "method"])
    .agg(
        observations=("plant_id", "size"),
        feasible=("success", "sum"),
        infeasible=("success", lambda x: (~x).sum()),
        feasibility_rate=("success", "mean"),
        mean_efficiency=("efficiency", "mean"),
        median_efficiency=("efficiency", "median"),
        sd_efficiency=("efficiency", "std")
    )
    .reset_index()
)

summary["feasibility_rate"] *= 100

display(summary.round(4))

print("\nEDRO INFEASIBLE DMUS")
display(
    res[
        (res.method.str.upper() == "EDRO") &
        (~res.success)
    ][["year", "plant_id", "status"]]
)


In [ ]:

import os
import pandas as pd
import numpy as np

path = os.path.join(PROC, "robust_dea_corrected_results_final.csv")
res = pd.read_csv(path)

res = res[
    res["status"].astype(str).str.lower().eq("optimal")
    & res["efficiency"].notna()
].copy()

# One row per plant-year, with one column per method
wide = res.pivot_table(
    index=["year", "plant_id"],
    columns="method",
    values="efficiency",
    aggfunc="first"
).reset_index()

methods = ["EDRO", "GBU_equal", "BS_separate"]
wide = wide.dropna(subset=methods)

print("Matched plant-year observations:", len(wide))

rows = []

for year, g in wide.groupby("year"):
    rows.append({
        "year": year,
        "matched_plants": len(g),
        "EDRO_minus_GBU": (g["EDRO"] - g["GBU_equal"]).mean(),
        "EDRO_minus_BS": (g["EDRO"] - g["BS_separate"]).mean(),
        "BS_minus_GBU": (g["BS_separate"] - g["GBU_equal"]).mean(),
        "EDRO_below_GBU_pct": 100 * (g["EDRO"] < g["GBU_equal"]).mean(),
        "EDRO_below_BS_pct": 100 * (g["EDRO"] < g["BS_separate"]).mean()
    })

paired = pd.DataFrame(rows)
display(paired.round(5))


In [ ]:

import numpy as np
import pandas as pd

# Reuse the matched wide table if it is still available.
# Otherwise reconstruct it.
res = pd.read_csv(
    os.path.join(PROC, "robust_dea_corrected_results_final.csv")
)

res = res[
    res["status"].astype(str).str.lower().eq("optimal")
    & res["efficiency"].notna()
].copy()

wide = res.pivot_table(
    index=["year", "plant_id"],
    columns="method",
    values="efficiency",
    aggfunc="first"
).reset_index()

wide = wide.dropna(
    subset=["EDRO", "GBU_equal", "BS_separate"]
)

wide["EDRO_GBU"] = wide["EDRO"] - wide["GBU_equal"]
wide["EDRO_BS"] = wide["EDRO"] - wide["BS_separate"]

rng = np.random.default_rng(20261120)
B = 2000

def cluster_bootstrap(data, value_col, B=2000):
    plants = data["plant_id"].unique()
    grouped = {
        p: data.loc[data.plant_id == p, value_col].to_numpy()
        for p in plants
    }

    estimates = np.empty(B)

    for b in range(B):
        sampled = rng.choice(plants, size=len(plants), replace=True)
        vals = np.concatenate([grouped[p] for p in sampled])
        estimates[b] = np.mean(vals)

    return (
        data[value_col].mean(),
        np.quantile(estimates, 0.025),
        np.quantile(estimates, 0.975)
    )

results = []

for year, g in wide.groupby("year"):
    for comparison in ["EDRO_GBU", "EDRO_BS"]:
        mean, low, high = cluster_bootstrap(g, comparison, B)
        results.append({
            "year": year,
            "comparison": comparison,
            "mean_difference": mean,
            "CI_lower": low,
            "CI_upper": high
        })

# Pooled estimate: plant-cluster resampling across all years
for comparison in ["EDRO_GBU", "EDRO_BS"]:
    mean, low, high = cluster_bootstrap(wide, comparison, B)
    results.append({
        "year": "Pooled",
        "comparison": comparison,
        "mean_difference": mean,
        "CI_lower": low,
        "CI_upper": high
    })

bootstrap_results = pd.DataFrame(results)
display(bootstrap_results.round(5))

bootstrap_results.to_csv(
    os.path.join(PROC, "edro_paired_cluster_bootstrap.csv"),
    index=False
)


In [ ]:

from pathlib import Path
import shutil
import zipfile
import hashlib
import os

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
PROJECT = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = PROJECT / "processed"

PACKAGE = PROJECT / "EDRO_reproducibility_package"
ZIP_PATH = PROJECT / "EDRO_reproducibility_package.zip"

# Set True only if you also want the large original EIA files.
INCLUDE_RAW = False

if not PROJECT.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT}")

if not PROCESSED.exists():
    raise FileNotFoundError(f"Processed data folder not found: {PROCESSED}")

# Start clean
if PACKAGE.exists():
    shutil.rmtree(PACKAGE)

PACKAGE.mkdir(parents=True)
(PACKAGE / "data").mkdir()
(PACKAGE / "code").mkdir()
(PACKAGE / "manuscript").mkdir()
(PACKAGE / "documentation").mkdir()

# ------------------------------------------------------------
# 1. COPY ALL PROCESSED DATA
# ------------------------------------------------------------
data_files = list(PROCESSED.glob("*.csv"))

for src in data_files:
    shutil.copy2(src, PACKAGE / "data" / src.name)

print("Processed data files copied:", len(data_files))

# ------------------------------------------------------------
# 2. COPY EXISTING CODE AND NOTEBOOKS
# ------------------------------------------------------------
code_extensions = {".py", ".ipynb", ".R", ".jl"}

code_files = []

for src in PROJECT.rglob("*"):
    if not src.is_file():
        continue

    if PACKAGE in src.parents:
        continue

    if any(part in {
        ".ipynb_checkpoints", "__pycache__", ".git",
        ".venv", "venv", "site-packages"
    } for part in src.parts):
        continue

    if src.suffix.lower() in code_extensions:
        code_files.append(src)

for src in code_files:
    relative = src.relative_to(PROJECT)
    destination = PACKAGE / "code" / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, destination)

print("Code/notebook files copied:", len(code_files))

# ------------------------------------------------------------
# 3. OPTIONALLY INCLUDE ORIGINAL DATA
# ------------------------------------------------------------
if INCLUDE_RAW:
    for folder_name in ["raw", "extracted"]:
        src = PROJECT / folder_name
        if src.exists():
            shutil.copytree(
                src,
                PACKAGE / folder_name,
                dirs_exist_ok=True
            )

# ------------------------------------------------------------
# 4. INCLUDE THE MANUSCRIPT IF IT IS AVAILABLE
# ------------------------------------------------------------
manuscript_name = "event_responsive_robust_DEA_full_manuscript.tex"

possible_manuscripts = [
    PROJECT / manuscript_name,
    Path("/mnt/data") / manuscript_name
]

manuscript_found = False

for src in possible_manuscripts:
    if src.exists():
        shutil.copy2(src, PACKAGE / "manuscript" / manuscript_name)
        manuscript_found = True
        print("Manuscript copied from:", src)
        break

# Also copy any other TeX files already in the project
for src in PROJECT.rglob("*.tex"):
    if PACKAGE in src.parents:
        continue
    if src.name == manuscript_name and manuscript_found:
        continue
    shutil.copy2(src, PACKAGE / "manuscript" / src.name)

# ------------------------------------------------------------
# 5. CREATE REQUIREMENTS FILE
# ------------------------------------------------------------
requirements = """numpy
pandas
scipy
openpyxl
requests
statsmodels
matplotlib
"""

(PACKAGE / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

# ------------------------------------------------------------
# 6. CREATE README
# ------------------------------------------------------------
readme = f"""
EVENT-DRIVEN ROBUST DEA
REPRODUCIBILITY PACKAGE
=============================================

Purpose
-------
This folder contains the processed data, available source
code/notebooks, and manuscript files for the empirical
Event-Driven Robust DEA study.

Project source:
{PROJECT}

Data
----
The data/ folder contains the processed CSV files generated
during the study. These include, where available:

- Corrected NGCC plant-year panel
- Primary analysis sample
- Nominal VRS DEA results
- Gas-price event data
- Temperature event data
- Combined event exposure data
- EDRO budget allocations
- Leakage-safe uncertainty calibration
- Robust DEA results and summaries
- Feasibility and diagnostic results

Code
----
The code/ folder contains Python scripts and Jupyter notebooks
found inside the project directory.

Important:
If the current Deepnote notebook is not included, export it
from Deepnote as an .ipynb file, place it in the project folder,
and rerun the packaging cell.

Manuscript
----------
The manuscript/ folder contains the TeX manuscript if it was
available in the project or /mnt/data when this package was built.

If the manuscript is absent, upload the TeX file to the project
folder and rerun the packaging cell.

Reproduction notes
------------------
1. Install the packages listed in requirements.txt.
2. Run the notebook or scripts in code/.
3. The processed data in data/ are the intermediate inputs
   and outputs used in the empirical analysis.
4. The original EIA and NOAA source files are not included
   unless INCLUDE_RAW=True was selected.
5. Check file paths in the notebook before running it outside
   the original Deepnote project.

Methodological notes
--------------------
- Nominal DEA uses annual input-oriented VRS frontiers.
- The robust evaluation uses the separate fuel and output
  uncertainty channels described in the manuscript.
- EDRO allocations are based on lagged event information.
- Uncertainty radii are operational variability proxies;
  they should not be interpreted as causal event effects.
- The separate-channel BS benchmark uses its own budget
  in each physical constraint. Its protection level is not
  directly equivalent to the EDRO total allocation.

This package is an archive of available project files.
It does not independently certify that every notebook cell
has been executed successfully from a clean environment.
"""

(PACKAGE / "README.txt").write_text(readme, encoding="utf-8")

# ------------------------------------------------------------
# 7. CREATE FILE MANIFEST WITH CHECKSUMS
# ------------------------------------------------------------
manifest_lines = [
    "relative_path,size_bytes,sha256"
]

for src in sorted(PACKAGE.rglob("*")):
    if src.is_file():
        digest = hashlib.sha256(src.read_bytes()).hexdigest()
        relative = src.relative_to(PACKAGE)
        manifest_lines.append(
            f'"{relative}",{src.stat().st_size},{digest}'
        )

(PACKAGE / "documentation" / "file_manifest.csv").write_text(
    "\n".join(manifest_lines), encoding="utf-8"
)

# ------------------------------------------------------------
# 8. ZIP THE COMPLETE PACKAGE
# ------------------------------------------------------------
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for src in PACKAGE.rglob("*"):
        if src.is_file():
            zf.write(src, src.relative_to(PROJECT))

# ------------------------------------------------------------
# 9. REPORT
# ------------------------------------------------------------
print("\n" + "=" * 55)
print("PACKAGE CREATED")
print("=" * 55)
print("Folder:", PACKAGE)
print("ZIP:", ZIP_PATH)
print("Processed CSV files:", len(data_files))
print("Code/notebook files:", len(code_files))
print("Manuscript included:", manuscript_found)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / 1e6, 2))

print("\nFiles in package:")
for src in sorted(PACKAGE.rglob("*")):
    if src.is_file():
        print(" -", src.relative_to(PACKAGE))


In [ ]:
from pathlib import Path

P = Path("/datasets/_deepnote_work/eia_edro_project")

print("Project exists:", P.exists())
print("Processed exists:", (P / "processed").exists())
print("Processed CSV count:", len(list((P / "processed").glob("*.csv"))))
print("Root notebooks:", [x.name for x in P.glob("*.ipynb")])
print("Root Python files:", [x.name for x in P.glob("*.py")])
print("Root TeX files:", [x.name for x in P.glob("*.tex")])

In [ ]:
from pathlib import Path
import shutil
import zipfile
import hashlib
import pandas as pd

PROJECT = Path("/datasets/_deepnote_work/eia_edro_project")
PROCESSED = PROJECT / "processed"

PACKAGE = PROJECT / "EDRO_reproducibility_package"
ZIPFILE = PROJECT / "EDRO_reproducibility_package.zip"

# Clean previous package
if PACKAGE.exists():
    shutil.rmtree(PACKAGE)
if ZIPFILE.exists():
    ZIPFILE.unlink()

# Create folders
DATA = PACKAGE / "data"
CODE = PACKAGE / "code"
DOCS = PACKAGE / "documentation"
MANUSCRIPT = PACKAGE / "manuscript"

for folder in [DATA, CODE, DOCS, MANUSCRIPT]:
    folder.mkdir(parents=True, exist_ok=True)

# Copy all processed datasets
csv_files = sorted(PROCESSED.glob("*.csv"))
for f in csv_files:
    shutil.copy2(f, DATA / f.name)

# Record exactly what is included
manifest = []
for f in sorted(DATA.glob("*.csv")):
    manifest.append({
        "file": f"data/{f.name}",
        "size_bytes": f.stat().st_size,
        "sha256": hashlib.sha256(f.read_bytes()).hexdigest()
    })

pd.DataFrame(manifest).to_csv(
    DOCS / "file_manifest.csv", index=False
)

# README
readme = """
EVENT-DRIVEN ROBUST DEA — REPRODUCIBILITY PACKAGE

FOLDER STRUCTURE
----------------
data/           Processed CSV datasets used in the project.
code/           Reserved for exported Python scripts and notebooks.
manuscript/     Reserved for the final LaTeX manuscript.
documentation/  File manifest and this README.

DATA
----
The data folder contains copies of the processed CSV files
from the project's processed directory.

IMPORTANT
---------
The original EIA, NOAA and FRED source files are not included.
They must be downloaded separately to reproduce the entire
data preparation pipeline.

The Deepnote notebook containing the actual code has not yet
been exported into this project. Export the notebook as an
.ipynb file and place it in the code folder.

The manuscript must also be copied into the manuscript folder.

The robust DEA results should be checked against the repaired
EDRO zero-budget results before final publication.
"""

(PACKAGE / "README.txt").write_text(readme, encoding="utf-8")

# Create ZIP
with zipfile.ZipFile(ZIPFILE, "w", zipfile.ZIP_DEFLATED) as z:
    for f in PACKAGE.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(PROJECT))

print("PACKAGE CREATED")
print("CSV datasets copied:", len(csv_files))
print("Package folder:", PACKAGE)
print("ZIP file:", ZIPFILE)
print("ZIP size (MB):", round(ZIPFILE.stat().st_size / 1e6, 2))
print("First 15 datasets:")
for f in csv_files[:15]:
    print(" -", f.name)

In [ ]:
from pathlib import Path

P = Path("/datasets/_deepnote_work/eia_edro_project")

for name in [
    "EDRO_reproducibility_package",
    "EDRO_reproducibility_package.zip"
]:
    x = P / name
    print(name, "exists:", x.exists(),
          "size (MB):", round(x.stat().st_size / 1e6, 2) if x.is_file() else "-")

In [ ]:
from pathlib import Path
import zipfile
import os

PROJECT = Path("/datasets/_deepnote_work/eia_edro_project")
PACKAGE = PROJECT / "EDRO_reproducibility_package"
ZIPFILE = PROJECT / "EDRO_reproducibility_package.zip"

# Remove incomplete ZIP
if ZIPFILE.exists():
    ZIPFILE.unlink()

files = [f for f in PACKAGE.rglob("*") if f.is_file()]
print("Files to archive:", len(files))
print("Total size (MB):", round(sum(f.stat().st_size for f in files)/1e6, 2))

with zipfile.ZipFile(ZIPFILE, "w", compression=zipfile.ZIP_STORED) as z:
    for i, f in enumerate(files, 1):
        z.write(f, f.relative_to(PROJECT))
        if i % 10 == 0:
            print("Archived", i, "of", len(files))

print("ZIP CREATED:", ZIPFILE.exists())
print("ZIP SIZE (MB):", round(os.path.getsize(ZIPFILE)/1e6, 2))

In [ ]:
from pathlib import Path

P = Path("/datasets/_deepnote_work/eia_edro_project/EDRO_reproducibility_package")

for folder in ["data", "code", "manuscript", "documentation"]:
    d = P / folder
    files = list(d.glob("*")) if d.exists() else []
    print(f"\n{folder}: {len(files)} files")
    for f in files[:10]:
        print(" ", f.name)

In [ ]:
from pathlib import Path
import shutil

PROJECT = Path("/datasets/_deepnote_work/eia_edro_project")
SOURCE = PROJECT / "processed"
DATA = PROJECT / "EDRO_reproducibility_package" / "data"

DATA.mkdir(parents=True, exist_ok=True)

files = list(SOURCE.glob("*.csv"))
copied = 0
failed = []

for f in files:
    try:
        shutil.copy2(f, DATA / f.name)
        copied += 1
    except Exception as e:
        failed.append((f.name, str(e)))

print("Source CSV files:", len(files))
print("Successfully copied:", copied)
print("Failed:", len(failed))
print("Destination CSV files:", len(list(DATA.glob("*.csv"))))

if failed:
    print("Failures:", failed)

In [ ]:
from pathlib import Path

P = Path("/datasets/_deepnote_work/eia_edro_project/processed")

for f in sorted(P.glob("*.csv"), key=lambda x: x.stat().st_size, reverse=True):
    print(f"{f.name}: {f.stat().st_size / 1e6:.2f} MB")

In [ ]:
from pathlib import Path
import zipfile
import os

PROJECT = Path("/datasets/_deepnote_work/eia_edro_project")
SOURCE = PROJECT / "processed"
ZIPFILE = PROJECT / "EDRO_reproducibility_package.zip"

if ZIPFILE.exists():
    ZIPFILE.unlink()

files = sorted(SOURCE.glob("*.csv"))

with zipfile.ZipFile(
    ZIPFILE, "w", compression=zipfile.ZIP_STORED
) as z:
    for i, f in enumerate(files, 1):
        z.write(f, f"data/{f.name}")
        print(f"{i}/{len(files)}: {f.name}")

print("\nZIP CREATED")
print("CSV files:", len(files))
print("ZIP size (MB):", round(os.path.getsize(ZIPFILE)/1e6, 2))
print("Location:", ZIPFILE)

In [ ]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("Directory contents:")
for x in list(Path.cwd().iterdir())[:20]:
    print(x.name)

In [ ]:
from pathlib import Path
import shutil

source = Path("/datasets/_deepnote_work/eia_edro_project/EDRO_reproducibility_package.zip")
folder = source.parent

part_size = 20 * 1024 * 1024  # 20 MB

with open(source, "rb") as f:
    part = 1
    while True:
        chunk = f.read(part_size)
        if not chunk:
            break
        output = folder / f"EDRO_package_part{part}.bin"
        with open(output, "wb") as out:
            out.write(chunk)
        print(output.name, round(output.stat().st_size / 1e6, 2), "MB")
        part += 1

print("Splitting completed.")

In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

if "solve_one_dea_corrected" not in globals():
    raise RuntimeError("Execute corrected solver cell 54 first.")

# Load original calibration data and annual budget shares
df_s = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)
budgets = budgets.rename(columns={
    "Gamma_gas": "gamma_gas",
    "Gamma_heat": "gamma_heat"
})
df_s["year"] = df_s["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df_s = df_s.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

df_s = df_s[
    df_s.year.between(2022, 2025) &
    (df_s.state != "AK")
].copy()

# Original results, using Gamma = 3
baseline = pd.read_csv(
    os.path.join(PROC, "robust_dea_corrected_results_final.csv")
)

baseline = baseline[
    (baseline.method == "EDRO") &
    (baseline.status == "optimal")
][["year", "plant_id", "efficiency"]].rename(
    columns={"efficiency": "efficiency_Gamma3"}
)

# Alternative total budgets
BUDGET_LEVELS = [2.0, 4.0]

results = []
output_file = os.path.join(
    PROC, "edro_budget_sensitivity.csv"
)

for year, g in df_s.groupby("year"):

    g = g.sort_values("plant_id").reset_index(drop=True)

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)

    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    original_g = float(g.gamma_gas.iloc[0])
    original_h = float(g.gamma_heat.iloc[0])

    total_original = original_g + original_h
    gas_share = original_g / total_original

    print(f"\nYEAR {year}", flush=True)
    print("Original allocation:", original_g, original_h)

    for total_budget in BUDGET_LEVELS:

        gamma_g = total_budget * gas_share
        gamma_h = total_budget - gamma_g

        print(
            f"\nGamma={total_budget}: "
            f"gas={gamma_g:.4f}, heat={gamma_h:.4f}",
            flush=True
        )

        successful = 0

        for i in range(len(g)):

            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY,
                target=i,
                gamma_gas=gamma_g,
                gamma_heat=gamma_h
            )

            if status == "optimal":
                successful += 1

            results.append({
                "year": year,
                "plant_id": g.loc[i, "plant_id"],
                "gamma_total": total_budget,
                "gamma_gas": gamma_g,
                "gamma_heat": gamma_h,
                "efficiency": score,
                "status": status
            })

            if (i + 1) % 50 == 0:
                print(
                    f"{i+1}/{len(g)} "
                    f"(successful={successful})",
                    flush=True
                )

        print(
            "Successful:", successful,
            "of", len(g),
            flush=True
        )

    # Save checkpoint after each year
    pd.DataFrame(results).to_csv(output_file, index=False)

# Compare with original Gamma=3
alternative = pd.DataFrame(results)

comparison = alternative.merge(
    baseline,
    on=["year", "plant_id"],
    how="left"
)

comparison["difference"] = (
    comparison["efficiency"] -
    comparison["efficiency_Gamma3"]
)

summary = (
    comparison.groupby(["year", "gamma_total"])
    .agg(
        observations=("plant_id", "size"),
        feasible=("efficiency", "count"),
        mean_efficiency=("efficiency", "mean"),
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median")
    )
    .reset_index()
)

print("\nBUDGET SENSITIVITY RESULTS")
print(summary.round(5).to_string(index=False))

summary.to_csv(
    os.path.join(PROC, "edro_budget_sensitivity_summary.csv"),
    index=False
)

print("\nSaved:")
print(output_file)
print(os.path.join(PROC, "edro_budget_sensitivity_summary.csv"))


In [ ]:

import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# Load the original calibrated data and annual budgets
df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def find_col(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for x in candidates:
        key = x.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    find_col(budgets, ["year"]): "year",
    find_col(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    find_col(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

scales = [0.8, 1.0, 1.2]
results = []

for year in sorted(df["year"].unique()):

    g = (
        df[(df["year"] == year) & (df["state"] != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g["capacity_mw"].to_numpy(float)
    F = g["fuel_consumption_mmbtu"].to_numpy(float)
    Y = g["generation_mwh"].to_numpy(float)

    # Original calibrated radii
    rF_original = g["fuel_uncertainty_radius"].to_numpy(float)
    rY_original = g["output_uncertainty_radius"].to_numpy(float)

    gamma_g = float(g["gamma_gas"].iloc[0])
    gamma_h = float(g["gamma_heat"].iloc[0])

    print(f"\nYEAR {year}: {len(g)} DMUs")
    print(f"Fixed budgets: gas={gamma_g:.4f}, heat={gamma_h:.4f}")

    for scale in scales:

        rF = scale * rF_original
        rY = scale * rY_original

        scores = []
        failures = 0

        for target in range(len(g)):

            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY,
                target=target,
                gamma_gas=gamma_g,
                gamma_heat=gamma_h
            )

            if status == "optimal" and np.isfinite(score):
                scores.append(score)
            else:
                failures += 1

        mean_score = (
            float(np.mean(scores)) if scores else np.nan
        )

        results.append({
            "year": year,
            "radius_scale": scale,
            "observations": len(g),
            "feasible": len(scores),
            "failures": failures,
            "mean_efficiency": mean_score
        })

        print(
            f"Scale={scale:.1f}: "
            f"feasible={len(scores)}/{len(g)}, "
            f"mean efficiency={mean_score:.6f}, "
            f"failures={failures}"
        )

# Save detailed summary
sensitivity = pd.DataFrame(results)

output = os.path.join(
    PROC, "edro_calibration_scaling_sensitivity.csv"
)

sensitivity.to_csv(output, index=False)

print("\nCALIBRATION-SCALING SENSITIVITY")
print(sensitivity.to_string(index=False))
print("\nSaved:", output)


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def find_col(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for x in candidates:
        key = x.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    find_col(budgets, ["year"]): "year",
    find_col(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    find_col(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

scales = [0.8, 1.0, 1.2]
score_records = []

for year in sorted(df["year"].unique()):

    g = (
        df[(df["year"] == year) & (df["state"] != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g["capacity_mw"].to_numpy(float)
    F = g["fuel_consumption_mmbtu"].to_numpy(float)
    Y = g["generation_mwh"].to_numpy(float)

    rF0 = g["fuel_uncertainty_radius"].to_numpy(float)
    rY0 = g["output_uncertainty_radius"].to_numpy(float)

    gamma_g = float(g["gamma_gas"].iloc[0])
    gamma_h = float(g["gamma_heat"].iloc[0])

    print(f"\nScoring plants in {year}...")

    for scale in scales:

        for target in range(len(g)):

            score, status = solve_one_dea_corrected(
                K, F, Y,
                scale * rF0,
                scale * rY0,
                target=target,
                gamma_gas=gamma_g,
                gamma_heat=gamma_h
            )

            score_records.append({
                "year": year,
                "plant_id": g.loc[target, "plant_id"],
                "radius_scale": scale,
                "efficiency": (
                    score if status == "optimal" else np.nan
                )
            })

        print(f"  Scale {scale:.1f} completed")

scores = pd.DataFrame(score_records)

# Rank plants within each year and scale.
# Higher efficiency receives rank 1.
scores["rank"] = scores.groupby(
    ["year", "radius_scale"]
)["efficiency"].rank(
    ascending=False,
    method="average"
)

# Compare rankings using plants feasible at both scales.
comparisons = [(0.8, 1.0), (1.0, 1.2), (0.8, 1.2)]
rank_results = []
rank_changes = []

for year in sorted(scores["year"].unique()):

    sy = scores[scores["year"] == year]

    for scale_a, scale_b in comparisons:

        pair = sy[sy["radius_scale"].isin(
            [scale_a, scale_b]
        )].pivot(
            index="plant_id",
            columns="radius_scale",
            values="rank"
        ).dropna()

        if len(pair) >= 3:
            rho, pvalue = spearmanr(
                pair[scale_a], pair[scale_b]
            )
        else:
            rho, pvalue = np.nan, np.nan

        rank_results.append({
            "year": year,
            "comparison": f"{scale_a:.1f} vs {scale_b:.1f}",
            "common_plants": len(pair),
            "spearman_rho": rho,
            "p_value": pvalue
        })

        if scale_a == 0.8 and scale_b == 1.2:
            changes = pair.copy()
            changes["rank_change"] = (
                changes[scale_b] - changes[scale_a]
            )
            changes["absolute_rank_change"] = (
                changes["rank_change"].abs()
            )
            changes = changes.reset_index()
            changes["year"] = year
            rank_changes.append(changes)

rank_summary = pd.DataFrame(rank_results)

print("\nRANK STABILITY RESULTS")
print(rank_summary.to_string(index=False))

if rank_changes:
    changes_all = pd.concat(rank_changes, ignore_index=True)

    print("\nLARGEST RANK CHANGES (0.8 vs 1.2)")
    for year in sorted(changes_all["year"].unique()):
        top = (
            changes_all[changes_all["year"] == year]
            .sort_values("absolute_rank_change", ascending=False)
            .head(10)
        )
        print(f"\nYEAR {year}")
        print(top[
            ["plant_id", "0.8", "1.2", "rank_change"]
        ].to_string(index=False))

    changes_all.to_csv(
        os.path.join(PROC, "edro_plant_rank_changes.csv"),
        index=False
    )

scores.to_csv(
    os.path.join(PROC, "edro_individual_scaling_scores.csv"),
    index=False
)

rank_summary.to_csv(
    os.path.join(PROC, "edro_rank_stability_summary.csv"),
    index=False
)

print("\nSaved:")
print(os.path.join(PROC, "edro_individual_scaling_scores.csv"))
print(os.path.join(PROC, "edro_rank_stability_summary.csv"))
print(os.path.join(PROC, "edro_plant_rank_changes.csv"))


In [ ]:

import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

changes_all = pd.read_csv(
    os.path.join(PROC, "edro_plant_rank_changes.csv")
)

print("LARGEST RANK CHANGES (0.8 vs 1.2)")

for year in sorted(changes_all["year"].unique()):
    top = (
        changes_all[changes_all["year"] == year]
        .sort_values("absolute_rank_change", ascending=False)
        .head(10)
    )

    print(f"\nYEAR {year}")
    print(
        top[["plant_id", "0.8", "1.2", "rank_change"]]
        .to_string(index=False)
    )


In [ ]:
import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

# Save the results already calculated in memory
if "scores" in globals():
    scores.to_csv(
        os.path.join(PROC, "edro_individual_scaling_scores.csv"),
        index=False
    )
    print("Individual scores saved.")

if "rank_summary" in globals():
    rank_summary.to_csv(
        os.path.join(PROC, "edro_rank_stability_summary.csv"),
        index=False
    )
    print("\nRANK STABILITY")
    print(rank_summary.to_string(index=False))

if "changes_all" in globals():
    changes_all.to_csv(
        os.path.join(PROC, "edro_plant_rank_changes.csv"),
        index=False
    )

    print("\nLARGEST RANK CHANGES")
    for year in sorted(changes_all["year"].unique()):
        top = (
            changes_all[changes_all["year"] == year]
            .sort_values("absolute_rank_change", ascending=False)
            .head(10)
        )
        print(f"\nYEAR {year}")
        print(top.to_string(index=False))
else:
    print("\nchanges_all is not available in memory.")

In [ ]:

import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

scores = pd.read_csv(
    os.path.join(PROC, "edro_individual_scaling_scores.csv")
)

scales = [0.8, 1.0, 1.2]

for year in sorted(scores["year"].unique()):

    sy = scores[scores["year"] == year]

    # One row per plant, one column per calibration scale
    wide = sy.pivot(
        index="plant_id",
        columns="radius_scale",
        values="efficiency"
    ).reindex(columns=scales)

    # A missing efficiency indicates solver failure
    failed = wide.isna()

    always_failed = failed.all(axis=1)
    sometimes_failed = failed.any(axis=1) & ~always_failed

    print(f"\nYEAR {year}")
    print("-" * 45)
    print("Plants failing at all three scales:", int(always_failed.sum()))
    print("Plants failing at only some scales:", int(sometimes_failed.sum()))

    print("\nFailure counts by scale:")
    print(failed.sum().to_string())

    if sometimes_failed.any():
        print("\nPlants with scale-dependent failures:")
        print(failed[sometimes_failed].to_string())

    if always_failed.any():
        print("\nPlants failing at every scale:")
        print(list(wide.index[always_failed]))

# Save a complete failure-status table
failure_table = scores.copy()
failure_table["failed"] = failure_table["efficiency"].isna()

failure_table.to_csv(
    os.path.join(PROC, "edro_solver_failure_diagnostics.csv"),
    index=False
)

print("\nSaved solver failure diagnostics.")


In [ ]:

import os
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def find_col(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for x in candidates:
        key = x.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    find_col(budgets, ["year"]): "year",
    find_col(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    find_col(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

diagnostic = df[
    df["plant_id"].isin([710, 56407])
].copy()

cols = [
    "plant_id", "year", "state",
    "capacity_mw", "fuel_consumption_mmbtu",
    "generation_mwh",
    "gas_relative_radius", "heat_relative_radius",
    "fuel_uncertainty_radius", "output_uncertainty_radius",
    "gamma_gas", "gamma_heat"
]

print(diagnostic[cols].sort_values(
    ["plant_id", "year"]
).to_string(index=False))

print("\nMissing values:")
print(diagnostic[cols].isna().sum().to_string())

output = os.path.join(PROC, "edro_failed_plant_diagnostic.csv")
diagnostic[cols].to_csv(output, index=False)
print("\nSaved:", output)


In [ ]:

import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def find_col(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for x in candidates:
        key = x.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    find_col(budgets, ["year"]): "year",
    find_col(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    find_col(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

results = []

for year in sorted(df["year"].unique()):

    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF0 = g.fuel_uncertainty_radius.to_numpy(float)
    rY0 = g.output_uncertainty_radius.to_numpy(float)

    gamma_g = float(g.gamma_gas.iloc[0])
    gamma_h = float(g.gamma_heat.iloc[0])

    for plant in [710, 56407]:

        matches = np.where(g.plant_id.to_numpy() == plant)[0]

        if len(matches) == 0:
            continue

        target = int(matches[0])

        for scale in [0.8, 1.0, 1.2]:

            score, status = solve_one_dea_corrected(
                K, F, Y,
                scale * rF0,
                scale * rY0,
                target=target,
                gamma_gas=gamma_g,
                gamma_heat=gamma_h
            )

            results.append({
                "year": year,
                "plant_id": plant,
                "scale": scale,
                "score": score,
                "solver_status": status
            })

diagnostic = pd.DataFrame(results)

print(diagnostic.to_string(index=False))

diagnostic.to_csv(
    os.path.join(PROC, "edro_failed_plant_solver_messages.csv"),
    index=False
)

print("\nSaved solver messages.")


In [ ]:

import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

for year in [2022, 2023, 2024, 2025]:

    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)

    print(f"\nYEAR {year}")

    for plant in [710, 56407]:

        matches = np.where(g.plant_id.to_numpy() == plant)[0]

        if len(matches) == 0:
            continue

        target = int(matches[0])

        score, status = solve_one_dea_corrected(
            K, F, Y,
            np.zeros_like(F),
            np.zeros_like(Y),
            target=target,
            gamma_gas=0,
            gamma_heat=0
        )

        print(
            f"Plant {plant}: "
            f"score={score}, status={status}"
        )


In [ ]:
import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)

for year in [2022, 2023, 2024, 2025]:
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    print(f"\nYEAR {year}")

    for plant in [710, 56407]:
        matches = np.where(g.plant_id.to_numpy() == plant)[0]
        if len(matches) == 0:
            continue

        target = int(matches[0])
        row = g.iloc[target]

        tests = [
            ("Nominal", 0, 0),
            ("Gas only", row.gamma_gas, 0),
            ("Heat only", 0, row.gamma_heat),
            ("Both", row.gamma_gas, row.gamma_heat),
        ]

        for label, gg, gh in tests:
            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY,
                target=target,
                gamma_gas=gg,
                gamma_heat=gh
            )
            print(
                f"Plant {plant}, {label}: "
                f"score={score}, status={status}"
            )

In [ ]:
print(g.columns.tolist())
print(g.filter(regex="gamma|budget|gas|heat").columns.tolist())

In [ ]:
for name in ["annual_budget", "budget_allocation", "gamma_gas",
             "gamma_heat", "gamma_gas_by_year", "gamma_heat_by_year"]:
    if name in globals():
        print(name, "=", globals()[name])

In [ ]:
budgets = {
    2022: (2.1122911458, 0.8877088542),
    2023: (1.8892635042, 1.1107364958),
    2024: (0.0, 3.0),
    2025: (0.0, 3.0),
}

for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    print(f"\nYEAR {year}")

    for plant in [710, 56407]:
        matches = np.where(g.plant_id.to_numpy() == plant)[0]
        if len(matches) == 0:
            continue

        target = int(matches[0])

        for label, gas, heat in [
            ("Nominal", 0, 0),
            ("Gas only", gg, 0),
            ("Heat only", 0, gh),
            ("Both", gg, gh),
        ]:
            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY,
                target, gas, heat
            )
            print(f"Plant {plant}, {label}: {score}, {status}")

In [ ]:
for year, (_, gamma_heat) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    print(f"\nYEAR {year}, original heat budget={gamma_heat}")

    for plant in [710, 56407]:
        target = np.where(g.plant_id.to_numpy() == plant)[0][0]

        for gh in np.linspace(0, gamma_heat, 6):
            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY,
                target=target,
                gamma_gas=0,
                gamma_heat=float(gh)
            )

            print(
                f"Plant {plant}, heat budget={gh:.3f}: "
                f"{'feasible' if status == 'optimal' else 'infeasible'}"
            )

In [ ]:

def solve_one_dea_common_shock(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    # Scale physical quantities
    ks, fs, ys = np.median(K), np.median(F), np.median(Y)

    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    # Variables: theta, lambda, and active uncertainty blocks
    it = 0
    il = 1
    next_idx = il + n

    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg = next_idx
        iqg = next_idx + 1
        next_idx += n + 1

    if gamma_heat > 0:
        iph = next_idx
        iqh = next_idx + 1
        next_idx += n + 1

    nv = next_idx

    c = np.zeros(nv)
    c[it] = 1.0

    rows, rhs = [], []

    # Capacity constraint
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Fuel constraint
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    # Gas support-function constraints
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Common-shock robust output constraint
    row = np.zeros(nv)
    row[il:il+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat

        # Exclude the target plant: its shock affects
        # both sides of the comparison and cancels
        for j in range(n):
            if j != target:
                row[iqh+j] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Heat support-function constraints, excluding target
    if gamma_heat > 0:
        for j in range(n):
            if j != target:
                row = np.zeros(nv)
                row[il+j] = ry[j]
                row[iph] = -1.0
                row[iqh+j] = -1.0
                rows.append(row)
                rhs.append(0.0)

    # VRS constraint
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
       

In [ ]:
def solve_one_dea_common_shock(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    it, il = 0, 1
    next_idx = 1 + n

    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = next_idx, next_idx + 1
        next_idx += n + 1

    if gamma_heat > 0:
        iph, iqh = next_idx, next_idx + 1
        next_idx += n + 1

    nv = next_idx
    c = np.zeros(nv)
    c[it] = 1.0

    rows, rhs = [], []

    # Capacity
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Fuel
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    # Gas dual constraints
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

In [ ]:
print("Function defined:",
      callable(globals().get("solve_one_dea_common_shock")))

In [ ]:
import inspect

src = inspect.getsource(solve_one_dea_common_shock)
print(src)

In [ ]:
    # Common-shock heat constraint
    row = np.zeros(nv)
    row[il:il+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        for j in range(n):
            if j != target:
                row[iqh+j] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Heat dual constraints
    if gamma_heat > 0:
        for j in range(n):
            if j != target:
                row = np.zeros(nv)
                row[il+j] = ry[j]
                row[iph] = -1.0
                row[iqh+j] = -1.0
                rows.append(row)
                rhs.append(0.0)

    # VRS constraint
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=[(0, None)] * nv,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message

In [ ]:
print("Original solver preserved:",
      callable(solve_one_dea_corrected))

In [ ]:
import inspect

original_solver_source = inspect.getsource(solve_one_dea_corrected)

print("Original solver source captured:",
      len(original_solver_source), "characters")

In [ ]:
common_shock_source = original_solver_source.replace(
    "def solve_one_dea_corrected(",
    "def solve_one_dea_common_shock("
)

old_heat_block = """        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = ry[j]
            row[iph] = -1.0
            row[iqh+j] = -1.0
            rows.append(row)
            rhs.append(0.0)"""

new_heat_block = """        for j in range(n):
            if j == target:
                continue
            row = np.zeros(nv)
            row[il+j] = ry[j]
            row[iph] = -1.0
            row[iqh+j] = -1.0
            rows.append(row)
            rhs.append(0.0)"""

if old_heat_block not in common_shock_source:
    print("ERROR: Original heat block not found.")
else:
    common_shock_source = common_shock_source.replace(
        old_heat_block, new_heat_block, 1
    )
    exec(common_shock_source, globals())
    print("Alternative solver created:",
          callable(solve_one_dea_common_shock))

In [ ]:
budgets = {
    2022: (2.1122911458, 0.8877088542),
    2023: (1.8892635042, 1.1107364958),
    2024: (0.0, 3.0),
    2025: (0.0, 3.0)
}

for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    print(f"\nYEAR {year}")

    for plant in [710, 56407]:
        target = np.where(g.plant_id.to_numpy() == plant)[0][0]

        old_score, old_status = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, gg, gh
        )

        new_score, new_status = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )

        print(
            f"Plant {plant}: "
            f"Original={old_status}; "
            f"Common-shock={new_score}, {new_status}"
        )

In [ ]:
common_results = []

for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    for target in range(len(g)):
        score, status = solve_one_dea_common_shock(
            K, F, Y, rF, rY,
            target, gg, gh
        )

        common_results.append({
            "year": year,
            "plant_id": g.iloc[target].plant_id,
            "score": score,
            "status": status
        })

common_df = pd.DataFrame(common_results)

summary = common_df.groupby("year").agg(
    observations=("plant_id", "count"),
    feasible=("score", "count"),
    mean_efficiency=("score", "mean"),
    median_efficiency=("score", "median")
)

print(summary)

common_df.to_csv(
    os.path.join(PROC, "edro_common_shock_results.csv"),
    index=False
)

In [ ]:
budgets = {
    2022: (2.1122911458, 0.8877088542),
    2023: (1.8892635042, 1.1107364958),
    2024: (0.0, 3.0),
    2025: (0.0, 3.0)
}

print("Annual budgets restored:", budgets)

In [ ]:
import os
import pandas as pd
import numpy as np

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

df = pd.read_csv(
    os.path.join(
        PROC,
        "ngcc_lagged_uncertainty_coefficients.csv"
    )
)

print("Data loaded:", df.shape)
print("Years:", sorted(df["year"].unique()))

In [ ]:
print("Original solver:",
      callable(globals().get("solve_one_dea_corrected")))

print("Common-shock solver:",
      callable(globals().get("solve_one_dea_common_shock")))

In [ ]:
print("Original solver:",
      callable(globals().get("solve_one_dea_corrected")))

In [ ]:
def solve_one_dea_corrected(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    it, il = 0, 1
    next_idx = 1 + n

    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = next_idx, next_idx + 1
        next_idx += n + 1

    if gamma_heat > 0:
        iph, iqh = next_idx, next_idx + 1
        next_idx += n + 1

    nv = next_idx
    c = np.zeros(nv)
    c[it] = 1.0

    rows, rhs = [], []

    # Capacity
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Fuel
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    # Gas support constraints
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Output
    row = np.zeros(nv)
    row[il:il+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Heat support constraints
    if gamma_heat > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = ry[j]
            row[iph] = -1.0
            row[iqh+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # VRS
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=[(0, None)] * nv,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message


print("Original solver restored:",
      callable(solve_one_dea_corrected))

In [ ]:
import numpy as np

test_plants = [710, 56407]

for year in [2022, 2023, 2024, 2025]:
    d = df[df["year"] == year].copy().reset_index(drop=True)

    K = d["capacity_mw"].to_numpy(dtype=float)
    F = d["fuel_consumption_mmbtu"].to_numpy(dtype=float)
    Y = d["generation_mwh"].to_numpy(dtype=float)

    # Nominal model: no uncertainty
    rF = np.zeros(len(d))
    rY = np.zeros(len(d))

    print(f"\nYEAR {year}")

    for plant in test_plants:
        positions = np.where(d["plant_id"].to_numpy() == plant)[0]

        if len(positions) == 0:
            print(f"Plant {plant}: NOT FOUND")
            continue

        target = int(positions[0])

        score, status = solve_one_dea_corrected(
            K, F, Y, rF, rY, target,
            gamma_gas=0.0,
            gamma_heat=0.0
        )

        print(f"Plant {plant}: score={score:.8f}, status={status}")

In [ ]:
from scipy.optimize import linprog

print("linprog available:", callable(linprog))

In [ ]:
for year in [2022, 2023, 2024, 2025]:
    d = df[df["year"] == year].copy().reset_index(drop=True)

    K = d["capacity_mw"].to_numpy(dtype=float)
    F = d["fuel_consumption_mmbtu"].to_numpy(dtype=float)
    Y = d["generation_mwh"].to_numpy(dtype=float)
    rF = d["fuel_uncertainty_radius"].to_numpy(dtype=float)
    rY = d["output_uncertainty_radius"].to_numpy(dtype=float)

    gamma_gas, gamma_heat = budgets[year]

    print(f"\nYEAR {year} | gas={gamma_gas:.4f}, heat={gamma_heat:.4f}")

    for plant in [710, 56407]:
        positions = np.where(d["plant_id"].to_numpy() == plant)[0]

        if len(positions) == 0:
            print(f"Plant {plant}: NOT FOUND")
            continue

        target = int(positions[0])

        score, status = solve_one_dea_corrected(
            K, F, Y, rF, rY, target,
            gamma_gas=gamma_gas,
            gamma_heat=gamma_heat
        )

        print(f"Plant {plant}: score={score}, status={status}")

In [ ]:
cols = [
    "plant_id",
    "year",
    "state",
    "heat_relative_radius",
    "output_uncertainty_radius"
]

print("Available columns:")
print([c for c in cols if c in df.columns])

print("\nHeat uncertainty by state and year:")
print(
    df.groupby(["year", "state"])
      .agg(
          plants=("plant_id", "nunique"),
          heat_radius_mean=("heat_relative_radius", "mean"),
          heat_radius_sd=("heat_relative_radius", "std"),
          heat_radius_min=("heat_relative_radius", "min"),
          heat_radius_max=("heat_relative_radius", "max")
      )
      .round(6)
      .head(30)
      .to_string()
)

In [ ]:
print("All dataset columns:")
for c in df.columns:
    print(c)

print("\nExample plant-level heat calibration:")
display(
    df.loc[
        df["year"].eq(2022) & df["state"].eq("GA"),
        [
            c for c in df.columns
            if any(k in c.lower() for k in
                   ["plant", "state", "year", "heat", "temp", "degree", "radius"])
        ]
    ].head(10)
)

In [ ]:
# Show the heat-calibration fields without truncating columns
cols = [
    "plant_id", "plant_name", "state", "year",
    "n_heat", "heat_median", "delta_heat",
    "heat_relative_radius", "output_uncertainty_radius"
]

print(
    df.loc[
        (df["year"] == 2022) & (df["state"] == "GA"),
        cols
    ].to_string(index=False)
)

print("\nWithin-state variation in heat calibration:")
print(
    df.groupby(["year", "state"])
      .agg(
          plants=("plant_id", "nunique"),
          n_heat_values=("n_heat", "nunique"),
          heat_median_values=("heat_median", "nunique"),
          delta_heat_values=("delta_heat", "nunique"),
          radius_values=("heat_relative_radius", "nunique")
      )
      .head(20)
      .to_string()
)

In [ ]:
# Check whether the heat impact is proportional to plant output
df["heat_impact_ratio"] = (
    df["output_uncertainty_radius"] / df["generation_mwh"]
)

print(
    df.groupby("year")
      .agg(
          median_heat_radius=("heat_relative_radius", "median"),
          median_impact_ratio=("heat_impact_ratio", "median"),
          corr_delta_radius=(
              "delta_heat",
              lambda x: x.corr(df.loc[x.index, "heat_relative_radius"])
          ),
          corr_impact_radius=(
              "heat_impact_ratio",
              lambda x: x.corr(df.loc[x.index, "heat_relative_radius"])
          )
      )
      .round(5)
)

In [ ]:
# Inspect the notebook's calibration-related variables and formulas
for name in [
    "delta_heat",
    "heat_relative_radius",
    "output_uncertainty_radius",
    "heat_median"
]:
    print(f"\n{name}:")
    print(df[name].describe().to_string())

In [ ]:

# Inspect calibration-related variables and their values
terms = [
    "heat_median", "delta_heat",
    "heat_relative_radius", "output_uncertainty_radius",
    "n_heat", "heat_variability", "heat_rate_mmbtu_per_mwh",
    "capacity_factor", "generation_mwh"
]

print("CALIBRATION VARIABLES")
for name in terms:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: type={type(obj).__name__}")
        if np.isscalar(obj):
            print("  value:", obj)

print("\nDATAFRAME CALIBRATION COLUMNS")
for name in terms:
    if name in df.columns:
        print(f"\n{name}:")
        print(df[name].describe().to_string())


In [ ]:

import os
import pandas as pd
import numpy as np

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

path = os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")

df = pd.read_csv(path)

print("Loaded:", path)
print("Shape:", df.shape)
print("\nCalibration columns:")
for c in df.columns:
    if any(term in c.lower() for term in
           ["heat", "output_uncertainty", "generation", "radius"]):
        print(c)


In [ ]:

cols = [
    "plant_id", "year", "state",
    "n_heat", "heat_median", "delta_heat",
    "heat_relative_radius", "output_uncertainty_radius",
    "generation_mwh"
]

print(df[cols].head(12).to_string(index=False))

print("\nChecking exact relationships:")
for name, expression in {
    "radius / delta_heat": df["heat_relative_radius"] / df["delta_heat"],
    "output radius / generation": (
        df["output_uncertainty_radius"] / df["generation_mwh"]
    )
}.items():
    print(f"\n{name}:")
    print(expression.describe().to_string())


In [ ]:

# Identify how the plant-specific heat radius relates to heat_median
d = df.dropna(subset=[
    "heat_median", "delta_heat", "heat_relative_radius"
]).copy()

d["radius_ratio"] = (
    d["heat_relative_radius"] / d["delta_heat"]
)

print("CORRELATIONS")
print(d[[
    "heat_median", "delta_heat",
    "heat_relative_radius", "radius_ratio"
]].corr().round(5).to_string())

print("\nRATIO BY HEAT-MEDIAN QUANTILE")
d["heat_median_group"] = pd.qcut(
    d["heat_median"], 5, duplicates="drop"
)
print(
    d.groupby("heat_median_group", observed=True)
     ["radius_ratio"].agg(["count", "mean", "min", "max"])
     .round(5).to_string()
)


In [ ]:

d = df.dropna(subset=[
    "heat_median", "delta_heat", "heat_relative_radius"
]).copy()

d["ratio"] = d["heat_relative_radius"] / d["delta_heat"]

# Test whether the adjustment follows a simple inverse-exposure rule
d["candidate_1"] = 1 / (1 + d["heat_median"])
d["candidate_2"] = np.exp(-d["heat_median"])
d["candidate_3"] = 1 / (1 + d["heat_median"] / d["delta_heat"])

print("Correlation with observed radius ratio:")
for c in ["candidate_1", "candidate_2", "candidate_3"]:
    print(c, round(d["ratio"].corr(d[c]), 6))

print("\nMean absolute approximation errors:")
for c in ["candidate_1", "candidate_2", "candidate_3"]:
    print(c, round((d["ratio"] - d[c]).abs().mean(), 6))


In [ ]:

d = df.dropna(subset=[
    "heat_median", "delta_heat", "heat_relative_radius"
]).copy()

d["ratio"] = d["heat_relative_radius"] / d["delta_heat"]

# Test whether the adjustment follows a simple inverse-exposure rule
d["candidate_1"] = 1 / (1 + d["heat_median"])
d["candidate_2"] = np.exp(-d["heat_median"])
d["candidate_3"] = 1 / (1 + d["heat_median"] / d["delta_heat"])

print("Correlation with observed radius ratio:")
for c in ["candidate_1", "candidate_2", "candidate_3"]:
    print(c, round(d["ratio"].corr(d[c]), 6))

print("\nMean absolute approximation errors:")
for c in ["candidate_1", "candidate_2", "candidate_3"]:
    print(c, round((d["ratio"] - d[c]).abs().mean(), 6))


In [ ]:

import os

BASE = "/datasets/_deepnote_work/eia_edro_project"

terms = [
    "heat_relative_radius",
    "output_uncertainty_radius",
    "heat_median"
]

print("PROJECT FILES CONTAINING CALIBRATION FORMULAS")
for root, dirs, files in os.walk(BASE):
    dirs[:] = [x for x in dirs if x not in [
        ".git", "__pycache__", ".ipynb_checkpoints"
    ]]
    for filename in files:
        if filename.endswith((".py", ".ipynb", ".md")):
            path = os.path.join(root, filename)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                found = [t for t in terms if t in content]
                if found:
                    print("\nFILE:", path)
                    print("MATCHES:", found)
                    for term in found:
                        pos = content.find(term)
                        print("\n", content[max(0,pos-350):pos+500])
            except Exception:
                pass


In [ ]:

d = df.dropna(subset=[
    "heat_median", "delta_heat", "heat_relative_radius"
]).copy()

d["ratio"] = d["heat_relative_radius"] / d["delta_heat"]

# Fit a simple exponential adjustment
from scipy.optimize import curve_fit

def model(x, a, b):
    return a * np.exp(-b * x)

params, _ = curve_fit(
    model,
    d["heat_median"].values,
    d["ratio"].values,
    p0=[1.0, 0.5],
    maxfev=10000
)

d["fitted_ratio"] = model(d["heat_median"], *params)

print("Estimated adjustment:")
print("a =", round(params[0], 6))
print("b =", round(params[1], 6))
print("Mean absolute error =", round(
    np.mean(np.abs(d["ratio"] - d["fitted_ratio"])), 6
))
print("Maximum absolute error =", round(
    np.max(np.abs(d["ratio"] - d["fitted_ratio"])), 6
))
print("\nObserved versus fitted:")
print(d[["heat_median", "ratio", "fitted_ratio"]].head(15).round(5).to_string(index=False))


In [ ]:

d["absolute_error"] = (
    d["ratio"] - d["fitted_ratio"]
).abs()

print("LARGEST CALIBRATION DISCREPANCIES")
print(
    d.nlargest(15, "absolute_error")[
        ["plant_id", "year", "state", "heat_median",
         "delta_heat", "ratio", "fitted_ratio", "absolute_error"]
    ].round(5).to_string(index=False)
)

print("\nERROR BY YEAR")
print(
    d.groupby("year")["absolute_error"]
     .agg(["count", "mean", "median", "max"])
     .round(5).to_string()
)


In [ ]:

plant = df[df["plant_id"] == 54592].sort_values("year")

cols = [
    "plant_id", "year", "state", "n_heat",
    "heat_median", "delta_heat",
    "heat_relative_radius", "output_uncertainty_radius",
    "generation_mwh"
]

print(plant[cols].to_string(index=False))


In [ ]:
d = df.dropna(subset=[
    "n_heat", "heat_median",
    "delta_heat", "heat_relative_radius"
]).copy()

d["radius_ratio"] = (
    d["heat_relative_radius"] / d["delta_heat"]
)

print(
    d.groupby("n_heat")
     .agg(
         plants=("plant_id", "nunique"),
         median_heat=("heat_median", "median"),
         median_ratio=("radius_ratio", "median"),
         min_ratio=("radius_ratio", "min"),
         max_ratio=("radius_ratio", "max")
     )
     .round(5)
     .to_string()
)

In [ ]:
required = [
    "plant_id", "year", "state",
    "generation_mwh", "output_uncertainty_radius",
    "gamma_heat"
]

print("Required columns:")
for c in required:
    print(f"{c}: {'OK' if c in df.columns else 'MISSING'}")

In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.optimize import linprog

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

# Reload calibrated panel and annual budgets
df = pd.read_csv(
    os.path.join(PROC, "ngcc_lagged_uncertainty_coefficients.csv")
)
budgets = pd.read_csv(
    os.path.join(PROC, "edro_annual_budget_allocation.csv")
)

def find_col(data, candidates):
    lookup = {
        c.lower().replace("_", "").replace(" ", ""): c
        for c in data.columns
    }
    for x in candidates:
        key = x.lower().replace("_", "").replace(" ", "")
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column: {candidates}")

budgets = budgets.rename(columns={
    find_col(budgets, ["year"]): "year",
    find_col(budgets, ["gamma_gas", "gas_gamma"]): "gamma_gas",
    find_col(budgets, ["gamma_heat", "heat_gamma"]): "gamma_heat"
})

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year", how="left", validate="many_to_one"
)

df = df[
    df["year"].between(2022, 2025) &
    (df["state"] != "AK")
].copy()

print("Panel loaded:", df.shape)
print("Annual heat budgets:")
print(df.groupby("year")["gamma_heat"].first())

# Cell 61 diagnostic
failed_ids = [(2024, 56407), (2025, 710), (2025, 56407)]
diagnostics = []

for year, plant_id in failed_ids:
    g = df[df.year == year].reset_index(drop=True)

    target_rows = g.index[g.plant_id == plant_id].tolist()
    if not target_rows:
        print("Not found:", year, plant_id)
        continue

    target = target_rows[0]
    Y = g.generation_mwh.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    ys = np.median(Y)
    y, ry = Y / ys, rY / ys
    gamma = float(g.gamma_heat.iloc[0])
    n = len(g)

    c = np.r_[-y, gamma, np.ones(n)]
    A_ub = []
    for j in range(n):
        row = np.zeros(2*n + 1)
        row[j] = ry[j]
        row[n] = -1
        row[n+1+j] = -1
        A_ub.append(row)

    A_eq = np.zeros((1, 2*n + 1))
    A_eq[0, :n] = 1

    result = linprog(
        c,
        A_ub=np.asarray(A_ub),
        b_ub=np.zeros(n),
        A_eq=A_eq,
        b_eq=[1.0],
        bounds=[(0, None)] * (2*n + 1),
        method="highs"
    )

    if result.success:
        maximum = -result.fun * ys
        target_output = Y[target]
        shortfall = target_output - maximum

        diagnostics.append({
            "year": year,
            "plant_id": plant_id,
            "target_output": target_output,
            "max_robust_output": maximum,
            "shortfall_mwh": shortfall,
            "shortfall_percent": 100 * shortfall / target_output,
            "output_feasible": maximum >= target_output - 1e-6
        })
    else:
        print("Diagnostic LP failed:", year, plant_id, result.message)

print("\nROBUST OUTPUT FEASIBILITY")
print(pd.DataFrame(diagnostics).round(3).to_string(index=False))


In [ ]:

from scipy.optimize import linprog
import numpy as np
import pandas as pd

failed_ids = [(2024, 56407), (2025, 710), (2025, 56407)]

def maximum_robust_output(Y, rY, gamma):
    scale = np.median(Y)
    y, ry = Y / scale, rY / scale
    n = len(Y)

    c = np.r_[-y, gamma, np.ones(n)]

    A = []
    for j in range(n):
        row = np.zeros(2*n + 1)
        row[j] = ry[j]
        row[n] = -1
        row[n+1+j] = -1
        A.append(row)

    Aeq = np.zeros((1, 2*n + 1))
    Aeq[0, :n] = 1

    res = linprog(
        c,
        A_ub=np.asarray(A),
        b_ub=np.zeros(n),
        A_eq=Aeq,
        b_eq=[1],
        bounds=[(0, None)] * (2*n + 1),
        method="highs"
    )

    if not res.success:
        raise RuntimeError(res.message)

    return -res.fun * scale


rows = []

for year, plant_id in failed_ids:
    g = df[
        (df.year == year) & (df.state != "AK")
    ].reset_index(drop=True)

    target = g.index[g.plant_id == plant_id][0]
    Y = g.generation_mwh.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)
    target_output = Y[target]

    low, high = 0.0, 3.0

    # Check whether feasibility is possible at zero budget
    if maximum_robust_output(Y, rY, low) < target_output:
        critical = np.nan
    else:
        for _ in range(40):
            mid = (low + high) / 2
            attainable = maximum_robust_output(Y, rY, mid)

            if attainable >= target_output:
                low = mid
            else:
                high = mid

        critical = low

    rows.append({
        "year": year,
        "plant_id": plant_id,
        "actual_heat_budget": float(g.gamma_heat.iloc[0]),
        "critical_heat_budget": critical
    })

print(pd.DataFrame(rows).round(5).to_string(index=False))


In [ ]:
for year, plant_id in [(2024, 56407), (2025, 710), (2025, 56407)]:
    g = df[df.year == year].sort_values(
        "generation_mwh", ascending=False
    ).reset_index(drop=True)

    target = g[g.plant_id == plant_id].iloc[0]
    rank = g.index[g.plant_id == plant_id][0] + 1

    print(f"\nYEAR {year} — PLANT {plant_id}")
    print("Output rank:", rank, "of", len(g))
    print("Target output:", round(target.generation_mwh, 2))
    print("Largest output:", round(g.generation_mwh.max(), 2))
    print("Second-largest output:", round(
        g.generation_mwh.iloc[1], 2
    ))
    print("\nTop five reference plants:")
    print(
        g[["plant_id", "plant_name", "generation_mwh"]]
        .head(5).to_string(index=False)
    )

In [ ]:
for year in sorted(df.year.unique()):
    g = df[df.year == year].copy()

    top = g.loc[g.generation_mwh.idxmax()]

    print(
        f"Year {year}: "
        f"top plant={int(top.plant_id)}, "
        f"name={top.plant_name}, "
        f"output={top.generation_mwh:,.1f}, "
        f"heat radius={top.output_uncertainty_radius:,.1f}, "
        f"heat budget={top.gamma_heat if 'gamma_heat' in g.columns else 'not merged'}"
    )

In [ ]:
import pandas as pd
import numpy as np
import time

start = time.time()
results = []

for year in sorted(df["year"].unique()):
    g = df[df["year"] == year].copy()

    for _, target in g.iterrows():
        try:
            result = solve_one_dea_corrected(
                target=target,
                reference=g,
                gamma_gas=float(target["gamma_gas"]),
                gamma_heat=float(target["gamma_heat"])
            )

            # Adapt only if your solver returns a different structure.
            if isinstance(result, dict):
                score = result.get("theta", result.get("score", np.nan))
                status = result.get("status", "unknown")
                feasible = result.get("success", np.isfinite(score))
            else:
                score = result
                status = "returned"
                feasible = np.isfinite(score)

            results.append({
                "year": int(year),
                "plant_id": target["plant_id"],
                "plant_name": target["plant_name"],
                "feasible": bool(feasible),
                "score": score,
                "status": status
            })

        except Exception as e:
            results.append({
                "year": int(year),
                "plant_id": target["plant_id"],
                "plant_name": target["plant_name"],
                "feasible": False,
                "score": np.nan,
                "status": str(e)[:150]
            })

    print(f"Year {year} completed: {len(g)} plants")

results_df = pd.DataFrame(results)

print("\nSUMMARY")
print(
    results_df.groupby("year").agg(
        plants=("plant_id", "count"),
        feasible=("feasible", "sum"),
        infeasible=("feasible", lambda x: (~x).sum())
    ).to_string()
)

print("\nINFEASIBLE PLANTS")
print(
    results_df.loc[~results_df["feasible"],
                   ["year", "plant_id", "plant_name", "status"]]
    .to_string(index=False)
)

print(f"\nElapsed time: {(time.time()-start)/60:.2f} minutes")

In [ ]:
# Classify infeasible DMUs by their nominal output position.

classification = []

for year in sorted(df["year"].unique()):
    g = df[df.year == year].copy()
    max_output = g["generation_mwh"].max()

    bad = results_df[
        (results_df.year == year) &
        (~results_df.feasible)
    ]

    for _, r in bad.iterrows():
        target = g[g.plant_id == r.plant_id].iloc[0]

        classification.append({
            "year": year,
            "plant_id": r.plant_id,
            "plant_name": r.plant_name,
            "output_rank_1": np.isclose(
                target["generation_mwh"], max_output
            ),
            "output": target["generation_mwh"],
            "maximum_output": max_output,
            "heat_radius": target["output_uncertainty_radius"],
            "gamma_heat": target["gamma_heat"]
        })

classification_df = pd.DataFrame(classification)

print(classification_df.to_string(index=False))

In [ ]:
import inspect

print("SOLVER SIGNATURE:")
print(inspect.signature(solve_one_dea_corrected))

print("\nRESULTS DIAGNOSTIC:")
print(results_df["status"].value_counts(dropna=False).head(10))

print("\nFIRST FIVE RESULTS:")
print(results_df.head().to_string(index=False))

In [ ]:
%whos

In [ ]:
print("Result columns:", results_df.columns.tolist())

print("\nStatus frequencies:")
print(results_df["status"].value_counts(dropna=False).head(10))

print("\nFirst five solver results:")
print(results_df.head().to_string(index=False))

In [ ]:
for i, cell in enumerate(In[-100:]):
    if "def solve_one_dea_corrected" in cell:
        print("Found solver definition at recent history index:", i)
        print(cell)

In [ ]:
matches = [
    cell for cell in In[-100:]
    if "def solve_one_dea_corrected" in cell
]

print(matches[0] if matches else "Solver definition not found")

In [ ]:
matches = [
    cell for cell in In
    if cell.lstrip().startswith("def solve_one_dea_corrected")
]

print("Number of definitions found:", len(matches))

if matches:
    print(matches[-1])

In [ ]:
import inspect
print(inspect.signature(solve_one_dea_corrected))

In [ ]:
import numpy as np
import pandas as pd
import inspect

print(inspect.signature(solve_one_dea_corrected))

In [ ]:
import inspect
print(inspect.getsource(solve_one_dea_corrected))

In [ ]:
print(df.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd
import os
import time

# Work on a clean copy, grouped by year
all_results = []
start_time = time.time()
total = len(df)
completed = 0

for year, g in df.groupby("year", sort=True):

    g = g.reset_index(drop=True)

    K  = g["capacity_mw"].to_numpy(dtype=float)
    F  = g["fuel_consumption_mmbtu"].to_numpy(dtype=float)
    Y  = g["generation_mwh"].to_numpy(dtype=float)
    rF = g["fuel_uncertainty_radius"].to_numpy(dtype=float)
    rY = g["output_uncertainty_radius"].to_numpy(dtype=float)

    gamma_gas = float(g["gamma_gas"].iloc[0])
    gamma_heat = float(g["gamma_heat"].iloc[0])

    for target in range(len(g)):
        try:
            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY, target,
                gamma_gas=gamma_gas,
                gamma_heat=gamma_heat
            )
        except Exception as e:
            score, status = np.nan, str(e)

        all_results.append({
            "year": int(year),
            "plant_id": g.loc[target, "plant_id"],
            "plant_name": g.loc[target, "plant_name"],
            "state": g.loc[target, "state"],
            "score": score,
            "status": status,
            "gamma_gas": gamma_gas,
            "gamma_heat": gamma_heat
        })

        completed += 1

    print(f"Year {year}: {len(g)} plants completed "
          f"({completed}/{total} total).")

# Assemble and save
results_df = pd.DataFrame(all_results)

out_file = os.path.join(PROC, "edro_robust_dea_results.csv")
results_df.to_csv(out_file, index=False)

print("\nSOLVER SUMMARY")
print("Total plants:", len(results_df))
print("Optimal:", (results_df["status"] == "optimal").sum())
print("Unsuccessful:", (results_df["status"] != "optimal").sum())
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Saved:", out_file)

In [ ]:
# Inspect unsuccessful LPs
failed = results_df[results_df["status"] != "optimal"].copy()

print("UNSUCCESSFUL CASES")
print(
    failed[
        ["year", "plant_id", "plant_name", "state",
         "gamma_gas", "gamma_heat", "status"]
    ].to_string(index=False)
)

print("\nSOLVER MESSAGE COUNTS")
print(failed["status"].value_counts().to_string())

print("\nRESULT VALIDATION")
print("Duplicate plant-year records:",
      results_df.duplicated(["plant_id", "year"]).sum())
print("Missing scores among optimal solves:",
      results_df.loc[results_df["status"] == "optimal", "score"].isna().sum())
print("Non-finite scores among optimal solves:",
      (~np.isfinite(results_df.loc[
          results_df["status"] == "optimal", "score"
      ])).sum())

In [ ]:
from scipy.optimize import linprog
import numpy as np
import pandas as pd

failed = results_df[results_df["status"] != "optimal"]
diagnostics = []

for _, case in failed.iterrows():

    year = int(case["year"])
    plant_id = case["plant_id"]

    g = df[df["year"] == year].reset_index(drop=True)

    K = g["capacity_mw"].to_numpy(float)
    Y = g["generation_mwh"].to_numpy(float)
    rY = g["output_uncertainty_radius"].to_numpy(float)

    target = int(np.flatnonzero(g["plant_id"].to_numpy() == plant_id)[0])

    n = len(g)
    gamma = float(case["gamma_heat"])

    # Maximize robust reference output under VRS
    # Variables: lambda[0:n], pi, q[0:n]
    c = np.r_[-Y, gamma, np.ones(n)]

    A = np.zeros((n, 2*n + 1))
    for j in range(n):
        A[j, j] = rY[j]
        A[j, n] = -1
        A[j, n+1+j] = -1

    result = linprog(
        c,
        A_ub=A,
        b_ub=np.zeros(n),
        A_eq=np.r_[np.ones(n), 0, np.zeros(n)].reshape(1, -1),
        b_eq=[1],
        bounds=[(0, None)] * (2*n + 1),
        method="highs"
    )

    if result.success:
        max_robust_output = -result.fun
        target_output = Y[target]
        shortfall = target_output - max_robust_output

        diagnostics.append({
            "year": year,
            "plant_id": plant_id,
            "target_output": target_output,
            "max_robust_output": max_robust_output,
            "shortfall": shortfall,
            "shortfall_pct": 100 * shortfall / target_output,
            "output_feasible": shortfall <= 1e-6
        })
    else:
        diagnostics.append({
            "year": year,
            "plant_id": plant_id,
            "output_feasible": "Diagnostic LP failed: " + result.message
        })

diagnostics_df = pd.DataFrame(diagnostics)
print(diagnostics_df.to_string(index=False))

In [ ]:
from scipy.optimize import linprog
import numpy as np
import pandas as pd

def max_robust_output(Y, rY, gamma):
    n = len(Y)
    c = np.r_[-Y, gamma, np.ones(n)]

    A = np.zeros((n, 2*n + 1))
    for j in range(n):
        A[j, j] = rY[j]
        A[j, n] = -1
        A[j, n+1+j] = -1

    Aeq = np.zeros((1, 2*n + 1))
    Aeq[0, :n] = 1

    res = linprog(
        c,
        A_ub=A,
        b_ub=np.zeros(n),
        A_eq=Aeq,
        b_eq=[1],
        bounds=[(0, None)] * (2*n + 1),
        method="highs"
    )

    if not res.success:
        raise RuntimeError(res.message)

    return -res.fun


critical_results = []

for _, case in failed.iterrows():
    year = int(case["year"])
    plant_id = case["plant_id"]

    g = df[df["year"] == year].reset_index(drop=True)
    Y = g["generation_mwh"].to_numpy(float)
    rY = g["output_uncertainty_radius"].to_numpy(float)

    target = int(np.flatnonzero(g["plant_id"].to_numpy() == plant_id)[0])
    target_output = Y[target]
    actual_gamma = float(case["gamma_heat"])

    low, high = 0.0, actual_gamma

    # Verify feasibility at zero budget
    if max_robust_output(Y, rY, 0.0) < target_output - 1e-6:
        critical = np.nan
        status = "Nominally infeasible"
    else:
        for _ in range(35):
            mid = (low + high) / 2
            if max_robust_output(Y, rY, mid) >= target_output - 1e-6:
                low = mid
            else:
                high = mid

        critical = low
        status = "Threshold found"

    critical_results.append({
        "year": year,
        "plant_id": plant_id,
        "actual_gamma_heat": actual_gamma,
        "critical_gamma_heat": critical,
        "status": status
    })

critical_df = pd.DataFrame(critical_results)
print(critical_df.to_string(index=False))

In [ ]:
for year in [2022, 2023, 2024, 2025]:
    g = df[df["year"] == year].copy()
    top = g.loc[g["generation_mwh"].idxmax()]

    print(
        year,
        "Top plant:", top["plant_id"],
        "Output:", top["generation_mwh"],
        "Output radius:", top["output_uncertainty_radius"],
        "Relative radius:", top["heat_relative_radius"]
    )

In [ ]:
valid = results_df.loc[
    results_df["status"] == "optimal"
].copy()

print("ROBUST DEA EFFICIENCY SUMMARY")
print("=" * 55)

print("\nOverall:")
print(valid["score"].describe().round(4))

print("\nBy year:")
print(
    valid.groupby("year")["score"]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .round(4)
)

print("\nEfficiency classification:")
print("Score < 1:", (valid["score"] < 1 - 1e-6).sum())
print("Score = 1:", np.isclose(valid["score"], 1, atol=1e-6).sum())
print("Score > 1:", (valid["score"] > 1 + 1e-6).sum())

print("\nHighest scores:")
print(
    valid.nlargest(10, "score")[
        ["year", "plant_id", "plant_name", "score"]
    ].to_string(index=False)
)

print("\nLowest scores:")
print(
    valid.nsmallest(10, "score")[
        ["year", "plant_id", "plant_name", "score"]
    ].to_string(index=False)
)

In [ ]:
print("SCORE DEFINITION CHECK")
print("Objective: minimize theta")
print("Capacity constraint: sum(K*lambda) >= theta*K_target")
print("Fuel constraint: robust fuel <= theta*F_target")
print("Output constraint: robust output >= Y_target")
print("VRS constraint: sum(lambda) = 1")

print("\nScores above one by year:")
print(
    valid.assign(above_one=valid["score"] > 1 + 1e-6)
    .groupby("year")["above_one"]
    .agg(["sum", "count"])
)

print("\nScores exactly one within tolerance:")
print(np.isclose(valid["score"], 1, atol=1e-6).sum())

In [ ]:
import numpy as np
import pandas as pd
import os
import time

nominal_results = []
start_time = time.time()

for year, g in df.groupby("year", sort=True):

    g = g.reset_index(drop=True)

    K  = g["capacity_mw"].to_numpy(dtype=float)
    F  = g["fuel_consumption_mmbtu"].to_numpy(dtype=float)
    Y  = g["generation_mwh"].to_numpy(dtype=float)
    rF = g["fuel_uncertainty_radius"].to_numpy(dtype=float)
    rY = g["output_uncertainty_radius"].to_numpy(dtype=float)

    for target in range(len(g)):
        try:
            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY, target,
                gamma_gas=0.0,
                gamma_heat=0.0
            )
        except Exception as e:
            score, status = np.nan, str(e)

        nominal_results.append({
            "year": int(year),
            "plant_id": g.loc[target, "plant_id"],
            "plant_name": g.loc[target, "plant_name"],
            "state": g.loc[target, "state"],
            "nominal_score": score,
            "status": status
        })

    print(f"Year {year}: {len(g)} plants completed.")

nominal_results_df = pd.DataFrame(nominal_results)

out_file = os.path.join(PROC, "edro_nominal_dea_results.csv")
nominal_results_df.to_csv(out_file, index=False)

print("\nNOMINAL DEA SUMMARY")
print("Total:", len(nominal_results_df))
print("Optimal:",
      (nominal_results_df["status"] == "optimal").sum())
print("Unsuccessful:",
      (nominal_results_df["status"] != "optimal").sum())
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Saved:", out_file)

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# Prepare and merge results
nom = nominal_results_df[
    nominal_results_df["status"] == "optimal"
][["year", "plant_id", "nominal_score"]].copy()

rob = results_df[
    results_df["status"] == "optimal"
][["year", "plant_id", "score"]].copy()

rob = rob.rename(columns={"score": "robust_score"})

comparison = nom.merge(
    rob,
    on=["year", "plant_id"],
    how="left",
    validate="one_to_one"
)

comparison["score_change"] = (
    comparison["robust_score"] - comparison["nominal_score"]
)

paired = comparison.dropna(subset=["robust_score"]).copy()

print("PAIRED COMPARISON")
print("=" * 55)
print("Nominal observations:", len(nom))
print("Robust feasible observations:", len(paired))
print("Robust infeasible observations:", comparison["robust_score"].isna().sum())

print("\nOVERALL SCORE COMPARISON")
print(
    paired[["nominal_score", "robust_score", "score_change"]]
    .agg(["mean", "std", "median", "min", "max"])
    .round(5)
)

print("\nDIRECTION OF CHANGE")
print("Robust score > nominal:",
      (paired["score_change"] > 1e-6).sum())
print("Approximately equal:",
      (paired["score_change"].abs() <= 1e-6).sum())
print("Robust score < nominal:",
      (paired["score_change"] < -1e-6).sum())

print("\nBY YEAR")
print(
    paired.groupby("year").agg(
        observations=("score_change", "count"),
        nominal_mean=("nominal_score", "mean"),
        robust_mean=("robust_score", "mean"),
        mean_change=("score_change", "mean"),
        median_change=("score_change", "median"),
        maximum_change=("score_change", "max")
    ).round(5)
)

print("\nRANK ASSOCIATION")
rho, pvalue = spearmanr(
    paired["nominal_score"],
    paired["robust_score"]
)
print("Spearman correlation:", round(rho, 5))
print("P-value:", pvalue)

# Save the paired comparison
comparison_file = os.path.join(
    PROC, "edro_nominal_robust_comparison.csv"
)
comparison.to_csv(comparison_file, index=False)
print("\nSaved:", comparison_file)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import SVG, display
import os

# Paired sample: only plants with feasible robust scores
plot_data = paired.groupby("year").agg(
    Nominal=("nominal_score", "mean"),
    Robust=("robust_score", "mean")
)

fig, ax = plt.subplots(figsize=(8, 5))

plot_data.plot(
    kind="bar",
    ax=ax,
    width=0.72,
    edgecolor="black"
)

ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Mean efficiency / expansion factor", fontsize=11)
ax.set_title("Nominal versus Robust DEA", fontsize=13)
ax.set_xticklabels(plot_data.index.astype(str), rotation=0)
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

fig.tight_layout()

svg_path = os.path.join(PROC, "nominal_vs_robust_DEA.svg")
fig.savefig(svg_path, format="svg", bbox_inches="tight")
plt.close(fig)

print("SVG saved:", svg_path)
display(SVG(filename=svg_path))

In [ ]:
import numpy as np
import pandas as pd

# Paired score differences
d = paired.copy()
d["plant_id"] = d["plant_id"].astype(str)

plants = d["plant_id"].unique()
years = sorted(d["year"].unique())

B = 2000
rng = np.random.default_rng(20261120)

boot_overall = np.empty(B)
boot_yearly = {year: np.empty(B) for year in years}

# Resample whole plants, preserving their observations across years
plant_rows = {
    p: d[d["plant_id"] == p]
    for p in plants
}

for b in range(B):
    sampled = rng.choice(plants, size=len(plants), replace=True)

    sample = pd.concat(
        [plant_rows[p] for p in sampled],
        ignore_index=True
    )

    boot_overall[b] = sample["score_change"].mean()

    for year in years:
        boot_yearly[year][b] = (
            sample.loc[
                sample["year"] == year, "score_change"
            ].mean()
        )

def summarize_bootstrap(values, observed):
    return {
        "mean_change": observed,
        "CI_lower": np.quantile(values, 0.025),
        "CI_upper": np.quantile(values, 0.975)
    }

overall = summarize_bootstrap(
    boot_overall, d["score_change"].mean()
)

print("PLANT-CLUSTER BOOTSTRAP")
print("Replications:", B)
print("Unique plants:", len(plants))
print("Paired observations:", len(d))

print("\nOVERALL MEAN ROBUST SCORE INCREASE")
print(pd.Series(overall).round(6))

print("\nYEARLY RESULTS")
rows = []

for year in years:
    observed = d.loc[
        d["year"] == year, "score_change"
    ].mean()

    result = summarize_bootstrap(
        boot_yearly[year], observed
    )

    rows.append({"year": year, **result})

print(pd.DataFrame(rows).round(6).to_string(index=False))

In [ ]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

exposure = df[
    ["year", "plant_id", "gas_relative_radius",
     "heat_relative_radius", "gamma_gas", "gamma_heat"]
].copy()

analysis_df = paired.merge(
    exposure,
    on=["year", "plant_id"],
    how="left",
    validate="one_to_one"
)

variables = [
    "gas_relative_radius",
    "heat_relative_radius",
    "gamma_gas",
    "gamma_heat"
]

print("ASSOCIATION WITH ROBUST SCORE INCREASE")
print("=" * 60)

for var in variables:
    temp = analysis_df[["score_change", var]].dropna()
    rho, p = spearmanr(temp["score_change"], temp[var])

    print(
        f"{var:25s} Spearman rho={rho: .4f}, "
        f"p-value={p:.4g}, n={len(temp)}"
    )

print("\nYEAR-SPECIFIC UNCERTAINTY ASSOCIATIONS")

for year in sorted(analysis_df["year"].unique()):
    sub = analysis_df[analysis_df["year"] == year]

    print(f"\nYear {year}")
    for var in ["gas_relative_radius", "heat_relative_radius"]:
        rho, p = spearmanr(sub["score_change"], sub[var])
        print(f"  {var}: rho={rho:.4f}, p={p:.4g}")

In [ ]:
import numpy as np
import pandas as pd
import os
import time

separate_results = []
start_time = time.time()

for year, g in df.groupby("year", sort=True):

    g = g.reset_index(drop=True)

    K  = g["capacity_mw"].to_numpy(float)
    F  = g["fuel_consumption_mmbtu"].to_numpy(float)
    Y  = g["generation_mwh"].to_numpy(float)
    rF = g["fuel_uncertainty_radius"].to_numpy(float)
    rY = g["output_uncertainty_radius"].to_numpy(float)

    gg = float(g["gamma_gas"].iloc[0])
    gh = float(g["gamma_heat"].iloc[0])

    for target in range(len(g)):
        for specification, gas_budget, heat_budget in [
            ("gas_only", gg, 0.0),
            ("heat_only", 0.0, gh)
        ]:
            try:
                score, status = solve_one_dea_corrected(
                    K, F, Y, rF, rY, target,
                    gamma_gas=gas_budget,
                    gamma_heat=heat_budget
                )
            except Exception as e:
                score, status = np.nan, str(e)

            separate_results.append({
                "year": int(year),
                "plant_id": g.loc[target, "plant_id"],
                "specification": specification,
                "score": score,
                "status": status
            })

    print(f"Year {year}: {len(g)} plants completed.")

separate_df = pd.DataFrame(separate_results)

out_file = os.path.join(
    PROC, "edro_separate_uncertainty_results.csv"
)
separate_df.to_csv(out_file, index=False)

print("\nSUMMARY")
print(separate_df.groupby(
    ["specification", "status"]
).size())

print("Elapsed minutes:",
      round((time.time() - start_time) / 60, 2))
print("Saved:", out_file)

In [ ]:
import numpy as np
import pandas as pd

# Put gas-only and heat-only scores into separate columns
sep = separate_df.pivot(
    index=["year", "plant_id"],
    columns="specification",
    values="score"
).reset_index()

# Use the paired nominal/full-robust results
comparison = paired.merge(
    sep,
    on=["year", "plant_id"],
    how="inner",
    validate="one_to_one"
)

# Restrict to observations feasible in every specification
comparison = comparison.dropna(
    subset=["nominal_score", "robust_score",
            "gas_only", "heat_only"]
).copy()

comparison["gas_effect"] = (
    comparison["gas_only"] - comparison["nominal_score"]
)
comparison["heat_effect"] = (
    comparison["heat_only"] - comparison["nominal_score"]
)
comparison["full_effect"] = (
    comparison["robust_score"] - comparison["nominal_score"]
)
comparison["interaction"] = (
    comparison["robust_score"]
    - comparison["gas_only"]
    - comparison["heat_only"]
    + comparison["nominal_score"]
)

print("COMMON FEASIBLE SAMPLE:", len(comparison))

effects = ["gas_effect", "heat_effect",
           "full_effect", "interaction"]

print("\nOVERALL EFFECTS")
print(comparison[effects].agg(
    ["mean", "median", "std", "min", "max"]
).T.to_string())

print("\nYEARLY MEAN EFFECTS")
print(
    comparison.groupby("year")[effects]
    .mean()
    .round(6)
    .to_string()
)

print("\nFEASIBILITY COUNTS")
print("Gas-only:", separate_df.query(
    "specification == 'gas_only'"
)["score"].notna().sum())
print("Heat-only:", separate_df.query(
    "specification == 'heat_only'"
)["score"].notna().sum())
print("Common sample:", len(comparison))

In [ ]:
import numpy as np
import pandas as pd

components = [
    "gas_effect",
    "heat_effect",
    "full_effect",
    "interaction"
]

d = comparison.copy()
d["plant_id"] = d["plant_id"].astype(str)

plants = d["plant_id"].unique()
plant_rows = {p: d[d["plant_id"] == p] for p in plants}

B = 2000
rng = np.random.default_rng(20261121)

boot = {v: np.empty(B) for v in components}

for b in range(B):
    sampled = rng.choice(plants, size=len(plants), replace=True)
    sample = pd.concat(
        [plant_rows[p] for p in sampled],
        ignore_index=True
    )

    for v in components:
        boot[v][b] = sample[v].mean()

rows = []

for v in components:
    rows.append({
        "component": v,
        "observed_mean": d[v].mean(),
        "CI_lower": np.quantile(boot[v], 0.025),
        "CI_upper": np.quantile(boot[v], 0.975)
    })

bootstrap_decomposition = pd.DataFrame(rows)

print("PLANT-CLUSTER BOOTSTRAP: DECOMPOSITION")
print("Replications:", B)
print("Unique plants:", len(plants))
print(bootstrap_decomposition.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

d = comparison.copy()
d["gas_increment_given_heat"] = (
    d["full_effect"] - d["heat_effect"]
)

plants = d["plant_id"].astype(str).unique()
plant_rows = {
    p: d[d["plant_id"].astype(str) == p]
    for p in plants
}

B = 2000
rng = np.random.default_rng(20261122)
boot = np.empty(B)

for b in range(B):
    sampled = rng.choice(plants, size=len(plants), replace=True)
    sample = pd.concat(
        [plant_rows[p] for p in sampled],
        ignore_index=True
    )
    boot[b] = sample["gas_increment_given_heat"].mean()

print("INCREMENTAL GAS EFFECT, CONDITIONAL ON HEAT UNCERTAINTY")
print("Mean:", d["gas_increment_given_heat"].mean())
print("95% CI:", np.quantile(boot, [0.025, 0.975]))

print("\nBY YEAR")
print(
    d.groupby("year")["gas_increment_given_heat"]
     .agg(["mean", "median", "std"])
     .to_string()
)

In [ ]:
# EDRO matched-allocation experiment: input verification

import pandas as pd
import numpy as np
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

required = [
    "plant_id", "plant_name", "state", "year",
    "capacity_mw", "fuel_consumption_mmbtu",
    "generation_mwh", "fuel_uncertainty_radius",
    "output_uncertainty_radius",
    "gamma_gas", "gamma_heat"
]

missing = [c for c in required if c not in df.columns]

if missing:
    print("MISSING COLUMNS:", missing)
else:
    print("Observations:", len(df))
    print("Unique plants:", df["plant_id"].nunique())
    print("\nAnnual EDRO allocations:")
    print(
        df.groupby("year")[["gamma_gas", "gamma_heat"]]
          .first()
          .assign(
              total=lambda x: x.gamma_gas + x.gamma_heat
          ).round(6).to_string()
    )

    print("\nMissing values:")
    print(df[required].isna().sum().to_string())

    print("\nExisting robust results:")
    result_path = os.path.join(
        PROC, "edro_robust_dea_results.csv"
    )

    if os.path.exists(result_path):
        old = pd.read_csv(result_path)
        print("Rows:", len(old))
        print(old["status"].value_counts(dropna=False))
    else:
        print("Existing robust results file not found.")

In [ ]:
import os
import numpy as np
import pandas as pd

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

data_path = os.path.join(
    PROC, "ngcc_lagged_uncertainty_coefficients.csv"
)
budget_path = os.path.join(
    PROC, "edro_annual_budget_allocation.csv"
)

assert os.path.exists(data_path), f"Missing: {data_path}"
assert os.path.exists(budget_path), f"Missing: {budget_path}"

df = pd.read_csv(data_path)
budgets = pd.read_csv(budget_path)

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

df = df[
    df["year"].between(2022, 2025)
    & (df["state"] != "AK")
].copy()

print("Data loaded successfully")
print("Observations:", len(df))
print("Unique plants:", df["plant_id"].nunique())
print("\nAnnual EDRO budgets:")
print(
    df.groupby("year")[["gamma_gas", "gamma_heat"]]
      .first()
      .assign(total=lambda x: x.gamma_gas + x.gamma_heat)
      .round(6)
)

In [ ]:
import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

print("PROCESSED FILES:")
for f in sorted(os.listdir(PROC)):
    if "budget" in f.lower() or "event" in f.lower() or "allocation" in f.lower():
        print(f)

budget_path = os.path.join(PROC, "edro_annual_budget_allocation.csv")

if os.path.exists(budget_path):
    budgets = pd.read_csv(budget_path)
    print("\nBUDGET FILE COLUMNS:")
    print(budgets.columns.tolist())
    print("\nFIRST ROWS:")
    print(budgets.head(10).to_string(index=False))
else:
    print("\nAnnual budget file not found.")

In [ ]:
import os
import pandas as pd
import numpy as np

BASE = "/datasets/_deepnote_work/eia_edro_project"
PROC = os.path.join(BASE, "processed")

data_path = os.path.join(
    PROC, "ngcc_lagged_uncertainty_coefficients.csv"
)
budget_path = os.path.join(
    PROC, "edro_annual_budget_allocation.csv"
)

assert os.path.exists(data_path), f"Missing: {data_path}"
assert os.path.exists(budget_path), f"Missing: {budget_path}"

df = pd.read_csv(data_path)
budgets = pd.read_csv(budget_path)

df["year"] = df["year"].astype(int)
budgets["year"] = budgets["year"].astype(int)

budgets = budgets.rename(columns={
    "Gamma_gas": "gamma_gas",
    "Gamma_heat": "gamma_heat"
})

df = df.merge(
    budgets[["year", "gamma_gas", "gamma_heat"]],
    on="year",
    how="left",
    validate="many_to_one"
)

df = df[
    df["year"].between(2022, 2025)
    & (df["state"] != "AK")
].copy()

assert df[["gamma_gas", "gamma_heat"]].notna().all().all()
assert np.allclose(
    df["gamma_gas"] + df["gamma_heat"], 3.0
)

print("DATA INITIALIZATION SUCCESSFUL")
print("Observations:", len(df))
print("Unique plants:", df["plant_id"].nunique())

print("\nAnnual budget allocation:")
print(
    df.groupby("year")[["gamma_gas", "gamma_heat"]]
      .first()
      .round(6)
)

print("\nAvailable columns:")
print(df.columns.tolist())

In [ ]:
import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

# 1. Check whether the LP solver survived the kernel restart
print("SOLVER STATUS:")
print("solve_one_dea_corrected defined:",
      "solve_one_dea_corrected" in globals())

# 2. Check existing result files
result_files = [
    "edro_robust_dea_results.csv",
    "edro_nominal_dea_results.csv",
    "edro_separate_uncertainty_results.csv",
    "robust_dea_edro_zero_budget_repaired.csv"
]

print("\nEXISTING RESULT FILES:")
for filename in result_files:
    path = os.path.join(PROC, filename)
    if os.path.exists(path):
        temp = pd.read_csv(path)
        print(f"\n{filename}: {len(temp)} rows")
        print("Columns:", temp.columns.tolist())
    else:
        print(f"{filename}: NOT FOUND")

# 3. Verify uncertainty-radius fields
radius_cols = [
    "fuel_uncertainty_radius",
    "output_uncertainty_radius"
]

print("\nUNCERTAINTY-RADIUS CHECK:")
print(df[radius_cols].describe().round(6))

print("\nMissing radius values:")
print(df[radius_cols].isna().sum())

In [ ]:
import os
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

robust = pd.read_csv(
    os.path.join(PROC, "edro_robust_dea_results.csv")
)

nominal = pd.read_csv(
    os.path.join(PROC, "edro_nominal_dea_results.csv")
)

keys = ["year", "plant_id"]

print("ROBUST STATUS COUNTS:")
print(robust["status"].value_counts(dropna=False))

print("\nNOMINAL STATUS COUNTS:")
print(nominal["status"].value_counts(dropna=False))

comparison = robust.merge(
    nominal,
    on=keys,
    how="inner",
    validate="one_to_one"
)

print("\nMATCHED RESULTS:")
print("Matched observations:", len(comparison))
print("Nominal optimal:",
      (comparison["status_y"] == "optimal").sum())
print("Robust optimal:",
      (comparison["status_x"] == "optimal").sum())

print("\nROBUST RESULT BUDGETS:")
print(
    robust.groupby("year")[["gamma_gas", "gamma_heat"]]
          .first()
          .round(6)
)

print("\nDuplicate plant-year keys:")
print("Robust:", robust.duplicated(keys).sum())
print("Nominal:", nominal.duplicated(keys).sum())

In [ ]:
import os

BASE = "/datasets/_deepnote_work/eia_edro_project"

print("PYTHON FILES:")
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if d not in [
        "__pycache__", ".ipynb_checkpoints"
    ]]
    for f in files:
        if f.endswith((".py", ".ipynb")):
            print(os.path.join(root, f))

print("\nPROJECT ROOT CONTENTS:")
print(sorted(os.listdir(BASE)))

In [ ]:
print("SCALING AND INPUT CHECK")

for year, g in df.groupby("year"):
    print(f"\nYear: {year}")
    print("Observations:", len(g))
    print("Median fuel:", g["fuel_consumption_mmbtu"].median())
    print("Median output:", g["generation_mwh"].median())
    print("Median capacity:", g["capacity_mw"].median())
    print("Fuel radius median:", g["fuel_uncertainty_radius"].median())
    print("Output radius median:", g["output_uncertainty_radius"].median())

print("\nRequired columns present:",
      all(c in df.columns for c in [
          "capacity_mw",
          "fuel_consumption_mmbtu",
          "generation_mwh",
          "fuel_uncertainty_radius",
          "output_uncertainty_radius"
      ]))

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

saved = pd.read_csv(
    f"{PROC}/edro_robust_dea_results.csv"
)

def test_dea_scaling(g, target, gamma_gas, gamma_heat,
                     output_radius_scale="output"):

    K = g["capacity_mw"].to_numpy(float)
    F = g["fuel_consumption_mmbtu"].to_numpy(float)
    Y = g["generation_mwh"].to_numpy(float)

    rF = g["fuel_uncertainty_radius"].to_numpy(float)
    rY = g["output_uncertainty_radius"].to_numpy(float)

    n = len(g)
    ks, fs, ys = np.median(K), np.median(F), np.median(Y)

    k, f, y = K/ks, F/fs, Y/ys
    rf = rF/fs
    ry = rY/(ys if output_radius_scale == "output" else fs)

    it, il = 0, 1
    next_idx = 1+n

    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = next_idx, next_idx+1
        next_idx += n+1

    if gamma_heat > 0:
        iph, iqh = next_idx, next_idx+1
        next_idx += n+1

    nv = next_idx
    c = np.zeros(nv)
    c[it] = 1.0
    rows, rhs = [], []

    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f
    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0
    rows.append(row)
    rhs.append(0.0)

    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1
            row[iqg+j] = -1
            rows.append(row)
            rhs.append(0.0)

    row = np.zeros(nv)
    row[il:il+n] = -y
    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0
    rows.append(row)
    rhs.append(-y[target])

    if gamma_heat > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = ry[j]
            row[iph] = -1
            row[iqh+j] = -1
            rows.append(row)
            rhs.append(0.0)

    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=[1.0],
        bounds=[(0, None)]*nv,
        method="highs"
    )

    return result.fun if result.success else np.nan


checks = []

for year in sorted(df["year"].unique()):
    g = df[df["year"] == year].reset_index(drop=True)

    old = saved[
        (saved["year"] == year)
        & (saved["status"] == "optimal")
    ].iloc[0]

    target = g.index[g["plant_id"] == old["plant_id"]][0]

    score_output = test_dea_scaling(
        g, target, old["gamma_gas"], old["gamma_heat"], "output"
    )

    score_fuel = test_dea_scaling(
        g, target, old["gamma_gas"], old["gamma_heat"], "fuel"
    )

    checks.append({
        "year": year,
        "plant_id": old["plant_id"],
        "saved_score": old["score"],
        "output_scaled_score": score_output,
        "fuel_scaled_score": score_fuel,
        "output_abs_error": abs(score_output-old["score"]),
        "fuel_abs_error": abs(score_fuel-old["score"])
    })

check_df = pd.DataFrame(checks)
print(check_df.to_string(index=False))

In [ ]:
import numpy as np
from scipy.optimize import linprog

def solve_one_dea_corrected(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    ks, fs, ys = np.median(K), np.median(F), np.median(Y)

    k = K / ks
    f = F / fs
    y = Y / ys

    # Correct, separately normalized uncertainty radii
    rf = rF / fs
    ry = rY / ys

    it, il = 0, 1
    next_idx = 1 + n

    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = next_idx, next_idx + 1
        next_idx += n + 1

    if gamma_heat > 0:
        iph, iqh = next_idx, next_idx + 1
        next_idx += n + 1

    nv = next_idx
    c = np.zeros(nv)
    c[it] = 1.0

    rows, rhs = [], []

    # Capacity
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Robust fuel constraint
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    # Fuel uncertainty support
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Robust output constraint
    row = np.zeros(nv)
    row[il:il+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Output uncertainty support
    if gamma_heat > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = ry[j]
            row[iph] = -1.0
            row[iqh+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Variable returns to scale
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=[(0, None)] * nv,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message


print("Validated DEA solver defined:",
      callable(solve_one_dea_corrected))

In [ ]:
import numpy as np
import pandas as pd
import os

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

saved = pd.read_csv(
    os.path.join(PROC, "edro_robust_dea_results.csv")
)

test_cases = [
    (2022, 201),  # feasible
    (2023, 56),   # feasible
    (2024, 56),   # feasible
    (2025, 56),   # feasible
    (2022, 710)   # known infeasible
]

checks = []

for year, plant_id in test_cases:
    g = df[df["year"] == year].reset_index(drop=True)
    matches = g.index[g["plant_id"] == plant_id].tolist()

    if not matches:
        print(f"Missing plant {plant_id} in {year}")
        continue

    target = matches[0]
    old = saved[
        (saved["year"] == year)
        & (saved["plant_id"] == plant_id)
    ].iloc[0]

    score, status = solve_one_dea_corrected(
        g["capacity_mw"].to_numpy(float),
        g["fuel_consumption_mmbtu"].to_numpy(float),
        g["generation_mwh"].to_numpy(float),
        g["fuel_uncertainty_radius"].to_numpy(float),
        g["output_uncertainty_radius"].to_numpy(float),
        target,
        float(old["gamma_gas"]),
        float(old["gamma_heat"])
    )

    checks.append({
        "year": year,
        "plant_id": plant_id,
        "saved_status": old["status"],
        "new_status": status,
        "saved_score": old["score"],
        "new_score": score,
        "difference": (
            abs(score - old["score"])
            if np.isfinite(score) and np.isfinite(old["score"])
            else np.nan
        )
    })

print(pd.DataFrame(checks).to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import time

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

specifications = {
    "Equal": (1.50, 1.50),
    "Gas_dominant": (2.25, 0.75),
    "Heat_dominant": (0.75, 2.25)
}

results = []
start_time = time.time()

for spec_name, (gamma_gas, gamma_heat) in specifications.items():

    print(f"\nStarting: {spec_name}")
    spec_start = time.time()

    for year, g_original in df.groupby("year", sort=True):

        g = g_original.reset_index(drop=True)

        K = g["capacity_mw"].to_numpy(float)
        F = g["fuel_consumption_mmbtu"].to_numpy(float)
        Y = g["generation_mwh"].to_numpy(float)
        rF = g["fuel_uncertainty_radius"].to_numpy(float)
        rY = g["output_uncertainty_radius"].to_numpy(float)

        year_results = []

        for target in range(len(g)):

            score, status = solve_one_dea_corrected(
                K, F, Y, rF, rY, target,
                gamma_gas=gamma_gas,
                gamma_heat=gamma_heat
            )

            year_results.append({
                "year": int(year),
                "plant_id": g.loc[target, "plant_id"],
                "plant_name": g.loc[target, "plant_name"],
                "state": g.loc[target, "state"],
                "method": spec_name,
                "gamma_gas": gamma_gas,
                "gamma_heat": gamma_heat,
                "score": score,
                "status": status
            })

        results.extend(year_results)

        year_optimal = sum(
            x["status"] == "optimal" for x in year_results
        )

        print(
            f"  {year}: {len(g)} solved; "
            f"{year_optimal} optimal; "
            f"{len(g)-year_optimal} infeasible"
        )

    print(
        f"Completed {spec_name} in "
        f"{(time.time()-spec_start)/60:.2f} minutes"
    )

matched = pd.DataFrame(results)

output_path = os.path.join(
    PROC, "edro_matched_budget_sensitivity.csv"
)

matched.to_csv(output_path, index=False)

print("\nALL THREE SPECIFICATIONS COMPLETED")
print("Total LPs:", len(matched))
print("Saved:", output_path)
print("Total elapsed minutes:", round((time.time()-start_time)/60, 2))

print("\nSTATUS SUMMARY:")
print(
    matched.groupby(["method", "status"])
           .size()
           .unstack(fill_value=0)
)

print("\nYEAR-METHOD SUMMARY:")
print(
    matched.groupby(["year", "method"])
           .agg(
               observations=("plant_id", "size"),
               feasible=("score", "count"),
               mean_score=("score", "mean"),
               median_score=("score", "median")
           )
           .round(6)
)

In [ ]:
import os

p = "/datasets/_deepnote_work/eia_edro_project/processed/edro_matched_budget_sensitivity.csv"

print("File exists:", os.path.exists(p))

if os.path.exists(p):
    import pandas as pd
    x = pd.read_csv(p)
    print("Rows saved:", len(x))
    print(x.head())

In [ ]:
import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

alt = pd.read_csv(
    os.path.join(PROC, "edro_matched_budget_sensitivity.csv")
)

edro = pd.read_csv(
    os.path.join(PROC, "edro_robust_dea_results.csv")
)

keys = ["year", "plant_id"]

edro = edro[keys + ["score", "status"]].rename(columns={
    "score": "edro_score",
    "status": "edro_status"
})

alt = alt.merge(
    edro,
    on=keys,
    how="left",
    validate="many_to_one"
)

print("1. FEASIBILITY COUNTS")
print(
    alt.groupby(["method", "status"])
       .size()
       .unstack(fill_value=0)
)

print("\n2. SCORE SUMMARY")
print(
    alt[alt["status"] == "optimal"]
       .groupby("method")
       .agg(
           feasible=("score", "count"),
           mean_score=("score", "mean"),
           median_score=("score", "median"),
           sd_score=("score", "std")
       )
       .round(6)
)

print("\n3. PAIRED COMPARISON WITH EDRO")
for method in ["Equal", "Gas_dominant", "Heat_dominant"]:

    g = alt[
        (alt["method"] == method)
        & (alt["status"] == "optimal")
        & (alt["edro_status"] == "optimal")
    ].copy()

    g["difference"] = g["score"] - g["edro_score"]

    print(f"\n{method}")
    print("Common feasible observations:", len(g))
    print("Alternative mean:", round(g["score"].mean(), 6))
    print("EDRO mean:", round(g["edro_score"].mean(), 6))
    print("Mean difference (alternative - EDRO):",
          round(g["difference"].mean(), 6))
    print("Median difference:",
          round(g["difference"].median(), 6))
    print("Alternative score higher:",
          int((g["difference"] > 1e-8).sum()))
    print("Scores effectively equal:",
          int((g["difference"].abs() <= 1e-8).sum()))
    print("Alternative score lower:",
          int((g["difference"] < -1e-8).sum()))

print("\n4. YEARLY PAIRED MEAN DIFFERENCES")
for method in ["Equal", "Gas_dominant", "Heat_dominant"]:

    g = alt[
        (alt["method"] == method)
        & (alt["status"] == "optimal")
        & (alt["edro_status"] == "optimal")
    ].copy()

    g["difference"] = g["score"] - g["edro_score"]

    print(f"\n{method}")
    print(
        g.groupby("year")
         .agg(
             common_n=("difference", "size"),
             mean_difference=("difference", "mean"),
             median_difference=("difference", "median")
         )
         .round(6)
    )

In [ ]:
import os
import numpy as np
import pandas as pd

PROC = "/datasets/_deepnote_work/eia_edro_project/processed"

alt = pd.read_csv(
    os.path.join(PROC, "edro_matched_budget_sensitivity.csv")
)

edro = pd.read_csv(
    os.path.join(PROC, "edro_robust_dea_results.csv")
)

keys = ["year", "plant_id"]
methods = ["Equal", "Gas_dominant", "Heat_dominant"]

# Pivot alternative results into one row per plant-year
alt_scores = alt.pivot(
    index=keys, columns="method", values="score"
).reset_index()

alt_status = alt.pivot(
    index=keys, columns="method", values="status"
).reset_index()

alt_scores.columns.name = None
alt_status.columns.name = None

alt_status = alt_status.rename(columns={
    m: f"{m}_status" for m in methods
})

edro = edro[keys + ["score", "status"]].rename(columns={
    "score": "EDRO",
    "status": "EDRO_status"
})

data = alt_scores.merge(
    alt_status, on=keys, validate="one_to_one"
).merge(
    edro, on=keys, validate="one_to_one"
)

# Strict common sample: all four methods must be feasible
status_columns = ["EDRO_status"] + [
    f"{m}_status" for m in methods
]

common = data[
    data[status_columns].eq("optimal").all(axis=1)
].copy()

score_columns = ["EDRO"] + methods

common = common.dropna(subset=score_columns)

for m in methods:
    common[f"diff_{m}"] = common[m] - common["EDRO"]

print("COMMON SAMPLE")
print("Plant-year observations:", len(common))
print("Unique plants:", common["plant_id"].nunique())
print("Years:", sorted(common["year"].unique()))

print("\nOBSERVED PAIRED DIFFERENCES")
for m in methods:
    print(
        m,
        "mean =", round(common[f"diff_{m}"].mean(), 7),
        "median =", round(common[f"diff_{m}"].median(), 7)
    )

# Plant-cluster bootstrap
B = 2000
SEED = 20261120
rng = np.random.default_rng(SEED)

plants = np.sort(common["plant_id"].unique())
nplants = len(plants)

metrics = [f"diff_{m}" for m in methods]
years = sorted(common["year"].unique())

# Precompute cluster sums and observation counts
overall_stats = (
    common.groupby("plant_id")[metrics]
    .sum()
    .reindex(plants)
    .fillna(0)
)

overall_counts = (
    common.groupby("plant_id")
    .size()
    .reindex(plants)
    .fillna(0)
    .to_numpy()
)

year_stats = {}
year_counts = {}

for year in years:
    gy = common[common["year"] == year]

    year_stats[year] = (
        gy.groupby("plant_id")[metrics]
        .sum()
        .reindex(plants)
        .fillna(0)
        .to_numpy()
    )

    year_counts[year] = (
        gy.groupby("plant_id")
        .size()
        .reindex(plants)
        .fillna(0)
        .to_numpy()
    )

overall_values = np.zeros((B, len(methods)))
annual_values = {
    year: np.zeros((B, len(methods))) for year in years
}

overall_sum_array = overall_stats.to_numpy()

for b in range(B):
    sampled = rng.integers(0, nplants, size=nplants)
    weights = np.bincount(sampled, minlength=nplants)

    denom = np.dot(weights, overall_counts)
    overall_values[b] = (
        weights @ overall_sum_array / denom
    )

    for year in years:
        denom_y = np.dot(weights, year_counts[year])
        annual_values[year][b] = (
            weights @ year_stats[year] / denom_y
        )

# Percentile confidence intervals
print("\nPLANT-CLUSTER BOOTSTRAP")
print("Replications:", B)
print("Seed:", SEED)

print("\nPOOLED 95% CONFIDENCE INTERVALS")
pooled_rows = []

for j, m in enumerate(methods):
    lo, hi = np.quantile(overall_values[:, j], [0.025, 0.975])
    observed = common[f"diff_{m}"].mean()

    pooled_rows.append({
        "comparison": f"{m} - EDRO",
        "mean_difference": observed,
        "CI_lower": lo,
        "CI_upper": hi,
        "bootstrap_positive_share":
            np.mean(overall_values[:, j] > 0)
    })

pooled_table = pd.DataFrame(pooled_rows)
print(pooled_table.round(6).to_string(index=False))

print("\nYEAR-SPECIFIC 95% CONFIDENCE INTERVALS")

annual_rows = []

for year in years:
    gy = common[common["year"] == year]

    for j, m in enumerate(methods):
        lo, hi = np.quantile(
            annual_values[year][:, j], [0.025, 0.975]
        )

        annual_rows.append({
            "year": year,
            "comparison": f"{m} - EDRO",
            "common_n": len(gy),
            "mean_difference": gy[f"diff_{m}"].mean(),
            "CI_lower": lo,
            "CI_upper": hi
        })

annual_table = pd.DataFrame(annual_rows)
print(annual_table.round(6).to_string(index=False))

# Save reproducible results
pooled_table.to_csv(
    os.path.join(PROC, "edro_matched_bootstrap_pooled.csv"),
    index=False
)

annual_table.to_csv(
    os.path.join(PROC, "edro_matched_bootstrap_annual.csv"),
    index=False
)

print("\nSaved pooled and annual bootstrap tables.")

In [ ]:
import re

patterns = re.compile(
    r"J_t|cohort|eligible|operating|capacity|generation|"
    r"heat_rate|heat.rate|median|quantile|percentile|"
    r"gas_exposure|heat_exposure|exposure",
    re.IGNORECASE
)

print("NOTEBOOK CELLS RELEVANT TO COHORT / EXPOSURE CONSTRUCTION")
print("=" * 75)

for cell_no, source in enumerate(In):
    if not source or not patterns.search(source):
        continue

    lines = source.splitlines()
    matching = [
        (i, line) for i, line in enumerate(lines)
        if patterns.search(line)
    ]

    if matching:
        print(f"\n--- Executed cell {cell_no} ---")
        shown = set()
        for i, _ in matching:
            for j in range(max(0, i - 2), min(len(lines), i + 3)):
                if j not in shown:
                    print(f"{j+1:3}: {lines[j]}")
                    shown.add(j)

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/datasets/_deepnote_work/eia_edro_project")
processed = root / "processed"

keywords = [
    "event", "exposure", "heat", "gas", "calibration",
    "cohort", "panel", "uncertainty", "budget"
]

print("RELEVANT PROJECT FILES")
print("=" * 75)

for folder in [root, processed]:
    print(f"\nFOLDER: {folder}")
    if not folder.exists():
        print("Not found")
        continue

    for p in sorted(folder.iterdir()):
        if p.is_file() and any(k in p.name.lower() for k in keywords):
            print(p.name)

print("\nCSV SCHEMAS OF RELEVANT PROCESSED FILES")
print("=" * 75)

for p in sorted(processed.glob("*.csv")):
    if any(k in p.name.lower() for k in keywords):
        try:
            header = pd.read_csv(p, nrows=0).columns.tolist()
            print(f"\n{p.name}")
            print(header)
        except Exception as e:
            print(f"\n{p.name}: could not inspect header: {e}")

In [ ]:
from pathlib import Path

root = Path("/datasets/_deepnote_work/eia_edro_project")

print("ROOT EXISTS:", root.exists())
print("ROOT PATH:", root.resolve() if root.exists() else "Unavailable")

if root.exists():
    print("\nALL FILES (up to 3 directory levels):")
    for p in sorted(root.rglob("*")):
        if p.is_file() and len(p.relative_to(root).parts) <= 4:
            print(p.relative_to(root), f"({p.stat().st_size:,} bytes)")

    print("\nALL SUBDIRECTORIES:")
    for p in sorted(root.rglob("*")):
        if p.is_dir():
            print(p.relative_to(root))

In [ ]:
from pathlib import Path

for folder in [
    Path("/datasets/_deepnote_work"),
    Path("/datasets/_deepnote_work/eia_edro_project"),
    Path("/datasets/_deepnote_work/eia_edro_project/processed"),
]:
    print(f"\n{folder}")
    if folder.exists():
        items = list(folder.iterdir())
        print("Items:", len(items))
        for p in items[:60]:
            kind = "DIR " if p.is_dir() else "FILE"
            size = f"{p.stat().st_size:,} bytes" if p.is_file() else ""
            print(f"  {kind} {p.name} {size}")
    else:
        print("DOES NOT EXIST")

In [ ]:
print(callable(globals().get("solve_one_dea_common_shock")))

In [ ]:
print("NumPy:", "np" in globals())
print("Pandas:", "pd" in globals())
print("SciPy LP:", "linprog" in globals())
print("Original solver:",
      callable(globals().get("solve_one_dea_corrected")))

In [ ]:

def solve_one_dea_common_shock(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)
    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    # Variables: theta, lambda, gas duals, heat duals
    ng = n + 1 if gamma_gas > 0 else 0
    nh = n if gamma_heat > 0 else 0
    nv = 1 + n + ng + nh

    it, il = 0, 1
    ipg = 1+n if gamma_gas > 0 else None
    iqg = ipg+1 if gamma_gas > 0 else None
    iph = 1+n+ng if gamma_heat > 0 else None
    iqh = iph+1 if gamma_heat > 0 else None

    c = np.zeros(nv)
    c[it] = 1.0
    rows, rhs = [], []

    # Capacity
    row = np.zeros(nv)
    row[it] = -k[target]
    row[il:il+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Fuel
    row = np.zeros(nv)
    row[it] = -f[target]
    row[il:il+n] = f
    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0
    rows.append(row)
    rhs.append(0.0)


In [ ]:
print(callable(globals().get("solve_one_dea_common_shock")))

In [ ]:

    # Gas dual constraints
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[il+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Common-shock heat constraint
    row = np.zeros(nv)
    row[il:il+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Heat dual constraints; target coefficient cancels
    if gamma_heat > 0:
        for j in range(n):
            if j != target:
                row = np.zeros(nv)
                row[il+j] = ry[j]
                row[iph] = -1.0
                row[iqh+j] = -1.0
                rows.append(row)
                rhs.append(0.0)

    # VRS constraint
    Aeq = np.zeros((1, nv))
    Aeq[0, il:il+n] = 1.0

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=[(0, None)] * nv,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message


In [ ]:
print("Original solver intact:",
      callable(globals().get("solve_one_dea_corrected")))
print("Alternative solver exists:",
      callable(globals().get("solve_one_dea_common_shock")))

In [ ]:

def solve_one_dea_common_shock(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    # Variable blocks
    idx = 1 + n
    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = idx, idx + 1
        idx += n + 1

    if gamma_heat > 0:
        iph, iqh = idx, idx + 1
        idx += n + 1

    nv = idx
    c = np.zeros(nv)
    c[0] = 1.0
    rows, rhs = [], []

    # Capacity
    row = np.zeros(nv)
    row[0] = -k[target]
    row[1:1+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Robust fuel
    row = np.zeros(nv)
    row[0] = -f[target]
    row[1:1+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    # Gas uncertainty dual
    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[1+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Common-shock robust output
    row = np.zeros(nv)
    row[1:1+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    # Heat uncertainty dual; target coefficient cancels
    if gamma_heat > 0:
        for j in range(n):
            if j != target:
                row = np.zeros(nv)
                row[1+j] = ry[j]
                row[iph] = -1.0
                row[iqh+j] = -1.0
                rows.append(row)
                rhs.append(0.0)

    # VRS
    Aeq = np.zeros((1, nv))
    Aeq[0, 1:1+n] = 1.0

    # Nonnegative variables; target heat-dual variable fixed at zero
    bounds = [(0, None)] * nv
    if gamma_heat > 0:
        bounds[iqh + target] = (0, 0)

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=bounds,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message


print("Alternative solver defined:",
      callable(solve_one_dea_common_shock))


In [ ]:
g = (
    df[(df.year == 2022) & (df.state != "AK")]
    .sort_values("plant_id")
    .reset_index(drop=True)
)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)

target = np.where(g.plant_id.to_numpy() == 710)[0][0]

score, status = solve_one_dea_common_shock(
    K, F, Y, rF, rY,
    target=target,
    gamma_gas=0,
    gamma_heat=0.8877088542
)

print("Score:", score)
print("Status:", status)

In [ ]:
budgets = {
    2022: (2.1122911458, 0.8877088542),
    2023: (1.8892635042, 1.1107364958),
    2024: (0.0, 3.0),
    2025: (0.0, 3.0)
}

for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    print(f"\nYEAR {year}")

    for plant in [710, 56407]:
        target = np.where(g.plant_id.to_numpy() == plant)[0][0]

        old_score, old_status = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, gg, gh
        )

        new_score, new_status = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )

        print(
            f"Plant {plant}: "
            f"Original={old_status}; "
            f"Common-shock={new_status}; "
            f"New score={new_score}"
        )

In [ ]:
year = 2024

g = (
    df[(df.year == year) & (df.state != "AK")]
    .sort_values("plant_id")
    .reset_index(drop=True)
)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)

results = []

for target in range(len(g)):
    nominal, _ = solve_one_dea_corrected(
        K, F, Y, rF, rY, target, 0, 0
    )

    robust, status = solve_one_dea_common_shock(
        K, F, Y, rF, rY, target, 0, 3
    )

    results.append((g.plant_id.iloc[target], nominal, robust, status))

comparison = pd.DataFrame(
    results,
    columns=["plant_id", "nominal_score", "common_shock_score", "status"]
)

print(comparison["common_shock_score"].describe())
print(
    "Scores equal to 1:",
    np.isclose(comparison["common_shock_score"], 1.0).sum(),
    "out of", len(comparison)
)
print(
    "Feasible:",
    (comparison["status"] == "optimal").sum(),
    "out of", len(comparison)
)

In [ ]:
comparison["score_change"] = (
    comparison["common_shock_score"]
    - comparison["nominal_score"]
)

print("NOMINAL VS COMMON-SHOCK, 2024")
print(comparison[
    ["nominal_score", "common_shock_score", "score_change"]
].describe().round(6))

print(
    "\nSpearman rank correlation:",
    comparison["nominal_score"].corr(
        comparison["common_shock_score"],
        method="spearman"
    )
)

print(
    "\nMean absolute score change:",
    comparison["score_change"].abs().mean()
)

print(
    "\nPlants with changed scores:",
    (comparison["score_change"].abs() > 1e-7).sum()
)

In [ ]:
all_year_comparisons = []

for year, (gg, gh) in budgets.items():

    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    for target in range(len(g)):

        nominal, ns = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, 0, 0
        )

        robust, rs = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )

        all_year_comparisons.append({
            "year": year,
            "plant_id": g.plant_id.iloc[target],
            "nominal_score": nominal,
            "common_shock_score": robust,
            "nominal_status": ns,
            "common_shock_status": rs
        })

comparison_all = pd.DataFrame(all_year_comparisons)

summary = comparison_all.groupby("year").apply(
    lambda x: pd.Series({
        "plants": len(x),
        "nominal_feasible": (x.nominal_status == "optimal").sum(),
        "common_shock_feasible": (x.common_shock_status == "optimal").sum(),
        "nominal_mean": x.nominal_score.mean(),
        "common_shock_mean": x.common_shock_score.mean(),
        "mean_abs_change": (
            x.common_shock_score - x.nominal_score
        ).abs().mean(),
        "spearman": x.nominal_score.corr(
            x.common_shock_score, method="spearman"
        )
    })
)

print(summary.round(6).to_string())

In [ ]:
def solve_one_dea_common_shock(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)

    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys

    # Correct scaling: heat uncertainty uses output scale ys
    rf, ry = rF/fs, rY/ys

    idx = 1 + n
    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = idx, idx + 1
        idx += n + 1

    if gamma_heat > 0:
        iph, iqh = idx, idx + 1
        idx += n + 1

    nv = idx
    c = np.zeros(nv)
    c[0] = 1.0
    rows, rhs = [], []

    # Capacity
    row = np.zeros(nv)
    row[0] = -k[target]
    row[1:1+n] = k
    rows.append(row)
    rhs.append(0.0)

    # Robust fuel
    row = np.zeros(nv)
    row[0] = -f[target]
    row[1:1+n] = f

    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0

    rows.append(row)
    rhs.append(0.0)

    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[1+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row)
            rhs.append(0.0)

    # Common-realization robust output
    row = np.zeros(nv)
    row[1:1+n] = -y

    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0

    rows.append(row)
    rhs.append(-y[target])

    if gamma_heat > 0:
        for j in range(n):
            if j != target:
                row = np.zeros(nv)
                row[1+j] = ry[j]
                row[iph] = -1.0
                row[iqh+j] = -1.0
                rows.append(row)
                rhs.append(0.0)

    # VRS
    Aeq = np.zeros((1, nv))
    Aeq[0, 1:1+n] = 1.0

    bounds = [(0, None)] * nv
    if gamma_heat > 0:
        bounds[iqh + target] = (0, 0)

    result = linprog(
        c,
        A_ub=np.asarray(rows),
        b_ub=np.asarray(rhs),
        A_eq=Aeq,
        b_eq=np.array([1.0]),
        bounds=bounds,
        method="highs"
    )

    if result.success:
        return result.fun, "optimal"

    return np.nan, result.message

In [ ]:
for year in [2024, 2025]:
    gg, gh = budgets[year]
    g = df[(df.year == year) & (df.state != "AK")].reset_index(drop=True)

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    print(f"\nYEAR {year}")
    for plant_id in [710, 56407]:
        target = g.index[g.plant_id == plant_id][0]

        score, status = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )

        print(
            f"Plant {plant_id}: score={score}, status={status}"
        )

In [ ]:
all_year_comparisons = []

for year, (gg, gh) in budgets.items():

    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    for target in range(len(g)):

        nominal, ns = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, 0, 0
        )

        robust, rs = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )

        all_year_comparisons.append({
            "year": year,
            "plant_id": g.plant_id.iloc[target],
            "nominal_score": nominal,
            "common_shock_score": robust,
            "nominal_status": ns,
            "common_shock_status": rs
        })

comparison_all = pd.DataFrame(all_year_comparisons)

rows = []
for year, x in comparison_all.groupby("year"):
    rows.append({
        "year": year,
        "plants": len(x),
        "nominal_feasible": (x.nominal_status == "optimal").sum(),
        "common_shock_feasible": (x.common_shock_status == "optimal").sum(),
        "nominal_mean": x.nominal_score.mean(),
        "common_shock_mean": x.common_shock_score.mean(),
        "mean_abs_change": (
            x.common_shock_score - x.nominal_score
        ).abs().mean(),
        "spearman": x.nominal_score.corr(
            x.common_shock_score, method="spearman"
        )
    })

print(pd.DataFrame(rows).round(6).to_string(index=False))

In [ ]:
print("Rows completed:", len(all_year_comparisons))
print("Comparison table exists:", "comparison_all" in globals())

if "comparison_all" in globals():
    print(comparison_all.groupby("year").size())

In [ ]:
for year in [2022, 2023, 2024, 2025]:
    g = df[(df.year == year) & (df.state != "AK")]

    fs = g.fuel_consumption_mmbtu.median()
    ys = g.generation_mwh.median()

    print(
        year,
        "fuel scale =", round(fs, 3),
        "output scale =", round(ys, 3),
        "ratio fs/ys =", round(fs / ys, 6)
    )

print("\nMaximum score difference from nominal:")
for year, x in comparison_all.groupby("year"):
    d = x.common_shock_score - x.nominal_score
    print(year, "min =", d.min(), "max =", d.max())

In [ ]:
import inspect

source = inspect.getsource(solve_one_dea_common_shock)

print("Current heat scaling:")
for line in source.splitlines():
    if "rf, ry" in line:
        print(line)

print("\nFunction contains output-scale correction:",
      "rY/ys" in source)
print("Function contains old scaling:",
      "rY/fs" in source)

In [ ]:
print("Comparison table timestamp-independent checks")
print("Rows:", len(comparison_all))
print("Years:", sorted(comparison_all.year.unique()))

print("\nCorrected solver spot checks:")
for year in [2022, 2023, 2024, 2025]:
    gg, gh = budgets[year]
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    target = g.index[g.plant_id == 710][0]

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    score, status = solve_one_dea_common_shock(
        K, F, Y, rF, rY, target, gg, gh
    )

    stored = comparison_all.loc[
        (comparison_all.year == year) &
        (comparison_all.plant_id == 710),
        "common_shock_score"
    ].iloc[0]

    print(
        year,
        "fresh =", score,
        "stored =", stored,
        "difference =", abs(score - stored),
        "status =", status
    )

In [ ]:
for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    gas_only = []
    common = []

    for target in range(len(g)):
        s1, _ = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, gg, 0
        )
        s2, _ = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )
        gas_only.append(s1)
        common.append(s2)

    d = np.array(common) - np.array(gas_only)

    print(
        year,
        "mean common-minus-gas-only =", round(np.nanmean(d), 8),
        "maximum absolute difference =", round(np.nanmax(np.abs(d)), 8)
    )

In [ ]:
for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    differences = []

    for target in range(len(g)):
        gas_score, _ = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, gg, 0
        )
        common_score, _ = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )
        differences.append(common_score - gas_score)

    d = np.asarray(differences)

    print(
        year,
        "median =", round(np.median(d), 6),
        "75th percentile =", round(np.quantile(d, 0.75), 6),
        "plants affected (%) =", round(100*np.mean(d > 1e-7), 2),
        "increase > 0.02 (%) =", round(100*np.mean(d > 0.02), 2)
    )

In [ ]:
budgets = {
    2022: (2.1122911458, 0.8877088542),
    2023: (1.8892635042, 1.1107364958),
    2024: (0.0, 3.0),
    2025: (0.0, 3.0),
}

print("Annual budgets restored:")
for year, (gas, heat) in budgets.items():
    print(year, "Gas:", gas, "Heat:", heat)

In [ ]:
import os

root = "/datasets/_deepnote_work/eia_edro_project/processed"

filename = "ngcc_lagged_uncertainty_coefficients.csv"
filepath = os.path.join(root, filename)

print("File exists:", os.path.exists(filepath))
if os.path.exists(filepath):
    print("File size (MB):", round(os.path.getsize(filepath)/1e6, 2))

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

filepath = "/datasets/_deepnote_work/eia_edro_project/processed/ngcc_lagged_uncertainty_coefficients.csv"

df = pd.read_csv(filepath)

print("Dataset reloaded")
print("Rows:", len(df))
print("Years:", sorted(df.year.unique()))
print("Columns:", df.columns.tolist())

In [ ]:
print("Original solver available:",
      "solve_one_dea_corrected" in globals())

print("Common-shock solver available:",
      "solve_one_dea_common_shock" in globals())

print("Budgets available:", "budgets" in globals())

In [ ]:
def solve_one_dea_corrected(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)
    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    idx = 2 + n
    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = idx, idx+1
        idx += n+1
    if gamma_heat > 0:
        iph, iqh = idx, idx+1
        idx += n+1

    nv = idx
    c = np.zeros(nv)
    c[0] = 1.0
    rows, rhs = [], []

    row = np.zeros(nv)
    row[0] = -k[target]
    row[2:2+n] = k
    rows.append(row); rhs.append(0.0)

    row = np.zeros(nv)
    row[0] = -f[target]
    row[2:2+n] = f
    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0
    rows.append(row); rhs.append(0.0)

    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[2+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row); rhs.append(0.0)

    row = np.zeros(nv)
    row[2:2+n] = -y
    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0
    rows.append(row); rhs.append(-y[target])

    if gamma_heat > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[2+j] = ry[j]
            row[iph] = -1.0
            row[iqh+j] = -1.0
            rows.append(row); rhs.append(0.0)

    Aeq = np.zeros((1, nv))
    Aeq[0, 2:2+n] = 1.0
    Aeq[0, 1] = 0.0

    # Variable 1 is unused; fix it at zero.
    bounds = [(0, None)] * nv
    bounds[1] = (0, 0)

    result = linprog(
        c, A_ub=np.asarray(rows), b_ub=np.asarray(rhs),
        A_eq=Aeq, b_eq=[1.0], bounds=bounds, method="highs"
    )
    return (result.fun, "optimal") if result.success else (np.nan, result.message)


def solve_one_dea_common_shock(
    K, F, Y, rF, rY, target,
    gamma_gas=0.0, gamma_heat=0.0
):
    n = len(K)
    ks, fs, ys = np.median(K), np.median(F), np.median(Y)
    k, f, y = K/ks, F/fs, Y/ys
    rf, ry = rF/fs, rY/ys

    idx = 1+n
    ipg = iqg = iph = iqh = None

    if gamma_gas > 0:
        ipg, iqg = idx, idx+1
        idx += n+1
    if gamma_heat > 0:
        iph, iqh = idx, idx+1
        idx += n+1

    nv = idx
    c = np.zeros(nv)
    c[0] = 1.0
    rows, rhs = [], []

    row = np.zeros(nv)
    row[0] = -k[target]
    row[1:1+n] = k
    rows.append(row); rhs.append(0.0)

    row = np.zeros(nv)
    row[0] = -f[target]
    row[1:1+n] = f
    if gamma_gas > 0:
        row[ipg] = gamma_gas
        row[iqg:iqg+n] = 1.0
    rows.append(row); rhs.append(0.0)

    if gamma_gas > 0:
        for j in range(n):
            row = np.zeros(nv)
            row[1+j] = rf[j]
            row[ipg] = -1.0
            row[iqg+j] = -1.0
            rows.append(row); rhs.append(0.0)

    row = np.zeros(nv)
    row[1:1+n] = -y
    if gamma_heat > 0:
        row[iph] = gamma_heat
        row[iqh:iqh+n] = 1.0
    rows.append(row); rhs.append(-y[target])

    if gamma_heat > 0:
        for j in range(n):
            if j != target:
                row = np.zeros(nv)
                row[1+j] = ry[j]
                row[iph] = -1.0
                row[iqh+j] = -1.0
                rows.append(row); rhs.append(0.0)

    Aeq = np.zeros((1, nv))
    Aeq[0, 1:1+n] = 1.0
    bounds = [(0, None)] * nv

    if gamma_heat > 0:
        bounds[iqh+target] = (0, 0)

    result = linprog(
        c, A_ub=np.asarray(rows), b_ub=np.asarray(rhs),
        A_eq=Aeq, b_eq=[1.0], bounds=bounds, method="highs"
    )
    return (result.fun, "optimal") if result.success else (np.nan, result.message)


print("Both solver functions restored.")

In [ ]:
g = (
    df[(df.year == 2024) & (df.state != "AK")]
    .sort_values("plant_id")
    .reset_index(drop=True)
)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)

for plant_id in [710, 56407, int(g.plant_id.iloc[0])]:
    target = g.index[g.plant_id == plant_id][0]

    score, status = solve_one_dea_corrected(
        K, F, Y, rF, rY, target, 0, 0
    )

    print(plant_id, "score =", score, "status =", status)

In [ ]:
for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    differences = []

    for target in range(len(g)):
        gas_score, _ = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, gg, 0
        )
        common_score, _ = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )
        differences.append(common_score - gas_score)

    d = np.asarray(differences)
    print(
        year,
        "median =", round(np.median(d), 6),
        "75th percentile =", round(np.quantile(d, 0.75), 6),
        "plants affected (%) =", round(100*np.mean(d > 1e-7), 2),
        "increase > 0.02 (%) =", round(100*np.mean(d > 0.02), 2)
    )

In [ ]:
heat_effect_summary = []

for year, (gg, gh) in budgets.items():
    g = (
        df[(df.year == year) & (df.state != "AK")]
        .sort_values("plant_id")
        .reset_index(drop=True)
    )

    K = g.capacity_mw.to_numpy(float)
    F = g.fuel_consumption_mmbtu.to_numpy(float)
    Y = g.generation_mwh.to_numpy(float)
    rF = g.fuel_uncertainty_radius.to_numpy(float)
    rY = g.output_uncertainty_radius.to_numpy(float)

    differences = []

    for target in range(len(g)):
        gas_score, _ = solve_one_dea_corrected(
            K, F, Y, rF, rY, target, gg, 0
        )
        common_score, _ = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )
        differences.append(common_score - gas_score)

    d = np.asarray(differences)

    heat_effect_summary.append({
        "year": year,
        "plants": len(d),
        "median_increase": np.median(d),
        "q75_increase": np.quantile(d, 0.75),
        "affected_pct": 100*np.mean(d > 1e-7),
        "above_002_pct": 100*np.mean(d > 0.02)
    })

heat_effect_df = pd.DataFrame(heat_effect_summary)

out = "/datasets/_deepnote_work/eia_edro_project/processed/edro_common_shock_heat_effect.csv"
heat_effect_df.to_csv(out, index=False)

print(heat_effect_df.round(6).to_string(index=False))
print("\nSaved:", out)

In [ ]:
g = (
    df[(df.year == 2022) & (df.state != "AK")]
    .sort_values("plant_id")
    .reset_index(drop=True)
)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)

allocations = {
    "All gas": (3.0, 0.0),
    "Original": budgets[2022],
    "All heat": (0.0, 3.0)
}

allocation_results = []

for label, (gg, gh) in allocations.items():
    scores = []
    statuses = []

    for target in range(len(g)):
        score, status = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )
        scores.append(score)
        statuses.append(status)

    s = np.asarray(scores, dtype=float)

    allocation_results.append({
        "allocation": label,
        "gas_budget": gg,
        "heat_budget": gh,
        "feasible": np.isfinite(s).sum(),
        "mean_efficiency": np.nanmean(s),
        "median_efficiency": np.nanmedian(s)
    })

allocation_2022_df = pd.DataFrame(allocation_results)

print(allocation_2022_df.round(6).to_string(index=False))

In [ ]:
g = (
    df[(df.year == 2023) & (df.state != "AK")]
    .sort_values("plant_id")
    .reset_index(drop=True)
)

K = g.capacity_mw.to_numpy(float)
F = g.fuel_consumption_mmbtu.to_numpy(float)
Y = g.generation_mwh.to_numpy(float)
rF = g.fuel_uncertainty_radius.to_numpy(float)
rY = g.output_uncertainty_radius.to_numpy(float)

allocations = {
    "All gas": (3.0, 0.0),
    "Original": budgets[2023],
    "All heat": (0.0, 3.0)
}

allocation_results = []

for label, (gg, gh) in allocations.items():
    scores = []

    for target in range(len(g)):
        score, status = solve_one_dea_common_shock(
            K, F, Y, rF, rY, target, gg, gh
        )
        scores.append(score)

    s = np.asarray(scores, dtype=float)

    allocation_results.append({
        "allocation": label,
        "gas_budget": gg,
        "heat_budget": gh,
        "feasible": np.isfinite(s).sum(),
        "mean_efficiency": np.nanmean(s),
        "median_efficiency": np.nanmedian(s)
    })

allocation_2023_df = pd.DataFrame(allocation_results)

print(allocation_2023_df.round(6).to_string(index=False))

In [ ]:
allocation_all = pd.concat(
    [allocation_2022_df.assign(year=2022),
     allocation_2023_df.assign(year=2023)],
    ignore_index=True
)

allocation_all = allocation_all[
    ["year", "allocation", "gas_budget", "heat_budget",
     "feasible", "mean_efficiency", "median_efficiency"]
]

out = (
    "/datasets/_deepnote_work/eia_edro_project/processed/"
    "edro_event_budget_allocation_sensitivity.csv"
)

allocation_all.to_csv(out, index=False)

print(allocation_all.round(6).to_string(index=False))
print("\nSaved:", out)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c305b7bf-dde8-4aad-a973-346837e5e665' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>